In [ ]:
# ============================================================
# NOTEBOOK 03 — DATA CLEANING & WRANGLING
# 03.1 — IMPORT LIBRARIES & PROJECT CONFIGURATION
# ============================================================

import os
import sys
import pandas as pd
import numpy as np

from pathlib import Path

# ------------------------------------------------------------
# LOAD LOCAL CONFIGURATION
# ------------------------------------------------------------

PROJECT_DIR = Path(
    r"G:\.shortcut-targets-by-id\1Xk-_mYwUHPWs3pKBzHBs1yQGslb7jsqJ\PROJECT SAINS DATA"
)

sys.path.insert(0, str(PROJECT_DIR))

from config_local import (
    DATA_DIR,
    COMPOUND_DIR,
    MAOB_DIR,
    ALPHA_SYNUCLEIN_DIR,
    OUTPUT_DIR,
    CLEANED_DATA_DIR
)

# ------------------------------------------------------------
# VALIDATE INPUT DIRECTORIES
# ------------------------------------------------------------

directories = {
    "PROJECT": PROJECT_DIR,
    "DATA": DATA_DIR,
    "COMPOUND": COMPOUND_DIR,
    "MAO-B": MAOB_DIR,
    "Alpha-Synuclein": ALPHA_SYNUCLEIN_DIR,
    "OUTPUT": OUTPUT_DIR,
    "CLEANED_DATA": CLEANED_DATA_DIR
}

print("=" * 65)
print("PROJECT CONFIGURATION VALIDATION")
print("=" * 65)

for name, path in directories.items():
    print(
        f"{name:<20}:",
        "OK" if path.is_dir() else "NOT FOUND"
    )

print("\nPROJECT DIR:", PROJECT_DIR)
print("OUTPUT DIR :", OUTPUT_DIR)

In [ ]:
# ============================================================
# NOTEBOOK 03 — DATA CLEANING & WRANGLING
# 03.2 — LOAD RAW DATASETS
# ============================================================

# ------------------------------------------------------------
# DEFINE INPUT FILES
# ------------------------------------------------------------

dataset_files = {
    "pubchem_compounds": COMPOUND_DIR / "pubchem_compounds.csv",
    "validated_compounds": COMPOUND_DIR / "validated_compounds.csv",
    "compound_identity_validation": COMPOUND_DIR / "compound_identity_validation.csv",
    "compound_duplicate_validation": COMPOUND_DIR / "compound_duplicate_validation.csv",

    "maob_target_metadata": MAOB_DIR / "maob_target_metadata.csv",
    "maob_activity": MAOB_DIR / "maob_activity_raw.csv",

    "alpha_target_metadata": ALPHA_SYNUCLEIN_DIR / "alpha_synuclein_target_metadata.csv",
    "alpha_activity": ALPHA_SYNUCLEIN_DIR / "alpha_synuclein_activity_raw.csv",
    "alpha_assay_metadata": ALPHA_SYNUCLEIN_DIR / "alpha_synuclein_assay_metadata_raw.csv",
}

# ------------------------------------------------------------
# LOAD DATASETS
# ------------------------------------------------------------

datasets = {}
missing_files = []

for name, file_path in dataset_files.items():

    if file_path.is_file():
        datasets[name] = pd.read_csv(file_path)
        print(f"[FOUND] {name:<35} {datasets[name].shape}")

    else:
        missing_files.append(name)
        print(f"[MISSING] {name:<33} {file_path}")

# ------------------------------------------------------------
# DATASET INVENTORY
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("DATASET LOADING SUMMARY")
print("=" * 70)

print("Successfully loaded :", len(datasets))
print("Missing files       :", len(missing_files))

for name, df in datasets.items():
    print(f"{name:<35} {len(df):>8,} rows | {len(df.columns):>3} columns")

In [ ]:
# ============================================================
# 03.3 — CHECK ALPHA-SYNUCLEIN DIRECTORY
# ============================================================

print("=" * 65)
print("ALPHA-SYNUCLEIN DIRECTORY INSPECTION")
print("=" * 65)

print("Directory:", ALPHA_SYNUCLEIN_DIR)
print("Exists   :", ALPHA_SYNUCLEIN_DIR.exists())

print("\nFiles available:")

for file in ALPHA_SYNUCLEIN_DIR.iterdir():
    print(f"{file.name:<50} {file.stat().st_size:,} bytes")

In [ ]:
# ============================================================
# SEARCH MISSING ASSAY METADATA
# ============================================================

target_filename = "alpha_synuclein_assay_metadata_raw.csv"

print("Searching:", PROJECT_DIR)

found_files = list(PROJECT_DIR.rglob(target_filename))

if found_files:
    print("\nFILE FOUND:")
    for file in found_files:
        print(file)
else:
    print("\nFILE NOT FOUND IN PROJECT DIRECTORY")

In [ ]:
# ============================================================
# 03.4 — DATASET STRUCTURE INSPECTION
# ============================================================

for name, df in datasets.items():

    print("\n" + "=" * 70)
    print(f"DATASET: {name.upper()}")
    print("=" * 70)

    print("Shape:", df.shape)

    print("\nColumn names:")
    print(df.columns.tolist())

    print("\nData types:")
    print(df.dtypes.value_counts())

    print("\nFirst 3 rows:")
    display(df.head(3))

In [ ]:
# ============================================================
# 03.5 — MISSING VALUES AUDIT
# ============================================================

activity_datasets = {
    "MAO-B": datasets["maob_activity"],
    "Alpha-Synuclein": datasets["alpha_activity"]
}

missing_summary = []

for target_name, df in activity_datasets.items():

    print("\n" + "=" * 70)
    print(f"MISSING VALUES AUDIT — {target_name}")
    print("=" * 70)

    missing_count = df.isnull().sum()
    missing_percent = (missing_count / len(df)) * 100

    summary = pd.DataFrame({
        "Missing_Count": missing_count,
        "Missing_Percentage": missing_percent
    })

    summary = summary.sort_values(
        "Missing_Percentage",
        ascending=False
    )

    display(summary[summary["Missing_Count"] > 0])

    missing_summary.append(
        summary.assign(Target=target_name)
    )

# Gabungkan ringkasan kedua target
missing_summary_df = pd.concat(missing_summary)

print("\nMissing values audit completed.")

In [ ]:
# ============================================================
# CELL 03.6 — ENDPOINT AND UNIT DISTRIBUTION AUDIT
# ============================================================

audit_columns = [
    "type",
    "standard_type",
    "relation",
    "standard_relation",
    "units",
    "standard_units",
    "assay_type",
    "bao_endpoint"
]

activity_datasets = {
    "MAO-B": datasets["maob_activity"],
    "Alpha-Synuclein": datasets["alpha_activity"]
}

for target_name, df in activity_datasets.items():

    print("\n" + "=" * 75)
    print(f"ENDPOINT AND UNIT AUDIT — {target_name}")
    print("=" * 75)

    for col in audit_columns:

        if col not in df.columns:
            continue

        print(f"\n--- {col} ---")

        distribution = (
            df[col]
            .fillna("[MISSING]")
            .value_counts(dropna=False)
            .head(20)
            .rename_axis(col)
            .reset_index(name="Count")
        )

        distribution["Percentage"] = (
            distribution["Count"] / len(df) * 100
        ).round(2)

        display(distribution)

print("\nEndpoint and unit audit completed.")

In [ ]:

# ============================================================
# CELL 03.7 — DETAIL AUDIT ENDPOINT UTAMA
# ============================================================

endpoint_groups = {
    "MAO-B IC50": {
        "df": datasets["maob_activity"],
        "endpoint": "IC50"
    },
    "Alpha-Synuclein Potency": {
        "df": datasets["alpha_activity"],
        "endpoint": "Potency"
    },
    "Alpha-Synuclein IC50": {
        "df": datasets["alpha_activity"],
        "endpoint": "IC50"
    }
}

for group_name, config in endpoint_groups.items():

    df = config["df"]
    endpoint = config["endpoint"]

    subset = df[
        df["standard_type"].astype(str).str.strip() == endpoint
    ].copy()

    print("\n" + "=" * 75)
    print(f"DETAIL AUDIT — {group_name}")
    print("=" * 75)

    print(f"Total records       : {len(subset):,}")
    print(f"Unique molecules    : {subset['molecule_chembl_id'].nunique():,}")
    print(f"Unique assays       : {subset['assay_chembl_id'].nunique():,}")
    print(f"Unique documents    : {subset['document_chembl_id'].nunique():,}")

    # Distribusi satuan, relasi, dan assay
    for column in [
        "standard_units",
        "standard_relation",
        "assay_type",
        "bao_endpoint"
    ]:
        print(f"\n--- {column} ---")
        print(
            subset[column]
            .fillna("[MISSING]")
            .value_counts()
            .head(15)
            .to_string()
        )

    # Pemeriksaan nilai numerik
    numeric_values = pd.to_numeric(
        subset["standard_value"],
        errors="coerce"
    )

    print("\n--- STANDARD VALUE ---")
    print("Valid numeric       :", numeric_values.notna().sum())
    print("Missing/non-numeric :", numeric_values.isna().sum())

    if numeric_values.notna().any():
        print(numeric_values.describe())

    # Pemeriksaan deskripsi assay
    print("\n--- ASSAY DESCRIPTION (TOP 15) ---")
    print(
        subset["assay_description"]
        .fillna("[MISSING]")
        .value_counts()
        .head(15)
        .to_string()
    )

print("\nDetail endpoint audit completed.")


In [ ]:
# ============================================================
# CELL 03.7A — RINGKASAN DETAIL ENDPOINT
# ============================================================

endpoint_groups = {
    "MAO-B IC50": (datasets["maob_activity"], "IC50"),
    "Alpha-Synuclein Potency": (datasets["alpha_activity"], "Potency"),
    "Alpha-Synuclein IC50": (datasets["alpha_activity"], "IC50")
}

for group_name, (df, endpoint) in endpoint_groups.items():

    subset = df[
        df["standard_type"].astype(str).str.strip() == endpoint
    ].copy()

    print("\n" + "=" * 70)
    print(f"{group_name}")
    print("=" * 70)

    print("Total records:", len(subset))
    print("Unique assays:", subset["assay_chembl_id"].nunique())
    print("Unique documents:", subset["document_chembl_id"].nunique())

    print("\n[UNIT × RELATION]")
    print(pd.crosstab(
        subset["standard_units"].fillna("MISSING"),
        subset["standard_relation"].fillna("MISSING")
    ).to_string())

    values = pd.to_numeric(
        subset["standard_value"], errors="coerce"
    )

    print("\n[NUMERIC VALUE]")
    print("Valid:", values.notna().sum())
    print("Missing/non-numeric:", values.isna().sum())
    print("Zero or negative:", (values <= 0).sum())

    if values.notna().any():
        print(values.describe(
            percentiles=[0.01, 0.25, 0.50, 0.75, 0.99]
        ).to_string())

    print("\n[ASSAY DESCRIPTION — TOP 10]")

    descriptions = (
        subset["assay_description"]
        .fillna("MISSING")
        .value_counts()
        .head(10)
    )

    for description, count in descriptions.items():
        print(f"- ({count}) {str(description)[:180]}")

    print("\n[TYPE VS STANDARD_TYPE]")
    print("Original type:")
    print(subset["type"].value_counts(dropna=False).head(10).to_string())

    print("\nStandard type:")
    print(subset["standard_type"].value_counts(dropna=False).head(10).to_string())

In [ ]:
# ============================================================
# CELL 03.8 — ASSAY CONSISTENCY AUDIT
# ============================================================

audit_groups = {
    "MAO-B IC50 (nM, exact)": {
        "df": datasets["maob_activity"],
        "endpoint": "IC50",
        "unit": "nM",
        "relation": "="
    },
    "Alpha-Synuclein IC50 (nM, exact)": {
        "df": datasets["alpha_activity"],
        "endpoint": "IC50",
        "unit": "nM",
        "relation": "="
    }
}

for group_name, config in audit_groups.items():

    df = config["df"]

    subset = df[
        (df["standard_type"] == config["endpoint"]) &
        (df["standard_units"] == config["unit"]) &
        (df["standard_relation"] == config["relation"])
    ].copy()

    print("\n" + "=" * 80)
    print(group_name)
    print("=" * 80)

    print("Records:", len(subset))
    print("Unique assays:", subset["assay_chembl_id"].nunique())
    print("Unique documents:", subset["document_chembl_id"].nunique())

    # Ringkasan kontribusi setiap assay
    assay_summary = (
        subset.groupby("assay_chembl_id", dropna=False)
        .agg(
            record_count=("molecule_chembl_id", "size"),
            unique_molecules=("molecule_chembl_id", "nunique"),
            document_id=("document_chembl_id", "first"),
            assay_type=("assay_type", "first"),
            assay_description=("assay_description", "first")
        )
        .sort_values("record_count", ascending=False)
    )

    print("\n--- TOP 20 ASSAYS ---")

    for assay_id, row in assay_summary.head(20).iterrows():
        print(f"\nAssay ID       : {assay_id}")
        print(f"Records        : {row['record_count']}")
        print(f"Unique molecules: {row['unique_molecules']}")
        print(f"Document ID    : {row['document_id']}")
        print(f"Assay type     : {row['assay_type']}")
        print(
            "Description    :",
            str(row["assay_description"])[:300]
        )

    print("\n--- ASSAY RECORD DISTRIBUTION ---")
    print(assay_summary["record_count"].describe().to_string())

    print("\n--- ORIGINAL TYPE DISTRIBUTION ---")
    print(subset["type"].value_counts(dropna=False).to_string())

print("\nAssay consistency audit completed.")

In [ ]:
# ============================================================
# CELL 03.10 — ORIGINAL VS STANDARDIZED ACTIVITY AUDIT
# ============================================================

maob = datasets["maob_activity"].copy()

subset = maob[
    (maob["standard_type"] == "IC50") &
    (maob["standard_units"] == "nM") &
    (maob["standard_relation"] == "=")
].copy()

print("=" * 75)
print("ORIGINAL VS STANDARDIZED ACTIVITY — MAO-B")
print("=" * 75)

for original_type, group in subset.groupby("type", dropna=False):

    print(f"\n{'-' * 60}")
    print("Original type:", original_type)
    print("Records:", len(group))

    print("\nOriginal units:")
    print(group["units"].value_counts(dropna=False).to_string())

    print("\nOriginal relation:")
    print(group["relation"].value_counts(dropna=False).to_string())

    print("\nOriginal value:")
    print(
        pd.to_numeric(group["value"], errors="coerce")
        .describe()
        .to_string()
    )

    print("\nStandardized value:")
    print(
        pd.to_numeric(group["standard_value"], errors="coerce")
        .describe()
        .to_string()
    )

print("\n--- SAMPLE RECORDS ---")

columns_to_show = [
    "molecule_chembl_id",
    "type",
    "relation",
    "value",
    "units",
    "standard_type",
    "standard_relation",
    "standard_value",
    "standard_units",
    "assay_chembl_id"
]

display(subset[columns_to_show].groupby("type", dropna=False).head(5))

print("\nOriginal vs standardized audit completed.")

In [ ]:
# ============================================================
# CELL 03.11 — STANDARDIZATION VERIFICATION
# ============================================================

maob = datasets["maob_activity"]

subset = maob[
    (maob["standard_type"] == "IC50") &
    (maob["standard_units"] == "nM") &
    (maob["standard_relation"] == "=")
].copy()

columns = [
    "molecule_chembl_id",
    "assay_chembl_id",
    "type",
    "value",
    "units",
    "standard_type",
    "standard_value",
    "standard_units"
]

for original_type in ["IC50", "Log IC50", "pIC50"]:

    group = subset[subset["type"] == original_type]

    print("\n" + "=" * 70)
    print(f"STANDARDIZATION CHECK — {original_type}")
    print("=" * 70)

    sample = group[columns].head(10).copy()

    display(sample)

    original = pd.to_numeric(group["value"], errors="coerce")
    standardized = pd.to_numeric(
        group["standard_value"], errors="coerce"
    )

    if original_type == "pIC50":
        expected = 10 ** (9 - original)

    elif original_type == "Log IC50":
        expected = 10 ** (original + 9)

    else:
        expected = None

    if expected is not None:
        comparison = pd.DataFrame({
            "original_value": original,
            "expected_nM": expected,
            "standard_value": standardized
        })

        comparison["relative_error"] = (
            (comparison["standard_value"] - comparison["expected_nM"]).abs()
            / comparison["expected_nM"]
        )

        print("\nConversion comparison:")
        display(comparison.head(10))

        print(
            "Median relative error:",
            comparison["relative_error"].median()
        )

print("\nStandardization verification completed.")

In [ ]:

# ============================================================
# CELL 03.11 — REPEATED MOLECULE & ACTIVITY AUDIT
# ============================================================

maob = datasets["maob_activity"].copy()

exact_ic50 = maob[
    (maob["standard_type"] == "IC50") &
    (maob["standard_units"] == "nM") &
    (maob["standard_relation"] == "=")
].copy()

exact_ic50["standard_value"] = pd.to_numeric(
    exact_ic50["standard_value"],
    errors="coerce"
)

exact_ic50 = exact_ic50[
    exact_ic50["standard_value"] > 0
].copy()

print("=" * 70)
print("REPEATED MOLECULE AUDIT — MAO-B")
print("=" * 70)

# 1. Pengulangan molekul
molecule_counts = (
    exact_ic50.groupby("molecule_chembl_id")
    .size()
    .sort_values(ascending=False)
)

print("\nTotal records:", len(exact_ic50))
print("Unique molecules:", exact_ic50["molecule_chembl_id"].nunique())
print("Molecules tested more than once:",
      (molecule_counts > 1).sum())

print("\nTop 15 repeated molecules:")
print(molecule_counts.head(15).to_string())

# 2. Pengukuran berulang dalam assay yang sama
within_assay = (
    exact_ic50.groupby(
        ["molecule_chembl_id", "assay_chembl_id"]
    )
    .size()
)

print("\nRepeated molecule within same assay:")
print("Groups with repeated measurements:",
      (within_assay > 1).sum())

# 3. Pengukuran dalam assay berbeda
assay_per_molecule = (
    exact_ic50.groupby("molecule_chembl_id")["assay_chembl_id"]
    .nunique()
)

print("\nMolecules tested in multiple assays:")
print((assay_per_molecule > 1).sum())

# 4. Distribusi jumlah pengukuran
print("\nRecords per molecule:")
print(molecule_counts.describe().to_string())

# 5. Duplikasi record
duplicate_columns = [
    "molecule_chembl_id",
    "assay_chembl_id",
    "standard_type",
    "standard_relation",
    "standard_value",
    "standard_units"
]

print("\nExact duplicate records:",
      exact_ic50.duplicated(subset=duplicate_columns).sum())

print("\nRepeated molecule audit completed.")



In [ ]:
# ============================================================
# CELL 03.12 — INSPECT EXACT DUPLICATES
# ============================================================

df_maob_raw = datasets["maob_activity"].copy()

# Cari duplikasi berdasarkan seluruh kolom
duplicate_mask = df_maob_raw.duplicated(keep=False)

df_exact_duplicates = (
    df_maob_raw.loc[duplicate_mask]
    .sort_values(
        by=[
            "molecule_chembl_id",
            "assay_chembl_id",
            "standard_value"
        ]
    )
)

print("=" * 70)
print("EXACT DUPLICATE INSPECTION — MAO-B")
print("=" * 70)

print("Total duplicate rows:", len(df_exact_duplicates))

display(df_exact_duplicates)

# Simpan kandidat duplikasi untuk dokumentasi audit
df_exact_duplicates.to_csv(
    CLEANED_DATA_DIR / "maob_exact_duplicate_candidates.csv",
    index=False
)

In [ ]:
# ============================================================
# CELL 03.13 — DUPLICATE KEY RECHECK
# ============================================================

df_maob = datasets["maob_activity"].copy()

# Filter subset aktivitas yang sedang kita audit
df_check = df_maob[
    (df_maob["standard_type"] == "IC50") &
    (df_maob["standard_units"] == "nM") &
    (df_maob["standard_relation"] == "=")
].copy()

# Definisi duplikasi berdasarkan identitas pengukuran
duplicate_keys = [
    "molecule_chembl_id",
    "assay_chembl_id",
    "standard_type",
    "standard_relation",
    "standard_value",
    "standard_units"
]

# Identifikasi semua baris dengan kombinasi key berulang
duplicate_mask = df_check.duplicated(
    subset=duplicate_keys,
    keep=False
)

df_duplicate_candidates = (
    df_check.loc[duplicate_mask]
    .sort_values(duplicate_keys)
)

print("=" * 70)
print("DUPLICATE KEY RECHECK — MAO-B")
print("=" * 70)

print("Total candidate rows:", len(df_duplicate_candidates))

display(
    df_duplicate_candidates[
        duplicate_keys + [
            "activity_id",
            "record_id",
            "document_chembl_id",
            "assay_description"
        ]
    ]
)

# Preparation Cleaning

In [ ]:

# ============================================================
# CELL 03.14 — CLEANING JUSTIFICATION & DECISION
# PROJECT: AURICULARIA POLYTRICHA × PARKINSON
# TARGET: HUMAN MAO-B
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

# ------------------------------------------------------------
# 1. LOAD RAW DATA
# ------------------------------------------------------------

df_audit = datasets["maob_activity"].copy()

df_audit["standard_value"] = pd.to_numeric(
    df_audit["standard_value"],
    errors="coerce"
)

total_records = len(df_audit)

print("=" * 75)
print("CLEANING JUSTIFICATION — MAO-B")
print("=" * 75)

print(f"Total original records : {total_records:,}")
print(f"Total columns          : {len(df_audit.columns)}")


# ------------------------------------------------------------
# 2. MISSING VALUE AUDIT
# ------------------------------------------------------------

missing_summary = pd.DataFrame({
    "Missing_Count": df_audit.isna().sum(),
    "Missing_Percentage": (
        df_audit.isna().mean() * 100
    ).round(2)
})

missing_summary = (
    missing_summary[missing_summary["Missing_Count"] > 0]
    .sort_values("Missing_Percentage", ascending=False)
)

print("\n" + "=" * 75)
print("A. MISSING VALUE EVIDENCE")
print("=" * 75)

display(missing_summary.head(20))


# ------------------------------------------------------------
# 3. ENDPOINT AUDIT
# ------------------------------------------------------------

endpoint_summary = (
    df_audit["standard_type"]
    .fillna("MISSING")
    .value_counts()
    .rename_axis("Standard_Type")
    .reset_index(name="Record_Count")
)

endpoint_summary["Percentage"] = (
    endpoint_summary["Record_Count"] / total_records * 100
).round(2)

print("\n" + "=" * 75)
print("B. ENDPOINT HETEROGENEITY")
print("=" * 75)

display(endpoint_summary)


# ------------------------------------------------------------
# 4. UNIT AUDIT
# ------------------------------------------------------------

unit_summary = (
    df_audit["standard_units"]
    .fillna("MISSING")
    .value_counts()
    .rename_axis("Standard_Units")
    .reset_index(name="Record_Count")
)

unit_summary["Percentage"] = (
    unit_summary["Record_Count"] / total_records * 100
).round(2)

print("\n" + "=" * 75)
print("C. ACTIVITY UNIT HETEROGENEITY")
print("=" * 75)

display(unit_summary)


# ------------------------------------------------------------
# 5. RELATION AUDIT
# ------------------------------------------------------------

relation_summary = (
    df_audit["standard_relation"]
    .fillna("MISSING")
    .value_counts()
    .rename_axis("Standard_Relation")
    .reset_index(name="Record_Count")
)

relation_summary["Percentage"] = (
    relation_summary["Record_Count"] / total_records * 100
).round(2)

print("\n" + "=" * 75)
print("D. ACTIVITY RELATION AUDIT")
print("=" * 75)

display(relation_summary)


# ------------------------------------------------------------
# 6. NUMERIC VALUE AUDIT
# ------------------------------------------------------------

numeric_values = df_audit["standard_value"]

numeric_audit = pd.DataFrame({
    "Condition": [
        "Missing / non-numeric values",
        "Zero values",
        "Negative values",
        "Positive numeric values"
    ],
    "Record_Count": [
        numeric_values.isna().sum(),
        (numeric_values == 0).sum(),
        (numeric_values < 0).sum(),
        (numeric_values > 0).sum()
    ]
})

numeric_audit["Percentage"] = (
    numeric_audit["Record_Count"] / total_records * 100
).round(2)

print("\n" + "=" * 75)
print("E. NUMERIC VALUE VALIDITY")
print("=" * 75)

display(numeric_audit)


# ------------------------------------------------------------
# 7. REPEATED MOLECULE AUDIT
# ------------------------------------------------------------

molecule_counts = (
    df_audit["molecule_chembl_id"]
    .value_counts()
)

repeated_molecules = molecule_counts[
    molecule_counts > 1
]

print("\n" + "=" * 75)
print("F. REPEATED MOLECULE EVIDENCE")
print("=" * 75)

print("Unique molecules:", df_audit["molecule_chembl_id"].nunique())
print("Repeated molecules:", len(repeated_molecules))
print("Records belonging to repeated molecules:",
      int(molecule_counts[molecule_counts > 1].sum()))

display(repeated_molecules.head(15).to_frame(
    name="Record_Count"
))


# ------------------------------------------------------------
# 8. DUPLICATE CANDIDATE AUDIT
# ------------------------------------------------------------

duplicate_keys = [
    "molecule_chembl_id",
    "assay_chembl_id",
    "standard_type",
    "standard_relation",
    "standard_value",
    "standard_units"
]

# Fokus pada subset IC50 exact dalam nM
df_ic50_candidate = df_audit[
    (df_audit["standard_type"] == "IC50") &
    (df_audit["standard_units"] == "nM") &
    (df_audit["standard_relation"] == "=") &
    (df_audit["standard_value"] > 0)
].copy()

duplicate_mask = df_ic50_candidate.duplicated(
    subset=duplicate_keys,
    keep=False
)

df_duplicate_evidence = df_ic50_candidate[
    duplicate_mask
].copy()

print("\n" + "=" * 75)
print("G. DUPLICATE CANDIDATE EVIDENCE")
print("=" * 75)

print("IC50 exact subset:", len(df_ic50_candidate))
print("Candidate duplicate rows:", len(df_duplicate_evidence))

if len(df_duplicate_evidence) > 0:
    display(
        df_duplicate_evidence[
            duplicate_keys + [
                "activity_id",
                "record_id",
                "document_chembl_id"
            ]
        ].sort_values(duplicate_keys)
    )


# ------------------------------------------------------------
# 9. CLEANING DECISION TABLE
# ------------------------------------------------------------

decision_table = pd.DataFrame([
    {
        "Issue": "Missing values",
        "Evidence": (
            f"{int(df_audit.isna().any(axis=1).sum()):,} records "
            "memiliki minimal satu missing value"
        ),
        "Cleaning_Decision": (
            "Evaluasi berdasarkan kebutuhan kolom; "
            "jangan menghapus seluruh baris secara otomatis"
        )
    },
    {
        "Issue": "Endpoint heterogeneity",
        "Evidence": (
            f"{df_audit['standard_type'].nunique(dropna=True)} "
            "jenis endpoint ditemukan"
        ),
        "Cleaning_Decision": (
            "Pilih IC50 untuk dataset aktivitas utama MAO-B"
        )
    },
    {
        "Issue": "Unit heterogeneity",
        "Evidence": (
            f"{df_audit['standard_units'].nunique(dropna=True)} "
            "jenis satuan ditemukan"
        ),
        "Cleaning_Decision": (
            "Gunakan nM untuk dataset aktivitas utama"
        )
    },
    {
        "Issue": "Censored relations",
        "Evidence": (
            f"{int((df_audit['standard_relation'] != '=').sum()):,} "
            "record tidak memiliki relation exact"
        ),
        "Cleaning_Decision": (
            "Pisahkan relation selain '='; jangan perlakukan "
            "nilai tersensor sebagai nilai eksak"
        )
    },
    {
        "Issue": "Invalid numeric values",
        "Evidence": (
            f"{int((numeric_values.isna() | (numeric_values <= 0)).sum()):,} "
            "record missing, non-numeric, atau tidak positif"
        ),
        "Cleaning_Decision": (
            "Keluarkan dari dataset numerik utama dan dokumentasikan"
        )
    },
    {
        "Issue": "Repeated molecules",
        "Evidence": (
            f"{len(repeated_molecules):,} molekul memiliki "
            "lebih dari satu record"
        ),
        "Cleaning_Decision": (
            "Pertahankan; evaluasi assay dan pengukuran berulang"
        )
    },
    {
        "Issue": "Duplicate candidates",
        "Evidence": (
            f"{len(df_duplicate_evidence):,} baris kandidat "
            "berdasarkan kombinasi key"
        ),
        "Cleaning_Decision": (
            "Pertahankan sementara; periksa identitas record "
            "sebelum memutuskan penghapusan"
        )
    }
])

print("\n" + "=" * 75)
print("H. CLEANING DECISION SUMMARY")
print("=" * 75)

display(decision_table)


# ------------------------------------------------------------
# 10. SAVE AUDIT EVIDENCE
# ------------------------------------------------------------

CLEANED_DATA_DIR.mkdir(parents=True, exist_ok=True)

missing_summary.to_csv(
    CLEANED_DATA_DIR / "maob_cleaning_missing_audit.csv"
)

endpoint_summary.to_csv(
    CLEANED_DATA_DIR / "maob_cleaning_endpoint_audit.csv",
    index=False
)

unit_summary.to_csv(
    CLEANED_DATA_DIR / "maob_cleaning_unit_audit.csv",
    index=False
)

relation_summary.to_csv(
    CLEANED_DATA_DIR / "maob_cleaning_relation_audit.csv",
    index=False
)

numeric_audit.to_csv(
    CLEANED_DATA_DIR / "maob_cleaning_numeric_audit.csv",
    index=False
)

decision_table.to_csv(
    CLEANED_DATA_DIR / "maob_cleaning_decision.csv",
    index=False
)

df_duplicate_evidence.to_csv(
    CLEANED_DATA_DIR / "maob_duplicate_candidates.csv",
    index=False
)

print("\n" + "=" * 75)
print("CLEANING JUSTIFICATION COMPLETED")
print("=" * 75)
print("Audit evidence saved to:", CLEANED_DATA_DIR)


In [ ]:
# ============================================================
# CELL 03.14 — PREPARE MAO-B CLEANING
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

# Ambil data mentah
df_maob_cleaning = datasets["maob_activity"].copy()

# Pastikan kolom numerik dapat dibaca
df_maob_cleaning["standard_value"] = pd.to_numeric(
    df_maob_cleaning["standard_value"],
    errors="coerce"
)

print("=" * 70)
print("MAO-B CLEANING PREPARATION")
print("=" * 70)

print("Original records:", len(df_maob_cleaning))
print("Original columns:", len(df_maob_cleaning.columns))

# Pastikan output directory tersedia
CLEANED_DATA_DIR.mkdir(parents=True, exist_ok=True)

print("Output directory:", CLEANED_DATA_DIR)
print("Directory exists:", CLEANED_DATA_DIR.exists())

# Validasi kolom yang dibutuhkan
required_columns = [
    "molecule_chembl_id",
    "assay_chembl_id",
    "standard_type",
    "standard_relation",
    "standard_value",
    "standard_units"
]

missing_columns = [
    col for col in required_columns
    if col not in df_maob_cleaning.columns
]

if missing_columns:
    raise ValueError(
        f"Required columns missing: {missing_columns}"
    )

print("\nPreparation completed successfully.")

In [ ]:

# ============================================================
# CELL 03.15 — MAO-B CLEANING IMPLEMENTATION
# ============================================================

import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1. LOAD RAW DATA
# ------------------------------------------------------------

df_raw = datasets["maob_activity"].copy()

df_raw["standard_value"] = pd.to_numeric(
    df_raw["standard_value"],
    errors="coerce"
)

total_before = len(df_raw)

print("=" * 75)
print("MAO-B CLEANING IMPLEMENTATION")
print("=" * 75)

print("Original records:", total_before)


# ------------------------------------------------------------
# 2. DEFINE CLEANING CRITERIA
# ------------------------------------------------------------

condition_endpoint = (
    df_raw["standard_type"].eq("IC50")
)

condition_unit = (
    df_raw["standard_units"].eq("nM")
)

condition_relation = (
    df_raw["standard_relation"].eq("=")
)

condition_numeric = (
    df_raw["standard_value"].notna()
)

condition_positive = (
    df_raw["standard_value"] > 0
)

condition_molecule = (
    df_raw["molecule_chembl_id"].notna()
)

condition_assay = (
    df_raw["assay_chembl_id"].notna()
)

# Gabungan seluruh kriteria
valid_mask = (
    condition_endpoint &
    condition_unit &
    condition_relation &
    condition_numeric &
    condition_positive &
    condition_molecule &
    condition_assay
)


# ------------------------------------------------------------
# 3. CREATE MAIN CLEANED DATASET
# ------------------------------------------------------------

df_maob_cleaned = df_raw.loc[valid_mask].copy()

# Tandai kandidat duplikasi
duplicate_keys = [
    "molecule_chembl_id",
    "assay_chembl_id",
    "standard_type",
    "standard_relation",
    "standard_value",
    "standard_units"
]

df_maob_cleaned["duplicate_candidate"] = (
    df_maob_cleaned.duplicated(
        subset=duplicate_keys,
        keep=False
    )
)

print("\nMain cleaned dataset:", len(df_maob_cleaned))


# ------------------------------------------------------------
# 4. CREATE EXCLUSION REASONS
# ------------------------------------------------------------

df_excluded = df_raw.loc[~valid_mask].copy()

def get_exclusion_reasons(row):

    reasons = []

    if pd.isna(row["standard_type"]):
        reasons.append("Missing endpoint")
    elif row["standard_type"] != "IC50":
        reasons.append("Endpoint is not IC50")

    if pd.isna(row["standard_units"]):
        reasons.append("Missing standard unit")
    elif row["standard_units"] != "nM":
        reasons.append("Unit is not nM")

    if pd.isna(row["standard_relation"]):
        reasons.append("Missing relation")
    elif row["standard_relation"] != "=":
        reasons.append("Censored/non-exact relation")

    if pd.isna(row["standard_value"]):
        reasons.append("Missing/non-numeric activity value")
    elif row["standard_value"] <= 0:
        reasons.append("Non-positive IC50 value")

    if pd.isna(row["molecule_chembl_id"]):
        reasons.append("Missing molecule ID")

    if pd.isna(row["assay_chembl_id"]):
        reasons.append("Missing assay ID")

    return "; ".join(reasons)


df_excluded["exclusion_reason"] = df_excluded.apply(
    get_exclusion_reasons,
    axis=1
)


# ------------------------------------------------------------
# 5. VALIDATION SUMMARY
# ------------------------------------------------------------

total_after = len(df_maob_cleaned)
total_excluded = len(df_excluded)

summary = pd.DataFrame({
    "Category": [
        "Original records",
        "Cleaned records",
        "Excluded records"
    ],
    "Record_Count": [
        total_before,
        total_after,
        total_excluded
    ]
})

summary["Percentage"] = (
    summary["Record_Count"] / total_before * 100
).round(2)

print("\n" + "=" * 75)
print("CLEANING SUMMARY")
print("=" * 75)

display(summary)

# Pastikan semua record terpartisi
assert total_after + total_excluded == total_before

print("Partition validation: PASSED")


# ------------------------------------------------------------
# 6. CLEANED DATA VALIDATION
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("CLEANED DATA QUALITY CHECK")
print("=" * 75)

print("Endpoint unique:",
      df_maob_cleaned["standard_type"].unique())

print("Unit unique:",
      df_maob_cleaned["standard_units"].unique())

print("Relation unique:",
      df_maob_cleaned["standard_relation"].unique())

print("Missing activity values:",
      df_maob_cleaned["standard_value"].isna().sum())

print("Non-positive activity values:",
      (df_maob_cleaned["standard_value"] <= 0).sum())

print("Unique molecules:",
      df_maob_cleaned["molecule_chembl_id"].nunique())

print("Unique assays:",
      df_maob_cleaned["assay_chembl_id"].nunique())

print("Duplicate candidate rows:",
      df_maob_cleaned["duplicate_candidate"].sum())


# ------------------------------------------------------------
# 7. EXCLUSION REASON SUMMARY
# ------------------------------------------------------------

exclusion_summary = (
    df_excluded["exclusion_reason"]
    .value_counts()
    .rename_axis("Exclusion_Reason")
    .reset_index(name="Record_Count")
)

print("\n" + "=" * 75)
print("EXCLUSION REASON SUMMARY")
print("=" * 75)

display(exclusion_summary)


# ------------------------------------------------------------
# 8. SAVE CLEANING OUTPUTS
# ------------------------------------------------------------

CLEANED_DATA_DIR.mkdir(
    parents=True,
    exist_ok=True
)

df_maob_cleaned.to_csv(
    CLEANED_DATA_DIR / "maob_activity_cleaned.csv",
    index=False
)

df_excluded.to_csv(
    CLEANED_DATA_DIR / "maob_activity_excluded.csv",
    index=False
)

summary.to_csv(
    CLEANED_DATA_DIR / "maob_cleaning_summary.csv",
    index=False
)

exclusion_summary.to_csv(
    CLEANED_DATA_DIR / "maob_exclusion_summary.csv",
    index=False
)

print("\n" + "=" * 75)
print("MAO-B CLEANING OUTPUT SAVED")
print("=" * 75)

print("Cleaned dataset:",
      CLEANED_DATA_DIR / "maob_activity_cleaned.csv")

print("Excluded dataset:",
      CLEANED_DATA_DIR / "maob_activity_excluded.csv")


In [ ]:
import pandas as pd
import numpy as np

# Load saved cleaned dataset
cleaned_path = CLEANED_DATA_DIR / "maob_activity_cleaned.csv"
excluded_path = CLEANED_DATA_DIR / "maob_activity_excluded.csv"

df_maob_cleaned = pd.read_csv(cleaned_path)
df_maob_excluded = pd.read_csv(excluded_path)

print("=" * 70)
print("FINAL VALIDATION — MAO-B CLEANED DATA")
print("=" * 70)

# 1. Dataset dimensions
print("\nA. DATASET DIMENSIONS")
print("Cleaned:", df_maob_cleaned.shape)
print("Excluded:", df_maob_excluded.shape)

# 2. Required column validation
required_columns = [
    "molecule_chembl_id",
    "assay_chembl_id",
    "standard_type",
    "standard_relation",
    "standard_value",
    "standard_units"
]

missing_columns = [
    col for col in required_columns
    if col not in df_maob_cleaned.columns
]

assert not missing_columns, f"Missing columns: {missing_columns}"

# 3. Activity value validation
df_maob_cleaned["standard_value"] = pd.to_numeric(
    df_maob_cleaned["standard_value"],
    errors="coerce"
)

assert df_maob_cleaned["standard_value"].notna().all()
assert (df_maob_cleaned["standard_value"] > 0).all()

# 4. Endpoint consistency
assert df_maob_cleaned["standard_type"].eq("IC50").all()
assert df_maob_cleaned["standard_units"].eq("nM").all()
assert df_maob_cleaned["standard_relation"].eq("=").all()

# 5. Identifier validation
assert df_maob_cleaned["molecule_chembl_id"].notna().all()
assert df_maob_cleaned["assay_chembl_id"].notna().all()

# 6. Partition validation
assert (
    len(df_maob_cleaned) + len(df_maob_excluded)
    == 17947
)

# 7. Duplicate candidate validation
duplicate_count = df_maob_cleaned[
    "duplicate_candidate"
].sum()

print("\nB. VALIDATION RESULTS")
print("Missing required columns:", len(missing_columns))
print("Missing activity values:",
      df_maob_cleaned["standard_value"].isna().sum())
print("Non-positive values:",
      (df_maob_cleaned["standard_value"] <= 0).sum())
print("Unique molecules:",
      df_maob_cleaned["molecule_chembl_id"].nunique())
print("Unique assays:",
      df_maob_cleaned["assay_chembl_id"].nunique())
print("Duplicate candidate rows:", duplicate_count)

print("\nC. FINAL STATUS")
print("All validation checks: PASSED")

print("=" * 70)

In [ ]:
import numpy as np

# Preserve original standardized IC50 values
df_maob_cleaned["pIC50"] = (
    9 - np.log10(df_maob_cleaned["standard_value"])
)

# Validate pIC50
assert np.isfinite(df_maob_cleaned["pIC50"]).all()

print("=" * 65)
print("pIC50 TRANSFORMATION — MAO-B")
print("=" * 65)

print("Records:", len(df_maob_cleaned))
print("Missing pIC50:", df_maob_cleaned["pIC50"].isna().sum())

print("\nDescriptive statistics:")
display(df_maob_cleaned["pIC50"].describe())

# Save transformed dataset
df_maob_cleaned.to_csv(
    CLEANED_DATA_DIR / "maob_activity_cleaned.csv",
    index=False
)

print("\nUpdated cleaned dataset saved successfully.")

In [ ]:
#OUTLIER
columns_to_check = [
    "molecule_chembl_id",
    "assay_chembl_id",
    "standard_type",
    "standard_relation",
    "standard_value",
    "standard_units",
    "pIC50",
    "document_chembl_id"
]

print("10 LOWEST pIC50")
display(
    df_maob_cleaned
    .sort_values("pIC50")
    [columns_to_check]
    .head(10)
)

print("\n10 HIGHEST pIC50")
display(
    df_maob_cleaned
    .sort_values("pIC50", ascending=False)
    [columns_to_check]
    .head(10)
)

In [ ]:
# Select extreme records
df_extreme = pd.concat([
    df_maob_cleaned.nsmallest(10, "pIC50"),
    df_maob_cleaned.nlargest(10, "pIC50")
]).drop_duplicates()

# Relevant columns for audit
audit_columns = [
    "molecule_chembl_id",
    "assay_chembl_id",
    "document_chembl_id",
    "type",
    "value",
    "units",
    "standard_type",
    "standard_value",
    "standard_units",
    "standard_relation",
    "pIC50",  # Tambahkan kolom ini
    "pchembl_value",
    "data_validity_comment",
    "activity_comment",
    "bao_endpoint",
    "assay_description",
    "assay_type",
]

# Hanya gunakan kolom yang tersedia
audit_columns = [
    col for col in audit_columns
    if col in df_extreme.columns
]

display(
    df_extreme[audit_columns]
    .sort_values("pIC50")
)

In [ ]:
# Audit all ChEMBL validity comments
validity_audit = (
    df_maob_cleaned[
        df_maob_cleaned["data_validity_comment"].notna()
    ]
    ["data_validity_comment"]
    .value_counts()
    .reset_index()
)

validity_audit.columns = [
    "Validity_Comment",
    "Record_Count"
]

validity_audit["Percentage"] = (
    validity_audit["Record_Count"]
    / len(df_maob_cleaned) * 100
)

print("=" * 65)
print("MAO-B DATA VALIDITY AUDIT")
print("=" * 65)

display(validity_audit)

In [ ]:
flagged_maob = df_maob_cleaned[
    df_maob_cleaned["data_validity_comment"].notna()
].copy()

print("Flagged records:", len(flagged_maob))

display(
    flagged_maob[
        [
            "molecule_chembl_id",
            "assay_chembl_id",
            "standard_value",
            "standard_units",
            "pIC50",
            "data_validity_comment",
            "assay_description",
            "document_chembl_id"
        ]
    ].sort_values("pIC50")
)

In [ ]:
# Create a separate quality-controlled copy
df_maob_quality = df_maob_cleaned.copy()

# Flag ChEMBL validity warnings
df_maob_quality["quality_flag"] = (
    df_maob_quality["data_validity_comment"]
    .fillna("No warning")
)

# Create binary indicator
df_maob_quality["is_flagged"] = (
    df_maob_quality["data_validity_comment"].notna()
)

# Summarize quality status
quality_summary = (
    df_maob_quality["quality_flag"]
    .value_counts()
    .reset_index()
)

quality_summary.columns = [
    "Quality_Status",
    "Record_Count"
]

quality_summary["Percentage"] = (
    quality_summary["Record_Count"]
    / len(df_maob_quality) * 100
)

display(quality_summary)

# Save quality-annotated dataset
df_maob_quality.to_csv(
    CLEANED_DATA_DIR / "maob_activity_quality_annotated.csv",
    index=False
)

print("Quality-annotated dataset saved.")

In [ ]:
target_columns = [
    col for col in df_maob_cleaned.columns
    if "target" in col.lower()
    or "assay" in col.lower()
]

print("Available target/assay columns:")
print(target_columns)

In [ ]:
suspect_assay = df_maob_cleaned[
    df_maob_cleaned["assay_chembl_id"].eq("CHEMBL5732981")
]

display(suspect_assay)

In [ ]:
# Audit hubungan assay dengan target
assay_target_audit = (
    df_maob_cleaned
    .groupby(
        [
            "assay_chembl_id",
            "target_chembl_id",
            "target_pref_name",
            "target_organism"
        ],
        dropna=False
    )
    .agg(
        Record_Count=("activity_id", "count"),
        Assay_Description=("assay_description", "first")
    )
    .reset_index()
)

display(
    assay_target_audit[
        assay_target_audit["target_chembl_id"] != "CHEMBL2039"
    ]
)

print("Target distribution:")
display(
    df_maob_cleaned["target_chembl_id"]
    .value_counts(dropna=False)
)

In [ ]:
suspect_assay = df_maob_cleaned[
    df_maob_cleaned["assay_chembl_id"] == "CHEMBL5732981"
].copy()

print("Jumlah record:", len(suspect_assay))

display(
    suspect_assay[
        [
            "activity_id",
            "molecule_chembl_id",
            "assay_chembl_id",
            "assay_description",
            "target_chembl_id",
            "target_pref_name",
            "target_organism",
            "type",
            "standard_type",
            "standard_value",
            "standard_units",
            "pIC50",
            "document_chembl_id"
        ]
    ].head(20)
)

In [ ]:
assay_target_count = (
    df_maob_cleaned
    .groupby("assay_chembl_id")["target_chembl_id"]
    .nunique()
    .sort_values(ascending=False)
)

print("Assay dengan lebih dari satu target:")
display(assay_target_count[assay_target_count > 1])

In [ ]:
# Audit distribusi nilai IC50 pada assay mencurigakan
suspect_value_audit = (
    suspect_assay
    .groupby("standard_value")
    .agg(
        Record_Count=("activity_id", "count"),
        Unique_Compounds=("molecule_chembl_id", "nunique"),
        Activity_IDs=("activity_id", list)
    )
    .reset_index()
    .sort_values("Record_Count", ascending=False)
)

display(suspect_value_audit)

print("Jumlah record:", len(suspect_assay))
print(
    "Jumlah senyawa unik:",
    suspect_assay["molecule_chembl_id"].nunique()
)
print(
    "Jumlah nilai IC50 unik:",
    suspect_assay["standard_value"].nunique()
)

In [ ]:
display(
    suspect_assay[
        [
            "assay_chembl_id",
            "assay_description",
            "document_chembl_id",
            "target_chembl_id",
            "target_pref_name",
            "standard_type",
            "standard_value",
            "standard_units",
            "data_validity_comment"
        ]
    ].drop_duplicates()
)

In [ ]:
# Audit nilai asli dan nilai standardisasi
columns_to_audit = [
    "activity_id",
    "molecule_chembl_id",
    "assay_chembl_id",
    "type",
    "value",
    "units",
    "relation",
    "standard_type",
    "standard_value",
    "standard_units",
    "standard_relation",
    "pchembl_value",
    "data_validity_comment",
    "activity_comment",
    "document_chembl_id"
]

display(
    suspect_assay[columns_to_audit]
    .sort_values("standard_value")
    .to_string(index=False)
)

In [ ]:
repeated_value = 8776000.0

repeated_records = suspect_assay[
    suspect_assay["standard_value"] == repeated_value
].copy()

print("Jumlah records:", len(repeated_records))
print(
    "Jumlah senyawa unik:",
    repeated_records["molecule_chembl_id"].nunique()
)

display(repeated_records[columns_to_audit])

In [ ]:
# ============================================================
# ASSAY QUALITY REVIEW — MAO-B
# ============================================================

suspect_assay_id = "CHEMBL5732981"

# Identifikasi record berdasarkan assay
mask_suspect = (
    df_maob_cleaned["assay_chembl_id"]
    .eq(suspect_assay_id)
)

# Pisahkan dataset
df_maob_quarantine = df_maob_cleaned[mask_suspect].copy()

df_maob_review = df_maob_cleaned[~mask_suspect].copy()

# Tambahkan status audit
df_maob_quarantine["audit_status"] = (
    "Quarantined - assay description mismatch"
)

df_maob_review["audit_status"] = (
    "Retained - pending general assay review"
)

# Ringkasan
print("ORIGINAL CLEANED:", len(df_maob_cleaned))
print("QUARANTINE:", len(df_maob_quarantine))
print("REVIEW DATASET:", len(df_maob_review))

# Validasi
assert len(df_maob_review) + len(df_maob_quarantine) == len(df_maob_cleaned)

# Simpan
df_maob_quarantine.to_csv(
    CLEANED_DATA_DIR / "maob_assay_quarantine.csv",
    index=False
)

df_maob_review.to_csv(
    CLEANED_DATA_DIR / "maob_activity_review.csv",
    index=False
)

In [ ]:
# Periksa struktur kolom asli
print(df_maob_cleaned.columns.tolist())

# Periksa beberapa kolom yang relevan
display(
    df_maob_cleaned[
        [
            "activity_id",
            "record_id",
            "activity_comment",
            "document_chembl_id",
            "assay_chembl_id"
        ]
    ].head(15)
)

In [ ]:
suspect_assay_id = "CHEMBL5732981"

df_suspect = df_maob_cleaned[
    df_maob_cleaned["assay_chembl_id"] == suspect_assay_id
].copy()

# Periksa tipe data kolom penting
print("DATA TYPES:")
display(
    df_suspect[
        [
            "activity_id",
            "record_id",
            "activity_comment",
            "standard_value",
            "pIC50",
            "data_validity_comment"
        ]
    ].dtypes
)

# Tampilkan seluruh 40 record
audit_columns = [
    "activity_id",
    "record_id",
    "molecule_chembl_id",
    "standard_value",
    "standard_units",
    "pIC50",
    "pchembl_value",
    "data_validity_comment",
    "activity_comment",
    "document_chembl_id",
    "assay_chembl_id",
    "assay_description"
]

display(
    df_suspect[audit_columns]
    .sort_values("standard_value")
    .reset_index(drop=True)
)

In [ ]:
print("JUMLAH RECORD:", len(df_suspect))

print("\nDISTRIBUSI STANDARD VALUE:")
display(
    df_suspect["standard_value"]
    .describe()
    .to_frame()
)

print("\nDISTRIBUSI VALIDITY COMMENT:")
display(
    df_suspect["data_validity_comment"]
    .value_counts(dropna=False)
)

print("\nJUMLAH MOLEKUL UNIK:")
print(df_suspect["molecule_chembl_id"].nunique())

print("\nJUMLAH NILAI STANDARD VALUE UNIK:")
print(df_suspect["standard_value"].nunique())

In [ ]:
assay_id = "CHEMBL5732981"

# Ambil metadata assay dari data mentah
assay_metadata = df_maob_raw[
    df_maob_raw["assay_chembl_id"].eq(assay_id)
]

metadata_columns = [
    "assay_chembl_id",
    "assay_description",
    "assay_type",
    "target_chembl_id",
    "target_pref_name",
    "target_organism",
    "document_chembl_id",
    "document_year",
    "document_journal",
    "src_id",
    "bao_endpoint",
    "bao_label"
]

display(
    assay_metadata[metadata_columns]
    .drop_duplicates()
    .reset_index(drop=True)
)

In [ ]:
extreme_records = df_suspect[
    df_suspect["standard_value"].eq(8776000)
]

display(
    extreme_records[
        [
            "molecule_chembl_id",
            "activity_id",
            "record_id",
            "type",
            "value",
            "units",
            "standard_type",
            "standard_value",
            "relation",
            "standard_relation",
            "data_validity_comment"
        ]
    ]
)

In [ ]:
# 03.9 - ASSAY-LEVEL QUALITY REVIEW

from pathlib import Path

# Salin seluruh record yang dikarantina
df_maob_quarantine = df_suspect.copy()

df_maob_quarantine["audit_status"] = (
    "Quarantined - assay description and target annotation mismatch"
)

df_maob_quarantine["audit_reason"] = (
    "Assay description mentions LSD1, "
    "while ChEMBL target annotation indicates MAO-B"
)

# Tandai 14 record dengan nilai ekstrem
df_maob_quarantine["extreme_value_flag"] = (
    df_maob_quarantine["standard_value"] == 8776000
)

# Buat dataset provisional
df_maob_review = df_maob_cleaned[
    df_maob_cleaned["assay_chembl_id"] != "CHEMBL5732981"
].copy()

# Simpan hasil
df_maob_quarantine.to_csv(
    CLEANED_DATA_DIR / "maob_assay_quarantine.csv",
    index=False
)

df_maob_review.to_csv(
    CLEANED_DATA_DIR / "maob_activity_provisional.csv",
    index=False
)

print("ASSAY-LEVEL REVIEW SUMMARY")
print("=" * 55)
print("Original cleaned :", len(df_maob_cleaned))
print("Quarantine       :", len(df_maob_quarantine))
print("Extreme records  :", df_maob_quarantine["extreme_value_flag"].sum())
print("Provisional data :", len(df_maob_review))

In [ ]:
# 03.10 - FINAL DATASET VALIDATION

print("=" * 60)
print("FINAL DATASET VALIDATION — MAO-B")
print("=" * 60)

checks = {
    "Original dataset preserved":
        len(df_maob_cleaned) == 6006,

    "Quarantine contains 40 records":
        len(df_maob_quarantine) == 40,

    "Provisional dataset contains 5966 records":
        len(df_maob_review) == 5966,

    "No quarantined assay in provisional dataset":
        not df_maob_review["assay_chembl_id"].eq(
            "CHEMBL5732981"
        ).any(),

    "All endpoints are IC50":
        df_maob_review["standard_type"].eq("IC50").all(),

    "All units are nM":
        df_maob_review["standard_units"].eq("nM").all(),

    "All relations are exact":
        df_maob_review["standard_relation"].eq("=").all(),

    "No missing activity values":
        df_maob_review["standard_value"].notna().all(),

    "All activity values are positive":
        df_maob_review["standard_value"].gt(0).all(),

    "No missing pIC50":
        df_maob_review["pIC50"].notna().all()
}

for name, result in checks.items():
    print(f"[{'PASS' if result else 'FAIL'}] {name}")

print("\nTOTAL CHECKS:", len(checks))
print("PASSED:", sum(checks.values()))
print("FAILED:", len(checks) - sum(checks.values()))

assert all(checks.values()), "Final validation failed!"

print("\nFINAL VALIDATION: PASSED")

In [ ]:
# CHECK ALPHA-SYNUCLEIN ASSAY METADATA

alpha_metadata_path = (
    ALPHA_SYNUCLEIN_DIR /
    "alpha_synuclein_assay_metadata_raw.csv"
)

print("FILE PATH:", alpha_metadata_path)
print("FILE EXISTS:", alpha_metadata_path.is_file())

if alpha_metadata_path.is_file():
    df_alpha_assay_metadata = pd.read_csv(
        alpha_metadata_path
    )

    print("\nDATASET SHAPE:", df_alpha_assay_metadata.shape)
    print("\nCOLUMNS:")
    print(df_alpha_assay_metadata.columns.tolist())

    display(df_alpha_assay_metadata.head())
else:
    print("File belum terdeteksi. Periksa sinkronisasi Google Drive.")

In [ ]:
# ============================================================
# ALPHA-SYNUCLEIN ASSAY METADATA AUDIT
# ============================================================

print("=" * 65)
print("ALPHA-SYNUCLEIN ASSAY METADATA AUDIT")
print("=" * 65)

print("\nA. BASIC INFORMATION")
print("Rows:", len(df_alpha_assay_metadata))
print("Columns:", len(df_alpha_assay_metadata.columns))
print("Duplicate rows:", df_alpha_assay_metadata.duplicated().sum())

print("\nB. MISSING VALUES")
missing = (
    df_alpha_assay_metadata.isnull()
    .sum()
    .sort_values(ascending=False)
)

missing_summary = pd.DataFrame({
    "Missing_Count": missing,
    "Missing_Percentage": (
        missing / len(df_alpha_assay_metadata) * 100
    )
})

display(missing_summary[missing_summary["Missing_Count"] > 0])

print("\nC. TARGET DISTRIBUTION")
display(
    df_alpha_assay_metadata["target_chembl_id"]
    .value_counts(dropna=False)
)

print("\nD. ORGANISM DISTRIBUTION")
display(
    df_alpha_assay_metadata["assay_organism"]
    .value_counts(dropna=False)
)

print("\nE. ASSAY TYPE DISTRIBUTION")
display(
    df_alpha_assay_metadata["assay_type"]
    .value_counts(dropna=False)
)

print("\nF. RELATIONSHIP TYPE")
display(
    df_alpha_assay_metadata["relationship_type"]
    .value_counts(dropna=False)
)

print("\nG. BAO LABEL")
display(
    df_alpha_assay_metadata["bao_label"]
    .value_counts(dropna=False)
)

In [ ]:
# Tampilkan seluruh deskripsi assay

assay_descriptions = (
    df_alpha_assay_metadata[
        [
            "assay_chembl_id",
            "description",
            "assay_type",
            "target_chembl_id",
            "relationship_type",
            "confidence_score"
        ]
    ]
    .drop_duplicates()
    .sort_values("assay_chembl_id")
)

display(assay_descriptions)

In [ ]:

# ============================================================
# ALPHA-SYNUCLEIN ACTIVITY AND METADATA INTEGRATION AUDIT
# ============================================================

# Pastikan dataset aktivitas sudah tersedia
if "df_alpha_activity" not in globals():
    df_alpha_activity = pd.read_csv(
        ALPHA_SYNUCLEIN_DIR /
        "alpha_synuclein_activity_raw.csv"
    )

df_alpha_meta = df_alpha_assay_metadata.copy()
df_alpha_raw = df_alpha_activity.copy()

print("=" * 70)
print("ALPHA-SYNUCLEIN ACTIVITY AUDIT")
print("=" * 70)

# A. Basic information
print("\nA. BASIC INFORMATION")
print("Activity records:", len(df_alpha_raw))
print("Activity columns:", len(df_alpha_raw.columns))
print("Assay metadata:", len(df_alpha_meta))
print("Exact duplicate activity rows:",
      df_alpha_raw.duplicated().sum())

# B. Endpoint distribution
print("\nB. STANDARD TYPE DISTRIBUTION")

for col in ["type", "standard_type"]:
    if col in df_alpha_raw.columns:
        print(f"\n{col}:")
        display(
            df_alpha_raw[col]
            .value_counts(dropna=False)
            .to_frame("Record_Count")
        )

# C. Units
print("\nC. ORIGINAL UNITS")

if "units" in df_alpha_raw.columns:
    display(
        df_alpha_raw["units"]
        .value_counts(dropna=False)
        .to_frame("Record_Count")
    )

print("\nSTANDARD UNITS")

if "standard_units" in df_alpha_raw.columns:
    display(
        df_alpha_raw["standard_units"]
        .value_counts(dropna=False)
        .to_frame("Record_Count")
    )

# D. Relation distribution
print("\nD. RELATION DISTRIBUTION")

for col in ["relation", "standard_relation"]:
    if col in df_alpha_raw.columns:
        print(f"\n{col}:")
        display(
            df_alpha_raw[col]
            .value_counts(dropna=False)
            .to_frame("Record_Count")
        )

# E. Assay metadata coverage
print("\nE. ASSAY METADATA COVERAGE")

activity_assays = set(
    df_alpha_raw["assay_chembl_id"].dropna().unique()
)

metadata_assays = set(
    df_alpha_meta["assay_chembl_id"].dropna().unique()
)

matched = activity_assays & metadata_assays
unmatched = activity_assays - metadata_assays

print("Unique activity assays:", len(activity_assays))
print("Unique metadata assays:", len(metadata_assays))
print("Matched assays:", len(matched))
print("Unmatched activity assays:", len(unmatched))

print(
    "Metadata coverage (%):",
    round(len(matched) / len(activity_assays) * 100, 2)
    if activity_assays else 0
)

# F. Target distribution in activity
print("\nF. ACTIVITY TARGET DISTRIBUTION")

if "target_chembl_id" in df_alpha_raw.columns:
    display(
        df_alpha_raw["target_chembl_id"]
        .value_counts(dropna=False)
    )

# G. Join metadata
df_alpha_audit = df_alpha_raw.merge(
    df_alpha_meta[
        [
            "assay_chembl_id",
            "relationship_type",
            "confidence_score",
            "bao_label",
            "assay_type",
            "assay_organism",
            "target_chembl_id"
        ]
    ],
    on="assay_chembl_id",
    how="left",
    suffixes=("", "_metadata"),
    indicator=True
)

print("\nG. METADATA JOIN STATUS")
display(
    df_alpha_audit["_merge"]
    .value_counts()
    .to_frame("Record_Count")
)

print("\nH. RELATIONSHIP TYPE IN ACTIVITY RECORDS")
display(
    df_alpha_audit["relationship_type"]
    .value_counts(dropna=False)
)

print("\nI. BAO LABEL IN ACTIVITY RECORDS")
display(
    df_alpha_audit["bao_label"]
    .value_counts(dropna=False)
)


In [ ]:

# ============================================================
# ALPHA-SYNUCLEIN ASSAY RELATIONSHIP & ENDPOINT AUDIT
# ============================================================

# A. Relationship type per assay
relationship_by_assay = (
    df_alpha_audit.groupby(
        ["assay_chembl_id", "relationship_type"],
        dropna=False
    )
    .size()
    .reset_index(name="record_count")
)

print("A. RELATIONSHIP TYPE PER ASSAY")
display(
    relationship_by_assay["relationship_type"]
    .value_counts(dropna=False)
)

print("\nB. ASSAY COUNTS PER RELATIONSHIP")
display(
    relationship_by_assay.groupby("relationship_type")
    ["assay_chembl_id"]
    .nunique()
)

# B. Endpoint by relationship
print("\nC. ENDPOINT BY RELATIONSHIP")

endpoint_relationship = pd.crosstab(
    df_alpha_audit["standard_type"],
    df_alpha_audit["relationship_type"],
    margins=True
)

display(endpoint_relationship)

# C. Endpoint and BAO
print("\nD. ENDPOINT BY BAO LABEL")

endpoint_bao = pd.crosstab(
    df_alpha_audit["standard_type"],
    df_alpha_audit["bao_label"],
    margins=True
)

display(endpoint_bao)

# D. Confidence score
print("\nE. CONFIDENCE SCORE BY RELATIONSHIP")

display(
    df_alpha_audit.groupby(
        "relationship_type",
        dropna=False
    )["confidence_score"]
    .describe()
)

# E. Assay type F inspection
print("\nF. NON-BINDING ASSAY TYPE")

display(
    df_alpha_audit.loc[
        df_alpha_audit["assay_type_metadata"].eq("F"),
        [
            "assay_chembl_id",
            "standard_type",
            "standard_value",
            "standard_units",
            "relationship_type",
            "bao_label",
            "description"
        ]
    ].drop_duplicates()
)

# F. Assay descriptions for homologous records
print("\nG. HOMOLOGOUS ASSAY DESCRIPTIONS")

homologous_assays = (
    df_alpha_audit.loc[
        df_alpha_audit["relationship_type"].eq("H"),
        [
            "assay_chembl_id",
            "standard_type",
            "bao_label",
            "confidence_score",
            "description"
        ]
    ]
    .drop_duplicates("assay_chembl_id")
    .sort_values("assay_chembl_id")
)

display(homologous_assays)


In [ ]:

# ============================================================
# FIX: ASSAY TYPE AND DESCRIPTION AUDIT
# ============================================================

print("=" * 70)
print("ASSAY TYPE F — METADATA REVIEW")
print("=" * 70)

# A. Inspect assay type F
df_alpha_assay_F = df_alpha_assay_metadata.loc[
    df_alpha_assay_metadata["assay_type"].eq("F"),
    [
        "assay_chembl_id",
        "description",
        "assay_type",
        "target_chembl_id",
        "relationship_type",
        "confidence_score",
        "bao_label",
        "assay_organism",
        "document_chembl_id"
    ]
].copy()

display(df_alpha_assay_F)

# B. Inspect all homologous assays
print("\n" + "=" * 70)
print("HOMOLOGOUS ASSAY REVIEW")
print("=" * 70)

df_alpha_homologous = (
    df_alpha_assay_metadata.loc[
        df_alpha_assay_metadata["relationship_type"].eq("H"),
        [
            "assay_chembl_id",
            "description",
            "assay_type",
            "target_chembl_id",
            "confidence_score",
            "bao_label",
            "assay_organism",
            "document_chembl_id"
        ]
    ]
    .sort_values("assay_chembl_id")
    .copy()
)

print("Homologous assays:", len(df_alpha_homologous))

display(df_alpha_homologous)

# C. Summarize homologous assay endpoint distribution
print("\n" + "=" * 70)
print("HOMOLOGOUS ASSAY ENDPOINT DISTRIBUTION")
print("=" * 70)

homologous_ids = set(
    df_alpha_homologous["assay_chembl_id"]
)

df_alpha_homologous_activity = df_alpha_raw.loc[
    df_alpha_raw["assay_chembl_id"].isin(homologous_ids)
].copy()

display(
    df_alpha_homologous_activity[
        "standard_type"
    ].value_counts(dropna=False).to_frame("Record_Count")
)


In [ ]:

# ============================================================
# POTENCY ASSAY DESCRIPTION REVIEW
# ============================================================

potency_assay_ids = set(
    df_alpha_raw.loc[
        df_alpha_raw["standard_type"].eq("Potency"),
        "assay_chembl_id"
    ].dropna()
)

df_alpha_potency_review = (
    df_alpha_assay_metadata.loc[
        df_alpha_assay_metadata["assay_chembl_id"].isin(
            potency_assay_ids
        ),
        [
            "assay_chembl_id",
            "description",
            "relationship_type",
            "confidence_score",
            "bao_label",
            "assay_organism",
            "document_chembl_id"
        ]
    ]
    .copy()
)

print("Unique Potency assays:", len(df_alpha_potency_review))

print("\nPOTENCY ASSAY DESCRIPTION SAMPLE")
display(
    df_alpha_potency_review.head(30)
)


In [ ]:
# Ambil metadata assay Direct
df_alpha_direct_metadata = (
    df_alpha_assay_metadata.loc[
        df_alpha_assay_metadata["relationship_type"].eq("D"),
        [
            "assay_chembl_id",
            "description",
            "assay_type",
            "confidence_score",
            "bao_label",
            "assay_organism",
            "document_chembl_id"
        ]
    ]
    .copy()
)

# Hitung jumlah record per assay
direct_counts = (
    df_alpha_raw.loc[
        df_alpha_raw["assay_chembl_id"].isin(
            df_alpha_direct_metadata["assay_chembl_id"]
        )
    ]
    .groupby("assay_chembl_id")
    .agg(
        record_count=("molecule_chembl_id", "size"),
        endpoint_count=("standard_type", "nunique")
    )
    .reset_index()
)

# Gabungkan metadata dan jumlah record
df_alpha_direct_review = (
    df_alpha_direct_metadata
    .merge(
        direct_counts,
        on="assay_chembl_id",
        how="left"
    )
    .sort_values(
        ["record_count", "assay_chembl_id"],
        ascending=[False, True]
    )
)

print("Jumlah Direct assays:",
      len(df_alpha_direct_review))

display(df_alpha_direct_review)

In [ ]:
direct_ids = set(
    df_alpha_direct_metadata["assay_chembl_id"]
)

df_alpha_direct_activity = df_alpha_raw.loc[
    df_alpha_raw["assay_chembl_id"].isin(direct_ids)
].copy()

print("Direct activity records:",
      len(df_alpha_direct_activity))

print("\nEndpoint distribution:")
display(
    df_alpha_direct_activity[
        "standard_type"
    ].value_counts(dropna=False)
)

print("\nEndpoint × BAO label:")
display(
    pd.crosstab(
        df_alpha_direct_activity["standard_type"],
        df_alpha_direct_activity["bao_label"],
        dropna=False
    )
)

In [ ]:
# Distribusi jumlah endpoint dalam setiap assay
display(
    df_alpha_direct_review["endpoint_count"]
    .value_counts()
    .sort_index()
    .rename_axis("Endpoint_Count")
    .reset_index(name="Assay_Count")
)

In [ ]:
target_endpoints = ["IC50", "Ki", "Kd"]

df_alpha_binding_audit = df_alpha_direct_activity.loc[
    df_alpha_direct_activity["standard_type"].isin(
        target_endpoints
    )
].copy()

# Gabungkan metadata assay
assay_columns = [
    "assay_chembl_id",
    "description",
    "bao_label",
    "assay_organism",
    "document_chembl_id"
]

df_alpha_binding_audit = df_alpha_binding_audit.merge(
    df_alpha_assay_metadata[assay_columns],
    on="assay_chembl_id",
    how="left",
    validate="many_to_one"
)

# Ringkasan endpoint dan satuan
display(
    df_alpha_binding_audit.groupby(
        ["standard_type", "bao_label", "standard_units"],
        dropna=False
    ).agg(
        record_count=("molecule_chembl_id", "size"),
        unique_assays=("assay_chembl_id", "nunique"),
        unique_molecules=("molecule_chembl_id", "nunique")
    ).reset_index()
)

In [ ]:
display(
    df_alpha_binding_audit.groupby(
        ["standard_type", "standard_relation", "standard_units"],
        dropna=False
    ).agg(
        record_count=("standard_value", "size"),
        missing_value=("standard_value", lambda x: x.isna().sum()),
        unique_assays=("assay_chembl_id", "nunique")
    ).reset_index()
)

In [ ]:
for endpoint in ["IC50", "Ki", "Kd"]:
    print(f"\n{'=' * 70}")
    print(f"ASSAY DESCRIPTION — {endpoint}")
    print("=" * 70)

    df_review = (
        df_alpha_binding_audit.loc[
            df_alpha_binding_audit["standard_type"].eq(endpoint),
            [
                "assay_chembl_id",
                "description",
                "bao_label",
                "standard_units",
                "document_chembl_id"
            ]
        ]
        .drop_duplicates("assay_chembl_id")
        .sort_values("assay_chembl_id")
    )

    print("Unique assays:", len(df_review))
    display(df_review)

In [ ]:
# ============================================================
# REBUILD BINDING AUDIT — ALPHA-SYNUCLEIN DIRECT
# ============================================================

target_endpoints = ["IC50", "Ki", "Kd"]

# 1. Ambil aktivitas IC50, Ki, dan Kd
df_alpha_binding_audit = df_alpha_direct_activity.loc[
    df_alpha_direct_activity["standard_type"].isin(target_endpoints)
].copy()

# 2. Siapkan metadata assay
metadata_columns = [
    "assay_chembl_id",
    "description",
    "bao_label",
    "assay_organism",
    "document_chembl_id"
]

df_alpha_metadata_subset = (
    df_alpha_assay_metadata[metadata_columns]
    .drop_duplicates(subset="assay_chembl_id")
    .rename(columns={
        "description": "assay_description_meta",
        "bao_label": "assay_bao_label",
        "assay_organism": "assay_organism_meta",
        "document_chembl_id": "document_chembl_id_meta"
    })
)

# 3. Gabungkan metadata
df_alpha_binding_audit = df_alpha_binding_audit.merge(
    df_alpha_metadata_subset,
    on="assay_chembl_id",
    how="left",
    validate="many_to_one"
)

# 4. Periksa kolom hasil merge
print("SHAPE:", df_alpha_binding_audit.shape)

print("\nMETADATA COVERAGE:")
print(
    df_alpha_binding_audit[
        "assay_bao_label"
    ].notna().value_counts()
)

# 5. Ringkasan endpoint, BAO, dan unit
display(
    df_alpha_binding_audit.groupby(
        ["standard_type", "assay_bao_label", "standard_units"],
        dropna=False
    ).agg(
        record_count=("molecule_chembl_id", "size"),
        unique_assays=("assay_chembl_id", "nunique"),
        unique_molecules=("molecule_chembl_id", "nunique")
    ).reset_index()
)

In [ ]:
# ============================================================
# ASSAY DESCRIPTION REVIEW
# ============================================================

for endpoint in ["IC50", "Ki", "Kd"]:

    print(f"\n{'=' * 70}")
    print(f"ASSAY DESCRIPTION — {endpoint}")
    print("=" * 70)

    df_review = (
        df_alpha_binding_audit.loc[
            df_alpha_binding_audit["standard_type"].eq(endpoint),
            [
                "assay_chembl_id",
                "assay_description_meta",
                "assay_bao_label",
                "standard_units",
                "standard_relation",
                "document_chembl_id_meta"
            ]
        ]
        .drop_duplicates("assay_chembl_id")
        .sort_values("assay_chembl_id")
    )

    print("Unique assays:", len(df_review))
    display(df_review)

In [ ]:
# ============================================================
# FULL ASSAY DESCRIPTION REVIEW
# ALPHA-SYNUCLEIN — DIRECT RELATIONSHIP
# ============================================================

import pandas as pd

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", None)
pd.set_option("display.width", 200)

for endpoint in ["IC50", "Ki", "Kd"]:

    print(f"\n{'=' * 100}")
    print(f"FULL ASSAY DESCRIPTION — {endpoint}")
    print("=" * 100)

    df_review = (
        df_alpha_binding_audit.loc[
            df_alpha_binding_audit["standard_type"].eq(endpoint),
            [
                "assay_chembl_id",
                "assay_description_meta",
                "assay_bao_label",
                "standard_units",
                "standard_relation",
                "document_chembl_id_meta"
            ]
        ]
        .drop_duplicates("assay_chembl_id")
        .sort_values("assay_chembl_id")
        .reset_index(drop=True)
    )

    display(df_review)

In [ ]:
# ============================================================
# ASSAY CLASSIFICATION — ALPHA-SYNUCLEIN
# ============================================================

assay_classification = {
    # IC50
    ("IC50", "CHEMBL2065090"): "Binding_affinity",
    ("IC50", "CHEMBL5551998"): "Fibril_depolymerization",
    ("IC50", "CHEMBL5529075"): "Needs_review",

    # Ki
    ("Ki", "CHEMBL2065090"): "Binding_affinity",
    ("Ki", "CHEMBL3594881"): "ThioT_displacement",
    ("Ki", "CHEMBL5256054"): "Fibril_binding",
    ("Ki", "CHEMBL5391420"): "Competitive_binding",
    ("Ki", "CHEMBL6106136"): "Radioligand_tissue_binding",
    ("Ki", "CHEMBL6106205"): "Radioligand_tissue_binding",

    # Kd
    ("Kd", "CHEMBL5241349"): "Needs_review"
}

df_alpha_binding_audit["assay_group"] = [
    assay_classification.get(
        (endpoint, assay_id),
        "Aggregation_inhibition_candidate"
        if endpoint == "IC50"
        else "Binding_affinity_candidate"
    )
    for endpoint, assay_id in zip(
        df_alpha_binding_audit["standard_type"],
        df_alpha_binding_audit["assay_chembl_id"]
    )
]

# Tambahkan status kualitas awal
df_alpha_binding_audit["audit_status"] = "Pending_review"

# ============================================================
# SUMMARY
# ============================================================

display(
    df_alpha_binding_audit.groupby(
        ["standard_type", "assay_group"],
        dropna=False
    ).agg(
        record_count=("molecule_chembl_id", "size"),
        unique_assays=("assay_chembl_id", "nunique"),
        unique_molecules=("molecule_chembl_id", "nunique")
    ).reset_index()
)

In [ ]:
# ============================================================
# DATA QUALITY AUDIT — ALPHA-SYNUCLEIN
# ============================================================

df_alpha_binding_quality = df_alpha_binding_audit.copy()

df_alpha_binding_quality["quality_issue"] = ""

# Missing value
mask_missing_value = (
    df_alpha_binding_quality["standard_value"].isna()
)

# Missing unit
mask_missing_unit = (
    df_alpha_binding_quality["standard_units"].isna()
)

# Non-exact relation
mask_censored = (
    df_alpha_binding_quality["standard_relation"].ne("=")
    & df_alpha_binding_quality["standard_relation"].notna()
)

# Non-nM unit
mask_non_nM = (
    df_alpha_binding_quality["standard_units"].notna()
    & df_alpha_binding_quality["standard_units"].ne("nM")
)

# Assign audit labels
df_alpha_binding_quality.loc[
    mask_missing_value, "quality_issue"
] += "Missing_value; "

df_alpha_binding_quality.loc[
    mask_missing_unit, "quality_issue"
] += "Missing_unit; "

df_alpha_binding_quality.loc[
    mask_censored, "quality_issue"
] += "Censored_relation; "

df_alpha_binding_quality.loc[
    mask_non_nM, "quality_issue"
] += "Non_nM_unit; "

# Ringkasan
display(
    df_alpha_binding_quality.groupby(
        ["standard_type", "quality_issue"],
        dropna=False
    ).size().reset_index(name="record_count")
)

# Tampilkan record bermasalah
df_alpha_problem_records = df_alpha_binding_quality.loc[
    df_alpha_binding_quality["quality_issue"].ne("")
].copy()

display(
    df_alpha_problem_records[
        [
            "molecule_chembl_id",
            "assay_chembl_id",
            "standard_type",
            "standard_value",
            "standard_units",
            "standard_relation",
            "assay_group",
            "quality_issue"
        ]
    ]
)

In [ ]:
df_alpha_binding_quality.to_csv(
    CLEANED_DATA_DIR / "alpha_synuclein_binding_audit.csv",
    index=False
)

df_alpha_problem_records.to_csv(
    CLEANED_DATA_DIR / "alpha_synuclein_binding_problem_records.csv",
    index=False
)

In [ ]:
# ============================================================
# ALPHA-SYNUCLEIN — PROVISIONAL DATASET SELECTION
# ============================================================

import pandas as pd
from pathlib import Path

# Salin agar dataset audit asli tetap utuh
df_alpha_provisional = df_alpha_binding_audit.copy()

# Pastikan nilai aktivitas numerik
df_alpha_provisional["standard_value"] = pd.to_numeric(
    df_alpha_provisional["standard_value"],
    errors="coerce"
)

# Kriteria teknis untuk record yang dapat dipakai
valid_mask = (
    df_alpha_provisional["standard_value"].notna()
    & df_alpha_provisional["standard_value"].gt(0)
    & df_alpha_provisional["standard_units"].eq("nM")
    & df_alpha_provisional["standard_relation"].eq("=")
)

df_alpha_provisional["technical_status"] = "Excluded - technical issue"
df_alpha_provisional.loc[
    valid_mask, "technical_status"
] = "Pass - technical criteria"

# Kandidat dataset IC50 agregasi
df_alpha_ic50_aggregation = df_alpha_provisional.loc[
    valid_mask
    & df_alpha_provisional["standard_type"].eq("IC50")
    & df_alpha_provisional["assay_group"].eq(
        "Aggregation_inhibition_candidate"
    )
].copy()

# Kandidat dataset IC50 binding
df_alpha_ic50_binding = df_alpha_provisional.loc[
    valid_mask
    & df_alpha_provisional["standard_type"].eq("IC50")
    & df_alpha_provisional["assay_group"].eq("Binding_affinity")
].copy()

# Kandidat dataset IC50 depolymerisasi
df_alpha_ic50_depolymerization = df_alpha_provisional.loc[
    valid_mask
    & df_alpha_provisional["standard_type"].eq("IC50")
    & df_alpha_provisional["assay_group"].eq(
        "Fibril_depolymerization"
    )
].copy()

# Dataset Ki: tetap dipisahkan berdasarkan assay_group
df_alpha_ki = df_alpha_provisional.loc[
    valid_mask
    & df_alpha_provisional["standard_type"].eq("Ki")
].copy()

# Dataset Kd binding
df_alpha_kd_binding = df_alpha_provisional.loc[
    valid_mask
    & df_alpha_provisional["standard_type"].eq("Kd")
    & df_alpha_provisional["assay_group"].eq(
        "Binding_affinity_candidate"
    )
].copy()

# Ringkasan jumlah record
dataset_summary = pd.DataFrame([
    {
        "Dataset": "IC50 Aggregation",
        "Records": len(df_alpha_ic50_aggregation),
        "Assays": df_alpha_ic50_aggregation["assay_chembl_id"].nunique(),
        "Molecules": df_alpha_ic50_aggregation["molecule_chembl_id"].nunique()
    },
    {
        "Dataset": "IC50 Binding",
        "Records": len(df_alpha_ic50_binding),
        "Assays": df_alpha_ic50_binding["assay_chembl_id"].nunique(),
        "Molecules": df_alpha_ic50_binding["molecule_chembl_id"].nunique()
    },
    {
        "Dataset": "IC50 Depolymerization",
        "Records": len(df_alpha_ic50_depolymerization),
        "Assays": df_alpha_ic50_depolymerization["assay_chembl_id"].nunique(),
        "Molecules": df_alpha_ic50_depolymerization["molecule_chembl_id"].nunique()
    },
    {
        "Dataset": "Ki",
        "Records": len(df_alpha_ki),
        "Assays": df_alpha_ki["assay_chembl_id"].nunique(),
        "Molecules": df_alpha_ki["molecule_chembl_id"].nunique()
    },
    {
        "Dataset": "Kd Binding",
        "Records": len(df_alpha_kd_binding),
        "Assays": df_alpha_kd_binding["assay_chembl_id"].nunique(),
        "Molecules": df_alpha_kd_binding["molecule_chembl_id"].nunique()
    }
])

display(dataset_summary)

In [ ]:
alpha_output_dir = CLEANED_DATA_DIR / "Alpha_Synuclein"
alpha_output_dir.mkdir(parents=True, exist_ok=True)

datasets_to_save = {
    "alpha_ic50_aggregation_provisional.csv":
        df_alpha_ic50_aggregation,

    "alpha_ic50_binding_provisional.csv":
        df_alpha_ic50_binding,

    "alpha_ic50_depolymerization_provisional.csv":
        df_alpha_ic50_depolymerization,

    "alpha_ki_provisional.csv":
        df_alpha_ki,

    "alpha_kd_binding_provisional.csv":
        df_alpha_kd_binding
}

for filename, dataframe in datasets_to_save.items():
    dataframe.to_csv(
        alpha_output_dir / filename,
        index=False
    )

dataset_summary.to_csv(
    alpha_output_dir / "alpha_binding_endpoint_summary.csv",
    index=False
)

print("Provisional datasets saved successfully.")

In [ ]:
# ============================================================
# REPEATED MEASUREMENT AUDIT
# ============================================================

datasets = {
    "IC50 Aggregation": df_alpha_ic50_aggregation,
    "IC50 Binding": df_alpha_ic50_binding,
    "IC50 Depolymerization": df_alpha_ic50_depolymerization,
    "Ki": df_alpha_ki,
    "Kd Binding": df_alpha_kd_binding
}

for dataset_name, df in datasets.items():

    print("\n" + "=" * 65)
    print(dataset_name)
    print("=" * 65)

    # Molecules measured more than once
    molecule_counts = (
        df.groupby("molecule_chembl_id")
        .size()
        .sort_values(ascending=False)
    )

    print("Total records:", len(df))
    print("Unique molecules:", df["molecule_chembl_id"].nunique())
    print("Repeated molecules:", (molecule_counts > 1).sum())

    # Assay distribution
    assay_counts = (
        df.groupby("assay_chembl_id")
        .size()
        .sort_values(ascending=False)
    )

    print("\nRecords per assay:")
    display(assay_counts.to_frame("record_count"))

    # Value distribution
    print("\nActivity value summary:")
    display(df["standard_value"].describe())

In [ ]:
# ============================================================
# IC50 BINDING — DATA INTEGRITY CHECK
# ============================================================

print("DATAFRAME SHAPE:")
print(df_alpha_ic50_binding.shape)

print("\nSTANDARD VALUE INFORMATION:")
print(df_alpha_ic50_binding["standard_value"].info())

print("\nMISSING VALUE:")
print(
    df_alpha_ic50_binding["standard_value"].isna().sum()
)

print("\nNON-NUMERIC VALUE:")
print(
    pd.to_numeric(
        df_alpha_ic50_binding["standard_value"],
        errors="coerce"
    ).isna().sum()
)

print("\nSTANDARD VALUE CONTENT:")
display(
    df_alpha_ic50_binding[
        [
            "molecule_chembl_id",
            "assay_chembl_id",
            "standard_type",
            "standard_value",
            "standard_units",
            "standard_relation"
        ]
    ]
)

In [ ]:
print("TOTAL:", len(df_alpha_ic50_binding))

print(
    "Missing value:",
    df_alpha_ic50_binding["standard_value"].isna().sum()
)

print(
    "Non-positive:",
    (
        df_alpha_ic50_binding["standard_value"] <= 0
    ).sum()
)

print(
    "Non-nM:",
    (
        df_alpha_ic50_binding["standard_units"] != "nM"
    ).sum()
)

print(
    "Non-exact relation:",
    (
        df_alpha_ic50_binding["standard_relation"] != "="
    ).sum()
)

In [ ]:
for name, df in {
    "IC50 Aggregation": df_alpha_ic50_aggregation,
    "Kd Binding": df_alpha_kd_binding
}.items():

    print("\n" + "=" * 60)
    print(name)
    print("=" * 60)

    repeated = (
        df.groupby("molecule_chembl_id")
        .agg(
            record_count=("molecule_chembl_id", "size"),
            assay_count=("assay_chembl_id", "nunique"),
            assay_ids=("assay_chembl_id", "unique")
        )
        .query("record_count > 1")
        .sort_values("record_count", ascending=False)
    )

    display(repeated)

In [ ]:
# ============================================================
# AUDIT INHIBITION DAN ACTIVITY — ALPHA-SYNUCLEIN
# ============================================================

df_alpha_inhibition_activity = df_alpha_binding_audit[
    df_alpha_binding_audit["standard_type"].isin(
        ["Inhibition", "Activity"]
    )
].copy()

print("TOTAL RECORDS:", len(df_alpha_inhibition_activity))

print("\nENDPOINT DISTRIBUTION:")
display(
    df_alpha_inhibition_activity["standard_type"]
    .value_counts()
)

print("\nRELATION DISTRIBUTION:")
display(
    pd.crosstab(
        df_alpha_inhibition_activity["standard_type"],
        df_alpha_inhibition_activity["standard_relation"],
        dropna=False
    )
)

print("\nUNIT DISTRIBUTION:")
display(
    pd.crosstab(
        df_alpha_inhibition_activity["standard_type"],
        df_alpha_inhibition_activity["standard_units"],
        dropna=False
    )
)

print("\nASSAY DISTRIBUTION:")
display(
    df_alpha_inhibition_activity.groupby(
        ["standard_type", "assay_group"],
        dropna=False
    ).agg(
        record_count=("molecule_chembl_id", "size"),
        unique_molecules=("molecule_chembl_id", "nunique"),
        unique_assays=("assay_chembl_id", "nunique")
    )
)

In [ ]:
# ============================================================
# AUDIT INHIBITION DAN ACTIVITY — ALPHA-SYNUCLEIN
# Menggunakan RAW DATA
# ============================================================

df_alpha_inhibition_activity = df_alpha_raw[
    df_alpha_raw["standard_type"].isin(["Inhibition", "Activity"])
].copy()

print("=" * 65)
print("INHIBITION & ACTIVITY AUDIT — ALPHA-SYNUCLEIN")
print("=" * 65)

print("TOTAL RECORDS:", len(df_alpha_inhibition_activity))

print("\nENDPOINT DISTRIBUTION:")
display(
    df_alpha_inhibition_activity["standard_type"].value_counts()
)

print("\nRELATION DISTRIBUTION:")
display(
    pd.crosstab(
        df_alpha_inhibition_activity["standard_type"],
        df_alpha_inhibition_activity["standard_relation"],
        dropna=False
    )
)

print("\nUNIT DISTRIBUTION:")
display(
    pd.crosstab(
        df_alpha_inhibition_activity["standard_type"],
        df_alpha_inhibition_activity["standard_units"],
        dropna=False
    )
)

print("\nTARGET DISTRIBUTION:")
display(
    df_alpha_inhibition_activity["target_chembl_id"].value_counts(
        dropna=False
    )
)

print("\nASSAY SUMMARY:")
display(
    df_alpha_inhibition_activity.groupby("standard_type").agg(
        record_count=("molecule_chembl_id", "size"),
        unique_molecules=("molecule_chembl_id", "nunique"),
        unique_assays=("assay_chembl_id", "nunique")
    )
)

In [ ]:
print("RAW DATA SHAPE:", df_alpha_raw.shape)

print("\nSTANDARD_TYPE VALUES:")
display(
    df_alpha_raw["standard_type"]
    .value_counts(dropna=False)
)

print("\nCOLUMN NAMES:")
print(df_alpha_raw.columns.tolist())

In [ ]:
# ============================================================
# ASSAY-LEVEL AUDIT — INHIBITION & ACTIVITY
# ============================================================

# Ambil metadata assay yang relevan
assay_metadata_cols = [
    "assay_chembl_id",
    "description",
    "assay_type",
    "relationship_type",
    "confidence_score",
    "bao_format",
    "bao_label",
    "assay_category"
]

# Gunakan hanya kolom yang tersedia
available_cols = [
    col for col in assay_metadata_cols
    if col in df_alpha_assay_metadata.columns
]

df_inhibition_activity_assay = (
    df_alpha_inhibition_activity
    .merge(
        df_alpha_assay_metadata[available_cols].drop_duplicates(
            subset="assay_chembl_id"
        ),
        on="assay_chembl_id",
        how="left",
        suffixes=("", "_metadata")
    )
)

# Ringkasan per assay dan endpoint
assay_summary = (
    df_inhibition_activity_assay
    .groupby(
        ["standard_type", "assay_chembl_id"],
        dropna=False
    )
    .agg(
        record_count=("molecule_chembl_id", "size"),
        unique_molecules=("molecule_chembl_id", "nunique"),
        units=("standard_units",
               lambda x: ", ".join(
                   sorted(x.dropna().astype(str).unique())
               )),
        relations=("standard_relation",
                   lambda x: ", ".join(
                       sorted(x.dropna().astype(str).unique())
                   )),
        assay_description=("assay_description", "first"),
        assay_type=("assay_type", "first")
    )
    .reset_index()
)

print("TOTAL ASSAYS:", assay_summary["assay_chembl_id"].nunique())

print("\nASSAY COUNTS PER ENDPOINT:")
print(
    assay_summary.groupby("standard_type")["assay_chembl_id"]
    .nunique()
)

print("\nTOP 20 ASSAYS — INHIBITION:")
display(
    assay_summary[
        assay_summary["standard_type"] == "Inhibition"
    ]
    .sort_values("record_count", ascending=False)
    .head(20)
)

print("\nTOP 20 ASSAYS — ACTIVITY:")
display(
    assay_summary[
        assay_summary["standard_type"] == "Activity"
    ]
    .sort_values("record_count", ascending=False)
    .head(20)
)

In [ ]:
# Assay yang memiliki lebih dari satu standard_type
assay_endpoint_overlap = (
    df_alpha_inhibition_activity
    .groupby("assay_chembl_id")["standard_type"]
    .nunique()
)

overlap_ids = assay_endpoint_overlap[
    assay_endpoint_overlap > 1
].index

print("ASSAY YANG MUNCUL DI LEBIH DARI SATU ENDPOINT:")
print(
    df_alpha_inhibition_activity[
        df_alpha_inhibition_activity["assay_chembl_id"].isin(overlap_ids)
    ][
        ["assay_chembl_id", "standard_type",
         "assay_description"]
    ].drop_duplicates().to_string(index=False)
)

In [ ]:
import re
import pandas as pd

# ============================================================
# PROVISIONAL ASSAY CLASSIFICATION
# ============================================================

def classify_alpha_assay(description):
    if pd.isna(description):
        return "Needs_review"

    desc = str(description).lower()

    # Prioritaskan model biologis khusus
    if "caenorhabditis elegans" in desc or "transgenic" in desc:
        return "In_vivo_model"

    if "neuro2a" in desc or "cell" in desc:
        if "dimer" in desc or "aggregation" in desc:
            return "Cell_based_aggregation"

    if "radioligand" in desc or "displacement of [3h]" in desc:
        return "Radioligand_displacement"

    if "picup" in desc or "oligomer formation" in desc:
        return "Oligomer_formation"

    if "depolymerization" in desc or "disaggregation" in desc:
        return "Fibril_disassembly"

    if (
        "binding affinity" in desc
        or "protein-compound complex" in desc
        or "surface plasmon resonance" in desc
    ):
        return "Binding_affinity"

    if "beta-sheet" in desc or "secondary structure" in desc:
        return "Secondary_structure"

    if (
        "aggregation" in desc
        and any(x in desc for x in ["tht", "thioflavin", "ths", "thioflavin s"])
    ):
        return "Aggregation_inhibition_candidate"

    return "Needs_review"


# Klasifikasikan berdasarkan assay_summary
assay_summary["provisional_assay_group"] = (
    assay_summary["assay_description"]
    .apply(classify_alpha_assay)
)

# ============================================================
# A. JUMLAH ASSAY PER KELOMPOK
# ============================================================

print("ASSAY COUNT PER ENDPOINT AND GROUP")

assay_group_summary = (
    assay_summary
    .groupby(
        ["standard_type", "provisional_assay_group"],
        dropna=False
    )
    .agg(
        assay_count=("assay_chembl_id", "nunique"),
        record_count=("record_count", "sum")
    )
    .reset_index()
)

display(
    assay_group_summary.sort_values(
        ["standard_type", "record_count"],
        ascending=[True, False]
    )
)

# ============================================================
# B. ASSAY YANG PERLU REVIEW MANUAL
# ============================================================

df_assay_review = assay_summary[
    assay_summary["provisional_assay_group"] == "Needs_review"
].copy()

print("\nASSAYS NEEDING MANUAL REVIEW:")
print("Total rows:", len(df_assay_review))

display(
    df_assay_review[
        [
            "standard_type",
            "assay_chembl_id",
            "record_count",
            "units",
            "relations",
            "assay_description"
        ]
    ].sort_values("record_count", ascending=False)
)

# ============================================================
# C. SIMPAN HASIL AUDIT SEMENTARA
# ============================================================

assay_summary.to_csv(
    CLEANED_DATA_DIR / "alpha_inhibition_activity_assay_classification.csv",
    index=False
)

assay_group_summary.to_csv(
    CLEANED_DATA_DIR / "alpha_inhibition_activity_assay_group_summary.csv",
    index=False
)

df_assay_review.to_csv(
    CLEANED_DATA_DIR / "alpha_inhibition_activity_assay_manual_review.csv",
    index=False
)

print("\nProvisional assay classification saved.")

In [ ]:
assay_id = "CHEMBL5330809"

df_check = df_alpha_inhibition_activity[
    df_alpha_inhibition_activity["assay_chembl_id"].eq(assay_id)
].copy()

print("RECORD PER ENDPOINT")
print(df_check["standard_type"].value_counts())

print("\nDETAIL RECORD")
display(
    df_check[
        [
            "molecule_chembl_id",
            "standard_type",
            "standard_relation",
            "standard_value",
            "standard_units",
            "value",
            "units",
            "activity_comment"
        ]
    ].sort_values(
        ["molecule_chembl_id", "standard_type"]
    )
)

In [ ]:
assay_id = "CHEMBL5330809"

# Filter assay
mask = df_alpha_inhibition_activity["assay_chembl_id"].eq(assay_id)

# Buat kolom jika belum tersedia
if "quality_issue" not in df_alpha_inhibition_activity.columns:
    df_alpha_inhibition_activity["quality_issue"] = ""

# Tandai record dengan nilai aktivitas kosong
df_alpha_inhibition_activity.loc[mask, "quality_issue"] = (
    "missing_activity_value"
)

# Tampilkan hasil
display(
    df_alpha_inhibition_activity.loc[
        mask,
        [
            "molecule_chembl_id",
            "standard_type",
            "standard_value",
            "standard_units",
            "quality_issue"
        ]
    ]
)

In [ ]:
print("KOLOM DATAFRAME:")
for col in df_alpha_inhibition_activity.columns:
    print(col)

In [ ]:
classification_columns = [
    col for col in df_alpha_inhibition_activity.columns
    if any(word in col.lower() for word in
           ["group", "class", "category", "review"])
]

print(classification_columns)

In [ ]:
print("DATAFRAME YANG TERSEDIA:")

for name, obj in list(globals().items()):
    if name.startswith("df_") and hasattr(obj, "columns"):
        if "assay_chembl_id" in obj.columns:
            print(name, obj.shape)

In [ ]:
candidate_names = [
    "df_assay_review",
    "df_alpha_direct_review",
    "df_alpha_potency_review",
    "df_alpha_assay_metadata",
    "df_alpha_direct_metadata",
    "df_review"
]

for name in candidate_names:
    if name in globals():
        obj = globals()[name]

        print("\n" + "=" * 70)
        print(f"DATAFRAME: {name}")
        print("Shape:", obj.shape)
        print("Columns:", obj.columns.tolist())
        display(obj.head(5))

In [ ]:
print("=" * 70)
print("MAO-B REVIEW COLUMNS")
print("=" * 70)

print(df_maob_review.columns.tolist())

print("\n" + "=" * 70)
print("MAO-B RAW COLUMNS")
print("=" * 70)

print(df_maob_raw.columns.tolist())

In [ ]:
display(
    df_maob_review[
        [
            col for col in [
                "assay_chembl_id",
                "standard_type",
                "standard_units",
                "standard_relation",
                "assay_description",
                "assay_type",
                "quality_issue"
            ]
            if col in df_maob_review.columns
        ]
    ].head(10)
)

In [ ]:
# ============================================================
# MAO-B ASSAY-LEVEL INVENTORY
# ============================================================

assay_columns = [
    "assay_chembl_id",
    "standard_type",
    "standard_units",
    "standard_relation",
    "assay_description",
    "assay_type"
]

df_maob_assay_inventory = (
    df_maob_review
    .groupby(assay_columns, dropna=False)
    .agg(
        record_count=("activity_id", "count"),
        unique_molecules=("molecule_chembl_id", "nunique")
    )
    .reset_index()
)

df_maob_assay_inventory = (
    df_maob_assay_inventory
    .sort_values(
        ["standard_type", "record_count"],
        ascending=[True, False]
    )
    .reset_index(drop=True)
)

print("=" * 75)
print("MAO-B ASSAY-LEVEL INVENTORY")
print("=" * 75)

print("Total assay-endpoint combinations:",
      len(df_maob_assay_inventory))

print("Unique assay IDs:",
      df_maob_assay_inventory["assay_chembl_id"].nunique())

display(df_maob_assay_inventory.head(20))

In [ ]:
print("=" * 75)
print("STANDARD TYPE DISTRIBUTION")
print("=" * 75)

display(
    df_maob_assay_inventory
    .groupby("standard_type", dropna=False)
    .agg(
        assay_count=("assay_chembl_id", "nunique"),
        record_count=("record_count", "sum")
    )
    .sort_values("record_count", ascending=False)
)

print("\n" + "=" * 75)
print("STANDARD UNITS DISTRIBUTION")
print("=" * 75)

display(
    df_maob_assay_inventory
    .groupby(
        ["standard_type", "standard_units"],
        dropna=False
    )
    .agg(
        assay_count=("assay_chembl_id", "nunique"),
        record_count=("record_count", "sum")
    )
    .sort_values("record_count", ascending=False)
)

In [ ]:
print("=" * 70)
print("MAO-B STANDARD RELATION AUDIT")
print("=" * 70)

display(
    df_maob_review["standard_relation"]
    .value_counts(dropna=False)
    .rename_axis("standard_relation")
    .reset_index(name="record_count")
)

In [ ]:
import re

df_maob_assay_inventory["description_lower"] = (
    df_maob_assay_inventory["assay_description"]
    .fillna("")
    .str.lower()
)

patterns = {
    "human": r"\bhuman\b",
    "recombinant": r"\brecombinant\b",
    "baboon": r"\bbaboon\b",
    "mouse": r"\bmouse\b|\bmice\b",
    "rat": r"\brat\b",
    "unknown_origin": r"unknown origin",
    "kynuramine": r"kynuramine",
    "p_tyramine": r"p[- ]?tyramine|para[- ]?tyramine",
    "benzylamine": r"benzylamine",
    "mao_a_and_b": r"mao[- ]?a.{0,30}mao[- ]?b|"
                   r"mao[- ]?b.{0,30}mao[- ]?a"
}

for category, pattern in patterns.items():
    mask = df_maob_assay_inventory[
        "description_lower"
    ].str.contains(pattern, regex=True, na=False)

    print(f"{category:20s}: {mask.sum():4d} assays")

In [ ]:
review_patterns = (
    r"unknown origin|baboon|mouse|mice|rat|"
    r"mao[- ]?a"
)

df_maob_assay_attention = (
    df_maob_assay_inventory[
        df_maob_assay_inventory[
            "description_lower"
        ].str.contains(
            review_patterns,
            regex=True,
            na=False
        )
    ]
    .sort_values("record_count", ascending=False)
)

print("Assays requiring additional review:",
      len(df_maob_assay_attention))

display(
    df_maob_assay_attention[
        [
            "assay_chembl_id",
            "standard_type",
            "standard_units",
            "record_count",
            "unique_molecules",
            "assay_description"
        ]
    ].head(30)
)

In [ ]:
import numpy as np

def classify_maob_assay(description):
    desc = str(description).lower()

    has_maoa = bool(re.search(r"\bmao[- ]?a\b", desc))
    has_maob = bool(re.search(r"\bmao[- ]?b\b", desc))

    # 1. Assay yang melibatkan kedua isoform
    if has_maoa and has_maob:
        return "MAO_A_and_B_mixed"

    # 2. Assay MAO-A saja
    if has_maoa and not has_maob:
        return "MAO_A_only"

    # 3. Asal spesies non-human
    if re.search(r"\bbaboon\b|\brat\b|\bmouse\b|\bmice\b", desc):
        return "Non_human"

    # 4. Asal protein tidak diketahui
    if "unknown origin" in desc:
        return "Unknown_origin"

    # 5. Human MAO-B
    if has_maob and "human" in desc:
        return "Human_MAO_B"

    # 6. MAO-B tanpa informasi asal yang cukup
    if has_maob:
        return "MAO_B_origin_unclear"

    return "Needs_manual_review"


df_maob_assay_inventory["assay_category"] = (
    df_maob_assay_inventory["assay_description"]
    .apply(classify_maob_assay)
)

display(
    df_maob_assay_inventory["assay_category"]
    .value_counts()
    .rename_axis("assay_category")
    .reset_index(name="assay_count")
)

In [ ]:
df_maob_assay_summary = (
    df_maob_review
    .merge(
        df_maob_assay_inventory[
            ["assay_chembl_id", "assay_category"]
        ],
        on="assay_chembl_id",
        how="left",
        validate="many_to_one"
    )
    .groupby("assay_category", dropna=False)
    .agg(
        record_count=("activity_id", "count"),
        unique_molecules=("molecule_chembl_id", "nunique"),
        assay_count=("assay_chembl_id", "nunique")
    )
    .reset_index()
    .sort_values("record_count", ascending=False)
)

display(df_maob_assay_summary)

In [ ]:
df_maob_manual_review = (
    df_maob_assay_inventory[
        df_maob_assay_inventory["assay_category"].isin([
            "MAO_A_and_B_mixed",
            "MAO_A_only",
            "Non_human",
            "Unknown_origin",
            "MAO_B_origin_unclear",
            "Needs_manual_review"
        ])
    ]
    .sort_values("record_count", ascending=False)
)

display(
    df_maob_manual_review[[
        "assay_chembl_id",
        "assay_category",
        "record_count",
        "unique_molecules",
        "assay_description"
    ]]
)

In [ ]:
# ============================================================
# MAO-B TARGET AUDIT
# ============================================================

# Pilih assay yang masih perlu ditinjau
df_flagged_assays = df_maob_assay_inventory[
    df_maob_assay_inventory["assay_category"].isin([
        "MAO_A_and_B_mixed",
        "Unknown_origin",
        "MAO_B_origin_unclear",
        "Needs_manual_review"
    ])
][["assay_chembl_id", "assay_category"]].copy()

# Hubungkan dengan data aktivitas
df_target_audit = df_maob_review.merge(
    df_flagged_assays,
    on="assay_chembl_id",
    how="inner",
    validate="many_to_one"
)

# Ringkasan target berdasarkan kategori assay
df_target_summary = (
    df_target_audit
    .groupby(
        [
            "assay_category",
            "target_chembl_id",
            "target_pref_name",
            "target_organism"
        ],
        dropna=False
    )
    .agg(
        record_count=("activity_id", "count"),
        assay_count=("assay_chembl_id", "nunique"),
        unique_molecules=("molecule_chembl_id", "nunique")
    )
    .reset_index()
    .sort_values(
        ["assay_category", "record_count"],
        ascending=[True, False]
    )
)

print("=" * 65)
print("MAO-B TARGET AUDIT SUMMARY")
print("=" * 65)

display(df_target_summary)

In [ ]:
# ============================================================
# MAO-B MIXED ASSAY TARGET DETAIL
# ============================================================

df_mixed_target_detail = (
    df_target_audit[
        df_target_audit["assay_category"] == "MAO_A_and_B_mixed"
    ]
    .groupby("assay_chembl_id", dropna=False)
    .agg(
        record_count=("activity_id", "count"),

        target_ids=(
            "target_chembl_id",
            lambda x: ", ".join(
                sorted(x.dropna().unique())
            )
        ),

        target_names=(
            "target_pref_name",
            lambda x: ", ".join(
                sorted(x.dropna().unique())
            )
        ),

        organisms=(
            "target_organism",
            lambda x: ", ".join(
                sorted(x.dropna().unique())
            )
        )
    )
    .reset_index()
)

print("=" * 65)
print("MAO-B MIXED ASSAY TARGET DETAIL")
print("=" * 65)

display(df_mixed_target_detail)

In [ ]:
# ============================================================
# TARGET AUDIT VALIDATION
# ============================================================

print("Total flagged records :", len(df_target_audit))

print(
    "Total mixed records   :",
    (df_target_audit["assay_category"] == "MAO_A_and_B_mixed").sum()
)

print(
    "Unique flagged assays :",
    df_target_audit["assay_chembl_id"].nunique()
)

print(
    "Missing target IDs    :",
    df_target_audit["target_chembl_id"].isna().sum()
)

print(
    "Non-MAO-B target IDs  :",
    (
        df_target_audit["target_chembl_id"].notna()
        & (df_target_audit["target_chembl_id"] != "CHEMBL2039")
    ).sum()
)

In [ ]:
# ============================================================
# MANUAL REVIEW: MIXED MAO-A AND MAO-B ASSAYS
# ============================================================

df_mixed_assay_review = (
    df_maob_assay_inventory[
        df_maob_assay_inventory["assay_category"]
        == "MAO_A_and_B_mixed"
    ][
        [
            "assay_chembl_id",
            "assay_type",
            "standard_type",
            "standard_units",
            "record_count",
            "unique_molecules",
            "assay_description"
        ]
    ]
    .sort_values("record_count", ascending=False)
    .reset_index(drop=True)
)

print("Total mixed assays:", len(df_mixed_assay_review))

display(df_mixed_assay_review)

In [ ]:
# ============================================================
# MANUAL REVIEW: MIXED MAO-B ASSAYS
# ============================================================

retain_assays = [
    "CHEMBL3887028",
    "CHEMBL5729814",
    "CHEMBL5730713",
    "CHEMBL5738570",
    "CHEMBL3707880",
    "CHEMBL5729821",
    "CHEMBL5729878",
    "CHEMBL5733762",
    "CHEMBL5734061",
    "CHEMBL3889174",
    "CHEMBL5730304",
    "CHEMBL3996824",
    "CHEMBL5732528",
    "CHEMBL5736769",
    "CHEMBL3707988",
    "CHEMBL5729884"
]

quarantine_assays = [
    "CHEMBL3707812",
    "CHEMBL5729971",
    "CHEMBL5736053",
    "CHEMBL5731899"
]

exclude_assays = [
    "CHEMBL3708014",
    "CHEMBL5729983"
]

# Assign review decisions
df_maob_review["manual_assay_decision"] = "Not_Mixed_Assay"

df_maob_review.loc[
    df_maob_review["assay_chembl_id"].isin(retain_assays),
    "manual_assay_decision"
] = "Provisional_Retain"

df_maob_review.loc[
    df_maob_review["assay_chembl_id"].isin(quarantine_assays),
    "manual_assay_decision"
] = "Quarantine"

df_maob_review.loc[
    df_maob_review["assay_chembl_id"].isin(exclude_assays),
    "manual_assay_decision"
] = "Exclude_Nonhuman"

# Summary
display(
    df_maob_review["manual_assay_decision"]
    .value_counts()
    .to_frame("record_count")
)

In [ ]:
print("Kolom df_maob_review:")
print(df_maob_review.columns.tolist())

print("\nKolom df_maob_assay_inventory:")
print(df_maob_assay_inventory.columns.tolist())

In [ ]:
# Gabungkan kategori assay ke dataset review
df_decision_audit_data = df_maob_review.merge(
    df_maob_assay_inventory[
        ["assay_chembl_id", "assay_category"]
    ].drop_duplicates(),
    on="assay_chembl_id",
    how="left",
    validate="many_to_one"
)

# Ringkasan keputusan
df_decision_audit = (
    df_decision_audit_data
    .groupby(
        ["assay_category", "manual_assay_decision"],
        dropna=False
    )
    .agg(
        record_count=("activity_id", "count"),
        assay_count=("assay_chembl_id", "nunique"),
        unique_molecules=("molecule_chembl_id", "nunique")
    )
    .reset_index()
    .sort_values(
        ["assay_category", "manual_assay_decision"]
    )
)

display(df_decision_audit)

In [ ]:
print("Original records :", len(df_maob_review))
print("Merged records   :", len(df_decision_audit_data))
print(
    "Missing category :",
    df_decision_audit_data["assay_category"].isna().sum()
)

In [ ]:
# ============================================================
# MAO-B ORIGIN UNCLEAR: MANUAL REVIEW
# ============================================================

df_unclear_review = (
    df_maob_assay_inventory[
        df_maob_assay_inventory["assay_category"]
        == "MAO_B_origin_unclear"
    ][
        [
            "assay_chembl_id",
            "assay_type",
            "standard_type",
            "standard_units",
            "record_count",
            "unique_molecules",
            "assay_description"
        ]
    ]
    .sort_values("record_count", ascending=False)
    .reset_index(drop=True)
)

print("Total assays:", len(df_unclear_review))
print("Total records:", df_unclear_review["record_count"].sum())

display(df_unclear_review)

In [ ]:
df_unclear_target = (
    df_decision_audit_data[
        df_decision_audit_data["assay_category"]
        == "MAO_B_origin_unclear"
    ]
    .groupby(
        ["target_chembl_id", "target_pref_name",
         "target_organism", "target_tax_id"],
        dropna=False
    )
    .agg(
        record_count=("activity_id", "count"),
        assay_count=("assay_chembl_id", "nunique"),
        unique_molecules=("molecule_chembl_id", "nunique")
    )
    .reset_index()
)

display(df_unclear_target)

In [ ]:
# MAO-B origin unclear: provisional retention
unclear_assays = [
    "CHEMBL1261744",
    "CHEMBL1059215",
    "CHEMBL5732585",
    "CHEMBL945895",
    "CHEMBL2215741",
    "CHEMBL5730350",
    "CHEMBL5730874",
    "CHEMBL5730999",
    "CHEMBL5731037",
    "CHEMBL5733463",
    "CHEMBL5734386",
    "CHEMBL5736641",
    "CHEMBL706869",
    "CHEMBL1275162",
    "CHEMBL5735933",
    "CHEMBL874045",
    "CHEMBL988334"
]

df_maob_review.loc[
    df_maob_review["assay_chembl_id"].isin(unclear_assays),
    "manual_assay_decision"
] = "Provisional_Retain"

In [ ]:
df_unclear_decision_check = (
    df_maob_review[
        df_maob_review["assay_chembl_id"].isin(unclear_assays)
    ]
    .groupby("manual_assay_decision")
    .agg(
        record_count=("activity_id", "count"),
        assay_count=("assay_chembl_id", "nunique"),
        unique_molecules=("molecule_chembl_id", "nunique")
    )
    .reset_index()
)

display(df_unclear_decision_check)

In [ ]:
df_needs_manual_review = (
    df_maob_assay_inventory[
        df_maob_assay_inventory["assay_category"] == "Needs_manual_review"
    ][[
        "assay_chembl_id",
        "assay_type",
        "standard_type",
        "standard_units",
        "record_count",
        "unique_molecules",
        "assay_description"
    ]]
    .sort_values("record_count", ascending=False)
    .reset_index(drop=True)
)

print("Total assays:", len(df_needs_manual_review))
print("Total records:", df_needs_manual_review["record_count"].sum())

display(df_needs_manual_review)

In [ ]:
# 9 assay: provisional retain
retain_assays = [
    "CHEMBL2394315",
    "CHEMBL1949363",
    "CHEMBL732527",
    "CHEMBL733090",
    "CHEMBL734178",
    "CHEMBL828050",
    "CHEMBL732535",
    "CHEMBL1663829",
    "CHEMBL949319"
]

# 1 assay: quarantine
quarantine_assays = [
    "CHEMBL5303895"
]

# 8 assay: exclude
exclude_assays = [
    "CHEMBL5739141",
    "CHEMBL814332",
    "CHEMBL814335",
    "CHEMBL814336",
    "CHEMBL814333",
    "CHEMBL814334",
    "CHEMBL814337",
    "CHEMBL814339"
]

# Terapkan keputusan
df_maob_review.loc[
    df_maob_review["assay_chembl_id"].isin(retain_assays),
    "manual_assay_decision"
] = "Provisional_Retain"

df_maob_review.loc[
    df_maob_review["assay_chembl_id"].isin(quarantine_assays),
    "manual_assay_decision"
] = "Quarantine"

df_maob_review.loc[
    df_maob_review["assay_chembl_id"].isin(exclude_assays),
    "manual_assay_decision"
] = "Exclude_Nonhuman"

In [ ]:
df_manual_check = (
    df_maob_review[
        df_maob_review["assay_chembl_id"].isin(
            retain_assays + quarantine_assays + exclude_assays
        )
    ]
    .groupby("manual_assay_decision")
    .agg(
        record_count=("activity_id", "count"),
        assay_count=("assay_chembl_id", "nunique"),
        unique_molecules=("molecule_chembl_id", "nunique")
    )
    .reset_index()
)

display(df_manual_check)

In [ ]:
df_unknown_review = (
    df_maob_assay_inventory[
        df_maob_assay_inventory["assay_category"] == "Unknown_origin"
    ][[
        "assay_chembl_id",
        "assay_type",
        "standard_type",
        "standard_units",
        "record_count",
        "unique_molecules",
        "assay_description"
    ]]
    .sort_values("record_count", ascending=False)
    .reset_index(drop=True)
)

print("Total assays:", len(df_unknown_review))
print("Total records:", df_unknown_review["record_count"].sum())

display(df_unknown_review)

In [ ]:
df_unknown_target = (
    df_decision_audit_data[
        df_decision_audit_data["assay_category"] == "Unknown_origin"
    ]
    .groupby(
        [
            "target_chembl_id",
            "target_pref_name",
            "target_organism",
            "target_tax_id"
        ],
        dropna=False
    )
    .agg(
        record_count=("activity_id", "count"),
        assay_count=("assay_chembl_id", "nunique"),
        unique_molecules=("molecule_chembl_id", "nunique")
    )
    .reset_index()
)

display(df_unknown_target)

In [ ]:
# Semua assay kategori Unknown_origin
unknown_assays = df_unknown_review["assay_chembl_id"].tolist()

# Tandai sebagai provisional retain
df_maob_review.loc[
    df_maob_review["assay_chembl_id"].isin(unknown_assays),
    "manual_assay_decision"
] = "Provisional_Retain"

# Tambahkan kolom catatan audit
df_maob_review["assay_review_note"] = ""

# Assay dengan tipe A atau T
unusual_assay_type = [
    "CHEMBL4197202",
    "CHEMBL4321226",
    "CHEMBL4378700",
    "CHEMBL4837696",
    "CHEMBL5393271",
    "CHEMBL5551943"
]

df_maob_review.loc[
    df_maob_review["assay_chembl_id"].isin(unusual_assay_type),
    "assay_review_note"
] = "Review assay type A/T"

# Assay dengan konteks binding atau displacement
unusual_assay_context = [
    "CHEMBL3050772",
    "CHEMBL6089657",
    "CHEMBL5216439"
]

df_maob_review.loc[
    df_maob_review["assay_chembl_id"].isin(unusual_assay_context),
    "assay_review_note"
] = "Review binding/displacement context"

In [ ]:
df_unknown_validation = df_maob_review[
    df_maob_review["assay_chembl_id"].isin(unknown_assays)
]

print("Records:", len(df_unknown_validation))
print(
    "Assays:",
    df_unknown_validation["assay_chembl_id"].nunique()
)
print(
    "\nDecision:"
)
display(
    df_unknown_validation["manual_assay_decision"]
    .value_counts()
)

print("\nAssay review notes:")
display(
    df_unknown_validation[
        df_unknown_validation["assay_review_note"] != ""
    ][[
        "assay_chembl_id",
        "assay_review_note"
    ]].drop_duplicates()
)

In [ ]:
# Gabungkan kategori assay ke dataset review
df_final_audit = df_maob_review.merge(
    df_maob_assay_inventory[
        ["assay_chembl_id", "assay_category"]
    ].drop_duplicates(),
    on="assay_chembl_id",
    how="left",
    validate="many_to_one"
)

# Rekap kategori dan keputusan
df_decision_summary = (
    df_final_audit
    .groupby(
        ["assay_category", "manual_assay_decision"],
        dropna=False
    )
    .agg(
        record_count=("activity_id", "count"),
        assay_count=("assay_chembl_id", "nunique"),
        unique_molecules=("molecule_chembl_id", "nunique")
    )
    .reset_index()
    .sort_values(
        ["assay_category", "manual_assay_decision"]
    )
)

display(df_decision_summary)

# Cari keputusan yang masih default
df_pending = df_final_audit[
    df_final_audit["manual_assay_decision"] == "Not_Mixed_Assay"
]

print("Records with default decision:", len(df_pending))
print(
    "Assays with default decision:",
    df_pending["assay_chembl_id"].nunique()
)

display(
    df_pending.groupby("assay_category").agg(
        record_count=("activity_id", "count"),
        assay_count=("assay_chembl_id", "nunique"),
        unique_molecules=("molecule_chembl_id", "nunique")
    ).reset_index()
)

In [ ]:
# 1. Human MAO-B
df_maob_review.loc[
    df_maob_review["assay_chembl_id"].isin(
        df_maob_assay_inventory.loc[
            df_maob_assay_inventory["assay_category"] == "Human_MAO_B",
            "assay_chembl_id"
        ]
    ),
    "manual_assay_decision"
] = "Retain"


# 2. MAO-A only
df_maob_review.loc[
    df_maob_review["assay_chembl_id"].isin(
        df_maob_assay_inventory.loc[
            df_maob_assay_inventory["assay_category"] == "MAO_A_only",
            "assay_chembl_id"
        ]
    ),
    "manual_assay_decision"
] = "Exclude_Not_Human_MAO_B"


# 3. Non-human
df_maob_review.loc[
    df_maob_review["assay_chembl_id"].isin(
        df_maob_assay_inventory.loc[
            df_maob_assay_inventory["assay_category"] == "Non_human",
            "assay_chembl_id"
        ]
    ),
    "manual_assay_decision"
] = "Exclude_Not_Human_MAO_B"


# 4. Samakan label eksklusi lama agar lebih jelas
df_maob_review["manual_assay_decision"] = (
    df_maob_review["manual_assay_decision"].replace({
        "Exclude_Nonhuman": "Exclude_Not_Human_MAO_B"
    })
)

In [ ]:
print("TOTAL RECORDS:", len(df_maob_review))

display(
    df_maob_review.groupby("manual_assay_decision").agg(
        record_count=("activity_id", "count"),
        assay_count=("assay_chembl_id", "nunique"),
        unique_molecules=("molecule_chembl_id", "nunique")
    ).reset_index()
)

print(
    "\nRemaining default decisions:",
    (df_maob_review["manual_assay_decision"] == "Not_Mixed_Assay").sum()
)

In [ ]:
# Pisahkan berdasarkan keputusan akhir
df_maob_retain = df_maob_review[
    df_maob_review["manual_assay_decision"] == "Retain"
].copy()

df_maob_provisional = df_maob_review[
    df_maob_review["manual_assay_decision"] == "Provisional_Retain"
].copy()

df_maob_quarantine = df_maob_review[
    df_maob_review["manual_assay_decision"] == "Quarantine"
].copy()

df_maob_excluded = df_maob_review[
    df_maob_review["manual_assay_decision"] == "Exclude_Not_Human_MAO_B"
].copy()

In [ ]:
datasets = {
    "Retain": df_maob_retain,
    "Provisional": df_maob_provisional,
    "Quarantine": df_maob_quarantine,
    "Excluded": df_maob_excluded
}

for name, df in datasets.items():
    print(f"{name}: {len(df)} records")

In [ ]:
from pathlib import Path

output_dir = Path("results/MAO-B/assay_review")
output_dir.mkdir(parents=True, exist_ok=True)

df_maob_retain.to_csv(
    output_dir / "maob_retain.csv", index=False
)

df_maob_provisional.to_csv(
    output_dir / "maob_provisional.csv", index=False
)

df_maob_quarantine.to_csv(
    output_dir / "maob_quarantine.csv", index=False
)

df_maob_excluded.to_csv(
    output_dir / "maob_excluded.csv", index=False
)

In [ ]:
# Audit struktur dataset utama

print("Shape:", df_maob_retain.shape)

print("\nMissing values:")
display(
    df_maob_retain.isna().sum()
    .sort_values(ascending=False)
    .head(20)
)

print("\nDuplicate activity_id:")
print(df_maob_retain["activity_id"].duplicated().sum())

print("\nStandard type:")
print(df_maob_retain["standard_type"].value_counts(dropna=False))

print("\nStandard units:")
print(df_maob_retain["standard_units"].value_counts(dropna=False))

print("\nStandard relation:")
print(df_maob_retain["standard_relation"].value_counts(dropna=False))

In [ ]:
import numpy as np
import pandas as pd

# Pastikan standard_value numerik
df_maob_retain["standard_value"] = pd.to_numeric(
    df_maob_retain["standard_value"],
    errors="coerce"
)

# Statistik deskriptif
print("STATISTIK IC50 (nM)")
display(
    df_maob_retain["standard_value"].describe(
        percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
    )
)

# Pemeriksaan nilai tidak valid
print("\nNILAI TIDAK VALID")

print(
    "Missing:",
    df_maob_retain["standard_value"].isna().sum()
)

print(
    "Nilai nol:",
    (df_maob_retain["standard_value"] == 0).sum()
)

print(
    "Nilai negatif:",
    (df_maob_retain["standard_value"] < 0).sum()
)

print(
    "Nilai positif:",
    (df_maob_retain["standard_value"] > 0).sum()
)

# Nilai ekstrem menggunakan IQR
q1 = df_maob_retain["standard_value"].quantile(0.25)
q3 = df_maob_retain["standard_value"].quantile(0.75)

iqr = q3 - q1

lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

df_maob_extreme_check = df_maob_retain[
    (df_maob_retain["standard_value"] < lower_bound) |
    (df_maob_retain["standard_value"] > upper_bound)
].copy()

print("\nIQR BOUNDS")
print("Lower bound:", lower_bound)
print("Upper bound:", upper_bound)
print("Jumlah kandidat outlier:", len(df_maob_extreme_check))

In [ ]:
# Urutkan kandidat outlier dari IC50 tertinggi
df_maob_extreme_check = df_maob_extreme_check.sort_values(
    "standard_value",
    ascending=False
)

display(
    df_maob_extreme_check[
        [
            "molecule_chembl_id",
            "assay_chembl_id",
            "standard_value",
            "standard_units",
            "standard_type",
            "pchembl_value",
            "document_chembl_id"
        ]
    ].head(20)
)

In [ ]:
outlier_by_assay = (
    df_maob_extreme_check
    .groupby("assay_chembl_id")
    .agg(
        outlier_count=("activity_id", "count"),
        unique_molecules=("molecule_chembl_id", "nunique"),
        min_ic50=("standard_value", "min"),
        max_ic50=("standard_value", "max")
    )
    .sort_values("outlier_count", ascending=False)
    .reset_index()
)

display(outlier_by_assay.head(20))

In [ ]:
total_by_assay = (
    df_maob_retain
    .groupby("assay_chembl_id")
    .size()
    .rename("total_records")
)

outlier_summary = outlier_by_assay.merge(
    total_by_assay,
    on="assay_chembl_id",
    how="left"
)

outlier_summary["outlier_percentage"] = (
    outlier_summary["outlier_count"]
    / outlier_summary["total_records"]
    * 100
)

display(
    outlier_summary.sort_values(
        "outlier_percentage",
        ascending=False
    ).head(20)
)

In [ ]:
# Ambil metadata assay
assay_metadata = df_maob_assay_inventory[
    [
        "assay_chembl_id",
        "assay_description",
        "assay_type",
        "assay_category"
    ]
].drop_duplicates("assay_chembl_id")

# Gabungkan dengan ringkasan outlier
outlier_assay_detail = outlier_summary.merge(
    assay_metadata,
    on="assay_chembl_id",
    how="left",
    validate="one_to_one"
)

# Tampilkan assay dengan jumlah outlier terbanyak
display(
    outlier_assay_detail.sort_values(
        ["outlier_count", "outlier_percentage"],
        ascending=False
    )[
        [
            "assay_chembl_id",
            "assay_category",
            "assay_type",
            "outlier_count",
            "total_records",
            "outlier_percentage",
            "min_ic50",
            "max_ic50",
            "assay_description"
        ]
    ].head(30)
)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Transformasi log10 untuk visualisasi
log_ic50 = np.log10(df_maob_retain["standard_value"])

plt.figure(figsize=(10, 5))
plt.hist(log_ic50, bins=50, edgecolor="black")

plt.xlabel("log10(IC50 in nM)")
plt.ylabel("Frequency")
plt.title("Distribution of MAO-B IC50 Values (Log10 Scale)")

plt.tight_layout()
plt.show()

In [ ]:
# Statistik distribusi IC50 per assay

assay_ic50_summary = (
    df_maob_retain
    .groupby("assay_chembl_id")
    .agg(
        record_count=("activity_id", "count"),
        unique_molecules=("molecule_chembl_id", "nunique"),
        mean_ic50=("standard_value", "mean"),
        median_ic50=("standard_value", "median"),
        min_ic50=("standard_value", "min"),
        max_ic50=("standard_value", "max"),
        std_ic50=("standard_value", "std")
    )
    .reset_index()
)

assay_ic50_summary = assay_ic50_summary.sort_values(
    "median_ic50",
    ascending=False
)

display(assay_ic50_summary.head(20))

In [ ]:
assay_description = (
    df_maob_assay_inventory[
        [
            "assay_chembl_id",
            "assay_description",
            "assay_type",
            "assay_category"
        ]
    ]
    .drop_duplicates("assay_chembl_id")
)

assay_ic50_summary = assay_ic50_summary.merge(
    assay_description,
    on="assay_chembl_id",
    how="left",
    validate="one_to_one"
)

display(
    assay_ic50_summary[
        [
            "assay_chembl_id",
            "record_count",
            "median_ic50",
            "min_ic50",
            "max_ic50",
            "assay_description"
        ]
    ].head(20)
)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.figure(figsize=(12, 6))

plt.hist(
    np.log10(assay_ic50_summary["median_ic50"].dropna()),
    bins=30,
    edgecolor="black"
)

plt.xlabel("log10 Median IC50 (nM)")
plt.ylabel("Number of Assays")
plt.title("Distribution of Median IC50 Across MAO-B Assays")

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd

# Salinan data Retain agar data utama tidak berubah
df_maob_assay_analysis = df_maob_retain.copy()

# Konversi IC50 nM menjadi pIC50
df_maob_assay_analysis["pIC50"] = (
    9 - np.log10(df_maob_assay_analysis["standard_value"])
)

# Ringkasan per assay
assay_distribution = (
    df_maob_assay_analysis
    .groupby("assay_chembl_id")
    .agg(
        record_count=("activity_id", "count"),
        unique_molecules=("molecule_chembl_id", "nunique"),
        median_ic50=("standard_value", "median"),
        min_ic50=("standard_value", "min"),
        max_ic50=("standard_value", "max"),
        median_pIC50=("pIC50", "median"),
        std_pIC50=("pIC50", "std")
    )
    .reset_index()
)

# Tambahkan deskripsi assay
assay_info = (
    df_maob_assay_inventory[
        ["assay_chembl_id", "assay_description"]
    ]
    .drop_duplicates("assay_chembl_id")
)

assay_distribution = assay_distribution.merge(
    assay_info,
    on="assay_chembl_id",
    how="left",
    validate="one_to_one"
)

# Urutkan berdasarkan jumlah data
assay_distribution = assay_distribution.sort_values(
    "record_count", ascending=False
)

display(assay_distribution.head(20))

In [ ]:
import matplotlib.pyplot as plt

# Pilih 15 assay dengan jumlah data terbanyak
selected_assays = (
    assay_distribution[
        assay_distribution["record_count"] >= 5
    ]
    .nlargest(15, "record_count")["assay_chembl_id"]
    .tolist()
)

plot_data = df_maob_assay_analysis[
    df_maob_assay_analysis["assay_chembl_id"].isin(selected_assays)
].copy()

# Siapkan data untuk boxplot
boxplot_data = [
    np.log10(
        plot_data.loc[
            plot_data["assay_chembl_id"] == assay,
            "standard_value"
        ]
    )
    for assay in selected_assays
]

plt.figure(figsize=(14, 7))

plt.boxplot(
    boxplot_data,
    tick_labels=selected_assays,
    showfliers=True
)

plt.xticks(rotation=75, ha="right")
plt.ylabel("log10(IC50 in nM)")
plt.xlabel("Assay ChEMBL ID")
plt.title("IC50 Distribution Across the 15 Most Populated MAO-B Assays")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Hitung jumlah record setiap molekul dalam setiap assay
molecule_assay_counts = (
    df_maob_retain
    .groupby(["molecule_chembl_id", "assay_chembl_id"])
    .agg(
        record_count=("activity_id", "count"),
        unique_ic50=("standard_value", "nunique"),
        min_ic50=("standard_value", "min"),
        max_ic50=("standard_value", "max")
    )
    .reset_index()
)

# Molekul yang memiliki lebih dari satu record dalam assay yang sama
repeated_within_assay = molecule_assay_counts[
    molecule_assay_counts["record_count"] > 1
].sort_values("record_count", ascending=False)

print("Jumlah kombinasi molekul-assay berulang:",
      len(repeated_within_assay))

display(repeated_within_assay.head(20))

In [ ]:
molecule_across_assays = (
    df_maob_retain
    .groupby("molecule_chembl_id")
    .agg(
        total_records=("activity_id", "count"),
        number_of_assays=("assay_chembl_id", "nunique"),
        number_of_ic50=("standard_value", "nunique")
    )
    .reset_index()
)

repeated_across_assays = (
    molecule_across_assays[
        molecule_across_assays["number_of_assays"] > 1
    ]
    .sort_values("number_of_assays", ascending=False)
)

print("Jumlah molekul yang diuji pada beberapa assay:",
      len(repeated_across_assays))

display(repeated_across_assays.head(20))

In [ ]:
# Periksa nilai maksimum yang sering muncul
top_max_values = (
    df_maob_retain
    .groupby("standard_value")
    .agg(
        record_count=("activity_id", "count"),
        assay_count=("assay_chembl_id", "nunique")
    )
    .reset_index()
    .sort_values("record_count", ascending=False)
)

display(top_max_values.head(20))

# Periksa metadata untuk nilai 61.350 nM
df_max_61350 = df_maob_retain[
    df_maob_retain["standard_value"] == 61350
]

display(
    df_max_61350[
        [
            "molecule_chembl_id",
            "assay_chembl_id",
            "standard_value",
            "standard_relation",
            "standard_units",
            "data_validity_comment",
            "activity_comment",
            "document_chembl_id"
        ]
    ]
)

In [ ]:
# Hitung rentang variasi IC50 pada setiap kombinasi molekul-assay
repeated_within_assay = repeated_within_assay.copy()

repeated_within_assay["fold_difference"] = (
    repeated_within_assay["max_ic50"] /
    repeated_within_assay["min_ic50"]
)

repeated_within_assay["absolute_difference"] = (
    repeated_within_assay["max_ic50"] -
    repeated_within_assay["min_ic50"]
)

display(
    repeated_within_assay.sort_values(
        "fold_difference", ascending=False
    ).head(20)
)

In [ ]:
df_chembl8706 = df_maob_retain[
    df_maob_retain["molecule_chembl_id"] == "CHEMBL8706"
].copy()

columns_to_check = [
    "activity_id",
    "molecule_chembl_id",
    "assay_chembl_id",
    "standard_type",
    "standard_relation",
    "standard_value",
    "standard_units",
    "pchembl_value",
    "data_validity_comment",
    "activity_comment",
    "document_chembl_id"
]

display(
    df_chembl8706[columns_to_check]
    .sort_values("assay_chembl_id")
)

In [ ]:
df_61350 = df_maob_retain[
    df_maob_retain["standard_value"] == 61350
].copy()

columns_to_check = [
    "activity_id",
    "molecule_chembl_id",
    "assay_chembl_id",
    "standard_type",
    "standard_relation",
    "standard_value",
    "standard_units",
    "pchembl_value",
    "document_chembl_id",
    "assay_description"
]

# Ambil deskripsi dari inventory jika belum tersedia
if "assay_description" not in df_61350.columns:
    df_61350 = df_61350.merge(
        df_maob_assay_inventory[
            ["assay_chembl_id", "assay_description"]
        ].drop_duplicates("assay_chembl_id"),
        on="assay_chembl_id",
        how="left"
    )

display(df_61350[columns_to_check])

In [ ]:
# Ambil 5 pasangan dengan perbedaan IC50 terbesar
top_pairs = (
    repeated_within_assay
    .sort_values("fold_difference", ascending=False)
    .head(5)[["molecule_chembl_id", "assay_chembl_id"]]
)

# Ambil seluruh record asli dari pasangan tersebut
df_top_pairs_detail = df_maob_retain.merge(
    top_pairs,
    on=["molecule_chembl_id", "assay_chembl_id"],
    how="inner"
)

# Tampilkan kolom penting untuk pemeriksaan
display(
    df_top_pairs_detail[
        [
            "activity_id",
            "molecule_chembl_id",
            "assay_chembl_id",
            "standard_type",
            "standard_relation",
            "standard_value",
            "standard_units",
            "pchembl_value",
            "document_chembl_id",
            "document_year",
            "document_journal",
            "assay_description"
        ]
    ].sort_values(
        ["molecule_chembl_id", "assay_chembl_id", "standard_value"]
    )
)

In [ ]:
print(df_maob_retain.columns.tolist())

In [ ]:
# Salin dataset utama agar tidak mengubah dataset asli
df_maob_audit = df_maob_retain.copy()

# Hitung jumlah record pada setiap kombinasi molekul dan assay
df_maob_audit["records_per_molecule_assay"] = (
    df_maob_audit
    .groupby(["molecule_chembl_id", "assay_chembl_id"])["activity_id"]
    .transform("count")
)

# Berikan flag pada kombinasi yang memiliki lebih dari satu record
df_maob_audit["repeated_within_assay"] = (
    df_maob_audit["records_per_molecule_assay"] > 1
)

# Periksa hasil
print("Total record:", len(df_maob_audit))

print(
    "Record dalam kombinasi molekul-assay berulang:",
    df_maob_audit["repeated_within_assay"].sum()
)

display(
    df_maob_audit[
        df_maob_audit["repeated_within_assay"]
    ][
        [
            "activity_id",
            "molecule_chembl_id",
            "assay_chembl_id",
            "standard_value",
            "document_chembl_id",
            "records_per_molecule_assay"
        ]
    ].head(20)
)

In [ ]:
# Ringkasan kombinasi molekul dan assay
repeated_summary = (
    df_maob_audit
    .groupby(["molecule_chembl_id", "assay_chembl_id"])
    .agg(
        record_count=("activity_id", "count"),
        unique_values=("standard_value", "nunique"),
        min_ic50=("standard_value", "min"),
        max_ic50=("standard_value", "max"),
        document_count=("document_chembl_id", "nunique")
    )
    .reset_index()
)

# Ambil kombinasi yang memiliki lebih dari satu record
repeated_summary = repeated_summary[
    repeated_summary["record_count"] > 1
].copy()

# Kategorikan pola pengulangan
repeated_summary["repeat_category"] = "Different values"

repeated_summary.loc[
    repeated_summary["unique_values"] == 1,
    "repeat_category"
] = "Identical values"

display(
    repeated_summary["repeat_category"]
    .value_counts()
)

display(
    repeated_summary.sort_values(
        "record_count",
        ascending=False
    ).head(20)
)

In [ ]:
# Audit detail record berulang
df_repeat_detail = (
    df_maob_audit[
        df_maob_audit["repeated_within_assay"]
    ]
    .sort_values(
        ["molecule_chembl_id", "assay_chembl_id", "standard_value"]
    )
    [[
        "activity_id",
        "molecule_chembl_id",
        "assay_chembl_id",
        "standard_value",
        "standard_units",
        "standard_relation",
        "document_chembl_id",
        "pchembl_value",
        "data_validity_comment",
        "activity_comment"
    ]]
)

display(df_repeat_detail)

In [ ]:
# Ringkasan sumber pengulangan
repeat_document_summary = (
    df_repeat_detail
    .groupby("assay_chembl_id")
    .agg(
        total_records=("activity_id", "count"),
        unique_molecules=("molecule_chembl_id", "nunique"),
        unique_documents=("document_chembl_id", "nunique"),
        unique_ic50_values=("standard_value", "nunique")
    )
    .reset_index()
    .sort_values("total_records", ascending=False)
)

display(repeat_document_summary.head(20))

In [ ]:
# Identifikasi pasangan molekul-assay yang memiliki lebih dari satu record

df_repeat_detail = (
    df_maob_retain
    .groupby(
        ["assay_chembl_id", "molecule_chembl_id"]
    )
    .agg(
        record_count=("activity_id", "count"),
        unique_ic50=("standard_value", "nunique"),
        min_ic50=("standard_value", "min"),
        max_ic50=("standard_value", "max"),
        document_count=("document_chembl_id", "nunique")
    )
    .reset_index()
)

# Ambil pasangan yang memiliki pengukuran berulang
df_repeat_detail = df_repeat_detail[
    df_repeat_detail["record_count"] > 1
].copy()

# Hitung rentang perbedaan nilai IC50
df_repeat_detail["fold_difference"] = (
    df_repeat_detail["max_ic50"] /
    df_repeat_detail["min_ic50"]
)

# Urutkan berdasarkan perbedaan nilai terbesar
df_repeat_detail = df_repeat_detail.sort_values(
    "fold_difference",
    ascending=False
)

display(df_repeat_detail)

In [ ]:
print("Total pasangan molekul-assay berulang:",
      len(df_repeat_detail))

print("\nKategori pengulangan nilai IC50:")
display(
    df_repeat_detail["unique_ic50"]
    .value_counts()
    .sort_index()
)

In [ ]:
df_repeat_detail["difference_category"] = pd.cut(
    df_repeat_detail["fold_difference"],
    bins=[0, 1.05, 2, 5, float("inf")],
    labels=[
        "Very small difference (<=1.05x)",
        "Moderate difference (1.05-2x)",
        "Large difference (2-5x)",
        "Very large difference (>5x)"
    ],
    include_lowest=True
)

print("DISTRIBUSI PERBEDAAN IC50")
display(
    df_repeat_detail["difference_category"]
    .value_counts()
    .sort_index()
)

print("\nPASANGAN DENGAN PERBEDAAN > 2 KALI")
display(
    df_repeat_detail[
        df_repeat_detail["fold_difference"] > 2
    ].sort_values(
        "fold_difference",
        ascending=False
    )
)

In [ ]:
# Simpan hasil audit pengulangan molekul-assay
df_repeat_detail.to_csv(
    OUTPUT_DIR / "maob_repeated_molecule_assay_audit.csv",
    index=False
)

print("Audit pengulangan berhasil disimpan.")

In [ ]:

# ============================================================
# PROVISIONAL DATA REVIEW — MAO-B
# ============================================================

import pandas as pd
import numpy as np

# 1. Pemeriksaan dasar
print("=" * 65)
print("PROVISIONAL DATA REVIEW — MAO-B")
print("=" * 65)

print("Shape:", df_maob_provisional.shape)
print("Unique molecules:",
      df_maob_provisional["molecule_chembl_id"].nunique())
print("Unique assays:",
      df_maob_provisional["assay_chembl_id"].nunique())

# 2. Distribusi endpoint dan satuan
print("\nA. STANDARD TYPE")
display(df_maob_provisional["standard_type"].value_counts(dropna=False))

print("\nB. STANDARD UNITS")
display(df_maob_provisional["standard_units"].value_counts(dropna=False))

print("\nC. STANDARD RELATION")
display(df_maob_provisional["standard_relation"].value_counts(dropna=False))

# 3. Statistik nilai IC50
print("\nD. STANDARD VALUE SUMMARY")
display(
    df_maob_provisional["standard_value"]
    .describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99])
)

# 4. Distribusi assay category
assay_category_map = (
    df_maob_assay_inventory[
        ["assay_chembl_id", "assay_category"]
    ]
    .drop_duplicates("assay_chembl_id")
)

provisional_assay_review = df_maob_provisional.merge(
    assay_category_map,
    on="assay_chembl_id",
    how="left"
)

print("\nE. ASSAY CATEGORY DISTRIBUTION")
display(
    provisional_assay_review
    .groupby("assay_category", dropna=False)
    .agg(
        records=("molecule_chembl_id", "size"),
        unique_molecules=("molecule_chembl_id", "nunique"),
        unique_assays=("assay_chembl_id", "nunique")
    )
    .sort_values("records", ascending=False)
)

# 5. Jumlah record per assay
print("\nF. TOP ASSAYS BY RECORD COUNT")
display(
    provisional_assay_review
    .groupby(
        ["assay_chembl_id", "assay_category"],
        dropna=False
    )
    .agg(
        records=("molecule_chembl_id", "size"),
        unique_molecules=("molecule_chembl_id", "nunique")
    )
    .sort_values("records", ascending=False)
    .head(20)
)

# 6. Periksa nilai tidak valid
print("\nG. VALUE VALIDITY")
print("Missing standard_value:",
      df_maob_provisional["standard_value"].isna().sum())

print("Non-numeric standard_value:",
      pd.to_numeric(
          df_maob_provisional["standard_value"],
          errors="coerce"
      ).isna().sum())

print("Zero or negative values:",
      (
          pd.to_numeric(
              df_maob_provisional["standard_value"],
              errors="coerce"
          ) <= 0
      ).sum())

# 7. Periksa keputusan manual
if "manual_assay_decision" in df_maob_provisional.columns:
    print("\nH. MANUAL ASSAY DECISION")
    display(
        df_maob_provisional["manual_assay_decision"]
        .value_counts(dropna=False)
    )

if "assay_review_note" in df_maob_provisional.columns:
    print("\nI. ASSAY REVIEW NOTES")
    display(
        df_maob_provisional["assay_review_note"]
        .value_counts(dropna=False)
        .head(20)
    )


In [ ]:

# ============================================================
# MANUAL REVIEW — PROVISIONAL ASSAY EVIDENCE
# ============================================================

# A. Ambil informasi assay yang relevan
review_columns = [
    "assay_chembl_id",
    "assay_category",
    "assay_type",
    "assay_description",
    "record_count",
    "unique_molecules"
]

available_columns = [
    col for col in review_columns
    if col in df_maob_assay_inventory.columns
]

provisional_assays = (
    provisional_assay_review[
        ["assay_chembl_id", "assay_category"]
    ]
    .drop_duplicates()
    .merge(
        df_maob_assay_inventory[
            available_columns
        ].drop_duplicates("assay_chembl_id"),
        on=["assay_chembl_id", "assay_category"],
        how="left"
    )
    .sort_values(["assay_category", "assay_chembl_id"])
)

print("TOTAL PROVISIONAL ASSAYS:",
      provisional_assays["assay_chembl_id"].nunique())

# B. Tampilkan assay yang paling membutuhkan pemeriksaan
priority_categories = [
    "MAO_A_and_B_mixed",
    "MAO_B_origin_unclear",
    "Needs_manual_review"
]

priority_assays = provisional_assays[
    provisional_assays["assay_category"].isin(priority_categories)
]

print("\nPRIORITY ASSAY REVIEW")
display(priority_assays)

# C. Tampilkan record yang memiliki catatan manual
note_mask = (
    df_maob_provisional["assay_review_note"]
    .fillna("")
    .str.strip()
    .ne("")
)

print("\nRECORDS WITH MANUAL REVIEW NOTES:",
      note_mask.sum())

note_columns = [
    col for col in [
        "molecule_chembl_id",
        "assay_chembl_id",
        "assay_review_note",
        "assay_description",
        "assay_type",
        "standard_value",
        "standard_units",
        "standard_relation"
    ]
    if col in df_maob_provisional.columns
]

display(
    df_maob_provisional.loc[note_mask, note_columns]
    .sort_values("assay_chembl_id")
)

# D. Simpan daftar assay untuk pemeriksaan
provisional_assays.to_csv(
    OUTPUT_DIR / "maob_provisional_assay_review.csv",
    index=False
)


In [ ]:

# ============================================================
# TARGET ID VERIFICATION — PROVISIONAL MAO-B
# ============================================================

# Cari kolom identitas target yang tersedia
target_columns = [
    "target_chembl_id",
    "target_pref_name",
    "target_organism",
    "target_type",
    "assay_chembl_id",
    "molecule_chembl_id"
]

available_target_columns = [
    col for col in target_columns
    if col in df_maob_provisional.columns
]

print("AVAILABLE TARGET COLUMNS:")
print(available_target_columns)

# 1. Distribusi target
if "target_chembl_id" in df_maob_provisional.columns:
    print("\nA. TARGET CHEMBL ID")
    display(
        df_maob_provisional["target_chembl_id"]
        .value_counts(dropna=False)
    )

# 2. Nama target
if "target_pref_name" in df_maob_provisional.columns:
    print("\nB. TARGET NAME")
    display(
        df_maob_provisional["target_pref_name"]
        .value_counts(dropna=False)
    )

# 3. Organisme
if "target_organism" in df_maob_provisional.columns:
    print("\nC. TARGET ORGANISM")
    display(
        df_maob_provisional["target_organism"]
        .value_counts(dropna=False)
    )

# 4. Kombinasi target dan organisme
if {
    "target_chembl_id",
    "target_pref_name",
    "target_organism"
}.issubset(df_maob_provisional.columns):

    print("\nD. TARGET-ORGANISM COMBINATION")

    display(
        df_maob_provisional.groupby(
            [
                "target_chembl_id",
                "target_pref_name",
                "target_organism"
            ],
            dropna=False
        )
        .agg(
            records=("molecule_chembl_id", "size"),
            unique_molecules=("molecule_chembl_id", "nunique"),
            unique_assays=("assay_chembl_id", "nunique")
        )
        .reset_index()
        .sort_values("records", ascending=False)
    )

# 5. Target berdasarkan assay
if "target_chembl_id" in df_maob_provisional.columns:
    print("\nE. ASSAY-TARGET MAPPING")

    display(
        df_maob_provisional.groupby(
            ["assay_chembl_id", "target_chembl_id"],
            dropna=False
        )
        .agg(
            records=("molecule_chembl_id", "size"),
            unique_molecules=("molecule_chembl_id", "nunique")
        )
        .reset_index()
        .sort_values("records", ascending=False)
    )

# 6. Jika kolom target tidak tersedia
if "target_chembl_id" not in df_maob_provisional.columns:
    print(
        "\nTarget ID tidak tersedia dalam dataframe."
        "\nKita perlu mengambil informasi target dari data mentah "
        "atau melakukan pengambilan ulang melalui ChEMBL."
    )


In [ ]:

# ============================================================
# TARGET ID VERIFICATION — PROVISIONAL MAO-B
# ============================================================

print("AVAILABLE TARGET COLUMNS:")
print([
    col for col in [
        "target_chembl_id",
        "target_pref_name",
        "target_organism",
        "target_type"
    ]
    if col in df_maob_provisional.columns
])

target_cols = [
    "target_chembl_id",
    "target_pref_name",
    "target_organism",
    "target_type"
]

available = [
    col for col in target_cols
    if col in df_maob_provisional.columns
]

if available:
    print("\nA. TARGET DISTRIBUTION")

    for col in available:
        print(f"\n{col}")
        display(
            df_maob_provisional[col]
            .value_counts(dropna=False)
        )

    print("\nB. TARGET COMBINATION")

    if len(available) > 1:
        display(
            df_maob_provisional.groupby(
                available,
                dropna=False
            )
            .size()
            .reset_index(name="record_count")
            .sort_values("record_count", ascending=False)
        )
else:
    print(
        "Target metadata tidak tersedia di dataframe."
        "Perlu mengambilnya dari data mentah atau ChEMBL."
    )


In [ ]:

# ============================================================
# PROVISIONAL ASSAY REVIEW — HUMAN MAO-B
# ============================================================

# Ringkas data pada tingkat assay
df_provisional_assay_review = (
    df_maob_provisional
    .groupby("assay_chembl_id", dropna=False)
    .agg(
        record_count=("molecule_chembl_id", "size"),
        unique_molecules=("molecule_chembl_id", "nunique"),
        assay_type=("assay_type", "first"),
        assay_description=("assay_description", "first"),
        standard_type=("standard_type", "first"),
        standard_units=("standard_units", "first"),
        standard_relation=("standard_relation", "first"),
        manual_notes=("assay_review_note", 
                      lambda x: "; ".join(
                          sorted(x.dropna().astype(str).unique())
                      ))
    )
    .reset_index()
    .sort_values("record_count", ascending=False)
)

print("TOTAL ASSAYS:", len(df_provisional_assay_review))
print("TOTAL RECORDS:", df_provisional_assay_review["record_count"].sum())

print("\nASSAY TYPE DISTRIBUTION")
display(
    df_provisional_assay_review["assay_type"]
    .value_counts(dropna=False)
)

print("\nFULL PROVISIONAL ASSAY INVENTORY")
display(df_provisional_assay_review)


In [ ]:
# ============================================================
# CLASSIFY PROVISIONAL MAO-B ASSAYS
# ============================================================

df_maob_assay_classified = df_maob_provisional.copy()

# Assay yang deskripsinya menunjukkan binding/displacement
binding_assay_ids = [
    "CHEMBL3050772",
    "CHEMBL6089657",
    "CHEMBL5216439"
]

# Assay dengan assay_type A/T yang perlu ditinjau
at_assay_ids = [
    "CHEMBL5393271",
    "CHEMBL5551943",
    "CHEMBL4197202",
    "CHEMBL4321226",
    "CHEMBL4378700",
    "CHEMBL4837696"
]

# Klasifikasi berdasarkan assay ID dan deskripsi
def classify_maob_assay(row):
    assay_id = row["assay_chembl_id"]
    description = str(row["assay_description"]).lower()

    if assay_id in binding_assay_ids:
        return "Binding_Displacement"

    if any(keyword in description for keyword in [
        "inhibition",
        "inhibitory",
        "inhibitor",
        "ic50"
    ]):
        return "Functional_Inhibition_Candidate"

    return "Manual_Review"


df_maob_assay_classified["assay_category_final"] = (
    df_maob_assay_classified.apply(
        classify_maob_assay,
        axis=1
    )
)

# Flag khusus untuk assay type A/T
df_maob_assay_classified["assay_type_review_flag"] = (
    df_maob_assay_classified["assay_type"].isin(["A", "T"])
)

# Ringkasan hasil
print("ASSAY CLASSIFICATION")
display(
    df_maob_assay_classified[
        "assay_category_final"
    ].value_counts(dropna=False)
)

print("\nASSAY TYPE REVIEW")
display(
    df_maob_assay_classified.groupby(
        ["assay_category_final", "assay_type_review_flag"]
    ).size().rename("record_count").reset_index()
)

# Periksa semua record yang perlu ditinjau
df_maob_manual_review = (
    df_maob_assay_classified[
        df_maob_assay_classified["assay_category_final"]
        == "Manual_Review"
    ]
)

display(
    df_maob_manual_review[
        [
            "molecule_chembl_id",
            "assay_chembl_id",
            "assay_type",
            "assay_description",
            "standard_type",
            "standard_value",
            "standard_units"
        ]
    ]
)

In [ ]:
# Periksa DataFrame terkait MAO-B yang tersedia
for name in [
    "df_maob_provisional",
    "df_provisional_assay_review",
    "df_manual_assay_review",
    "df_type_flag_check",
    "df_manual_check"
]:
    if name in globals():
        obj = globals()[name]

        print(f"\n{'='*60}")
        print(f"VARIABLE: {name}")
        print("Shape:", obj.shape)
        print("Columns:")
        print(obj.columns.tolist())
    else:
        print(f"\n{name}: BELUM TERDEFINISI")

In [ ]:
if "df_provisional_assay_review" in globals():

    print("SHAPE:", df_provisional_assay_review.shape)

    display(
        df_provisional_assay_review.head()
    )

    print("\nKOLOM:")
    print(df_provisional_assay_review.columns.tolist())

In [ ]:
df_provisional_assay_review.to_csv(
    output_dir / "maob_provisional_assay_inventory.csv",
    index=False
)

print("Provisional assay inventory berhasil disimpan.")

In [ ]:
# Audit hasil provisional assay inventory

print("SHAPE:", df_provisional_assay_review.shape)

print("\nDAFTAR KOLOM:")
print(df_provisional_assay_review.columns.tolist())

print("\nDISTRIBUSI KLASIFIKASI:")
if "assay_category_final" in df_provisional_assay_review.columns:
    print(
        df_provisional_assay_review[
            "assay_category_final"
        ].value_counts(dropna=False)
    )
else:
    print("Kolom assay_category_final belum tersedia.")

In [ ]:
import pandas as pd

print("=" * 70)
print("DATAFRAME YANG MEMILIKI KOLOM KLASIFIKASI")
print("=" * 70)

for name, obj in list(globals().items()):
    if isinstance(obj, pd.DataFrame):
        cols = obj.columns.tolist()

        if (
            "assay_category_final" in cols
            or "assay_type_review_flag" in cols
        ):
            print(f"\nVariable: {name}")
            print("Shape:", obj.shape)

            relevant_cols = [
                col for col in [
                    "assay_chembl_id",
                    "assay_category_final",
                    "assay_type_review_flag"
                ]
                if col in cols
            ]

            display(obj[relevant_cols].head())

In [ ]:
df_maob_manual_review.to_csv(
    output_dir / "maob_manual_assay_review.csv",
    index=False
)

print("Manual assay review berhasil disimpan.")
print("Total records:", len(df_maob_manual_review))
print(
    "Unique assays:",
    df_maob_manual_review["assay_chembl_id"].nunique()
)

In [ ]:
print("=" * 60)
print("FINAL CLASSIFICATION AUDIT")
print("=" * 60)

print("Total records:", len(df_maob_assay_classified))

print("\nASSAY CATEGORY:")
print(
    df_maob_assay_classified[
        "assay_category_final"
    ].value_counts(dropna=False)
)

print("\nASSAY TYPE REVIEW FLAG:")
print(
    df_maob_assay_classified[
        "assay_type_review_flag"
    ].value_counts(dropna=False)
)

print("\nUNIQUE ASSAYS:")
print(
    df_maob_assay_classified["assay_chembl_id"].nunique()
)

In [ ]:
# Buat ringkasan klasifikasi pada tingkat assay
df_assay_class_summary = (
    df_maob_assay_classified
    .groupby("assay_chembl_id")
    .agg(
        classification_count=(
            "assay_category_final", "nunique"
        ),
        assay_category_final=(
            "assay_category_final",
            lambda x: x.iloc[0]
            if x.nunique() == 1 else "Conflicting"
        ),
        assay_type_review_flag=(
            "assay_type_review_flag", "max"
        )
    )
    .reset_index()
)

# Integrasikan ke inventory 136 assay
df_provisional_assay_inventory_final = (
    df_provisional_assay_review
    .merge(
        df_assay_class_summary,
        on="assay_chembl_id",
        how="left",
        validate="one_to_one"
    )
)

print("INVENTORY SHAPE:")
print(df_provisional_assay_inventory_final.shape)

print("\nCLASSIFICATION DISTRIBUTION:")
print(
    df_provisional_assay_inventory_final[
        "assay_category_final"
    ].value_counts(dropna=False)
)

print("\nCONFLICTING CLASSIFICATION:")
print(
    df_provisional_assay_inventory_final[
        df_provisional_assay_inventory_final[
            "assay_category_final"
        ] == "Conflicting"
    ]
)

In [ ]:
df_provisional_assay_inventory_final.to_csv(
    output_dir / "maob_provisional_assay_inventory_classified.csv",
    index=False
)

print("Classified assay inventory berhasil disimpan.")

In [ ]:
# Ambil 4 assay yang masih Manual_Review
df_manual_assay_detail = (
    df_provisional_assay_inventory_final.loc[
        df_provisional_assay_inventory_final[
            "assay_category_final"
        ] == "Manual_Review"
    ].copy()
)

display(
    df_manual_assay_detail[
        [
            "assay_chembl_id",
            "record_count",
            "unique_molecules",
            "assay_type",
            "assay_description",
            "manual_notes"
        ]
    ]
)

In [ ]:
df_flagged_records = df_maob_assay_classified.loc[
    df_maob_assay_classified[
        "assay_type_review_flag"
    ].eq(True)
].copy()

print("FLAGGED RECORDS:", len(df_flagged_records))
print(
    "UNIQUE ASSAYS:",
    df_flagged_records["assay_chembl_id"].nunique()
)

display(
    df_flagged_records[
        [
            "molecule_chembl_id",
            "assay_chembl_id",
            "assay_type",
            "assay_description",
            "standard_value",
            "standard_units",
            "assay_category_final",
            "assay_type_review_flag"
        ]
    ].drop_duplicates()
)

In [ ]:
# Salin dataset agar hasil klasifikasi awal tetap tersedia
df_maob_assay_final = df_maob_assay_classified.copy()

# Simpan klasifikasi awal
df_maob_assay_final["assay_category_original"] = (
    df_maob_assay_final["assay_category_final"]
)

# Tandai keputusan review
df_maob_assay_final["manual_review_decision"] = "Not_Required"

df_maob_assay_final["assay_context_note"] = ""

# Empat assay yang telah ditinjau
manual_assay_decisions = {
    "CHEMBL5732528": (
        "Functional_Inhibition_Candidate",
        "Recombinant human MAO-B enzyme assay"
    ),
    "CHEMBL5735933": (
        "Functional_Inhibition_Candidate",
        "MAO-Glo MAO-B selectivity assay"
    ),
    "CHEMBL3707988": (
        "Functional_Inhibition_Candidate",
        "MAO-A and MAO-B tested under separate reaction conditions"
    ),
    "CHEMBL5729884": (
        "Functional_Inhibition_Candidate",
        "MAO-A and MAO-B tested under separate reaction conditions"
    )
}

for assay_id, (decision, note) in manual_assay_decisions.items():

    mask = (
        df_maob_assay_final["assay_chembl_id"] == assay_id
    )

    df_maob_assay_final.loc[
        mask, "assay_category_final"
    ] = decision

    df_maob_assay_final.loc[
        mask, "manual_review_decision"
    ] = "Reviewed_Retain_As_Functional_Candidate"

    df_maob_assay_final.loc[
        mask, "assay_context_note"
    ] = note

# Tandai sembilan record dengan assay type flag
flag_mask = df_maob_assay_final[
    "assay_type_review_flag"
].eq(True)

df_maob_assay_final.loc[
    flag_mask, "assay_type_review_decision"
] = "Reviewed_Functional_Inhibition_Description"

df_maob_assay_final.loc[
    ~flag_mask, "assay_type_review_decision"
] = "Not_Flagged"

# Audit hasil
print("TOTAL RECORDS:", len(df_maob_assay_final))

print("\nFINAL ASSAY CLASSIFICATION:")
print(
    df_maob_assay_final[
        "assay_category_final"
    ].value_counts(dropna=False)
)

print("\nMANUAL REVIEW DECISIONS:")
print(
    df_maob_assay_final[
        "manual_review_decision"
    ].value_counts()
)

print("\nFLAGGED RECORDS:")
print(
    df_maob_assay_final[
        "assay_type_review_decision"
    ].value_counts()
)

In [ ]:
df_maob_assay_final.to_csv(
    output_dir / "maob_provisional_activity_classified_final.csv",
    index=False
)

print("Final provisional activity classification saved.")

In [ ]:
# Buat ringkasan klasifikasi final per assay
df_assay_final_summary = (
    df_maob_assay_final
    .groupby("assay_chembl_id")
    .agg(
        record_count_final=("assay_chembl_id", "size"),
        unique_molecules_final=("molecule_chembl_id", "nunique"),
        assay_category_final=("assay_category_final", "first"),
        assay_type_reviewed=("assay_type_review_decision", "first")
    )
    .reset_index()
)

# Gabungkan dengan inventory assay sebelumnya
df_maob_assay_inventory_final = (
    df_provisional_assay_review
    .drop(
        columns=["record_count", "unique_molecules"],
        errors="ignore"
    )
    .merge(
        df_assay_final_summary,
        on="assay_chembl_id",
        how="left",
        validate="one_to_one"
    )
)

print("INVENTORY SHAPE:", df_maob_assay_inventory_final.shape)

print("\nCATEGORY DISTRIBUTION:")
print(
    df_maob_assay_inventory_final[
        "assay_category_final"
    ].value_counts(dropna=False)
)

print("\nMISSING CLASSIFICATION:")
print(
    df_maob_assay_inventory_final[
        "assay_category_final"
    ].isna().sum()
)

print("\nRECORD COUNT CHECK:")
print(
    "Activity records:",
    len(df_maob_assay_final)
)

print(
    "Inventory records:",
    df_maob_assay_inventory_final[
        "record_count_final"
    ].sum()
)

In [ ]:
df_maob_assay_inventory_final.to_csv(
    output_dir / "maob_assay_inventory_final.csv",
    index=False
)

print("Final assay inventory berhasil disimpan.")

In [ ]:
# Rekonsiliasi kategori utama MAO-B

category_counts = {
    "Retain": len(df_maob_retain),
    "Provisional": len(df_maob_assay_final),
    "Quarantine": len(df_maob_quarantine),
    "Excluded": len(df_maob_excluded)
}

print("=" * 60)
print("MAO-B DATA RECONCILIATION")
print("=" * 60)

for category, count in category_counts.items():
    print(f"{category}: {count:,}")

total_selected = sum(category_counts.values())

print("-" * 60)
print("TOTAL:", f"{total_selected:,}")
print("Previous selected count:", "6,006")
print("Difference:", 6006 - total_selected)

In [ ]:
datasets = {
    "Retain": df_maob_retain,
    "Provisional": df_maob_assay_final,
    "Quarantine": df_maob_quarantine,
    "Excluded": df_maob_excluded
}

for name, df in datasets.items():
    print(
        f"{name}: "
        f"{len(df):,} records | "
        f"{df['molecule_chembl_id'].nunique():,} unique molecules"
    )

In [ ]:
datasets = {
    "Retain": df_maob_retain,
    "Provisional": df_maob_assay_final,
    "Quarantine": df_maob_quarantine,
    "Excluded": df_maob_excluded
}

print("=" * 60)
print("ACTIVITY ID OVERLAP CHECK")
print("=" * 60)

for name, df in datasets.items():
    print(
        f"{name}: {len(df):,} records | "
        f"{df['activity_id'].nunique():,} unique activity IDs | "
        f"{df['activity_id'].duplicated().sum():,} duplicate IDs"
    )

print("\nPAIRWISE OVERLAP")

names = list(datasets.keys())

for i in range(len(names)):
    for j in range(i + 1, len(names)):
        a, b = names[i], names[j]

        ids_a = set(datasets[a]["activity_id"].dropna())
        ids_b = set(datasets[b]["activity_id"].dropna())

        print(f"{a} vs {b}: {len(ids_a & ids_b)} overlapping IDs")

In [ ]:
import pandas as pd

print("=" * 70)
print("SEARCHING MAO-B DATAFRAME CANDIDATES")
print("=" * 70)

candidates = []

for name, obj in globals().copy().items():
    if isinstance(obj, pd.DataFrame) and "activity_id" in obj.columns:
        candidates.append({
            "variable_name": name,
            "records": len(obj),
            "unique_activity_id": obj["activity_id"].nunique(),
            "duplicate_activity_id": obj["activity_id"].duplicated().sum(),
            "has_molecule_id": "molecule_chembl_id" in obj.columns,
            "has_assay_id": "assay_chembl_id" in obj.columns
        })

df_candidates = pd.DataFrame(candidates)

if not df_candidates.empty:
    df_candidates = df_candidates.sort_values(
        by="records", ascending=False
    ).reset_index(drop=True)

    display(df_candidates)
else:
    print("Tidak ditemukan DataFrame dengan kolom activity_id.")

In [ ]:
# Dataset pembanding utama
df_baseline = df_maob_quality

# Gabungkan ID dari empat kategori final
final_datasets = {
    "Retain": df_maob_retain,
    "Provisional": df_maob_assay_final,
    "Quarantine": df_maob_quarantine,
    "Excluded": df_maob_excluded
}

final_ids = set()

for df_part in final_datasets.values():
    final_ids.update(df_part["activity_id"].dropna())

baseline_ids = set(df_baseline["activity_id"].dropna())

# Cari records yang belum masuk kategori final
missing_ids = baseline_ids - final_ids

df_missing_40 = df_baseline[
    df_baseline["activity_id"].isin(missing_ids)
].copy()

# Periksa hubungan dengan suspect records
suspect_ids = set(suspect_assay["activity_id"].dropna())

print("=" * 65)
print("FINAL RECONCILIATION: 6,006 vs 5,966")
print("=" * 65)

print("Baseline records:", len(df_baseline))
print("Baseline unique IDs:", len(baseline_ids))
print("Final categorized IDs:", len(final_ids))
print("Missing IDs:", len(missing_ids))
print("Suspect assay records:", len(suspect_ids))
print("Missing IDs matching suspect:", len(missing_ids & suspect_ids))
print("Suspect IDs outside baseline:", len(suspect_ids - baseline_ids))

print("\nMISSING RECORDS PREVIEW")
display(df_missing_40.head(10))

print("\nMISSING RECORDS COLUMNS")
print(df_missing_40.columns.tolist())

In [ ]:
if len(df_missing_40) > 0:

    print("ASSAY DISTRIBUTION")
    display(
        df_missing_40["assay_chembl_id"]
        .value_counts()
        .head(20)
    )

    print("\nENDPOINT DISTRIBUTION")
    display(
        df_missing_40["standard_type"]
        .value_counts(dropna=False)
    )

    print("\nMOLECULE SUMMARY")
    print(
        "Unique molecules:",
        df_missing_40["molecule_chembl_id"].nunique()
    )

In [ ]:
# Salin data 40 records yang belum terklasifikasi
df_maob_wrong_target = df_missing_40.copy()

# Tambahkan alasan eksklusi
df_maob_wrong_target["exclusion_reason"] = (
    "Wrong_Target_Assay_LSD1"
)

df_maob_wrong_target["review_decision"] = (
    "Confirmed_Exclude"
)

# Gabungkan dengan kategori Excluded
df_maob_excluded_final = pd.concat(
    [
        df_maob_excluded,
        df_maob_wrong_target
    ],
    ignore_index=True
)

print("EXCLUDED DATASET FINAL")
print("Records:", len(df_maob_excluded_final))
print(
    "Unique activity IDs:",
    df_maob_excluded_final["activity_id"].nunique()
)

print("\nEXCLUSION REASON")
display(
    df_maob_excluded_final["exclusion_reason"]
    .value_counts(dropna=False)
)

In [ ]:
print("ORIGINAL EXCLUDED DATASET")
print("Shape:", df_maob_excluded.shape)

print("\nAVAILABLE COLUMNS:")
print(df_maob_excluded.columns.tolist())

print("\nEXISTING EXCLUSION-RELATED COLUMNS:")
exclusion_cols = [
    col for col in df_maob_excluded.columns
    if any(keyword in col.lower() for keyword in [
        "reason", "decision", "review", "exclude", "flag"
    ])
]

display(df_maob_excluded[exclusion_cols].head(10))

print("\nEXCLUSION COLUMN VALUE COUNTS:")
for col in exclusion_cols:
    print(f"\n{col}")
    print(df_maob_excluded[col].value_counts(dropna=False))

In [ ]:
# Salin dataset excluded lama
df_maob_excluded_old = df_maob_excluded.copy()

# Isi alasan eksklusi berdasarkan keputusan manual yang sudah ada
df_maob_excluded_old["exclusion_reason"] = (
    df_maob_excluded_old["manual_assay_decision"]
    .map({
        "Exclude_Not_Human_MAO_B": "Not_Human_MAO_B"
    })
)

# Samakan nama kolom keputusan
df_maob_excluded_old["review_decision"] = (
    df_maob_excluded_old["manual_assay_decision"]
)

# Gabungkan dengan 40 records salah target LSD1
df_maob_excluded_final = pd.concat(
    [df_maob_excluded_old, df_maob_wrong_target],
    ignore_index=True
)

# Pemeriksaan
print("=" * 60)
print("FINAL EXCLUDED DATASET")
print("=" * 60)

print("Total records:", len(df_maob_excluded_final))
print("Unique activity IDs:",
      df_maob_excluded_final["activity_id"].nunique())

print("\nEXCLUSION REASON:")
display(
    df_maob_excluded_final["exclusion_reason"]
    .value_counts(dropna=False)
)

print("\nREVIEW DECISION:")
display(
    df_maob_excluded_final["review_decision"]
    .value_counts(dropna=False)
)

print("\nMISSING EXCLUSION REASON:",
      df_maob_excluded_final["exclusion_reason"].isna().sum())

In [ ]:
from collections import Counter
from itertools import combinations

# =========================================================
# 1. DEFINISI EMPAT KATEGORI FINAL
# =========================================================

maob_categories = {
    "Retain": df_maob_retain,
    "Provisional": df_maob_assay_final,
    "Quarantine": df_maob_quarantine,
    "Excluded": df_maob_excluded_final
}

baseline = df_maob_quality

# =========================================================
# 2. RINGKASAN JUMLAH RECORDS
# =========================================================

print("=" * 65)
print("FINAL MAO-B RECONCILIATION")
print("=" * 65)

print(f"Baseline records : {len(baseline)}")
print(f"Baseline unique IDs: {baseline['activity_id'].nunique()}")

print("\nCATEGORY SUMMARY")

for name, df in maob_categories.items():
    print(
        f"{name:15} | Records: {len(df):5} "
        f"| Unique IDs: {df['activity_id'].nunique():5} "
        f"| Missing IDs: {df['activity_id'].isna().sum():3}"
    )

total_records = sum(len(df) for df in maob_categories.values())

print("-" * 65)
print("Total categorized records:", total_records)

# =========================================================
# 3. CEK DUPLIKASI DALAM SETIAP KATEGORI
# =========================================================

print("\nDUPLICATE ACTIVITY IDs WITHIN CATEGORIES")

for name, df in maob_categories.items():
    duplicate_count = df["activity_id"].duplicated().sum()
    print(f"{name:15}: {duplicate_count}")

# =========================================================
# 4. CEK OVERLAP ANTARKATEGORI
# =========================================================

category_ids = {
    name: set(df["activity_id"].dropna())
    for name, df in maob_categories.items()
}

print("\nPAIRWISE OVERLAP")

for (name_a, ids_a), (name_b, ids_b) in combinations(
    category_ids.items(), 2
):
    overlap = ids_a.intersection(ids_b)

    print(f"{name_a} vs {name_b}: {len(overlap)}")

# =========================================================
# 5. CEK COVERAGE TERHADAP BASELINE
# =========================================================

baseline_ids = set(baseline["activity_id"].dropna())

all_category_id_list = [
    activity_id
    for df in maob_categories.values()
    for activity_id in df["activity_id"].dropna()
]

category_id_counts = Counter(all_category_id_list)

categorized_ids = set(category_id_counts.keys())

missing_ids = baseline_ids - categorized_ids
extra_ids = categorized_ids - baseline_ids
cross_category_duplicates = {
    activity_id: count
    for activity_id, count in category_id_counts.items()
    if count > 1
}

print("\nFINAL COVERAGE CHECK")
print("Baseline unique IDs:", len(baseline_ids))
print("Categorized unique IDs:", len(categorized_ids))
print("Missing IDs:", len(missing_ids))
print("Extra IDs:", len(extra_ids))
print("Cross-category duplicate IDs:",
      len(cross_category_duplicates))

# =========================================================
# 6. FINAL STATUS
# =========================================================

final_pass = (
    total_records == len(baseline)
    and all(
        df["activity_id"].notna().all()
        and df["activity_id"].is_unique
        for df in maob_categories.values()
    )
    and not missing_ids
    and not extra_ids
    and not cross_category_duplicates
)

print("\n" + "=" * 65)

if final_pass:
    print("RECONCILIATION: PASSED")
else:
    print("RECONCILIATION: NOT PASSED")

print("=" * 65)

In [ ]:
# =========================================================
# FINAL ASSAY SUBCATEGORY QC
# =========================================================

# Pisahkan berdasarkan klasifikasi assay final
df_maob_functional = df_maob_assay_final[
    df_maob_assay_final["assay_category_final"]
    == "Functional_Inhibition_Candidate"
].copy()

df_maob_binding = df_maob_assay_final[
    df_maob_assay_final["assay_category_final"]
    == "Binding_Displacement"
].copy()

print("=" * 65)
print("PROVISIONAL ASSAY SUBCATEGORY QC")
print("=" * 65)

print("Functional inhibition candidates:", len(df_maob_functional))
print("Binding displacement:", len(df_maob_binding))

# Periksa endpoint, satuan, dan relation
qc_columns = [
    "standard_type",
    "standard_units",
    "standard_relation"
]

for name, df in {
    "FUNCTIONAL INHIBITION": df_maob_functional,
    "BINDING DISPLACEMENT": df_maob_binding
}.items():

    print(f"\n{'=' * 65}")
    print(name)
    print("=" * 65)

    for col in qc_columns:
        print(f"\n{col}:")
        display(
            df[col].value_counts(dropna=False).to_frame("count")
        )

    print("\nUnique endpoint-unit combinations:")
    display(
        df.groupby(
            ["standard_type", "standard_units", "standard_relation"],
            dropna=False
        ).size().reset_index(name="records")
    )

In [ ]:
import numpy as np
import pandas as pd

def validate_ic50(df, dataset_name):

    values = pd.to_numeric(
        df["standard_value"],
        errors="coerce"
    )

    print("=" * 60)
    print(dataset_name)
    print("=" * 60)

    print("Total records:", len(df))
    print("Missing/non-numeric:", values.isna().sum())
    print("Zero values:", (values == 0).sum())
    print("Negative values:", (values < 0).sum())
    print("Infinite values:", np.isinf(values).sum())

    valid = values.notna() & np.isfinite(values) & (values > 0)

    print("Valid positive IC50:", valid.sum())
    print("Invalid IC50:", (~valid).sum())

    if valid.any():
        print("\nIC50 descriptive statistics (nM):")
        display(values[valid].describe().to_frame("IC50_nM"))

    return valid


valid_functional = validate_ic50(
    df_maob_functional,
    "FUNCTIONAL INHIBITION"
)

valid_binding = validate_ic50(
    df_maob_binding,
    "BINDING DISPLACEMENT"
)

In [ ]:
import numpy as np
import pandas as pd

# Validasi nilai IC50 pada dataset utama
values = pd.to_numeric(
    df_maob_retain["standard_value"],
    errors="coerce"
)

valid_retain = (
    values.notna()
    & np.isfinite(values)
    & (values > 0)
)

print("=" * 60)
print("CORE DATASET - MAO-B RETAIN")
print("=" * 60)

print("Total records:", len(df_maob_retain))
print("Missing/non-numeric:", values.isna().sum())
print("Zero values:", (values == 0).sum())
print("Negative values:", (values < 0).sum())
print("Infinite values:", np.isinf(values).sum())
print("Valid positive IC50:", valid_retain.sum())
print("Invalid IC50:", (~valid_retain).sum())

print("\nIC50 descriptive statistics (nM):")
display(
    values[valid_retain]
    .describe()
    .to_frame("IC50_nM")
)

In [ ]:
# Ambil 20 nilai IC50 tertinggi pada functional candidates
cols = [
    "molecule_chembl_id",
    "assay_chembl_id",
    "standard_type",
    "standard_value",
    "standard_units",
    "assay_description",
    "document_chembl_id"
]

available_cols = [
    col for col in cols
    if col in df_maob_functional.columns
]

display(
    df_maob_functional
    .sort_values("standard_value", ascending=False)
    [available_cols]
    .head(20)
)

In [ ]:
# Audit nilai IC50 ekstrem pada core dataset

df_extreme = df_maob_retain[
    df_maob_retain["standard_value"] >= 1_000_000
].copy()

print("TOTAL EXTREME RECORDS:", len(df_extreme))
print(
    "UNIQUE MOLECULES:",
    df_extreme["molecule_chembl_id"].nunique()
)
print(
    "UNIQUE ASSAYS:",
    df_extreme["assay_chembl_id"].nunique()
)

# Ringkasan per assay
assay_extreme_summary = (
    df_extreme
    .groupby("assay_chembl_id")
    .agg(
        records=("activity_id", "count"),
        unique_molecules=("molecule_chembl_id", "nunique"),
        min_ic50=("standard_value", "min"),
        max_ic50=("standard_value", "max"),
        mean_ic50=("standard_value", "mean"),
        description=("assay_description", "first"),
        document=("document_chembl_id", "first")
    )
    .sort_values("records", ascending=False)
)

display(assay_extreme_summary)

In [ ]:
# Konsistensi nilai IC50 pada core dataset

print("DATAFRAME SHAPE:", df_maob_retain.shape)

print(
    "MAX IC50:",
    df_maob_retain["standard_value"].max()
)

print(
    "EXTREME RECORDS >= 1,000,000 nM:",
    (df_maob_retain["standard_value"] >= 1_000_000).sum()
)

# Periksa tiga molekul dari tabel sebelumnya
check_molecules = [
    "CHEMBL1766622",
    "CHEMBL2418201",
    "CHEMBL1766620"
]

display(
    df_maob_retain[
        df_maob_retain["molecule_chembl_id"].isin(check_molecules)
    ][[
        "activity_id",
        "molecule_chembl_id",
        "assay_chembl_id",
        "standard_value",
        "standard_units",
        "document_chembl_id"
    ]].sort_values("standard_value", ascending=False)
)

In [ ]:
print("QUARANTINE SHAPE:", df_maob_quarantine.shape)

print("\nMISSING VALUES:")
display(
    df_maob_quarantine.isna().sum()
    .sort_values(ascending=False)
    .head(15)
)

print("\nREASON DISTRIBUTION:")
for col in [
    "quarantine_reason",
    "manual_review_decision",
    "review_decision",
    "assay_category_final"
]:
    if col in df_maob_quarantine.columns:
        print(f"\n{col}:")
        display(df_maob_quarantine[col].value_counts(dropna=False))

print("\nSAMPLE QUARANTINE RECORDS:")
display(
    df_maob_quarantine.head(10)
)

In [ ]:
# 1. Periksa kolom yang tersedia
print("KOLOM TERKAIT REVIEW:")
print([
    col for col in df_maob_quarantine.columns
    if any(x in col.lower() for x in [
        "reason", "review", "decision", "category",
        "flag", "target"
    ])
])

# 2. Jumlah records dan assay unik
print("\nTOTAL RECORDS:", len(df_maob_quarantine))
print(
    "UNIQUE ACTIVITY ID:",
    df_maob_quarantine["activity_id"].nunique()
)
print(
    "UNIQUE ASSAY:",
    df_maob_quarantine["assay_chembl_id"].nunique()
)

# 3. Distribusi keputusan yang sudah tersedia
for col in ["manual_assay_decision", "assay_review_note"]:
    if col in df_maob_quarantine.columns:
        print(f"\n{col.upper()}")
        display(
            df_maob_quarantine[col]
            .value_counts(dropna=False)
        )

# 4. Ringkasan berdasarkan assay
assay_summary = (
    df_maob_quarantine
    .groupby("assay_chembl_id", dropna=False)
    .agg(
        total_records=("activity_id", "count"),
        unique_molecules=("molecule_chembl_id", "nunique"),
        endpoint=("standard_type", "unique"),
        units=("standard_units", "unique"),
        relation=("standard_relation", "unique"),
        target=("target_pref_name", "unique"),
        organism=("target_organism", "unique"),
        assay_description=("assay_description", "first"),
        document=("document_chembl_id", "unique")
    )
    .reset_index()
    .sort_values("total_records", ascending=False)
)

display(assay_summary)

In [ ]:
# Kolom penting untuk audit per record
cols = [
    "activity_id",
    "molecule_chembl_id",
    "assay_chembl_id",
    "assay_type",
    "type",
    "standard_type",
    "standard_relation",
    "standard_value",
    "standard_units",
    "pchembl_value",
    "bao_endpoint",
    "target_chembl_id",
    "target_pref_name",
    "target_organism",
    "document_chembl_id",
    "activity_comment",
    "standard_flag",
    "manual_assay_decision",
    "assay_review_note"
]

cols = [c for c in cols if c in df_maob_quarantine.columns]

# A. Distribusi nilai per assay
display(
    df_maob_quarantine
    .groupby("assay_chembl_id")
    .agg(
        records=("activity_id", "count"),
        unique_molecules=("molecule_chembl_id", "nunique"),
        min_IC50=("standard_value", "min"),
        median_IC50=("standard_value", "median"),
        max_IC50=("standard_value", "max"),
        missing_activity_comment=("activity_comment", lambda x: x.isna().sum())
    )
)

# B. Periksa apakah dua assay 52-record menggunakan molekul yang sama
assay_a = set(
    df_maob_quarantine.loc[
        df_maob_quarantine["assay_chembl_id"] == "CHEMBL3707812",
        "molecule_chembl_id"
    ]
)

assay_b = set(
    df_maob_quarantine.loc[
        df_maob_quarantine["assay_chembl_id"] == "CHEMBL5729971",
        "molecule_chembl_id"
    ]
)

print("Molekul pada assay pertama:", len(assay_a))
print("Molekul pada assay kedua:", len(assay_b))
print("Molekul yang sama:", len(assay_a & assay_b))
print("Hanya assay pertama:", len(assay_a - assay_b))
print("Hanya assay kedua:", len(assay_b - assay_a))

# C. Tampilkan seluruh record dari tiga assay yang perlu ditinjau
display(
    df_maob_quarantine[
        df_maob_quarantine["assay_chembl_id"].isin([
            "CHEMBL5736053",
            "CHEMBL5303895",
            "CHEMBL5731899"
        ])
    ][cols]
    .sort_values(["assay_chembl_id", "standard_value"])
)

In [ ]:
# Ambil dua assay yang dicurigai memiliki data sama
assay_1 = df_maob_quarantine[
    df_maob_quarantine["assay_chembl_id"] == "CHEMBL3707812"
].copy()

assay_2 = df_maob_quarantine[
    df_maob_quarantine["assay_chembl_id"] == "CHEMBL5729971"
].copy()

# Bandingkan nilai aktivitas per molekul
comparison = assay_1[
    ["molecule_chembl_id", "standard_value"]
].merge(
    assay_2[
        ["molecule_chembl_id", "standard_value"]
    ],
    on="molecule_chembl_id",
    how="outer",
    suffixes=("_assay_1", "_assay_2"),
    indicator=True
)

# Periksa kesamaan nilai
comparison["same_value"] = (
    comparison["standard_value_assay_1"]
    == comparison["standard_value_assay_2"]
)

print("TOTAL MOLECULES:", len(comparison))
print("\nMERGE STATUS:")
display(comparison["_merge"].value_counts())

print("\nVALUE COMPARISON:")
display(comparison["same_value"].value_counts(dropna=False))

print("\nMOLECULES WITH DIFFERENT VALUES:")
display(
    comparison[
        comparison["same_value"] == False
    ]
)

In [ ]:
print("ASSAY 1 ACTIVITY IDs:")
print(assay_1["activity_id"].nunique())

print("\nASSAY 2 ACTIVITY IDs:")
print(assay_2["activity_id"].nunique())

print("\nIDENTICAL MOLECULE-VALUE PAIRS:")
print(
    comparison[
        comparison["same_value"] == True
    ].shape[0]
)

In [ ]:
# Periksa jumlah activity ID
print("ASSAY 1 ACTIVITY IDs:", assay_1["activity_id"].nunique())
print("ASSAY 2 ACTIVITY IDs:", assay_2["activity_id"].nunique())

# Bandingkan atribut tambahan
comparison_detail = assay_1[
    [
        "molecule_chembl_id",
        "activity_id",
        "standard_value",
        "standard_type",
        "standard_relation",
        "standard_units",
        "pchembl_value",
        "document_chembl_id"
    ]
].merge(
    assay_2[
        [
            "molecule_chembl_id",
            "activity_id",
            "standard_value",
            "standard_type",
            "standard_relation",
            "standard_units",
            "pchembl_value",
            "document_chembl_id"
        ]
    ],
    on="molecule_chembl_id",
    suffixes=("_assay_1", "_assay_2")
)

columns_to_compare = [
    "standard_value",
    "standard_type",
    "standard_relation",
    "standard_units",
    "pchembl_value",
    "document_chembl_id"
]

for col in columns_to_compare:
    comparison_detail[f"{col}_identical"] = (
        comparison_detail[f"{col}_assay_1"]
        == comparison_detail[f"{col}_assay_2"]
    )

print("\nATTRIBUTE COMPARISON:")
for col in columns_to_compare:
    flag = f"{col}_identical"
    print(f"{col}: {comparison_detail[flag].sum()}/52 identical")

display(comparison_detail.head())

In [ ]:
duplicate_assays = [
    "CHEMBL3707812",
    "CHEMBL5729971"
]

mask_duplicate = df_maob_quarantine[
    "assay_chembl_id"
].isin(duplicate_assays)

df_maob_quarantine.loc[
    mask_duplicate,
    "assay_review_note"
] = (
    "Suspected duplicate assay records: "
    "52 identical molecule-value pairs across two assay IDs; "
    "same endpoint, relation, units, pChEMBL values, "
    "and document. Retained in quarantine pending source verification."
)

print("ANNOTATED RECORDS:", mask_duplicate.sum())

display(
    df_maob_quarantine.loc[
        mask_duplicate,
        [
            "assay_chembl_id",
            "molecule_chembl_id",
            "activity_id",
            "assay_review_note"
        ]
    ].head()
)

In [ ]:
assay_id = "CHEMBL5736053"

df_audit = df_maob_quarantine[
    df_maob_quarantine["assay_chembl_id"] == assay_id
].copy()

print("TOTAL RECORDS:", len(df_audit))
print("UNIQUE MOLECULES:", df_audit["molecule_chembl_id"].nunique())
print("UNIQUE ACTIVITY IDs:", df_audit["activity_id"].nunique())

# Informasi assay
display(
    df_audit[
        [
            "assay_chembl_id",
            "assay_type",
            "assay_description",
            "target_chembl_id",
            "target_pref_name",
            "target_organism",
            "document_chembl_id"
        ]
    ].drop_duplicates()
)

# Seluruh aktivitas
display(
    df_audit[
        [
            "activity_id",
            "molecule_chembl_id",
            "type",
            "standard_type",
            "standard_relation",
            "standard_value",
            "standard_units",
            "pchembl_value",
            "activity_comment",
            "manual_assay_decision",
            "assay_review_note"
        ]
    ].sort_values("standard_value")
)

In [ ]:
assay_id = "CHEMBL5736053"

mask = (
    df_maob_quarantine["assay_chembl_id"] == assay_id
)

df_maob_quarantine.loc[
    mask, "assay_review_note"
] = (
    "Target specificity unclear: assay description covers "
    "both human MAO-A and MAO-B using MAO-Glo. "
    "The available records do not establish whether these "
    "12 IC50 values specifically correspond to MAO-B. "
    "Retained in quarantine pending source-level verification."
)

print(
    "ANNOTATED RECORDS:",
    df_maob_quarantine.loc[mask].shape[0]
)

display(
    df_maob_quarantine.loc[
        mask,
        [
            "assay_chembl_id",
            "activity_id",
            "molecule_chembl_id",
            "standard_value",
            "assay_review_note"
        ]
    ]
)

In [ ]:
assay_id = "CHEMBL5303895"

df_audit = df_maob_quarantine[
    df_maob_quarantine["assay_chembl_id"] == assay_id
].copy()

print("TOTAL RECORDS:", len(df_audit))

display(
    df_audit[
        [
            "activity_id",
            "molecule_chembl_id",
            "assay_chembl_id",
            "assay_type",
            "assay_description",
            "target_chembl_id",
            "target_pref_name",
            "target_organism",
            "document_chembl_id",
            "standard_type",
            "standard_relation",
            "standard_value",
            "standard_units",
            "pchembl_value",
            "activity_comment",
            "manual_assay_decision",
            "assay_review_note"
        ]
    ]
)

In [ ]:
assay_id = "CHEMBL5303895"

mask = (
    df_maob_quarantine["assay_chembl_id"] == assay_id
)

df_maob_quarantine.loc[
    mask, "assay_review_note"
] = (
    "Insufficient assay information: description only states "
    "'Cross screening panel'. The available metadata does not "
    "establish the experimental method or confirm MAO-B-specific "
    "functional inhibition. Retained in quarantine pending "
    "source-level verification."
)

print("ANNOTATED RECORDS:", mask.sum())

In [ ]:
assay_id = "CHEMBL5731899"

df_audit = df_maob_quarantine[
    df_maob_quarantine["assay_chembl_id"] == assay_id
].copy()

print("TOTAL RECORDS:", len(df_audit))

display(
    df_audit[
        [
            "activity_id",
            "molecule_chembl_id",
            "assay_chembl_id",
            "assay_type",
            "assay_description",
            "target_chembl_id",
            "target_pref_name",
            "target_organism",
            "document_chembl_id",
            "standard_type",
            "standard_relation",
            "standard_value",
            "standard_units",
            "pchembl_value",
            "activity_comment",
            "manual_assay_decision",
            "assay_review_note"
        ]
    ]
)

In [ ]:
assay_id = "CHEMBL5731899"

mask = (
    df_maob_quarantine["assay_chembl_id"] == assay_id
)

df_maob_quarantine.loc[
    mask, "manual_assay_decision"
] = "Reviewed_Retain_As_Functional_Candidate"

df_maob_quarantine.loc[
    mask, "assay_review_note"
] = (
    "Reviewed: assay description documents functional "
    "inhibition testing using recombinant human MAO-A "
    "and MAO-B, with separate enzyme addition to wells "
    "and concentration-dependent fluorescence measurement. "
    "ChEMBL target annotation maps this record to human MAO-B "
    "(CHEMBL2039). Retain as a functional inhibition candidate; "
    "individual result-to-enzyme mapping remains subject "
    "to source-level verification."
)

print("UPDATED RECORDS:", mask.sum())

display(
    df_maob_quarantine.loc[
        mask,
        [
            "activity_id",
            "assay_chembl_id",
            "manual_assay_decision",
            "assay_review_note"
        ]
    ]
)

In [ ]:
# ============================================================
# MOVE REVIEWED RECORD FROM QUARANTINE TO PROVISIONAL
# ============================================================

activity_id_to_move = 27086882

# Ambil record yang sudah disetujui
df_move = df_maob_quarantine[
    df_maob_quarantine["activity_id"] == activity_id_to_move
].copy()

assert len(df_move) == 1, (
    f"Expected 1 record, found {len(df_move)}"
)

# Pastikan keputusan manual benar
assert (
    df_move["manual_assay_decision"].iloc[0]
    == "Reviewed_Retain_As_Functional_Candidate"
)

# Tambahkan klasifikasi provisional
df_move["assay_category_final"] = (
    "Functional_Inhibition_Candidate"
)

df_move["manual_review_decision"] = (
    "Reviewed_Retain_As_Functional_Candidate"
)

df_move["assay_type_review_decision"] = (
    "Reviewed_Functional_Inhibition_Description"
)

# Samakan struktur kolom sebelum penggabungan
for col in df_maob_assay_final.columns:
    if col not in df_move.columns:
        df_move[col] = pd.NA

for col in df_move.columns:
    if col not in df_maob_assay_final.columns:
        df_maob_assay_final[col] = pd.NA

df_move = df_move[df_maob_assay_final.columns]

# Tambahkan ke provisional
df_maob_assay_final = pd.concat(
    [df_maob_assay_final, df_move],
    ignore_index=True
)

# Keluarkan dari quarantine
df_maob_quarantine = df_maob_quarantine[
    df_maob_quarantine["activity_id"]
    != activity_id_to_move
].copy()

print("MOVED RECORDS:", len(df_move))
print("PROVISIONAL RECORDS:", len(df_maob_assay_final))
print("QUARANTINE RECORDS:", len(df_maob_quarantine))

In [ ]:
print("PROVISIONAL CATEGORY DISTRIBUTION")
display(
    df_maob_assay_final[
        "assay_category_final"
    ].value_counts(dropna=False)
)

print("\nQUARANTINE ASSAY DISTRIBUTION")
display(
    df_maob_quarantine[
        "assay_chembl_id"
    ].value_counts()
)

print("\nCHECK ACTIVITY ID")
print(
    "Moved ID in provisional:",
    (
        df_maob_assay_final["activity_id"]
        == activity_id_to_move
    ).sum()
)

print(
    "Moved ID in quarantine:",
    (
        df_maob_quarantine["activity_id"]
        == activity_id_to_move
    ).sum()
)

In [ ]:
# ============================================================
# FINAL RECONCILIATION - MAO-B
# ============================================================

# 1. Gabungkan seluruh kategori
category_frames = {
    "Retain": df_maob_retain,
    "Provisional": df_maob_assay_final,
    "Quarantine": df_maob_quarantine,
    "Excluded": df_maob_excluded_final
}

# 2. Baseline activity IDs
baseline_ids = set(
    df_maob_quality["activity_id"].dropna()
)

# 3. Audit setiap kategori
all_category_ids = []
category_summary = []

for category, df in category_frames.items():

    ids = df["activity_id"].dropna()

    category_summary.append({
        "category": category,
        "records": len(df),
        "unique_activity_ids": ids.nunique(),
        "duplicate_activity_ids": ids.duplicated().sum(),
        "missing_activity_ids": df["activity_id"].isna().sum()
    })

    all_category_ids.extend(ids.tolist())

# 4. Gabungkan seluruh ID
from collections import Counter

category_id_counts = Counter(all_category_ids)

categorized_ids = set(category_id_counts.keys())

# 5. Periksa missing dan extra IDs
missing_ids = baseline_ids - categorized_ids
extra_ids = categorized_ids - baseline_ids

# 6. Periksa overlap antarkategori
overlap_results = {}

category_names = list(category_frames.keys())

for i in range(len(category_names)):
    for j in range(i + 1, len(category_names)):

        name_a = category_names[i]
        name_b = category_names[j]

        ids_a = set(
            category_frames[name_a]["activity_id"].dropna()
        )

        ids_b = set(
            category_frames[name_b]["activity_id"].dropna()
        )

        overlap_results[
            f"{name_a} vs {name_b}"
        ] = len(ids_a.intersection(ids_b))

# 7. Tampilkan hasil
print("=" * 65)
print("FINAL CATEGORY SUMMARY")
print("=" * 65)

display(pd.DataFrame(category_summary))

print("\nBASELINE RECORDS:", len(df_maob_quality))
print("BASELINE UNIQUE IDs:", len(baseline_ids))

print("\nCATEGORIZED RECORDS:", len(all_category_ids))
print("CATEGORIZED UNIQUE IDs:", len(categorized_ids))

print("\nMISSING IDs:", len(missing_ids))
print("EXTRA IDs:", len(extra_ids))

print("\nDUPLICATE IDs ACROSS CATEGORIES:",
      sum(count > 1 for count in category_id_counts.values()))

print("\nPAIRWISE OVERLAP:")
for pair, count in overlap_results.items():
    print(f"{pair}: {count}")

# 8. Final validation
no_internal_duplicates = all(
    item["duplicate_activity_ids"] == 0
    for item in category_summary
)

no_missing_ids = all(
    item["missing_activity_ids"] == 0
    for item in category_summary
)

no_overlap = all(
    value == 0 for value in overlap_results.values()
)

reconciliation_passed = (
    len(baseline_ids) == len(categorized_ids)
    and len(missing_ids) == 0
    and len(extra_ids) == 0
    and no_internal_duplicates
    and no_missing_ids
    and no_overlap
)

print("\n" + "=" * 65)

if reconciliation_passed:
    print("FINAL RECONCILIATION: PASSED")
else:
    print("FINAL RECONCILIATION: FAILED")

print("=" * 65)

In [ ]:
# ============================================================
# FINAL QC — PROVISIONAL MAO-B
# ============================================================

df_maob_functional = df_maob_assay_final[
    df_maob_assay_final["assay_category_final"]
    == "Functional_Inhibition_Candidate"
].copy()

df_maob_binding = df_maob_assay_final[
    df_maob_assay_final["assay_category_final"]
    == "Binding_Displacement"
].copy()

print("=" * 65)
print("PROVISIONAL DATASET QC")
print("=" * 65)

print("Total provisional:", len(df_maob_assay_final))
print("Functional candidates:", len(df_maob_functional))
print("Binding displacement:", len(df_maob_binding))

print("\nSTANDARD TYPE:")
print(df_maob_assay_final["standard_type"].value_counts(dropna=False))

print("\nSTANDARD UNITS:")
print(df_maob_assay_final["standard_units"].value_counts(dropna=False))

print("\nSTANDARD RELATION:")
print(df_maob_assay_final["standard_relation"].value_counts(dropna=False))

print("\nNUMERIC VALIDATION:")

values = pd.to_numeric(
    df_maob_assay_final["standard_value"],
    errors="coerce"
)

print("Missing/non-numeric:", values.isna().sum())
print("Zero:", (values == 0).sum())
print("Negative:", (values < 0).sum())
print("Infinite:", np.isinf(values).sum())
print("Valid positive:", (values.gt(0) & np.isfinite(values)).sum())

print("\nCATEGORY DISTRIBUTION:")
print(df_maob_assay_final["assay_category_final"].value_counts())

print("\nFUNCTIONAL UNIQUE ACTIVITY IDs:",
      df_maob_functional["activity_id"].nunique())

print("BINDING UNIQUE ACTIVITY IDs:",
      df_maob_binding["activity_id"].nunique())

In [ ]:
from pathlib import Path
import pandas as pd

# ============================================================
# 1. FINAL ANNOTATION CHECK
# ============================================================

print("=" * 65)
print("FINAL ANNOTATION CHECK")
print("=" * 65)

# Quarantine
quarantine_notes = (
    df_maob_quarantine["assay_review_note"]
    .fillna("")
    .astype(str)
    .str.strip()
)

print("Quarantine records:", len(df_maob_quarantine))
print("Quarantine without review note:", (quarantine_notes == "").sum())

# Excluded
excluded_reasons = (
    df_maob_excluded_final["exclusion_reason"]
    .fillna("")
    .astype(str)
    .str.strip()
)

print("\nExcluded records:", len(df_maob_excluded_final))
print("Excluded without reason:", (excluded_reasons == "").sum())

# Functional vs Binding
functional_ids = set(df_maob_functional["activity_id"])
binding_ids = set(df_maob_binding["activity_id"])

print("\nFunctional-Binding overlap:", len(functional_ids & binding_ids))

# Assertions
assert (quarantine_notes != "").all(), "Ada Quarantine tanpa review note."
assert (excluded_reasons != "").all(), "Ada Excluded tanpa exclusion reason."
assert len(functional_ids & binding_ids) == 0, "Ada overlap Functional-Binding."

print("\nANNOTATION CHECK: PASSED")


# ============================================================
# 2. SAVE FINAL PREPROCESSING OUTPUTS
# ============================================================

output_dir = Path("OUTPUTS/01_Cleaned_Data")
output_dir.mkdir(parents=True, exist_ok=True)

datasets = {
    "maob_retain_final.csv": df_maob_retain,
    "maob_provisional_final.csv": df_maob_assay_final,
    "maob_functional_inhibition_candidates.csv": df_maob_functional,
    "maob_binding_displacement.csv": df_maob_binding,
    "maob_quarantine_final.csv": df_maob_quarantine,
    "maob_excluded_final.csv": df_maob_excluded_final,
}

for filename, dataframe in datasets.items():
    dataframe.to_csv(output_dir / filename, index=False)

print("\n" + "=" * 65)
print("SAVED DATASETS")
print("=" * 65)

for filename in datasets:
    print(filename)


# ============================================================
# 3. READ-BACK VERIFICATION
# ============================================================

print("\n" + "=" * 65)
print("READ-BACK VERIFICATION")
print("=" * 65)

for filename, original_df in datasets.items():
    saved_df = pd.read_csv(output_dir / filename)

    passed = (
        len(saved_df) == len(original_df)
        and saved_df["activity_id"].nunique()
            == original_df["activity_id"].nunique()
    )

    print(
        f"{filename}: "
        f"{len(saved_df)} records | "
        f"{'PASSED' if passed else 'FAILED'}"
    )

    assert passed, f"Read-back verification failed: {filename}"

print("\nALL FILES SAVED AND VERIFIED")

# Lanjutan Ifnu


In [ ]:
from pathlib import Path
import pandas as pd

# Menggunakan konfigurasi proyek
CLEANED_DIR = OUTPUT_DIR / "01_Cleaned_Data"
ALPHA_DIR = CLEANED_DIR / "Alpha_Synuclein"

files = {
    "activity_classification":
        CLEANED_DIR / "alpha_inhibition_activity_assay_classification.csv",

    "activity_manual_review":
        CLEANED_DIR / "alpha_inhibition_activity_assay_manual_review.csv",

    "binding_audit":
        CLEANED_DIR / "alpha_synuclein_binding_audit.csv",

    "binding_problem_records":
        CLEANED_DIR / "alpha_synuclein_binding_problem_records.csv",

    "endpoint_summary":
        ALPHA_DIR / "alpha_binding_endpoint_summary.csv",

    "kd_binding":
        ALPHA_DIR / "alpha_kd_binding_provisional.csv",

    "ic50_depolymerization":
        ALPHA_DIR / "alpha_ic50_depolymerization_provisional.csv",

    "ki":
        ALPHA_DIR / "alpha_ki_provisional.csv",

    "ic50_aggregation":
        ALPHA_DIR / "alpha_ic50_aggregation_provisional.csv",

    "ic50_binding":
        ALPHA_DIR / "alpha_ic50_binding_provisional.csv",
}

for name, path in files.items():

    print("\n" + "=" * 75)
    print(name.upper())
    print("=" * 75)

    if not path.exists():
        print("FILE NOT FOUND:", path)
        continue

    df = pd.read_csv(path)

    print("File:", path.name)
    print("Shape:", df.shape)
    print("Total columns:", len(df.columns))

    print("\nCOLUMN NAMES:")
    print(df.columns.tolist())

    print("\nMISSING VALUES:")
    missing = df.isna().sum()
    print(missing[missing > 0].sort_values(ascending=False))

    print("\nDUPLICATE RECORDS:", df.duplicated().sum())

    for col in [
        "activity_id",
        "assay_chembl_id",
        "standard_type",
        "standard_units",
        "audit_status",
        "manual_review_decision",
        "assay_category_final",
        "provisional_assay_group",
    ]:
        if col in df.columns:
            print(f"\nDISTRIBUTION: {col}")
            print(df[col].value_counts(dropna=False).head(20))

    print("\nFIRST 3 RECORDS:")
    display(df.head(3))


ACTIVITY_CLASSIFICATION
File: alpha_inhibition_activity_assay_classification.csv
Shape: (313, 9)
Total columns: 9

COLUMN NAMES:
['standard_type', 'assay_chembl_id', 'record_count', 'unique_molecules', 'units', 'relations', 'assay_description', 'assay_type', 'provisional_assay_group']

MISSING VALUES:
relations    195
units         86
dtype: int64

DUPLICATE RECORDS: 0

DISTRIBUTION: assay_chembl_id
assay_chembl_id
CHEMBL5330809    2
CHEMBL1218780    1
CHEMBL1250260    1
CHEMBL1250261    1
CHEMBL1250262    1
CHEMBL1250263    1
CHEMBL1250264    1
CHEMBL1250265    1
CHEMBL1250266    1
CHEMBL1250267    1
CHEMBL1250268    1
CHEMBL1250269    1
CHEMBL1250270    1
CHEMBL1250271    1
CHEMBL1250272    1
CHEMBL1250273    1
CHEMBL2065092    1
CHEMBL3594886    1
CHEMBL4252525    1
CHEMBL4252526    1
Name: count, dtype: int64

DISTRIBUTION: standard_type
standard_type
Activity      162
Inhibition    151
Name: count, dtype: int64

DISTRIBUTION: provisional_assay_group
provisional_assay_group
Needs_

,standard_type,assay_chembl_id,record_count,unique_molecules,units,relations,assay_description,assay_type,provisional_assay_group
0,Activity,CHEMBL1218780,2,2,NaN,NaN,Inhibition of human alpha-synuclein-mediated a...,B,Needs_review
1,Activity,CHEMBL1250260,2,2,a.u.,=,Induction of alpha-synuclein aggregation in PB...,B,Aggregation_inhibition_candidate
2,Activity,CHEMBL1250261,2,2,a.u.,=,Induction of alpha-synuclein aggregation in PB...,B,Aggregation_inhibition_candidate



ACTIVITY_MANUAL_REVIEW
File: alpha_inhibition_activity_assay_manual_review.csv
Shape: (114, 9)
Total columns: 9

COLUMN NAMES:
['standard_type', 'assay_chembl_id', 'record_count', 'unique_molecules', 'units', 'relations', 'assay_description', 'assay_type', 'provisional_assay_group']

MISSING VALUES:
relations    79
units        17
dtype: int64

DUPLICATE RECORDS: 0

DISTRIBUTION: assay_chembl_id
assay_chembl_id
CHEMBL1218780    1
CHEMBL1250268    1
CHEMBL1250269    1
CHEMBL1250270    1
CHEMBL1250271    1
CHEMBL1250272    1
CHEMBL4252525    1
CHEMBL4252526    1
CHEMBL4252532    1
CHEMBL4430344    1
CHEMBL5144381    1
CHEMBL5144382    1
CHEMBL5325281    1
CHEMBL5325285    1
CHEMBL5325295    1
CHEMBL5325296    1
CHEMBL5325297    1
CHEMBL5325321    1
CHEMBL5325322    1
CHEMBL5325331    1
Name: count, dtype: int64

DISTRIBUTION: standard_type
standard_type
Inhibition    74
Activity      40
Name: count, dtype: int64

DISTRIBUTION: provisional_assay_group
provisional_assay_group
Needs_review

,standard_type,assay_chembl_id,record_count,unique_molecules,units,relations,assay_description,assay_type,provisional_assay_group
0,Activity,CHEMBL1218780,2,2,NaN,NaN,Inhibition of human alpha-synuclein-mediated a...,B,Needs_review
1,Activity,CHEMBL1250268,2,2,%,=,Induction of alpha-synuclein aggregation in PB...,B,Needs_review
2,Activity,CHEMBL1250269,1,1,%,=,Induction of alpha-synuclein aggregation in PB...,B,Needs_review



BINDING_AUDIT
File: alpha_synuclein_binding_audit.csv
Shape: (363, 54)
Total columns: 54

COLUMN NAMES:
['action_type', 'activity_comment', 'activity_id', 'activity_properties', 'assay_chembl_id', 'assay_description', 'assay_type', 'assay_variant_accession', 'assay_variant_mutation', 'bao_endpoint', 'bao_format', 'bao_label', 'canonical_smiles', 'data_validity_comment', 'data_validity_description', 'document_chembl_id', 'document_journal', 'document_year', 'ligand_efficiency', 'modality', 'molecule_chembl_id', 'molecule_pref_name', 'parent_molecule_chembl_id', 'pchembl_value', 'potential_duplicate', 'qudt_units', 'record_id', 'relation', 'src_id', 'standard_flag', 'standard_relation', 'standard_text_value', 'standard_type', 'standard_units', 'standard_upper_value', 'standard_value', 'target_chembl_id', 'target_organism', 'target_pref_name', 'target_tax_id', 'text_value', 'toid', 'type', 'units', 'uo_units', 'upper_value', 'value', 'assay_description_meta', 'assay_bao_label', 'assay_or

,action_type,activity_comment,activity_id,activity_properties,assay_chembl_id,assay_description,assay_type,assay_variant_accession,assay_variant_mutation,bao_endpoint,...,uo_units,upper_value,value,assay_description_meta,assay_bao_label,assay_organism_meta,document_chembl_id_meta,assay_group,audit_status,quality_issue
0,NaN,NaN,10978877,[],CHEMBL2065090,Binding affinity to human alpha-synuclein expr...,B,NaN,NaN,BAO_0000192,...,UO_0000065,NaN,500.0,Binding affinity to human alpha-synuclein expr...,cell-based format,Homo sapiens,CHEMBL2062425,Binding_affinity,Pending_review,Censored_relation;
1,NaN,NaN,10978878,[],CHEMBL2065090,Binding affinity to human alpha-synuclein expr...,B,NaN,NaN,BAO_0000192,...,UO_0000065,NaN,500.0,Binding affinity to human alpha-synuclein expr...,cell-based format,Homo sapiens,CHEMBL2062425,Binding_affinity,Pending_review,Censored_relation;
2,NaN,NaN,10978879,[],CHEMBL2065090,Binding affinity to human alpha-synuclein expr...,B,NaN,NaN,BAO_0000192,...,UO_0000065,NaN,500.0,Binding affinity to human alpha-synuclein expr...,cell-based format,Homo sapiens,CHEMBL2062425,Binding_affinity,Pending_review,Censored_relation;


... (10 outputs left out)


IC50_AGGREGATION
File: alpha_ic50_aggregation_provisional.csv
Shape: (109, 54)
Total columns: 54

COLUMN NAMES:
['action_type', 'activity_comment', 'activity_id', 'activity_properties', 'assay_chembl_id', 'assay_description', 'assay_type', 'assay_variant_accession', 'assay_variant_mutation', 'bao_endpoint', 'bao_format', 'bao_label', 'canonical_smiles', 'data_validity_comment', 'data_validity_description', 'document_chembl_id', 'document_journal', 'document_year', 'ligand_efficiency', 'modality', 'molecule_chembl_id', 'molecule_pref_name', 'parent_molecule_chembl_id', 'pchembl_value', 'potential_duplicate', 'qudt_units', 'record_id', 'relation', 'src_id', 'standard_flag', 'standard_relation', 'standard_text_value', 'standard_type', 'standard_units', 'standard_upper_value', 'standard_value', 'target_chembl_id', 'target_organism', 'target_pref_name', 'target_tax_id', 'text_value', 'toid', 'type', 'units', 'uo_units', 'upper_value', 'value', 'assay_description_meta', 'assay_bao_label', '

,action_type,activity_comment,activity_id,activity_properties,assay_chembl_id,assay_description,assay_type,assay_variant_accession,assay_variant_mutation,bao_endpoint,...,uo_units,upper_value,value,assay_description_meta,assay_bao_label,assay_organism_meta,document_chembl_id_meta,assay_group,audit_status,technical_status
0,NaN,NaN,18687739,[],CHEMBL4252521,Inhibition of wild type alpha-synuclein (unkno...,B,NaN,NaN,BAO_0000190,...,UO_0000065,NaN,322.0,Inhibition of wild type alpha-synuclein (unkno...,assay format,Homo sapiens,CHEMBL4251568,Aggregation_inhibition_candidate,Pending_review,Pass - technical criteria
1,NaN,NaN,18687740,[],CHEMBL4252521,Inhibition of wild type alpha-synuclein (unkno...,B,NaN,NaN,BAO_0000190,...,UO_0000065,NaN,5.4,Inhibition of wild type alpha-synuclein (unkno...,assay format,Homo sapiens,CHEMBL4251568,Aggregation_inhibition_candidate,Pending_review,Pass - technical criteria
2,NaN,NaN,18961421,[],CHEMBL4334285,Inhibition of alpha-synuclein (unknown origin)...,B,NaN,NaN,BAO_0000190,...,UO_0000065,NaN,1.9,Inhibition of alpha-synuclein (unknown origin)...,single protein format,Homo sapiens,CHEMBL4330177,Aggregation_inhibition_candidate,Pending_review,Pass - technical criteria



IC50_BINDING
File: alpha_ic50_binding_provisional.csv
Shape: (19, 54)
Total columns: 54

COLUMN NAMES:
['action_type', 'activity_comment', 'activity_id', 'activity_properties', 'assay_chembl_id', 'assay_description', 'assay_type', 'assay_variant_accession', 'assay_variant_mutation', 'bao_endpoint', 'bao_format', 'bao_label', 'canonical_smiles', 'data_validity_comment', 'data_validity_description', 'document_chembl_id', 'document_journal', 'document_year', 'ligand_efficiency', 'modality', 'molecule_chembl_id', 'molecule_pref_name', 'parent_molecule_chembl_id', 'pchembl_value', 'potential_duplicate', 'qudt_units', 'record_id', 'relation', 'src_id', 'standard_flag', 'standard_relation', 'standard_text_value', 'standard_type', 'standard_units', 'standard_upper_value', 'standard_value', 'target_chembl_id', 'target_organism', 'target_pref_name', 'target_tax_id', 'text_value', 'toid', 'type', 'units', 'uo_units', 'upper_value', 'value', 'assay_description_meta', 'assay_bao_label', 'assay_org

,action_type,activity_comment,activity_id,activity_properties,assay_chembl_id,assay_description,assay_type,assay_variant_accession,assay_variant_mutation,bao_endpoint,...,uo_units,upper_value,value,assay_description_meta,assay_bao_label,assay_organism_meta,document_chembl_id_meta,assay_group,audit_status,technical_status
0,NaN,NaN,10978896,[],CHEMBL2065090,Binding affinity to human alpha-synuclein expr...,B,NaN,NaN,BAO_0000190,...,UO_0000065,NaN,26000.0,Binding affinity to human alpha-synuclein expr...,cell-based format,Homo sapiens,CHEMBL2062425,Binding_affinity,Pending_review,Pass - technical criteria
1,NaN,NaN,10978897,[],CHEMBL2065090,Binding affinity to human alpha-synuclein expr...,B,NaN,NaN,BAO_0000190,...,UO_0000065,NaN,26000.0,Binding affinity to human alpha-synuclein expr...,cell-based format,Homo sapiens,CHEMBL2062425,Binding_affinity,Pending_review,Pass - technical criteria
2,NaN,NaN,10978898,[],CHEMBL2065090,Binding affinity to human alpha-synuclein expr...,B,NaN,NaN,BAO_0000190,...,UO_0000065,NaN,16000.0,Binding affinity to human alpha-synuclein expr...,cell-based format,Homo sapiens,CHEMBL2062425,Binding_affinity,Pending_review,Pass - technical criteria


In [ ]:
import pandas as pd

CLEANED_DIR = OUTPUT_DIR / "01_Cleaned_Data"

classification_path = (
    CLEANED_DIR / "alpha_inhibition_activity_assay_classification.csv"
)

review_path = (
    CLEANED_DIR / "alpha_inhibition_activity_assay_manual_review.csv"
)

group_summary_path = (
    CLEANED_DIR / "alpha_inhibition_activity_assay_group_summary.csv"
)

classification = pd.read_csv(classification_path)
manual_review = pd.read_csv(review_path)

# 1. Ringkasan seluruh kelompok assay
print("=" * 80)
print("PROVISIONAL ASSAY GROUP SUMMARY")
print("=" * 80)

print(
    classification.groupby(
        "provisional_assay_group",
        dropna=False
    ).agg(
        assay_groups=("assay_chembl_id", "nunique"),
        total_records=("record_count", "sum"),
        unique_molecules=("unique_molecules", "sum")
    ).sort_values("total_records", ascending=False)
)

# 2. Seluruh 114 assay yang perlu direview
print("\n" + "=" * 80)
print("ALL ASSAYS REQUIRING MANUAL REVIEW")
print("=" * 80)

review_columns = [
    "assay_chembl_id",
    "standard_type",
    "record_count",
    "unique_molecules",
    "units",
    "relations",
    "assay_type",
    "assay_description"
]

review_display = (
    manual_review[review_columns]
    .sort_values(["standard_type", "assay_chembl_id"])
)

pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 250)

display(review_display)

# 3. Cek apakah file group summary tersedia
print("\n" + "=" * 80)
print("GROUP SUMMARY FILE")
print("=" * 80)

if group_summary_path.exists():
    group_summary = pd.read_csv(group_summary_path)
    print("Shape:", group_summary.shape)
    display(group_summary)
else:
    print("File tidak ditemukan:", group_summary_path)

PROVISIONAL ASSAY GROUP SUMMARY
                                  assay_groups  total_records  \
provisional_assay_group                                         
Aggregation_inhibition_candidate            45            440   
Cell_based_aggregation                      41            238   
Needs_review                               114            234   
Fibril_disassembly                          56            130   
Binding_affinity                            23            118   
Oligomer_formation                           6             38   
Secondary_structure                         21             37   
In_vivo_model                                4             33   
Radioligand_displacement                     2             12   

                                  unique_molecules  
provisional_assay_group                             
Aggregation_inhibition_candidate               440  
Cell_based_aggregation                         238  
Needs_review                            

,assay_chembl_id,standard_type,record_count,unique_molecules,units,relations,assay_type,assay_description
0,CHEMBL1218780,Activity,2,2,NaN,NaN,B,Inhibition of human alpha-synuclein-mediated accumulation of stalled vesicle pools in yeast transfected with human alpha-synuclein after 8 to 12 hrs by electron microscopy
1,CHEMBL1250268,Activity,2,2,%,=,B,Induction of alpha-synuclein aggregation in PBS buffer assessed as alpha-synuclein monomer depletion after 48 hrs by size exclusion chromatography
2,CHEMBL1250269,Activity,1,1,%,=,B,Induction of alpha-synuclein aggregation in PBS buffer assessed as alpha-synuclein monomer depletion after 72 hrs by size exclusion chromatography
3,CHEMBL1250270,Activity,2,2,%,=,B,Induction of alpha-synuclein aggregation in PBS buffer assessed as alpha-synuclein monomer depletion after 96 hrs by size exclusion chromatography
4,CHEMBL1250271,Activity,1,1,NaN,NaN,B,Induction of alpha-synuclein aggregation in PBS buffer assessed as formation of alpha-synuclein fibrils after 48 hrs by electron microscopy
5,CHEMBL1250272,Activity,1,1,NaN,NaN,B,Induction of alpha-synuclein aggregation in PBS buffer assessed as formation of alpha-synuclein fibrils after 96 hrs by electron microscopy
6,CHEMBL4252525,Activity,2,2,nm,=,B,Inhibition of wild type alpha-synuclein (unknown origin) aggregation expressed in Escherichia coli BL21 assessed as oligomer height after 30 days by AFM analysis (Rvb = 1.9 +/- 1.5 nm)
7,CHEMBL4252526,Activity,2,2,%,=,B,Inhibition of wild type alpha-synuclein (unknown origin) aggregation expressed in Escherichia coli BL21 assessed as turbidity level after 30 days by UV-VIS spectrophotometric analysis (Rvb = 430.9 +/-100.7%)
8,CHEMBL4252532,Activity,2,2,%,=,B,Inhibition of wild type alpha-synuclein (unknown origin)-mediated cytotoxicity in human SH-SY5Y cells exposed to alpha-synuclein co-incubuted with with compound for 15 days assessed as cell viability by LDH release assay (Rvb = 42%)
9,CHEMBL4430344,Activity,2,2,NaN,NaN,B,Induction of recombinant alphaSN (unknown origin) fibrillation at 120 uM after 6 days by ThT fluorescence assay



GROUP SUMMARY FILE
Shape: (14, 4)


,standard_type,provisional_assay_group,assay_count,record_count
0,Activity,Aggregation_inhibition_candidate,10,102
1,Activity,Binding_affinity,23,118
2,Activity,Cell_based_aggregation,22,78
3,Activity,Fibril_disassembly,56,129
4,Activity,Needs_review,40,59
5,Activity,Radioligand_displacement,2,12
6,Activity,Secondary_structure,9,12
7,Inhibition,Aggregation_inhibition_candidate,35,338
8,Inhibition,Cell_based_aggregation,19,160
9,Inhibition,Fibril_disassembly,1,1


In [ ]:

import pandas as pd
from pathlib import Path

# Gunakan konfigurasi path proyek yang sudah ada
REVIEW_DIR = OUTPUT_DIR / "01_Cleaned_Data"

input_file = (
    REVIEW_DIR /
    "alpha_inhibition_activity_assay_manual_review.csv"
)

output_file = (
    REVIEW_DIR /
    "alpha_inhibition_activity_assay_manual_review_template.csv"
)

# Baca inventaris assay yang sudah tersedia
review_df = pd.read_csv(input_file)

# Pastikan kolom deskripsi tersedia
required_columns = [
    "assay_chembl_id",
    "standard_type",
    "assay_description"
]

missing_columns = [
    col for col in required_columns
    if col not in review_df.columns
]

if missing_columns:
    raise ValueError(
        f"Kolom wajib tidak ditemukan: {missing_columns}"
    )

# Tambahkan kolom keputusan manual
review_df["manual_assay_category"] = "Unresolved_Keep_Quarantine"
review_df["manual_assay_decision"] = "Pending_Review"
review_df["review_note"] = ""
review_df["review_status"] = "Not_Reviewed"

# Simpan sebagai file baru
review_df.to_csv(output_file, index=False)

print("TEMPLATE REVIEW BERHASIL DIBUAT")
print("Jumlah assay:", len(review_df))
print("Kolom:", review_df.columns.tolist())
print("Lokasi:", output_file)

display(
    review_df[
        [
            "assay_chembl_id",
            "standard_type",
            "assay_description",
            "manual_assay_category",
            "manual_assay_decision",
            "review_status"
        ]
    ].head(10)
)


TEMPLATE REVIEW BERHASIL DIBUAT
Jumlah assay: 114
Kolom: ['standard_type', 'assay_chembl_id', 'record_count', 'unique_molecules', 'units', 'relations', 'assay_description', 'assay_type', 'provisional_assay_group', 'manual_assay_category', 'manual_assay_decision', 'review_note', 'review_status']
Lokasi: H:\My Drive\PROJECT SAINS DATA\OUTPUT\01_Cleaned_Data\alpha_inhibition_activity_assay_manual_review_template.csv


,assay_chembl_id,standard_type,assay_description,manual_assay_category,manual_assay_decision,review_status
0,CHEMBL1218780,Activity,Inhibition of human alpha-synuclein-mediated accumulation of stalled vesicle pools in yeast transfected with human alpha-synuclein after 8 to 12 hrs by electron microscopy,Unresolved_Keep_Quarantine,Pending_Review,Not_Reviewed
1,CHEMBL1250268,Activity,Induction of alpha-synuclein aggregation in PBS buffer assessed as alpha-synuclein monomer depletion after 48 hrs by size exclusion chromatography,Unresolved_Keep_Quarantine,Pending_Review,Not_Reviewed
2,CHEMBL1250269,Activity,Induction of alpha-synuclein aggregation in PBS buffer assessed as alpha-synuclein monomer depletion after 72 hrs by size exclusion chromatography,Unresolved_Keep_Quarantine,Pending_Review,Not_Reviewed
3,CHEMBL1250270,Activity,Induction of alpha-synuclein aggregation in PBS buffer assessed as alpha-synuclein monomer depletion after 96 hrs by size exclusion chromatography,Unresolved_Keep_Quarantine,Pending_Review,Not_Reviewed
4,CHEMBL1250271,Activity,Induction of alpha-synuclein aggregation in PBS buffer assessed as formation of alpha-synuclein fibrils after 48 hrs by electron microscopy,Unresolved_Keep_Quarantine,Pending_Review,Not_Reviewed
5,CHEMBL1250272,Activity,Induction of alpha-synuclein aggregation in PBS buffer assessed as formation of alpha-synuclein fibrils after 96 hrs by electron microscopy,Unresolved_Keep_Quarantine,Pending_Review,Not_Reviewed
6,CHEMBL4252525,Activity,Inhibition of wild type alpha-synuclein (unknown origin) aggregation expressed in Escherichia coli BL21 assessed as oligomer height after 30 days by AFM analysis (Rvb = 1.9 +/- 1.5 nm),Unresolved_Keep_Quarantine,Pending_Review,Not_Reviewed
7,CHEMBL4252526,Activity,Inhibition of wild type alpha-synuclein (unknown origin) aggregation expressed in Escherichia coli BL21 assessed as turbidity level after 30 days by UV-VIS spectrophotometric analysis (Rvb = 430.9 +/-100.7%),Unresolved_Keep_Quarantine,Pending_Review,Not_Reviewed
8,CHEMBL4252532,Activity,Inhibition of wild type alpha-synuclein (unknown origin)-mediated cytotoxicity in human SH-SY5Y cells exposed to alpha-synuclein co-incubuted with with compound for 15 days assessed as cell viability by LDH release assay (Rvb = 42%),Unresolved_Keep_Quarantine,Pending_Review,Not_Reviewed
9,CHEMBL4430344,Activity,Induction of recombinant alphaSN (unknown origin) fibrillation at 120 uM after 6 days by ThT fluorescence assay,Unresolved_Keep_Quarantine,Pending_Review,Not_Reviewed


In [ ]:

import pandas as pd
from pathlib import Path

REVIEW_DIR = OUTPUT_DIR / "01_Cleaned_Data"

input_file = (
    REVIEW_DIR /
    "alpha_inhibition_activity_assay_manual_review_template.csv"
)

output_file = (
    REVIEW_DIR /
    "alpha_inhibition_activity_assay_suggested_review.csv"
)

df = pd.read_csv(input_file)

# Pastikan ID assay dan deskripsi tersedia
required = ["assay_chembl_id", "assay_description"]
missing = [c for c in required if c not in df.columns]

if missing:
    raise ValueError(f"Kolom tidak ditemukan: {missing}")

def suggest_assay_category(row):
    assay_id = str(row["assay_chembl_id"])
    desc = str(row["assay_description"]).lower()

    # Kasus kontradiktif yang sudah diketahui
    if assay_id == "CHEMBL5330806":
        return (
            "Unresolved_Keep_Quarantine",
            "Pending_Review",
            "Deskripsi menyebut induction, tetapi standard_type adalah Inhibition."
        )

    # Induksi agregasi
    if "induction" in desc and (
        "aggregation" in desc or "fibrillation" in desc
    ):
        return (
            "Aggregation_Induction",
            "Exclude_from_Inhibition_Analysis",
            "Assay mengukur induksi atau peningkatan agregasi."
        )

    # Degradasi / clearance
    if any(x in desc for x in [
        "degradation of alpha-synuclein",
        "alpha-synuclein degradation",
        "degradation of alpha synuclein"
    ]):
        return (
            "Protein_Degradation_Clearance",
            "Separate_Analysis",
            "Mengukur degradasi atau clearance protein."
        )

    # Toksisitas atau perlindungan sel
    if any(x in desc for x in [
        "cytotoxicity",
        "cell viability",
        "toxicity",
        "ldh release"
    ]):
        return (
            "Cellular_Toxicity_or_Protection",
            "Separate_Analysis",
            "Mengukur respons toksisitas atau viabilitas sel."
        )

    # Inklusi di dalam sel
    if any(x in desc for x in [
        "inclusion formation",
        "inclusions in",
        "inclusion bodies",
        "lewy body"
    ]):
        return (
            "Cellular_Inclusion_Modulation",
            "Separate_Analysis",
            "Mengukur pembentukan inklusi dalam sistem sel."
        )

    # Degradasi fibril / oligomer
    if any(x in desc for x in [
        "disassembly",
        "disassemble",
        "depolymerization"
    ]):
        return (
            "Fibril_Disassembly",
            "Separate_Analysis",
            "Mengukur penguraian fibril atau agregat yang telah terbentuk."
        )

    # Perubahan konformasi
    if any(x in desc for x in [
        "secondary structure",
        "conformational change",
        "alpha-helical conformation"
    ]):
        return (
            "Conformational_Modulation",
            "Separate_Analysis",
            "Mengukur perubahan struktur atau konformasi protein."
        )

    # Modulasi oligomer
    if "oligomer" in desc and any(x in desc for x in [
        "disassembly", "particle size", "oligomerization"
    ]):
        return (
            "Oligomer_Modulation",
            "Separate_Analysis",
            "Mengukur pembentukan, ukuran, atau perubahan oligomer."
        )

    # Pembentukan fibril
    if any(x in desc for x in [
        "fibrillation",
        "fibril formation",
        "fibril formation",
        "fibrillogenesis",
        "formation of alpha-synuclein fibrils"
    ]):
        if "inhibition" in desc:
            return (
                "Fibril_Formation_Inhibition",
                "Candidate_Separate_Endpoint",
                "Kandidat penghambatan pembentukan fibril."
            )

    # Penghambatan agregasi umum
    if "inhibition" in desc and any(x in desc for x in [
        "aggregation",
        "aggregates",
        "aggregation kinetics"
    ]):
        return (
            "Aggregation_Inhibition_Candidate",
            "Candidate_Separate_Endpoint",
            "Kandidat penghambatan agregasi; endpoint perlu diperiksa."
        )

    return (
        "Unresolved_Keep_Quarantine",
        "Pending_Review",
        "Belum cukup jelas dari deskripsi assay saja."
    )

suggestions = df.apply(
    suggest_assay_category,
    axis=1,
    result_type="expand"
)

suggestions.columns = [
    "suggested_assay_category",
    "suggested_assay_decision",
    "suggested_review_note"
]

df = pd.concat([df, suggestions], axis=1)

# Simpan file baru; keputusan manual tetap tidak berubah
df.to_csv(output_file, index=False)

print("SUGGESTED REVIEW BERHASIL DIBUAT")
print("Total assay:", len(df))
print("\nDISTRIBUSI SARAN KATEGORI:")
print(df["suggested_assay_category"].value_counts(dropna=False))

print("\nDISTRIBUSI SARAN KEPUTUSAN:")
print(df["suggested_assay_decision"].value_counts(dropna=False))

print("\nASSAY YANG MASIH UNRESOLVED:")
display(
    df.loc[
        df["suggested_assay_category"] == "Unresolved_Keep_Quarantine",
        [
            "assay_chembl_id",
            "standard_type",
            "assay_description",
            "suggested_review_note"
        ]
    ]
)

print("\nFILE TERSIMPAN:", output_file)


SUGGESTED REVIEW BERHASIL DIBUAT
Total assay: 114

DISTRIBUSI SARAN KATEGORI:
suggested_assay_category
Aggregation_Inhibition_Candidate    37
Unresolved_Keep_Quarantine          32
Fibril_Formation_Inhibition         24
Cellular_Inclusion_Modulation        7
Aggregation_Induction                6
Oligomer_Modulation                  3
Cellular_Toxicity_or_Protection      2
Conformational_Modulation            2
Fibril_Disassembly                   1
Name: count, dtype: int64

DISTRIBUSI SARAN KEPUTUSAN:
suggested_assay_decision
Candidate_Separate_Endpoint         61
Pending_Review                      32
Separate_Analysis                   15
Exclude_from_Inhibition_Analysis     6
Name: count, dtype: int64

ASSAY YANG MASIH UNRESOLVED:


,assay_chembl_id,standard_type,assay_description,suggested_review_note
0,CHEMBL1218780,Activity,Inhibition of human alpha-synuclein-mediated accumulation of stalled vesicle pools in yeast transfected with human alpha-synuclein after 8 to 12 hrs by electron microscopy,Belum cukup jelas dari deskripsi assay saja.
12,CHEMBL5325281,Activity,Induction of alpha-synuclein (unknown origin) degradation overexpressed in HEK293T cells at 50 to 100 uM incubated for 24 hrs by immunoblot analysis,Belum cukup jelas dari deskripsi assay saja.
13,CHEMBL5325285,Activity,Induction of EGFP-fused alpha-synuclein (unknown origin) degradation overexpressed in HEK293T cells co-expressing mCherry-fused LC3B assessed as puncta formation of alpha-synuclein at 50 uM incubated for 2 hrs by confocal laser scanning microscopy,Belum cukup jelas dari deskripsi assay saja.
14,CHEMBL5325295,Activity,Induction of alpha-synuclein (unknown origin) degradation overexpressed in HEK293T cells infected with adenovirus expressing mCherry-EGFP-LC3B incubated for 24 hrs by immunoblot analysis,Belum cukup jelas dari deskripsi assay saja.
15,CHEMBL5325296,Activity,Induction of alpha-synuclein (unknown origin) degradation overexpressed in HEK293T cells infected with adenovirus expressing mCherry-EGFP-LC3B at 50 uM incubated upto 24 hrs by immunoblot analysis,Belum cukup jelas dari deskripsi assay saja.
16,CHEMBL5325297,Activity,Induction of alpha-synuclein (unknown origin) degradation overexpressed in HEK293T cells infected with adenovirus expressing mCherry-EGFP-LC3B at 50 uM incubated for 3 hrs by immunoblot analysis,Belum cukup jelas dari deskripsi assay saja.
17,CHEMBL5325321,Activity,Induction of EGFP-fused alpha-synuclein (unknown origin) degradation overexpressed in HEK293T cells at 50 uM incubated for 2 hrs by confocal laser scanning microscopy,Belum cukup jelas dari deskripsi assay saja.
18,CHEMBL5325322,Activity,Induction of EGFP-fused alpha-synuclein (unknown origin) degradation overexpressed in HEK293T cells assessed as colocalization of alpha-synuclein with mCherry-fused LC3B at 50 uM incubated for 2 hrs by confocal laser scanning microscopy,Belum cukup jelas dari deskripsi assay saja.
19,CHEMBL5325331,Activity,Induction of alpha-synuclein (unknown origin) degradation overexpressed in HEK293T cells pretreated with BafA1 at 50 uM incubated for 24 hrs by immunoblot analysis,Belum cukup jelas dari deskripsi assay saja.
20,CHEMBL5325332,Activity,Induction of alpha-synuclein (unknown origin) degradation overexpressed in HEK293T cells at 50 uM incubated for 24 hrs in presence of MG132 by immunoblot analysis,Belum cukup jelas dari deskripsi assay saja.



FILE TERSIMPAN: H:\My Drive\PROJECT SAINS DATA\OUTPUT\01_Cleaned_Data\alpha_inhibition_activity_assay_suggested_review.csv


In [ ]:

import pandas as pd

input_file = (
    OUTPUT_DIR / "01_Cleaned_Data" /
    "alpha_inhibition_activity_assay_suggested_review.csv"
)

output_file = (
    OUTPUT_DIR / "01_Cleaned_Data" /
    "alpha_inhibition_activity_assay_review_revised.csv"
)

df = pd.read_csv(input_file)

# Salin saran awal sebagai dasar
df["reviewed_assay_category"] = df["suggested_assay_category"]
df["reviewed_assay_decision"] = df["suggested_assay_decision"]
df["reviewed_note"] = df["suggested_review_note"]

# Daftar assay yang perlu diperbaiki
overrides = {}

# Vesicle trafficking: bukan pengukuran agregasi langsung
overrides["CHEMBL1218780"] = (
    "Cellular_Vesicle_Trafficking",
    "Separate_Analysis",
    "Mengukur akumulasi vesikel pada yeast; bukan endpoint agregasi langsung."
)

# Degradasi protein
degradation_ids = [
    "CHEMBL5325281", "CHEMBL5325285",
    "CHEMBL5325295", "CHEMBL5325296",
    "CHEMBL5325297", "CHEMBL5325321",
    "CHEMBL5325322", "CHEMBL5325331",
    "CHEMBL5325332", "CHEMBL5325333",
    "CHEMBL5325334"
]

for assay_id in degradation_ids:
    overrides[assay_id] = (
        "Protein_Degradation_Clearance",
        "Separate_Analysis",
        "Mengukur induksi degradasi alpha-synuclein."
    )

# Perubahan konformasi berdasarkan circular dichroism
conformation_ids = [
    "CHEMBL5507748", "CHEMBL5507749",
    "CHEMBL5507750", "CHEMBL5507751",
    "CHEMBL5507756", "CHEMBL5507757",
    "CHEMBL5507758", "CHEMBL5507759",
    "CHEMBL5507760", "CHEMBL5507761",
    "CHEMBL5507762", "CHEMBL5507763"
]

for assay_id in conformation_ids:
    overrides[assay_id] = (
        "Conformational_Modulation",
        "Separate_Analysis",
        "Mengukur perubahan konformasi protein melalui circular dichroism."
    )

# Endpoint agregasi / fibril yang perlu dipisahkan
overrides["CHEMBL2065091"] = (
    "Aggregation_Inhibition_Candidate",
    "Candidate_Separate_Endpoint",
    "Assay ThT; kandidat pengukuran agregasi, perlu verifikasi detail endpoint."
)

overrides["CHEMBL4373561"] = (
    "Fibril_Elongation_Inhibition",
    "Candidate_Separate_Endpoint",
    "Mengukur inhibisi elongasi fibril yang telah terbentuk; bukan otomatis disassembly."
)

overrides["CHEMBL5347699"] = (
    "Aggregation_Inhibition_Candidate",
    "Candidate_Separate_Endpoint",
    "Mengukur inhibisi re-aggregation pada waktu pemberian tertentu."
)

overrides["CHEMBL5347700"] = (
    "Aggregation_Inhibition_Candidate",
    "Candidate_Separate_Endpoint",
    "Mengukur inhibisi re-aggregation pada waktu pemberian tertentu."
)

overrides["CHEMBL5347701"] = (
    "Aggregation_Inhibition_Candidate",
    "Candidate_Separate_Endpoint",
    "Mengukur inhibisi re-aggregation pada waktu pemberian tertentu."
)

overrides["CHEMBL5518544"] = (
    "Fibril_Formation_Inhibition",
    "Candidate_Separate_Endpoint",
    "Mengukur kemunculan amyloid fibrils menggunakan AFM."
)

overrides["CHEMBL5524395"] = (
    "Aggregation_Inhibition_Candidate",
    "Candidate_Separate_Endpoint",
    "Mengukur pembentukan pathological aggregates."
)

# Terapkan hanya pada ID yang ditemukan
found_ids = set(df["assay_chembl_id"].astype(str))
missing_override_ids = sorted(set(overrides) - found_ids)

for assay_id, decision in overrides.items():
    mask = df["assay_chembl_id"].astype(str) == assay_id

    if mask.any():
        category, action, note = decision
        df.loc[mask, "reviewed_assay_category"] = category
        df.loc[mask, "reviewed_assay_decision"] = action
        df.loc[mask, "reviewed_note"] = note

# CHEMBL5330806 tetap unresolved karena deskripsinya kontradiktif

df.to_csv(output_file, index=False)

print("REVISED REVIEW BERHASIL DISIMPAN")
print("Total assay:", len(df))
print("Override ditemukan:", len(overrides) - len(missing_override_ids))
print("Override tidak ditemukan:", missing_override_ids)

print("\nDISTRIBUSI KATEGORI REVISI:")
print(df["reviewed_assay_category"].value_counts())

print("\nDISTRIBUSI KEPUTUSAN REVISI:")
print(df["reviewed_assay_decision"].value_counts())

print("\nASSAY YANG MASIH PENDING:")
display(
    df.loc[
        df["reviewed_assay_decision"] == "Pending_Review",
        [
            "assay_chembl_id",
            "assay_description",
            "reviewed_assay_category",
            "reviewed_note"
        ]
    ]
)

print("\nFILE:", output_file)


REVISED REVIEW BERHASIL DISIMPAN
Total assay: 114
Override ditemukan: 31
Override tidak ditemukan: []

DISTRIBUSI KATEGORI REVISI:
reviewed_assay_category
Aggregation_Inhibition_Candidate    42
Fibril_Formation_Inhibition         25
Conformational_Modulation           14
Protein_Degradation_Clearance       11
Cellular_Inclusion_Modulation        7
Aggregation_Induction                6
Oligomer_Modulation                  3
Cellular_Toxicity_or_Protection      2
Cellular_Vesicle_Trafficking         1
Fibril_Elongation_Inhibition         1
Unresolved_Keep_Quarantine           1
Fibril_Disassembly                   1
Name: count, dtype: int64

DISTRIBUSI KEPUTUSAN REVISI:
reviewed_assay_decision
Candidate_Separate_Endpoint         68
Separate_Analysis                   39
Exclude_from_Inhibition_Analysis     6
Pending_Review                       1
Name: count, dtype: int64

ASSAY YANG MASIH PENDING:


,assay_chembl_id,assay_description,reviewed_assay_category,reviewed_note
62,CHEMBL5330806,Induction of alpha-Synuclein (unknown origin) aggregation assessed as dispersion of finer and shorter fibers at 30 uM incubated for 72 hrs by TEM microscopy,Unresolved_Keep_Quarantine,"Deskripsi menyebut induction, tetapi standard_type adalah Inhibition."



FILE: H:\My Drive\PROJECT SAINS DATA\OUTPUT\01_Cleaned_Data\alpha_inhibition_activity_assay_review_revised.csv


In [ ]:

import pandas as pd

REVIEW_DIR = OUTPUT_DIR / "01_Cleaned_Data"

input_file = (
    REVIEW_DIR /
    "alpha_inhibition_activity_assay_review_revised.csv"
)

df = pd.read_csv(input_file)

# Pastikan ID assay tidak berulang
if df["assay_chembl_id"].duplicated().any():
    duplicated = df.loc[
        df["assay_chembl_id"].duplicated(keep=False),
        "assay_chembl_id"
    ].unique()

    raise ValueError(
        f"Ada assay ID duplikat: {duplicated.tolist()}"
    )

# Kelompok inventaris berdasarkan keputusan review
groups = {
    "candidate": "Candidate_Separate_Endpoint",
    "separate": "Separate_Analysis",
    "excluded": "Exclude_from_Inhibition_Analysis",
    "pending": "Pending_Review"
}

# Kolom yang akan disimpan
columns = [
    "assay_chembl_id",
    "standard_type",
    "record_count",
    "unique_molecules",
    "units",
    "relations",
    "assay_type",
    "reviewed_assay_category",
    "reviewed_assay_decision",
    "assay_description",
    "reviewed_note"
]

summary = []

for name, decision in groups.items():

    subset = df.loc[
        df["reviewed_assay_decision"] == decision,
        columns
    ].copy()

    output_file = (
        REVIEW_DIR /
        f"alpha_assay_inventory_{name}.csv"
    )

    subset.to_csv(output_file, index=False)

    summary.append({
        "inventory": name,
        "assay_count": len(subset),
        "record_count_sum": pd.to_numeric(
            subset["record_count"], errors="coerce"
        ).sum(),
        "unique_molecules_sum": pd.to_numeric(
            subset["unique_molecules"], errors="coerce"
        ).sum(),
        "output_file": output_file.name
    })

summary_df = pd.DataFrame(summary)

print("ASSAY INVENTORY BERHASIL DIBUAT")
display(summary_df)

print("\nDISTRIBUSI KANDIDAT PER KATEGORI:")
display(
    df.loc[
        df["reviewed_assay_decision"] == "Candidate_Separate_Endpoint"
    ].groupby("reviewed_assay_category")
    .agg(
        assay_count=("assay_chembl_id", "nunique"),
        record_count_sum=("record_count", "sum"),
        molecule_count_sum=("unique_molecules", "sum")
    )
    .reset_index()
)

print("\nTOTAL ASSAY:", len(df))
print("TOTAL RECORD COUNT INVENTORY:",
      pd.to_numeric(df["record_count"], errors="coerce").sum())


ASSAY INVENTORY BERHASIL DIBUAT


,inventory,assay_count,record_count_sum,unique_molecules_sum,output_file
0,candidate,68,165,165,alpha_assay_inventory_candidate.csv
1,separate,39,57,57,alpha_assay_inventory_separate.csv
2,excluded,6,9,9,alpha_assay_inventory_excluded.csv
3,pending,1,3,3,alpha_assay_inventory_pending.csv



DISTRIBUSI KANDIDAT PER KATEGORI:


,reviewed_assay_category,assay_count,record_count_sum,molecule_count_sum
0,Aggregation_Inhibition_Candidate,42,111,111
1,Fibril_Elongation_Inhibition,1,1,1
2,Fibril_Formation_Inhibition,25,53,53



TOTAL ASSAY: 114
TOTAL RECORD COUNT INVENTORY: 234


In [ ]:

from pathlib import Path

# Folder sumber alpha-synuclein
alpha_dir = ALPHA_SYNUCLEIN_DIR

print("FOLDER ALPHA-SYNUCLEIN:")
print(alpha_dir)
print("Folder tersedia:", alpha_dir.exists())

print("\nDAFTAR FILE CSV:")
csv_files = sorted(alpha_dir.rglob("*.csv"))

for i, file in enumerate(csv_files, 1):
    print(f"{i}. {file.name}")


FOLDER ALPHA-SYNUCLEIN:
H:\My Drive\PROJECT SAINS DATA\DATA\α-Synuclein
Folder tersedia: True

DAFTAR FILE CSV:
1. alpha_synuclein_activity_raw.csv
2. alpha_synuclein_assay_metadata_raw (1).csv
3. alpha_synuclein_assay_metadata_raw.csv
4. alpha_synuclein_target_metadata.csv


In [ ]:

import pandas as pd

keywords = ["activity", "inhibition", "assay", "classification"]

candidate_files = [
    f for f in csv_files
    if any(word in f.name.lower() for word in keywords)
]

print("KANDIDAT FILE SUMBER:")

for file in candidate_files:
    try:
        temp = pd.read_csv(file, nrows=5)
        print("\nFILE:", file.name)
        print("Kolom:", temp.columns.tolist())
    except Exception as e:
        print("Gagal membaca:", file.name, e)


KANDIDAT FILE SUMBER:

FILE: alpha_synuclein_activity_raw.csv
Kolom: ['action_type', 'activity_comment', 'activity_id', 'activity_properties', 'assay_chembl_id', 'assay_description', 'assay_type', 'assay_variant_accession', 'assay_variant_mutation', 'bao_endpoint', 'bao_format', 'bao_label', 'canonical_smiles', 'data_validity_comment', 'data_validity_description', 'document_chembl_id', 'document_journal', 'document_year', 'ligand_efficiency', 'modality', 'molecule_chembl_id', 'molecule_pref_name', 'parent_molecule_chembl_id', 'pchembl_value', 'potential_duplicate', 'qudt_units', 'record_id', 'relation', 'src_id', 'standard_flag', 'standard_relation', 'standard_text_value', 'standard_type', 'standard_units', 'standard_upper_value', 'standard_value', 'target_chembl_id', 'target_organism', 'target_pref_name', 'target_tax_id', 'text_value', 'toid', 'type', 'units', 'uo_units', 'upper_value', 'value']

FILE: alpha_synuclein_assay_metadata_raw (1).csv
Kolom: ['aidx', 'assay_category', 'assay

In [ ]:
import pandas as pd
from pathlib import Path

ALPHA_SYNUCLEIN_DIR = Path(
    r"H:\My Drive\PROJECT SAINS DATA\DATA\α-Synuclein"
)

# =========================================================
# 1. LOAD FILE
# =========================================================

activity_path = ALPHA_SYNUCLEIN_DIR / "alpha_synuclein_activity_raw.csv"

metadata_path_1 = (
    ALPHA_SYNUCLEIN_DIR / "alpha_synuclein_assay_metadata_raw.csv"
)

metadata_path_2 = (
    ALPHA_SYNUCLEIN_DIR / "alpha_synuclein_assay_metadata_raw (1).csv"
)

alpha_activity = pd.read_csv(activity_path, low_memory=False)
alpha_metadata_1 = pd.read_csv(metadata_path_1, low_memory=False)
alpha_metadata_2 = pd.read_csv(metadata_path_2, low_memory=False)


# =========================================================
# 2. DATASET OVERVIEW
# =========================================================

print("=" * 65)
print("ALPHA-SYNUCLEIN SOURCE DATA AUDIT")
print("=" * 65)

print("\nACTIVITY RAW")
print("Shape:", alpha_activity.shape)
print("Unique activity_id:", alpha_activity["activity_id"].nunique())
print("Unique assay_chembl_id:", alpha_activity["assay_chembl_id"].nunique())
print("Unique molecule:", alpha_activity["molecule_chembl_id"].nunique())


# =========================================================
# 3. ACTIVITY ENDPOINT DISTRIBUTION
# =========================================================

print("\nSTANDARD TYPE DISTRIBUTION")
print(alpha_activity["standard_type"].value_counts(dropna=False))

print("\nSTANDARD UNITS DISTRIBUTION")
print(alpha_activity["standard_units"].value_counts(dropna=False))

print("\nTARGET DISTRIBUTION")
print(alpha_activity["target_chembl_id"].value_counts(dropna=False))


# =========================================================
# 4. DUPLICATE CHECK
# =========================================================

print("\nDUPLICATE CHECK")
print(
    "Duplicated activity_id:",
    alpha_activity["activity_id"].duplicated().sum()
)

print(
    "Duplicated full rows:",
    alpha_activity.duplicated().sum()
)


# =========================================================
# 5. METADATA COMPARISON
# =========================================================

print("\nMETADATA COMPARISON")

print("Metadata 1 shape:", alpha_metadata_1.shape)
print("Metadata 2 shape:", alpha_metadata_2.shape)

print(
    "Metadata 1 unique assay:",
    alpha_metadata_1["assay_chembl_id"].nunique()
)

print(
    "Metadata 2 unique assay:",
    alpha_metadata_2["assay_chembl_id"].nunique()
)

print(
    "Metadata identical:",
    alpha_metadata_1.equals(alpha_metadata_2)
)

print(
    "Metadata 1 duplicate assay IDs:",
    alpha_metadata_1["assay_chembl_id"].duplicated().sum()
)

print(
    "Metadata 2 duplicate assay IDs:",
    alpha_metadata_2["assay_chembl_id"].duplicated().sum()
)


# =========================================================
# 6. ASSAY COVERAGE
# =========================================================

activity_assays = set(
    alpha_activity["assay_chembl_id"].dropna().unique()
)

metadata_assays = set(
    alpha_metadata_1["assay_chembl_id"].dropna().unique()
)

print("\nASSAY COVERAGE")
print("Assays in activity:", len(activity_assays))
print("Assays in metadata:", len(metadata_assays))
print("Activity assays without metadata:", len(activity_assays - metadata_assays))
print("Metadata assays without activity:", len(metadata_assays - activity_assays))

ALPHA-SYNUCLEIN SOURCE DATA AUDIT

ACTIVITY RAW
Shape: (12221, 47)
Unique activity_id: 12221
Unique assay_chembl_id: 429
Unique molecule: 11054

STANDARD TYPE DISTRIBUTION
standard_type
Potency          10056
Inhibition         770
Activity           510
IC50               192
Ki                 145
Kd                 141
EC50                92
kon                 84
k_off               84
Flu intensity       59
TIME                23
Kapp                16
FC                  12
Ratio               12
T1/2                 9
Tlag                 9
DC50                 4
Kb                   2
Ratio IC50           1
Name: count, dtype: int64

STANDARD UNITS DISTRIBUTION
standard_units
nM          10593
%            1059
NaN           364
s-1            84
RU             47
hr             41
10^-3/hr        9
a.u.            8
/hr             7
nm              6
10^6/M          2
10^5/M          1
Name: count, dtype: int64

TARGET DISTRIBUTION
target_chembl_id
CHEMBL6152    12221
Name: c

In [ ]:
import pandas as pd
from pathlib import Path

# =========================================================
# 1. PATH
# =========================================================

DATA_DIR = Path(
    r"H:\My Drive\PROJECT SAINS DATA\DATA"
)

OUTPUT_DIR = DATA_DIR.parent / "OUTPUT"
CLEANED_DIR = OUTPUT_DIR / "01_Cleaned_Data"

activity_path = (
    DATA_DIR / "α-Synuclein" /
    "alpha_synuclein_activity_raw.csv"
)

review_path = (
    CLEANED_DIR /
    "alpha_inhibition_activity_assay_review_revised.csv"
)

# =========================================================
# 2. LOAD DATA
# =========================================================

activity = pd.read_csv(activity_path, low_memory=False)
review = pd.read_csv(review_path, low_memory=False)

print("ACTIVITY:", activity.shape)
print("ASSAY REVIEW:", review.shape)

# =========================================================
# 3. VALIDATE REQUIRED COLUMNS
# =========================================================

required_activity = [
    "activity_id",
    "assay_chembl_id",
    "standard_type",
    "standard_value",
    "standard_units",
    "standard_relation",
    "molecule_chembl_id"
]

required_review = [
    "assay_chembl_id",
    "reviewed_assay_category",
    "reviewed_assay_decision"
]

for col in required_activity:
    assert col in activity.columns, f"Kolom activity tidak ditemukan: {col}"

for col in required_review:
    assert col in review.columns, f"Kolom review tidak ditemukan: {col}"

# =========================================================
# 4. CHECK ASSAY ID UNIQUENESS
# =========================================================

print("\nASSAY REVIEW ID CHECK")

print(
    "Duplicate assay IDs:",
    review["assay_chembl_id"].duplicated().sum()
)

# =========================================================
# 5. MERGE REVIEW TO ACTIVITY
# =========================================================

review_columns = [
    "assay_chembl_id",
    "reviewed_assay_category",
    "reviewed_assay_decision"
]

activity_reviewed = activity.merge(
    review[review_columns],
    on="assay_chembl_id",
    how="left",
    validate="many_to_one",
    indicator=True
)

# =========================================================
# 6. MATCHING AUDIT
# =========================================================

matched = activity_reviewed[
    activity_reviewed["_merge"] == "both"
].copy()

unmatched = activity_reviewed[
    activity_reviewed["_merge"] == "left_only"
].copy()

print("\nMATCHING AUDIT")
print("Matched activity records:", len(matched))
print("Unmatched activity records:", len(unmatched))
print("Unique matched activity IDs:", matched["activity_id"].nunique())
print("Unique matched assays:", matched["assay_chembl_id"].nunique())

# =========================================================
# 7. ACTUAL RECORD COUNT PER ASSAY
# =========================================================

actual_counts = (
    matched.groupby("assay_chembl_id")
    .agg(
        actual_record_count=("activity_id", "nunique"),
        actual_unique_molecules=("molecule_chembl_id", "nunique")
    )
    .reset_index()
)

# =========================================================
# 8. COMPARE INVENTORY WITH RAW DATA
# =========================================================

comparison = review.merge(
    actual_counts,
    on="assay_chembl_id",
    how="left",
    validate="one_to_one"
)

comparison["actual_record_count"] = (
    comparison["actual_record_count"].fillna(0).astype(int)
)

comparison["actual_unique_molecules"] = (
    comparison["actual_unique_molecules"].fillna(0).astype(int)
)

comparison["record_count_difference"] = (
    comparison["actual_record_count"] -
    comparison["record_count"]
)

comparison["molecule_count_difference"] = (
    comparison["actual_unique_molecules"] -
    comparison["unique_molecules"]
)

print("\nINVENTORY RECONCILIATION")
print(
    comparison[
        [
            "record_count",
            "actual_record_count",
            "unique_molecules",
            "actual_unique_molecules"
        ]
    ].sum()
)

# =========================================================
# 9. DECISION DISTRIBUTION
# =========================================================

print("\nRECORDS BY REVIEW DECISION")

print(
    matched.groupby("reviewed_assay_decision")
    .agg(
        records=("activity_id", "nunique"),
        assays=("assay_chembl_id", "nunique"),
        molecules=("molecule_chembl_id", "nunique")
    )
)

print("\nRECORDS BY REVIEW CATEGORY")

print(
    matched.groupby("reviewed_assay_category")
    .agg(
        records=("activity_id", "nunique"),
        assays=("assay_chembl_id", "nunique"),
        molecules=("molecule_chembl_id", "nunique")
    )
    .sort_values("records", ascending=False)
)

# =========================================================
# 10. MISMATCH REPORT
# =========================================================

mismatches = comparison[
    (comparison["record_count_difference"] != 0) |
    (comparison["molecule_count_difference"] != 0)
].copy()

print("\nMISMATCH SUMMARY")
print("Assays with count mismatch:", len(mismatches))

print(
    mismatches[
        [
            "assay_chembl_id",
            "reviewed_assay_category",
            "record_count",
            "actual_record_count",
            "unique_molecules",
            "actual_unique_molecules",
            "record_count_difference",
            "molecule_count_difference"
        ]
    ].to_string(index=False)
)

# =========================================================
# 11. SAVE AUDIT OUTPUTS
# =========================================================

comparison.to_csv(
    CLEANED_DIR / "alpha_assay_activity_reconciliation.csv",
    index=False
)

matched.drop(columns=["_merge"]).to_csv(
    CLEANED_DIR / "alpha_activity_assay_review_merged.csv",
    index=False
)

print("\nAUDIT FILES SAVED.")

ACTIVITY: (12221, 47)
ASSAY REVIEW: (114, 19)

ASSAY REVIEW ID CHECK
Duplicate assay IDs: 0

MATCHING AUDIT
Matched activity records: 238
Unmatched activity records: 11983
Unique matched activity IDs: 238
Unique matched assays: 114

INVENTORY RECONCILIATION
record_count               234
actual_record_count        238
unique_molecules           234
actual_unique_molecules    234
dtype: int64

RECORDS BY REVIEW DECISION
                                  records  assays  molecules
reviewed_assay_decision                                     
Candidate_Separate_Endpoint           165      68         56
Exclude_from_Inhibition_Analysis        9       6          4
Pending_Review                          3       1          3
Separate_Analysis                      61      39         31

RECORDS BY REVIEW CATEGORY
                                  records  assays  molecules
reviewed_assay_category                                     
Aggregation_Inhibition_Candidate      111      42         33


In [ ]:
# =========================================================
# AUDIT CHEMBL5325295
# =========================================================

assay_id = "CHEMBL5325295"

# Ambil semua record dari raw activity
assay_records = activity[
    activity["assay_chembl_id"] == assay_id
].copy()

# Ambil baris inventaris review
assay_review = review[
    review["assay_chembl_id"] == assay_id
].copy()

print("=" * 70)
print("CHEMBL5325295 - DETAILED AUDIT")
print("=" * 70)

print("\nREVIEW INVENTORY")
print(
    assay_review[
        [
            "assay_chembl_id",
            "record_count",
            "unique_molecules",
            "reviewed_assay_category",
            "reviewed_assay_decision"
        ]
    ].to_string(index=False)
)

print("\nRAW ACTIVITY RECORDS")
print("Total records:", len(assay_records))
print("Unique activity IDs:", assay_records["activity_id"].nunique())
print("Unique molecules:", assay_records["molecule_chembl_id"].nunique())

# Tampilkan informasi penting setiap record
columns_to_show = [
    "activity_id",
    "molecule_chembl_id",
    "standard_type",
    "standard_value",
    "standard_units",
    "standard_relation",
    "type",
    "value",
    "units",
    "assay_description",
    "document_chembl_id"
]

print("\nDETAIL RECORDS")
print(
    assay_records[columns_to_show]
    .sort_values(["molecule_chembl_id", "activity_id"])
    .to_string(index=False)
)

print("\nSTANDARD TYPE DISTRIBUTION")
print(assay_records["standard_type"].value_counts(dropna=False))

print("\nSTANDARD UNITS DISTRIBUTION")
print(assay_records["standard_units"].value_counts(dropna=False))

print("\nRECORDS PER MOLECULE")
print(
    assay_records.groupby("molecule_chembl_id")
    .agg(
        record_count=("activity_id", "nunique"),
        activity_ids=("activity_id", lambda x: list(x)),
        standard_types=("standard_type", lambda x: list(x))
    )
    .to_string()
)

CHEMBL5325295 - DETAILED AUDIT

REVIEW INVENTORY
assay_chembl_id  record_count  unique_molecules       reviewed_assay_category reviewed_assay_decision
  CHEMBL5325295             4                 4 Protein_Degradation_Clearance       Separate_Analysis

RAW ACTIVITY RECORDS
Total records: 8
Unique activity IDs: 8
Unique molecules: 4

DETAIL RECORDS
 activity_id molecule_chembl_id standard_type  standard_value standard_units standard_relation     type  value units                                                                                                                                                                          assay_description document_chembl_id
    25472673      CHEMBL5412600      Activity             NaN            NaN               NaN Activity    NaN   NaN Induction of alpha-synuclein (unknown origin) degradation overexpressed in HEK293T cells infected with adenovirus expressing mCherry-EGFP-LC3B incubated for 24 hrs by immunoblot analysis      CHEMBL5321295
   

In [ ]:
# =========================================================
# UPDATE INVENTORY RECONCILIATION
# =========================================================

# Rekonsiliasi seluruh record aktual
actual_inventory = (
    activity.groupby("assay_chembl_id")
    .agg(
        total_raw_records=("activity_id", "nunique"),
        actual_unique_molecules=("molecule_chembl_id", "nunique")
    )
    .reset_index()
)

# Rekonsiliasi record kuantitatif
quantitative = activity[
    activity["standard_value"].notna()
    & activity["standard_units"].notna()
].copy()

quantitative_inventory = (
    quantitative.groupby("assay_chembl_id")
    .agg(
        quantitative_records=("activity_id", "nunique")
    )
    .reset_index()
)

# Rekonsiliasi record kualitatif Activity
qualitative = activity[
    activity["standard_type"].eq("Activity")
].copy()

qualitative_inventory = (
    qualitative.groupby("assay_chembl_id")
    .agg(
        qualitative_records=("activity_id", "nunique")
    )
    .reset_index()
)

# Gabungkan hasil
reconciliation_updated = (
    review
    .merge(
        actual_inventory,
        on="assay_chembl_id",
        how="left",
        validate="one_to_one"
    )
    .merge(
        quantitative_inventory,
        on="assay_chembl_id",
        how="left",
        validate="one_to_one"
    )
    .merge(
        qualitative_inventory,
        on="assay_chembl_id",
        how="left",
        validate="one_to_one"
    )
)

count_columns = [
    "total_raw_records",
    "actual_unique_molecules",
    "quantitative_records",
    "qualitative_records"
]

reconciliation_updated[count_columns] = (
    reconciliation_updated[count_columns].fillna(0).astype(int)
)

# Simpan
output_path = (
    CLEANED_DIR / "alpha_assay_activity_reconciliation_updated.csv"
)

reconciliation_updated.to_csv(output_path, index=False)

print("UPDATED RECONCILIATION SAVED")
print("File:", output_path)

print("\nCHEMBL5325295 CHECK")

print(
    reconciliation_updated[
        reconciliation_updated["assay_chembl_id"] == "CHEMBL5325295"
    ][
        [
            "assay_chembl_id",
            "reviewed_assay_category",
            "reviewed_assay_decision",
            "total_raw_records",
            "quantitative_records",
            "qualitative_records",
            "actual_unique_molecules"
        ]
    ].to_string(index=False)
)

UPDATED RECONCILIATION SAVED
File: H:\My Drive\PROJECT SAINS DATA\OUTPUT\01_Cleaned_Data\alpha_assay_activity_reconciliation_updated.csv

CHEMBL5325295 CHECK
assay_chembl_id       reviewed_assay_category reviewed_assay_decision  total_raw_records  quantitative_records  qualitative_records  actual_unique_molecules
  CHEMBL5325295 Protein_Degradation_Clearance       Separate_Analysis                  8                     4                    4                        4


In [ ]:
import pandas as pd
from pathlib import Path

# =====================================================
# 1. PATH
# =====================================================

ALPHA_CLEANED_DIR = CLEANED_DIR / "Alpha_Synuclein"

audit_path = CLEANED_DIR / "alpha_synuclein_binding_audit.csv"

provisional_files = [
    "alpha_kd_binding_provisional.csv",
    "alpha_ic50_depolymerization_provisional.csv",
    "alpha_ki_provisional.csv",
    "alpha_ic50_aggregation_provisional.csv",
    "alpha_ic50_binding_provisional.csv"
]

# =====================================================
# 2. VALIDATE FILES
# =====================================================

print("=" * 65)
print("FILE VALIDATION")
print("=" * 65)

assert audit_path.exists(), f"Audit file not found: {audit_path}"
assert ALPHA_CLEANED_DIR.exists(), "Alpha_Synuclein folder not found"

missing_files = [
    filename
    for filename in provisional_files
    if not (ALPHA_CLEANED_DIR / filename).exists()
]

if missing_files:
    raise FileNotFoundError(
        f"Missing provisional files: {missing_files}"
    )

print("Audit file: FOUND")
print("Provisional files: 5/5 FOUND")

# =====================================================
# 3. LOAD DATA
# =====================================================

audit = pd.read_csv(audit_path)

provisional_data = {
    filename: pd.read_csv(ALPHA_CLEANED_DIR / filename)
    for filename in provisional_files
}

# Validasi kolom ID
required_column = "activity_id"

assert required_column in audit.columns

for filename, df in provisional_data.items():
    assert required_column in df.columns, (
        f"activity_id missing in {filename}"
    )

# =====================================================
# 4. AUDIT DATASET
# =====================================================

print("\n" + "=" * 65)
print("DATASET RECORD AUDIT")
print("=" * 65)

print("Audit records:", len(audit))
print("Audit unique IDs:", audit["activity_id"].nunique())
print("Audit duplicated IDs:", audit["activity_id"].duplicated().sum())

provisional_id_frames = []

for filename, df in provisional_data.items():

    print(
        f"{filename}: {len(df)} records | "
        f"{df['activity_id'].nunique()} unique IDs"
    )

    temp = df[["activity_id"]].copy()
    temp["source_file"] = filename

    provisional_id_frames.append(temp)

provisional_ids = pd.concat(
    provisional_id_frames,
    ignore_index=True
)

# =====================================================
# 5. DUPLICATE CHECK
# =====================================================

print("\n" + "=" * 65)
print("PROVISIONAL DUPLICATE AUDIT")
print("=" * 65)

print("Total provisional rows:", len(provisional_ids))
print(
    "Unique provisional IDs:",
    provisional_ids["activity_id"].nunique()
)

duplicate_ids = provisional_ids[
    provisional_ids["activity_id"].duplicated(keep=False)
].sort_values("activity_id")

print("Duplicated IDs across files:", len(duplicate_ids))

if not duplicate_ids.empty:
    print("\nDUPLICATE ID DETAILS")
    print(duplicate_ids.to_string(index=False))
else:
    print("No duplicated activity IDs across provisional files.")

# =====================================================
# 6. ID-LEVEL RECONCILIATION
# =====================================================

audit_set = set(audit["activity_id"].dropna())
provisional_set = set(provisional_ids["activity_id"].dropna())

audit_only = audit_set - provisional_set
provisional_only = provisional_set - audit_set
overlap = audit_set & provisional_set

print("\n" + "=" * 65)
print("FINAL ID RECONCILIATION")
print("=" * 65)

print("Audit unique IDs:", len(audit_set))
print("Provisional unique IDs:", len(provisional_set))
print("Overlap:", len(overlap))
print("Audit only:", len(audit_only))
print("Provisional only:", len(provisional_only))

# =====================================================
# 7. SAVE UNMATCHED RECORDS
# =====================================================

audit_only_df = audit[
    audit["activity_id"].isin(audit_only)
].copy()

provisional_only_df = provisional_ids[
    provisional_ids["activity_id"].isin(provisional_only)
].copy()

audit_only_df.to_csv(
    ALPHA_CLEANED_DIR / "alpha_binding_audit_only_ids.csv",
    index=False
)

provisional_only_df.to_csv(
    ALPHA_CLEANED_DIR / "alpha_provisional_only_ids.csv",
    index=False
)

print("\nUNMATCHED ID FILES SAVED")
print("alpha_binding_audit_only_ids.csv")
print("alpha_provisional_only_ids.csv")

FILE VALIDATION
Audit file: FOUND
Provisional files: 5/5 FOUND

DATASET RECORD AUDIT
Audit records: 363
Audit unique IDs: 363
Audit duplicated IDs: 0
alpha_kd_binding_provisional.csv: 79 records | 79 unique IDs
alpha_ic50_depolymerization_provisional.csv: 4 records | 4 unique IDs
alpha_ki_provisional.csv: 88 records | 88 unique IDs
alpha_ic50_aggregation_provisional.csv: 109 records | 109 unique IDs
alpha_ic50_binding_provisional.csv: 19 records | 19 unique IDs

PROVISIONAL DUPLICATE AUDIT
Total provisional rows: 299
Unique provisional IDs: 299
Duplicated IDs across files: 0
No duplicated activity IDs across provisional files.

FINAL ID RECONCILIATION
Audit unique IDs: 363
Provisional unique IDs: 299
Overlap: 299
Audit only: 64
Provisional only: 0

UNMATCHED ID FILES SAVED
alpha_binding_audit_only_ids.csv
alpha_provisional_only_ids.csv


In [ ]:
# =====================================================
# INVESTIGASI 64 RECORD YANG BELUM MASUK PROVISIONAL
# =====================================================

unmatched = pd.read_csv(
    ALPHA_CLEANED_DIR / "alpha_binding_audit_only_ids.csv"
)

print("=" * 70)
print("AUDIT 64 UNMATCHED RECORDS")
print("=" * 70)

print("Total records:", len(unmatched))
print("Unique activity IDs:", unmatched["activity_id"].nunique())

# Kolom penting yang tersedia
preferred_columns = [
    "activity_id",
    "assay_chembl_id",
    "molecule_chembl_id",
    "standard_type",
    "standard_value",
    "standard_units",
    "standard_relation",
    "type",
    "value",
    "units",
    "assay_description",
    "document_chembl_id"
]

available_columns = [
    col for col in preferred_columns
    if col in unmatched.columns
]

print("\nDETAIL 64 RECORDS")
print(
    unmatched[available_columns]
    .sort_values(
        ["standard_type", "assay_chembl_id"],
        na_position="last"
    )
    .to_string(index=False)
)

# Distribusi endpoint
print("\n" + "=" * 70)
print("STANDARD TYPE DISTRIBUTION")
print("=" * 70)

print(
    unmatched["standard_type"]
    .value_counts(dropna=False)
)

# Distribusi satuan
print("\n" + "=" * 70)
print("STANDARD UNITS DISTRIBUTION")
print("=" * 70)

print(
    unmatched["standard_units"]
    .value_counts(dropna=False)
)

# Distribusi relation
print("\n" + "=" * 70)
print("STANDARD RELATION DISTRIBUTION")
print("=" * 70)

print(
    unmatched["standard_relation"]
    .value_counts(dropna=False)
)

# Distribusi assay
print("\n" + "=" * 70)
print("ASSAY DISTRIBUTION")
print("=" * 70)

print(
    unmatched["assay_chembl_id"]
    .value_counts(dropna=False)
)

# Pemeriksaan missing values
print("\n" + "=" * 70)
print("MISSING VALUE AUDIT")
print("=" * 70)

audit_columns = [
    col for col in [
        "standard_type",
        "standard_value",
        "standard_units",
        "standard_relation",
        "molecule_chembl_id",
        "assay_chembl_id"
    ]
    if col in unmatched.columns
]

print(unmatched[audit_columns].isna().sum())

AUDIT 64 UNMATCHED RECORDS
Total records: 64
Unique activity IDs: 64

DETAIL 64 RECORDS
 activity_id assay_chembl_id molecule_chembl_id standard_type  standard_value standard_units standard_relation type  value  units                                                                                                                                                                       assay_description document_chembl_id
    25888424   CHEMBL5518543      CHEMBL5567743          IC50         36000.0             nM                 ~ IC50   36.0     uM                                                              Inhibition of recombinant alpha-synuclein (unknown origin) fibrillogenesis measured after 24 hrs by ThT fluorescence assay      CHEMBL5518163
    25923053   CHEMBL5529075        CHEMBL32749          IC50         13000.0             nM                 = IC50   13.0     uM                                           Inhibition of wild type full length recombinant alpha-synuclein (unknown o

In [ ]:
# =====================================================
# METADATA REVIEW FOR 8 UNMATCHED ASSAYS
# =====================================================

metadata_path = (
    ALPHA_SYNUCLEIN_DIR /
    "alpha_synuclein_assay_metadata_raw.csv"
)

metadata = pd.read_csv(metadata_path)

unmatched = pd.read_csv(
    ALPHA_CLEANED_DIR /
    "alpha_binding_audit_only_ids.csv"
)

assay_ids = sorted(
    unmatched["assay_chembl_id"].dropna().unique()
)

print("ASSAY IDS:", len(assay_ids))

metadata_check = metadata[
    metadata["assay_chembl_id"].isin(assay_ids)
].copy()

print("\nMETADATA MATCHED:", len(metadata_check))
print(
    "UNMATCHED ASSAY IDS:",
    set(assay_ids) - set(metadata_check["assay_chembl_id"])
)

print("\nMETADATA COLUMNS:")
print(metadata_check.columns.tolist())

print("\n" + "=" * 70)
print("ASSAY METADATA DETAILS")
print("=" * 70)

print(metadata_check.to_string(index=False))

ASSAY IDS: 8

METADATA MATCHED: 8
UNMATCHED ASSAY IDS: set()

METADATA COLUMNS:
['aidx', 'assay_category', 'assay_cell_type', 'assay_chembl_id', 'assay_classifications', 'assay_group', 'assay_organism', 'assay_parameters', 'assay_strain', 'assay_subcellular_fraction', 'assay_tax_id', 'assay_test_type', 'assay_tissue', 'assay_type', 'assay_type_description', 'bao_format', 'bao_label', 'cell_chembl_id', 'confidence_description', 'confidence_score', 'description', 'document_chembl_id', 'relationship_description', 'relationship_type', 'src_assay_id', 'src_id', 'target_chembl_id', 'tissue_chembl_id', 'variant_sequence']

ASSAY METADATA DETAILS
   aidx assay_category assay_cell_type assay_chembl_id assay_classifications  assay_group assay_organism assay_parameters  assay_strain  assay_subcellular_fraction  assay_tax_id assay_test_type assay_tissue assay_type assay_type_description  bao_format             bao_label cell_chembl_id                confidence_description  confidence_score        

In [ ]:
# =====================================================
# MANUAL DECISION FOR 64 UNMATCHED RECORDS
# =====================================================

unmatched = pd.read_csv(
    ALPHA_CLEANED_DIR /
    "alpha_binding_audit_only_ids.csv"
)

assay_decisions = {
    "CHEMBL2065090": (
        "Binding_Affinity",
        "Separate_Analysis",
        "Ki binding affinity; not direct aggregation inhibition"
    ),

    "CHEMBL4628228": (
        "Binding_Affinity_Missing",
        "Quarantine_Missing_Value",
        "Kd records have no numerical values or units"
    ),

    "CHEMBL5040534": (
        "Aggregate_Binding_Censored",
        "Separate_Censored_Analysis",
        "Kd values are reported with relation >"
    ),

    "CHEMBL5241349": (
        "Unresolved_Kd_Inhibition",
        "Pending_Review",
        "Description mentions inhibition but endpoint is Kd"
    ),

    "CHEMBL5256056": (
        "Fibril_Binding",
        "Quarantine_Unit_Review",
        "Kd unit 10^5/M requires verification"
    ),

    "CHEMBL5391420": (
        "Binding_Affinity",
        "Separate_Analysis",
        "Competitive binding affinity; Ki endpoint"
    ),

    "CHEMBL5518543": (
        "Fibrillogenesis_Inhibition_Candidate",
        "Candidate_Separate_Endpoint",
        "IC50 assay directly describes inhibition of fibrillogenesis"
    ),

    "CHEMBL5529075": (
        "AlphaSyn_Inhibition_Candidate",
        "Candidate_Separate_Endpoint",
        "IC50 assay describes inhibition of full-length alpha-synuclein"
    )
}

unmatched["manual_assay_category"] = (
    unmatched["assay_chembl_id"]
    .map(lambda x: assay_decisions.get(x, (None, None, None))[0])
)

unmatched["manual_assay_decision"] = (
    unmatched["assay_chembl_id"]
    .map(lambda x: assay_decisions.get(x, (None, None, None))[1])
)

unmatched["review_note"] = (
    unmatched["assay_chembl_id"]
    .map(lambda x: assay_decisions.get(x, (None, None, None))[2])
)

unmatched["review_status"] = "Preliminary_Review"

# Validasi
assert unmatched["manual_assay_category"].notna().all()
assert unmatched["manual_assay_decision"].notna().all()

# Simpan sebagai file review baru
output_path = (
    ALPHA_CLEANED_DIR /
    "alpha_64_unmatched_manual_review.csv"
)

unmatched.to_csv(output_path, index=False)

print("SAVED:", output_path)

print("\nCATEGORY DISTRIBUTION")
print(unmatched["manual_assay_category"].value_counts())

print("\nDECISION DISTRIBUTION")
print(unmatched["manual_assay_decision"].value_counts())

print("\nRECORD COUNT:", len(unmatched))
print("UNIQUE ACTIVITY IDs:", unmatched["activity_id"].nunique())

SAVED: H:\My Drive\PROJECT SAINS DATA\OUTPUT\01_Cleaned_Data\Alpha_Synuclein\alpha_64_unmatched_manual_review.csv

CATEGORY DISTRIBUTION
manual_assay_category
Binding_Affinity_Missing                33
Binding_Affinity                        17
Aggregate_Binding_Censored               8
AlphaSyn_Inhibition_Candidate            3
Unresolved_Kd_Inhibition                 1
Fibril_Binding                           1
Fibrillogenesis_Inhibition_Candidate     1
Name: count, dtype: int64

DECISION DISTRIBUTION
manual_assay_decision
Quarantine_Missing_Value       33
Separate_Analysis              17
Separate_Censored_Analysis      8
Candidate_Separate_Endpoint     4
Pending_Review                  1
Quarantine_Unit_Review          1
Name: count, dtype: int64

RECORD COUNT: 64
UNIQUE ACTIVITY IDs: 64


In [ ]:
import requests
import pandas as pd

# =====================================================
# RETRIEVE CHEMBL DOCUMENT METADATA
# =====================================================

document_ids = [
    "CHEMBL5518163",
    "CHEMBL5528024"
]

base_url = "https://www.ebi.ac.uk/chembl/api/data/document"

documents = []

for doc_id in document_ids:

    url = f"{base_url}/{doc_id}.json"

    response = requests.get(url, timeout=30)
    response.raise_for_status()

    data = response.json()

    documents.append({
        "document_chembl_id": doc_id,
        "title": data.get("title"),
        "authors": data.get("authors"),
        "journal": data.get("journal"),
        "year": data.get("year"),
        "volume": data.get("volume"),
        "issue": data.get("issue"),
        "first_page": data.get("first_page"),
        "last_page": data.get("last_page"),
        "pubmed_id": data.get("pubmed_id"),
        "doi": data.get("doi"),
        "doc_type": data.get("doc_type")
    })

document_df = pd.DataFrame(documents)

print("PUBLICATION METADATA")
display(document_df.T)

# Simpan
output_path = (
    ALPHA_CLEANED_DIR /
    "alpha_candidate_publication_metadata.csv"
)

document_df.to_csv(output_path, index=False)

print("\nSAVED:", output_path)

PUBLICATION METADATA


,0,1
document_chembl_id,CHEMBL5518163,CHEMBL5528024
title,"Brazilin-7-acetate, a novel potential drug of Parkinson's disease, hinders the formation of α-synuclein fibril, mitigates cytotoxicity, and decreases oxidative stress.",Comprehensive and critical view on the anti-inflammatory and immunomodulatory role of natural phenolic antioxidants.
authors,"Cui Z, Guo FY, Li L, Lu F, Jin CH, Wang X, Liu F.","Pisoschi AM, Iordache F, Stanca L, Cimpeanu C, Furnaris F, Geicu OI, Bilteanu L, Serban AI."
journal,Eur J Med Chem,Eur J Med Chem
year,2024,2024
volume,264,265
issue,None,None
first_page,115965,116075
last_page,115965,116075
pubmed_id,38056304,38150963



SAVED: H:\My Drive\PROJECT SAINS DATA\OUTPUT\01_Cleaned_Data\Alpha_Synuclein\alpha_candidate_publication_metadata.csv


In [ ]:
# Ambil 64 record hasil review
review_df = pd.read_csv(
    ALPHA_CLEANED_DIR /
    "alpha_64_unmatched_manual_review.csv"
)

# Ambil hanya kandidat inhibisi
candidate_df = review_df[
    review_df["manual_assay_decision"]
    == "Candidate_Separate_Endpoint"
].copy()

columns_to_show = [
    "activity_id",
    "molecule_chembl_id",
    "assay_chembl_id",
    "standard_type",
    "standard_relation",
    "standard_value",
    "standard_units",
    "pchembl_value",
    "manual_assay_category",
    "review_note"
]

display(candidate_df[columns_to_show])

print("TOTAL CANDIDATES:", len(candidate_df))
print(
    "UNIQUE MOLECULES:",
    candidate_df["molecule_chembl_id"].nunique()
)

,activity_id,molecule_chembl_id,assay_chembl_id,standard_type,standard_relation,standard_value,standard_units,pchembl_value,manual_assay_category,review_note
60,25888424,CHEMBL5567743,CHEMBL5518543,IC50,~,36000.0,nM,NaN,Fibrillogenesis_Inhibition_Candidate,IC50 assay directly describes inhibition of fibrillogenesis
61,25923053,CHEMBL32749,CHEMBL5529075,IC50,=,13000.0,nM,4.89,AlphaSyn_Inhibition_Candidate,IC50 assay describes inhibition of full-length alpha-synuclein
62,25923054,CHEMBL5573471,CHEMBL5529075,IC50,=,50000.0,nM,4.30,AlphaSyn_Inhibition_Candidate,IC50 assay describes inhibition of full-length alpha-synuclein
63,25923055,CHEMBL320295,CHEMBL5529075,IC50,=,251000.0,nM,NaN,AlphaSyn_Inhibition_Candidate,IC50 assay describes inhibition of full-length alpha-synuclein


TOTAL CANDIDATES: 4
UNIQUE MOLECULES: 4


In [ ]:
# =====================================================
# PUBLICATION VALIDATION - ALPHA SYNuclein
# =====================================================

review_df = pd.read_csv(
    ALPHA_CLEANED_DIR /
    "alpha_64_unmatched_manual_review.csv"
)

publication_decisions = {
    "CHEMBL5518543": {
        "publication_review_status": "Publication_Supports_Assay_Context",
        "publication_review_note": (
            "Publication directly investigates alpha-synuclein "
            "fibril formation inhibition by brazilin-7-acetate."
        )
    },

    "CHEMBL5529075": {
        "publication_review_status": "Publication_Mismatch",
        "publication_review_note": (
            "Linked publication is a review on anti-inflammatory "
            "and immunomodulatory phenolic antioxidants. "
            "Direct support for this alpha-synuclein assay "
            "has not been established."
        )
    }
}

review_df["publication_review_status"] = (
    review_df["assay_chembl_id"]
    .map({
        k: v["publication_review_status"]
        for k, v in publication_decisions.items()
    })
    .fillna("Not_Checked")
)

review_df["publication_review_note"] = (
    review_df["assay_chembl_id"]
    .map({
        k: v["publication_review_note"]
        for k, v in publication_decisions.items()
    })
)

# Keputusan analisis setelah review publikasi
review_df["analysis_eligibility"] = (
    review_df["manual_assay_decision"]
)

review_df.loc[
    review_df["assay_chembl_id"] == "CHEMBL5529075",
    "analysis_eligibility"
] = "Pending_Publication_Verification"

output_path = (
    ALPHA_CLEANED_DIR /
    "alpha_64_unmatched_publication_reviewed.csv"
)

review_df.to_csv(output_path, index=False)

print("SAVED:", output_path)

print("\nPUBLICATION REVIEW STATUS")
print(review_df["publication_review_status"].value_counts())

print("\nANALYSIS ELIGIBILITY")
print(review_df["analysis_eligibility"].value_counts())

SAVED: H:\My Drive\PROJECT SAINS DATA\OUTPUT\01_Cleaned_Data\Alpha_Synuclein\alpha_64_unmatched_publication_reviewed.csv

PUBLICATION REVIEW STATUS
publication_review_status
Not_Checked                           60
Publication_Mismatch                   3
Publication_Supports_Assay_Context     1
Name: count, dtype: int64

ANALYSIS ELIGIBILITY
analysis_eligibility
Quarantine_Missing_Value            33
Separate_Analysis                   17
Separate_Censored_Analysis           8
Pending_Publication_Verification     3
Pending_Review                       1
Quarantine_Unit_Review               1
Candidate_Separate_Endpoint          1
Name: count, dtype: int64


In [ ]:

import pandas as pd
from pathlib import Path

# =========================================================
# 1. PATH
# =========================================================
ALPHA_DIR = Path(
    r"H:\My Drive\PROJECT SAINS DATA\DATA\α-Synuclein"
)

OUTPUT_DIR = Path(
    r"H:\My Drive\PROJECT SAINS DATA\OUTPUT\01_Cleaned_Data\Alpha_Synuclein"
)

reviewed_file = OUTPUT_DIR / "alpha_64_unmatched_publication_reviewed.csv"
raw_file = ALPHA_DIR / "alpha_synuclein_activity_raw.csv"
metadata_file = ALPHA_DIR / "alpha_synuclein_assay_metadata_raw.csv"

# =========================================================
# 2. LOAD DATA
# =========================================================
reviewed = pd.read_csv(reviewed_file)
raw = pd.read_csv(raw_file)
metadata = pd.read_csv(metadata_file)

TARGET_ASSAY = "CHEMBL5529075"

# =========================================================
# 3. AMBIL TIGA RECORD YANG BERMASALAH
# =========================================================
problem_records = reviewed[
    reviewed["assay_chembl_id"] == TARGET_ASSAY
].copy()

print("=" * 70)
print("A. RECORD DALAM PUBLICATION REVIEW")
print("=" * 70)

print("Jumlah record:", len(problem_records))
print("Unique activity ID:", problem_records["activity_id"].nunique())

display(problem_records.T)

# =========================================================
# 4. PERIKSA DATA RAW UNTUK ASSAY YANG SAMA
# =========================================================
raw_assay = raw[
    raw["assay_chembl_id"] == TARGET_ASSAY
].copy()

print("\n" + "=" * 70)
print("B. DATA RAW ASSAY")
print("=" * 70)

print("Jumlah record raw:", len(raw_assay))

raw_columns = [
    col for col in [
        "activity_id",
        "molecule_chembl_id",
        "assay_chembl_id",
        "document_chembl_id",
        "type",
        "relation",
        "value",
        "units",
        "standard_type",
        "standard_relation",
        "standard_value",
        "standard_units",
        "pchembl_value",
    ]
    if col in raw_assay.columns
]

display(raw_assay[raw_columns])

# =========================================================
# 5. PERIKSA METADATA ASSAY
# =========================================================
metadata_assay = metadata[
    metadata["assay_chembl_id"] == TARGET_ASSAY
].copy()

print("\n" + "=" * 70)
print("C. METADATA ASSAY")
print("=" * 70)

print("Jumlah baris metadata:", len(metadata_assay))

display(metadata_assay.T)

# =========================================================
# 6. BANDINGKAN DOCUMENT ID
# =========================================================
print("\n" + "=" * 70)
print("D. DOCUMENT ID COMPARISON")
print("=" * 70)

if "document_chembl_id" in raw_assay.columns:
    print("Document ID pada raw:")
    print(raw_assay["document_chembl_id"].value_counts(dropna=False))

if "document_chembl_id" in metadata_assay.columns:
    print("\nDocument ID pada metadata assay:")
    print(
        metadata_assay["document_chembl_id"].value_counts(
            dropna=False
        )
    )

print("\nAUDIT SELESAI — belum ada data yang diubah.")


A. RECORD DALAM PUBLICATION REVIEW
Jumlah record: 3
Unique activity ID: 3


,61,62,63
action_type,"{'action_type': 'INHIBITOR', 'description': 'Negatively effects (inhibits) the normal functioning of the protein e.g., prevention of enzymatic reaction or activation of downstream pathway', 'parent_type': 'NEGATIVE MODULATOR'}","{'action_type': 'INHIBITOR', 'description': 'Negatively effects (inhibits) the normal functioning of the protein e.g., prevention of enzymatic reaction or activation of downstream pathway', 'parent_type': 'NEGATIVE MODULATOR'}","{'action_type': 'INHIBITOR', 'description': 'Negatively effects (inhibits) the normal functioning of the protein e.g., prevention of enzymatic reaction or activation of downstream pathway', 'parent_type': 'NEGATIVE MODULATOR'}"
activity_comment,NaN,NaN,NaN
activity_id,25923053,25923054,25923055
activity_properties,[],[],[]
assay_chembl_id,CHEMBL5529075,CHEMBL5529075,CHEMBL5529075
assay_description,Inhibition of wild type full length recombinant alpha-synuclein (unknown origin) expressed in Escherichia coli by spectrophotometric analysis,Inhibition of wild type full length recombinant alpha-synuclein (unknown origin) expressed in Escherichia coli by spectrophotometric analysis,Inhibition of wild type full length recombinant alpha-synuclein (unknown origin) expressed in Escherichia coli by spectrophotometric analysis
assay_type,B,B,B
assay_variant_accession,NaN,NaN,NaN
assay_variant_mutation,NaN,NaN,NaN
bao_endpoint,BAO_0000190,BAO_0000190,BAO_0000190



B. DATA RAW ASSAY
Jumlah record raw: 3


,activity_id,molecule_chembl_id,assay_chembl_id,document_chembl_id,type,relation,value,units,standard_type,standard_relation,standard_value,standard_units,pchembl_value
11590,25923053,CHEMBL32749,CHEMBL5529075,CHEMBL5528024,IC50,=,13.0,uM,IC50,=,13000.0,nM,4.89
11591,25923054,CHEMBL5573471,CHEMBL5529075,CHEMBL5528024,IC50,=,50.0,uM,IC50,=,50000.0,nM,4.30
11592,25923055,CHEMBL320295,CHEMBL5529075,CHEMBL5528024,IC50,=,251.0,uM,IC50,=,251000.0,nM,NaN



C. METADATA ASSAY
Jumlah baris metadata: 1


,327
aidx,1654255
assay_category,NaN
assay_cell_type,NaN
assay_chembl_id,CHEMBL5529075
assay_classifications,[]
assay_group,NaN
assay_organism,Homo sapiens
assay_parameters,[]
assay_strain,NaN
assay_subcellular_fraction,NaN



D. DOCUMENT ID COMPARISON
Document ID pada raw:
document_chembl_id
CHEMBL5528024    3
Name: count, dtype: int64

Document ID pada metadata assay:
document_chembl_id
CHEMBL5528024    1
Name: count, dtype: int64

AUDIT SELESAI — belum ada data yang diubah.


In [ ]:

import requests
import pandas as pd
from pathlib import Path

BASE_URL = "https://www.ebi.ac.uk/chembl/api/data"

ASSAY_ID = "CHEMBL5529075"
DOCUMENT_ID = "CHEMBL5528024"

def fetch_chembl(endpoint):
    url = f"{BASE_URL}/{endpoint}.json"
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    return response.json()

# 1. Ambil metadata assay
assay_data = fetch_chembl(f"assay/{ASSAY_ID}")

# 2. Ambil metadata dokumen
document_data = fetch_chembl(f"document/{DOCUMENT_ID}")

# 3. Tampilkan informasi penting
assay_fields = [
    "assay_chembl_id",
    "description",
    "assay_type",
    "assay_type_description",
    "target_chembl_id",
    "document_chembl_id",
    "confidence_score",
    "relationship_type",
    "src_assay_id",
]

document_fields = [
    "document_chembl_id",
    "title",
    "authors",
    "journal",
    "year",
    "volume",
    "issue",
    "first_page",
    "last_page",
    "pubmed_id",
    "doi",
    "doc_type",
    "src_id",
]

print("=" * 70)
print("ASSAY METADATA FROM CHEMBL API")
print("=" * 70)

for field in assay_fields:
    print(f"{field}: {assay_data.get(field)}")

print("\n" + "=" * 70)
print("DOCUMENT METADATA FROM CHEMBL API")
print("=" * 70)

for field in document_fields:
    print(f"{field}: {document_data.get(field)}")

# 4. Simpan audit metadata tanpa mengubah dataset utama
audit_dir = Path(
    r"H:\My Drive\PROJECT SAINS DATA\OUTPUT\01_Cleaned_Data\Alpha_Synuclein"
)
audit_dir.mkdir(parents=True, exist_ok=True)

api_audit = pd.DataFrame([
    {"record_type": "assay", **assay_data},
    {"record_type": "document", **document_data}
])

audit_path = audit_dir / "alpha_chembl5529075_api_audit.csv"
api_audit.to_csv(audit_path, index=False)

print("\nAPI AUDIT SAVED:", audit_path)


ASSAY METADATA FROM CHEMBL API
assay_chembl_id: CHEMBL5529075
description: Inhibition of wild type full length recombinant alpha-synuclein (unknown origin) expressed in Escherichia coli by spectrophotometric analysis
assay_type: B
assay_type_description: Binding
target_chembl_id: CHEMBL6152
document_chembl_id: CHEMBL5528024
confidence_score: 9
relationship_type: D
src_assay_id: None

DOCUMENT METADATA FROM CHEMBL API
document_chembl_id: CHEMBL5528024
title: Comprehensive and critical view on the anti-inflammatory and immunomodulatory role of natural phenolic antioxidants.
authors: Pisoschi AM, Iordache F, Stanca L, Cimpeanu C, Furnaris F, Geicu OI, Bilteanu L, Serban AI.
journal: Eur J Med Chem
year: 2024
volume: 265
issue: None
first_page: 116075
last_page: 116075
pubmed_id: 38150963
doi: 10.1016/j.ejmech.2023.116075
doc_type: PUBLICATION
src_id: 1

API AUDIT SAVED: H:\My Drive\PROJECT SAINS DATA\OUTPUT\01_Cleaned_Data\Alpha_Synuclein\alpha_chembl5529075_api_audit.csv


In [ ]:

import pandas as pd
from pathlib import Path

OUTPUT_DIR = Path(
    r"H:\My Drive\PROJECT SAINS DATA\OUTPUT\01_Cleaned_Data\Alpha_Synuclein"
)

source_file = OUTPUT_DIR / "alpha_64_unmatched_publication_reviewed.csv"

df = pd.read_csv(source_file)

mask = df["assay_chembl_id"].eq("CHEMBL5529075")

# Catatan hasil verifikasi API
df.loc[mask, "provenance_audit_status"] = (
    "Confirmed_Source_Mismatch"
)

df.loc[mask, "provenance_audit_note"] = (
    "ChEMBL API confirms assay CHEMBL5529075 is linked "
    "to document CHEMBL5528024. The linked document is a "
    "review on natural phenolic antioxidants, not a primary "
    "alpha-synuclein experimental publication. Original "
    "experimental provenance remains unresolved."
)

df.loc[mask, "analysis_eligibility"] = (
    "Pending_Publication_Verification"
)

output_file = OUTPUT_DIR / (
    "alpha_64_unmatched_provenance_audited.csv"
)

df.to_csv(output_file, index=False)

print("SAVED:", output_file)
print("\nPROVENANCE AUDIT STATUS:")
print(df.loc[mask, "provenance_audit_status"].value_counts())

print("\nANALYSIS ELIGIBILITY:")
print(df.loc[mask, "analysis_eligibility"].value_counts())

print("\nTOTAL RECORDS:", len(df))
print("UNIQUE ACTIVITY IDs:", df["activity_id"].nunique())


SAVED: H:\My Drive\PROJECT SAINS DATA\OUTPUT\01_Cleaned_Data\Alpha_Synuclein\alpha_64_unmatched_provenance_audited.csv

PROVENANCE AUDIT STATUS:
provenance_audit_status
Confirmed_Source_Mismatch    3
Name: count, dtype: int64

ANALYSIS ELIGIBILITY:
analysis_eligibility
Pending_Publication_Verification    3
Name: count, dtype: int64

TOTAL RECORDS: 64
UNIQUE ACTIVITY IDs: 64


In [ ]:

from pathlib import Path
import pandas as pd

BASE_DIR = Path(
    r"H:\My Drive\PROJECT SAINS DATA\OUTPUT\01_Cleaned_Data"
)

ALPHA_DIR = BASE_DIR / "Alpha_Synuclein"

files_to_check = [
    "alpha_kd_binding_provisional.csv",
    "alpha_ic50_depolymerization_provisional.csv",
    "alpha_ki_provisional.csv",
    "alpha_ic50_aggregation_provisional.csv",
    "alpha_ic50_binding_provisional.csv",
    "alpha_64_unmatched_provenance_audited.csv",
]

print("=" * 75)
print("ALPHA-SYNUCLEIN FILE INVENTORY")
print("=" * 75)

inventory = []

for filename in files_to_check:
    path = ALPHA_DIR / filename

    if not path.exists():
        print("FILE NOT FOUND:", filename)
        continue

    data = pd.read_csv(path)

    inventory.append({
        "filename": filename,
        "records": len(data),
        "unique_activity_id": (
            data["activity_id"].nunique()
            if "activity_id" in data.columns else None
        ),
        "duplicate_activity_id": (
            data["activity_id"].duplicated().sum()
            if "activity_id" in data.columns else None
        ),
    })

inventory_df = pd.DataFrame(inventory)

display(inventory_df)

print("\nTOTAL RECORDS:", inventory_df["records"].sum())
print(
    "TOTAL UNIQUE IDs PER FILE:",
    inventory_df["unique_activity_id"].sum()
)

print("\nAUDIT SELESAI — belum ada data yang diubah.")


ALPHA-SYNUCLEIN FILE INVENTORY


,filename,records,unique_activity_id,duplicate_activity_id
0,alpha_kd_binding_provisional.csv,79,79,0
1,alpha_ic50_depolymerization_provisional.csv,4,4,0
2,alpha_ki_provisional.csv,88,88,0
3,alpha_ic50_aggregation_provisional.csv,109,109,0
4,alpha_ic50_binding_provisional.csv,19,19,0
5,alpha_64_unmatched_provenance_audited.csv,64,64,0



TOTAL RECORDS: 363
TOTAL UNIQUE IDs PER FILE: 363

AUDIT SELESAI — belum ada data yang diubah.


In [ ]:

from pathlib import Path
import pandas as pd

BASE_DIR = Path(
    r"H:\My Drive\PROJECT SAINS DATA\OUTPUT\01_Cleaned_Data"
)

ALPHA_DIR = BASE_DIR / "Alpha_Synuclein"

files = [
    "alpha_kd_binding_provisional.csv",
    "alpha_ic50_depolymerization_provisional.csv",
    "alpha_ki_provisional.csv",
    "alpha_ic50_aggregation_provisional.csv",
    "alpha_ic50_binding_provisional.csv",
    "alpha_64_unmatched_provenance_audited.csv",
]

# =========================================================
# 1. LOAD DAN GABUNGKAN ID
# =========================================================
id_frames = []

for filename in files:
    path = ALPHA_DIR / filename
    data = pd.read_csv(path)

    if "activity_id" not in data.columns:
        raise ValueError(f"Kolom activity_id tidak ditemukan: {filename}")

    temp = data[["activity_id"]].copy()
    temp["source_file"] = filename
    id_frames.append(temp)

all_ids = pd.concat(id_frames, ignore_index=True)

# =========================================================
# 2. CROSS-FILE OVERLAP
# =========================================================
id_source_count = (
    all_ids.groupby("activity_id")["source_file"]
    .nunique()
)

overlap_ids = id_source_count[id_source_count > 1]

unique_union_ids = all_ids["activity_id"].nunique()

print("=" * 70)
print("CROSS-FILE RECONCILIATION")
print("=" * 70)

print("Total rows dari seluruh file:", len(all_ids))
print("Unique activity IDs:", unique_union_ids)
print("ID yang muncul di lebih dari satu file:", len(overlap_ids))

# =========================================================
# 3. DETAIL OVERLAP
# =========================================================
if len(overlap_ids) > 0:
    overlap_detail = all_ids[
        all_ids["activity_id"].isin(overlap_ids.index)
    ].sort_values("activity_id")

    print("\nOVERLAPPING ACTIVITY IDs:")
    display(overlap_detail)
else:
    print("\nTidak ditemukan overlap antarfile.")

# =========================================================
# 4. BANDINKAN DENGAN RAW DATA
# =========================================================
RAW_DIR = Path(
    r"H:\My Drive\PROJECT SAINS DATA\DATA\α-Synuclein"
)

raw = pd.read_csv(
    RAW_DIR / "alpha_synuclein_activity_raw.csv"
)

raw_ids = set(raw["activity_id"].dropna())
classified_ids = set(all_ids["activity_id"].dropna())

raw_only = raw_ids - classified_ids
classified_only = classified_ids - raw_ids
matched = raw_ids & classified_ids

print("\n" + "=" * 70)
print("RAW DATA RECONCILIATION")
print("=" * 70)

print("Raw records:", len(raw))
print("Raw unique IDs:", len(raw_ids))
print("Matched IDs:", len(matched))
print("Raw-only IDs:", len(raw_only))
print("Classified-only IDs:", len(classified_only))

# =========================================================
# 5. SIMPAN ID YANG BELUM TERCAKUP
# =========================================================
raw_unmatched = raw[
    raw["activity_id"].isin(raw_only)
].copy()

output_file = ALPHA_DIR / "alpha_raw_unmatched_11858_review.csv"

raw_unmatched.to_csv(output_file, index=False)

print("\nRAW-ONLY DATA SAVED:", output_file)

print("\nAUDIT SELESAI — belum mengubah dataset sumber.")


CROSS-FILE RECONCILIATION
Total rows dari seluruh file: 363
Unique activity IDs: 363
ID yang muncul di lebih dari satu file: 0

Tidak ditemukan overlap antarfile.

RAW DATA RECONCILIATION
Raw records: 12221
Raw unique IDs: 12221
Matched IDs: 363
Raw-only IDs: 11858
Classified-only IDs: 0

RAW-ONLY DATA SAVED: H:\My Drive\PROJECT SAINS DATA\OUTPUT\01_Cleaned_Data\Alpha_Synuclein\alpha_raw_unmatched_11858_review.csv

AUDIT SELESAI — belum mengubah dataset sumber.


In [ ]:

import pandas as pd
from pathlib import Path

# =========================================================
# PATH
# =========================================================
ALPHA_DIR = Path(
    r"H:\My Drive\PROJECT SAINS DATA\OUTPUT\01_Cleaned_Data\Alpha_Synuclein"
)

DATA_DIR = Path(
    r"H:\My Drive\PROJECT SAINS DATA\DATA\α-Synuclein"
)

unmatched_file = ALPHA_DIR / "alpha_raw_unmatched_11858_review.csv"
metadata_file = DATA_DIR / "alpha_synuclein_assay_metadata_raw.csv"

OUTPUT_DIR = ALPHA_DIR / "Unmatched_Audit"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# =========================================================
# 1. LOAD DATA
# =========================================================
unmatched = pd.read_csv(unmatched_file)
metadata = pd.read_csv(metadata_file)

print("=" * 75)
print("UNMATCHED DATA OVERVIEW")
print("=" * 75)

print("Records:", len(unmatched))
print("Unique activity IDs:", unmatched["activity_id"].nunique())
print("Unique assays:", unmatched["assay_chembl_id"].nunique())
print("Unique molecules:", unmatched["molecule_chembl_id"].nunique())

# =========================================================
# 2. DISTRIBUTION STANDARD TYPE
# =========================================================
print("\n" + "=" * 75)
print("STANDARD TYPE DISTRIBUTION")
print("=" * 75)

type_dist = (
    unmatched["standard_type"]
    .value_counts(dropna=False)
    .rename_axis("standard_type")
    .reset_index(name="records")
)

display(type_dist)

# =========================================================
# 3. DISTRIBUTION STANDARD UNITS
# =========================================================
print("\n" + "=" * 75)
print("STANDARD UNITS DISTRIBUTION")
print("=" * 75)

units_dist = (
    unmatched["standard_units"]
    .value_counts(dropna=False)
    .rename_axis("standard_units")
    .reset_index(name="records")
)

display(units_dist)

# =========================================================
# 4. DISTRIBUTION RELATION
# =========================================================
print("\n" + "=" * 75)
print("STANDARD RELATION DISTRIBUTION")
print("=" * 75)

relation_dist = (
    unmatched["standard_relation"]
    .value_counts(dropna=False)
    .rename_axis("standard_relation")
    .reset_index(name="records")
)

display(relation_dist)

# =========================================================
# 5. ASSAY DISTRIBUTION
# =========================================================
print("\n" + "=" * 75)
print("ASSAY DISTRIBUTION")
print("=" * 75)

assay_summary = (
    unmatched.groupby("assay_chembl_id", dropna=False)
    .agg(
        records=("activity_id", "count"),
        unique_molecules=("molecule_chembl_id", "nunique"),
        standard_types=("standard_type",
                        lambda x: " | ".join(
                            sorted(x.dropna().astype(str).unique())
                        )),
        standard_units=("standard_units",
                        lambda x: " | ".join(
                            sorted(x.dropna().astype(str).unique())
                        )),
    )
    .reset_index()
    .sort_values("records", ascending=False)
)

print("Total unique assays:", len(assay_summary))
print("\nTOP 30 ASSAYS:")
display(assay_summary.head(30))

# =========================================================
# 6. JOIN ASSAY METADATA
# =========================================================
metadata_columns = [
    "assay_chembl_id",
    "description",
    "assay_type",
    "assay_type_description",
    "document_chembl_id",
    "confidence_score",
    "target_chembl_id",
]

available_columns = [
    col for col in metadata_columns if col in metadata.columns
]

metadata_subset = metadata[available_columns].copy()

metadata_subset = metadata_subset.drop_duplicates(
    subset=["assay_chembl_id"]
)

assay_detail = assay_summary.merge(
    metadata_subset,
    on="assay_chembl_id",
    how="left",
    indicator=True
)

print("\n" + "=" * 75)
print("ASSAY METADATA COVERAGE")
print("=" * 75)

print(assay_detail["_merge"].value_counts())

print("\nASSAY TANPA METADATA:")
display(
    assay_detail[
        assay_detail["_merge"] == "left_only"
    ].head(30)
)

# =========================================================
# 7. SIMPAN LAPORAN
# =========================================================
type_dist.to_csv(
    OUTPUT_DIR / "unmatched_standard_type_distribution.csv",
    index=False
)

units_dist.to_csv(
    OUTPUT_DIR / "unmatched_standard_units_distribution.csv",
    index=False
)

relation_dist.to_csv(
    OUTPUT_DIR / "unmatched_relation_distribution.csv",
    index=False
)

assay_summary.to_csv(
    OUTPUT_DIR / "unmatched_assay_summary.csv",
    index=False
)

assay_detail.to_csv(
    OUTPUT_DIR / "unmatched_assay_metadata_review.csv",
    index=False
)

print("\n" + "=" * 75)
print("AUDIT REPORT SAVED")
print("=" * 75)

print("Folder:", OUTPUT_DIR)
print("1. unmatched_standard_type_distribution.csv")
print("2. unmatched_standard_units_distribution.csv")
print("3. unmatched_relation_distribution.csv")
print("4. unmatched_assay_summary.csv")
print("5. unmatched_assay_metadata_review.csv")

print("\nAUDIT SELESAI — tidak ada data sumber yang diubah.")


UNMATCHED DATA OVERVIEW
Records: 11858
Unique activity IDs: 11858
Unique assays: 379
Unique molecules: 10858

STANDARD TYPE DISTRIBUTION


,standard_type,records
0,Potency,10056
1,Inhibition,770
2,Activity,510
3,EC50,92
4,kon,84
5,k_off,84
6,Flu intensity,59
7,IC50,56
8,Ki,40
9,TIME,23



STANDARD UNITS DISTRIBUTION


,standard_units,records
0,nM,10267
1,%,1059
2,NaN,328
3,s-1,84
4,RU,47
5,hr,41
6,10^-3/hr,9
7,a.u.,8
8,/hr,7
9,nm,6



STANDARD RELATION DISTRIBUTION


,standard_relation,records
0,NaN,10553
1,=,1282
2,>,10
3,>=,6
4,~,4
5,<,3



ASSAY DISTRIBUTION
Total unique assays: 379

TOP 30 ASSAYS:


,assay_chembl_id,records,unique_molecules,standard_types,standard_units
20,CHEMBL2354282,10019,9955,Potency,nM
24,CHEMBL3888008,90,89,EC50,nM
358,CHEMBL5736665,75,25,IC50 | k_off | kon,nM | s-1
276,CHEMBL5524380,68,68,Activity,%
356,CHEMBL5736378,57,11,Kd | k_off | kon,nM | s-1
57,CHEMBL4625544,47,47,Inhibition,%
61,CHEMBL4628227,47,47,Activity,RU
357,CHEMBL5736397,42,14,IC50 | k_off | kon,nM | s-1
283,CHEMBL5551997,41,41,Activity,%
162,CHEMBL5347686,40,40,Inhibition,%



ASSAY METADATA COVERAGE
_merge
both          379
left_only       0
right_only      0
Name: count, dtype: int64

ASSAY TANPA METADATA:


,assay_chembl_id,records,unique_molecules,standard_types,standard_units,description,assay_type,assay_type_description,document_chembl_id,confidence_score,target_chembl_id,_merge



AUDIT REPORT SAVED
Folder: H:\My Drive\PROJECT SAINS DATA\OUTPUT\01_Cleaned_Data\Alpha_Synuclein\Unmatched_Audit
1. unmatched_standard_type_distribution.csv
2. unmatched_standard_units_distribution.csv
3. unmatched_relation_distribution.csv
4. unmatched_assay_summary.csv
5. unmatched_assay_metadata_review.csv

AUDIT SELESAI — tidak ada data sumber yang diubah.


In [ ]:

import pandas as pd
from pathlib import Path

ALPHA_DIR = Path(
    r"H:\My Drive\PROJECT SAINS DATA\OUTPUT\01_Cleaned_Data\Alpha_Synuclein"
)

DATA_DIR = Path(
    r"H:\My Drive\PROJECT SAINS DATA\DATA\α-Synuclein"
)

assay_id = "CHEMBL2354282"

# Load data
unmatched = pd.read_csv(
    ALPHA_DIR / "alpha_raw_unmatched_11858_review.csv"
)

metadata = pd.read_csv(
    DATA_DIR / "alpha_synuclein_assay_metadata_raw.csv"
)

# =====================================================
# A. ASSAY METADATA
# =====================================================
assay_meta = metadata[
    metadata["assay_chembl_id"] == assay_id
]

print("=" * 75)
print("A. ASSAY METADATA")
print("=" * 75)

if len(assay_meta) == 0:
    print("Metadata tidak ditemukan.")
else:
    display(assay_meta.T)

# =====================================================
# B. DISTRIBUSI STANDARD TYPE DAN NILAI
# =====================================================
assay_data = unmatched[
    unmatched["assay_chembl_id"] == assay_id
].copy()

print("\n" + "=" * 75)
print("B. ACTIVITY RECORD OVERVIEW")
print("=" * 75)

print("Records:", len(assay_data))
print("Unique molecules:", assay_data["molecule_chembl_id"].nunique())

columns = [
    "activity_id",
    "molecule_chembl_id",
    "document_chembl_id",
    "type",
    "value",
    "units",
    "standard_type",
    "standard_relation",
    "standard_value",
    "standard_units",
    "pchembl_value",
]

columns = [c for c in columns if c in assay_data.columns]

display(assay_data[columns].head(15))

# =====================================================
# C. NILAI DAN RELATION
# =====================================================
print("\nSTANDARD RELATION:")
print(assay_data["standard_relation"].value_counts(dropna=False))

print("\nSTANDARD TYPE:")
print(assay_data["standard_type"].value_counts(dropna=False))

print("\nSTANDARD UNITS:")
print(assay_data["standard_units"].value_counts(dropna=False))

# =====================================================
# D. DOCUMENT DISTRIBUTION
# =====================================================
print("\n" + "=" * 75)
print("D. LINKED DOCUMENTS")
print("=" * 75)

print(
    assay_data["document_chembl_id"]
    .value_counts(dropna=False)
)

# =====================================================
# E. ASSAY DESCRIPTION
# =====================================================
if len(assay_meta) > 0:
    print("\nASSAY DESCRIPTION:")
    print(assay_meta.iloc[0].get("description"))

    print("\nASSAY TYPE:")
    print(assay_meta.iloc[0].get("assay_type_description"))

    print("\nTARGET:")
    print(assay_meta.iloc[0].get("target_chembl_id"))

    print("\nCONFIDENCE:")
    print(assay_meta.iloc[0].get("confidence_score"))

print("\nAUDIT SELESAI — tidak ada data yang diubah.")


A. ASSAY METADATA


,22
aidx,CLD0
assay_category,confirmatory
assay_cell_type,NaN
assay_chembl_id,CHEMBL2354282
assay_classifications,[]
assay_group,NaN
assay_organism,NaN
assay_parameters,[]
assay_strain,NaN
assay_subcellular_fraction,NaN



B. ACTIVITY RECORD OVERVIEW
Records: 10019
Unique molecules: 9955


,activity_id,molecule_chembl_id,document_chembl_id,type,value,units,standard_type,standard_relation,standard_value,standard_units,pchembl_value
78,13136034,CHEMBL1416096,CHEMBL1201862,Potency,1.5849,uM,Potency,NaN,1584.9,nM,NaN
79,13136035,CHEMBL1699082,CHEMBL1201862,Potency,35.4813,uM,Potency,NaN,35481.3,nM,NaN
80,13136036,CHEMBL1543347,CHEMBL1201862,Potency,14.1254,uM,Potency,NaN,14125.4,nM,NaN
81,13136037,CHEMBL1425843,CHEMBL1201862,Potency,35.4813,uM,Potency,NaN,35481.3,nM,NaN
82,13136038,CHEMBL1699039,CHEMBL1201862,Potency,15.8489,uM,Potency,NaN,15848.9,nM,NaN
83,13136039,CHEMBL1555805,CHEMBL1201862,Potency,35.4813,uM,Potency,NaN,35481.3,nM,NaN
84,13136040,CHEMBL1885038,CHEMBL1201862,Potency,25.1189,uM,Potency,NaN,25118.9,nM,NaN
85,13136041,CHEMBL1521536,CHEMBL1201862,Potency,12.5893,uM,Potency,NaN,12589.3,nM,NaN
86,13136042,CHEMBL3199539,CHEMBL1201862,Potency,17.7828,uM,Potency,NaN,17782.8,nM,NaN
87,13136043,CHEMBL5483017,CHEMBL1201862,Potency,10.0000,uM,Potency,NaN,10000.0,nM,NaN



STANDARD RELATION:
standard_relation
NaN    10019
Name: count, dtype: int64

STANDARD TYPE:
standard_type
Potency    10019
Name: count, dtype: int64

STANDARD UNITS:
standard_units
nM    10019
Name: count, dtype: int64

D. LINKED DOCUMENTS
document_chembl_id
CHEMBL1201862    10019
Name: count, dtype: int64

ASSAY DESCRIPTION:
PubChem BioAssay. qHTS of alpha-syn Inhibitors. (Class of assay: confirmatory) 

ASSAY TYPE:
Functional

TARGET:
CHEMBL6152

CONFIDENCE:
8

AUDIT SELESAI — tidak ada data yang diubah.


In [ ]:
from pathlib import Path
import pandas as pd

FILE = Path(
    r"H:\My Drive\PROJECT SAINS DATA\OUTPUT\01_Cleaned_Data\Alpha_Synuclein\alpha_raw_unmatched_11858_review.csv"
)

df = pd.read_csv(FILE)

print("FILE BERHASIL DIBACA")
print("Jumlah record:", len(df))
print("Jumlah kolom:", len(df.columns))

FILE BERHASIL DIBACA
Jumlah record: 11858
Jumlah kolom: 47


In [ ]:
assay = df[
    df["assay_chembl_id"] == "CHEMBL2354282"
].copy()

print("=" * 70)
print("CHEMBL2354282 — ACTIVITY AUDIT")
print("=" * 70)

print("Total records:", len(assay))
print("Unique activity:", assay["activity_id"].nunique())
print("Unique molecules:", assay["molecule_chembl_id"].nunique())

columns_to_check = [
    "activity_id",
    "molecule_chembl_id",
    "value",
    "units",
    "standard_type",
    "standard_value",
    "standard_units",
    "standard_relation",
    "pchembl_value",
    "activity_comment",
    "data_validity_comment",
    "document_chembl_id"
]

available = [
    col for col in columns_to_check
    if col in assay.columns
]

print("\nSAMPLE RECORDS")
display(assay[available].head(20))

for col in [
    "activity_comment",
    "data_validity_comment",
    "pchembl_value"
]:
    if col in assay.columns:
        print(f"\nDISTRIBUSI {col}")
        print(assay[col].value_counts(dropna=False).head(20))

print("\nSTATISTIK STANDARD VALUE")
if "standard_value" in assay.columns:
    display(assay["standard_value"].describe())

CHEMBL2354282 — ACTIVITY AUDIT
Total records: 10019
Unique activity: 10019
Unique molecules: 9955

SAMPLE RECORDS


,activity_id,molecule_chembl_id,value,units,standard_type,standard_value,standard_units,standard_relation,pchembl_value,activity_comment,data_validity_comment,document_chembl_id
78,13136034,CHEMBL1416096,1.5849,uM,Potency,1584.9,nM,NaN,NaN,active,NaN,CHEMBL1201862
79,13136035,CHEMBL1699082,35.4813,uM,Potency,35481.3,nM,NaN,NaN,inactive,NaN,CHEMBL1201862
80,13136036,CHEMBL1543347,14.1254,uM,Potency,14125.4,nM,NaN,NaN,inconclusive,NaN,CHEMBL1201862
81,13136037,CHEMBL1425843,35.4813,uM,Potency,35481.3,nM,NaN,NaN,inactive,NaN,CHEMBL1201862
82,13136038,CHEMBL1699039,15.8489,uM,Potency,15848.9,nM,NaN,NaN,inactive,NaN,CHEMBL1201862
83,13136039,CHEMBL1555805,35.4813,uM,Potency,35481.3,nM,NaN,NaN,inactive,NaN,CHEMBL1201862
84,13136040,CHEMBL1885038,25.1189,uM,Potency,25118.9,nM,NaN,NaN,inconclusive,NaN,CHEMBL1201862
85,13136041,CHEMBL1521536,12.5893,uM,Potency,12589.3,nM,NaN,NaN,inconclusive,NaN,CHEMBL1201862
86,13136042,CHEMBL3199539,17.7828,uM,Potency,17782.8,nM,NaN,NaN,active,NaN,CHEMBL1201862
87,13136043,CHEMBL5483017,10.0000,uM,Potency,10000.0,nM,NaN,NaN,inactive,NaN,CHEMBL1201862



DISTRIBUSI activity_comment
activity_comment
inconclusive    5687
inactive        3829
active           503
Name: count, dtype: int64

DISTRIBUSI data_validity_comment
data_validity_comment
NaN    10019
Name: count, dtype: int64

DISTRIBUSI pchembl_value
pchembl_value
NaN    10019
Name: count, dtype: int64

STATISTIK STANDARD VALUE


count    10019.000000
mean     19293.595119
std      11894.624262
min         20.000000
25%       7943.300000
50%      17782.800000
75%      31622.800000
max      56234.100000
Name: standard_value, dtype: float64

In [ ]:

import requests
import json
from pathlib import Path

AID = 652106

url = (
    f"https://pubchem.ncbi.nlm.nih.gov/"
    f"rest/pug/assay/aid/{AID}/description/JSON"
)

response = requests.get(url, timeout=60)

print("HTTP STATUS:", response.status_code)
response.raise_for_status()

assay_json = response.json()

# Simpan metadata mentah untuk dokumentasi
output_file = Path("results") / "alpha_AID652106_description.json"
output_file.parent.mkdir(parents=True, exist_ok=True)

with open(output_file, "w", encoding="utf-8") as f:
    json.dump(assay_json, f, indent=2, ensure_ascii=False)

print("Metadata berhasil disimpan:", output_file)
print(json.dumps(assay_json, indent=2, ensure_ascii=False)[:12000])



import requests
from pathlib import Path

AID = 652106

url = (
    f"https://pubchem.ncbi.nlm.nih.gov/"
    f"rest/pug/assay/aid/{AID}/CSV"
)

response = requests.get(url, timeout=120)

print("HTTP STATUS:", response.status_code)
response.raise_for_status()

output_dir = Path("results/Alpha_Synuclein/AID_652106")
output_dir.mkdir(parents=True, exist_ok=True)

output_file = output_dir / "AID652106_raw.csv"

output_file.write_bytes(response.content)

print("File tersimpan:", output_file)
print("Ukuran file:", round(len(response.content) / 1024, 2), "KB")


HTTP STATUS: 200
Metadata berhasil disimpan: results\alpha_AID652106_description.json
{
  "PC_AssayContainer": [
    {
      "assay": {
        "descr": {
          "aid": {
            "id": 652106,
            "version": 1
          },
          "aid_source": {
            "db": {
              "name": "NCGC",
              "source_id": {
                "str": "ASYN100"
              }
            }
          },
          "name": "qHTS of alpha-syn Inhibitors",
          "description": [
            "The goal of this project is to identify new chemical probes capable of reversing the toxicity associated with the Parkinson disease-related protein, alpha-synuclein.  Compounds will be selected on their ability to restore growth in yeast cells expressing toxic levels of alpha-synuclein.  These compounds will provide a complementary approach to deciphering the tangled mechanisms of toxicity and protection that we have uncovered through genome-wide genetic screens and are beginning to unc

HTTPError: 400 Client Error: PUGREST.BadRequest for url: https://pubchem.ncbi.nlm.nih.gov/rest/pug/assay/aid/652106/CSV

In [ ]:

import requests
import pandas as pd
from pathlib import Path
import time

AID = 652106
BASE_URL = "https://pubchem.ncbi.nlm.nih.gov/rest/pug"

# 1. Ambil daftar SID yang diuji pada assay
sid_url = f"{BASE_URL}/assay/aid/{AID}/sids/JSON"

response = requests.get(sid_url, timeout=120)
print("SID STATUS:", response.status_code)
response.raise_for_status()

sid_json = response.json()

# Periksa struktur respons
print(sid_json.keys())
print(str(sid_json)[:1000])


SID STATUS: 200
dict_keys(['InformationList'])
{'InformationList': {'Information': [{'AID': 652106, 'SID': [14725468, 7973576, 4250579, 4255920, 22409097, 17433468, 56316201, 56317518, 46500515, 49816723, 57262717, 24805640, 57257376, 57261400, 4244067, 14731980, 7972405, 22410268, 17505754, 16953003, 57263888, 57267912, 24804469, 56317372, 85270284, 17386278, 24832272, 26659530, 49817894, 49821918, 49677273, 17406982, 14734468, 17510949, 22416780, 24783765, 24838784, 24803298, 56463480, 89854487, 14733297, 14740980, 17412764, 17504437, 24781277, 24805786, 51090059, 56318689, 849207, 14739660, 49642958, 49669593, 85198585, 124754126, 99358865, 7965896, 14742148, 14746172, 24797960, 24838781, 24839955, 99358862, 26666039, 49719268, 49669590, 89854490, 842695, 99356377, 7965893, 17512266, 24833443, 24824592, 24789106, 24797957, 842698, 103074179, 93577484, 49646401, 47204420, 26666042, 89855804, 57256208, 125306877, 22401417, 24830955, 24791445, 26728011, 49823235, 49731562, 57262720, 10

In [ ]:

import requests
import json

AID = 652106

url = (
    f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/"
    f"assay/aid/{AID}/sids/JSON"
)

response = requests.get(
    url,
    params={"list_return": "listkey"},
    timeout=120
)

print("STATUS:", response.status_code)
response.raise_for_status()

listkey_json = response.json()

print(json.dumps(
    listkey_json,
    indent=2,
    ensure_ascii=False
)[:5000])


STATUS: 200
{
  "IdentifierList": {
    "ListKey": "2347507316992141553",
    "Size": 370177,
    "EntrezDB": "pcsubstance",
    "EntrezWebEnv": "MCID_6b0a25e547c217e74f043ea0",
    "EntrezQueryKey": 5,
    "EntrezURL": "https://www.ncbi.nlm.nih.gov/sites/entrez?cmd=HistorySearch&WebEnvRq=1&hinit=true&db=pcsubstance&WebEnv=MCID_6b0a25e547c217e74f043ea0&query_key=5"
  }
}


In [ ]:

import requests
import pandas as pd
from io import StringIO
from pathlib import Path

AID = 652106
LISTKEY = "2347507316992141553"

BASE_URL = (
    f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/"
    f"assay/aid/{AID}/CSV"
)

params = {
    "sid": "listkey",
    "listkey": LISTKEY,
    "listkey_start": 0,
    "listkey_count": 1000
}

response = requests.get(
    BASE_URL,
    params=params,
    timeout=180
)

print("STATUS:", response.status_code)
print("CONTENT TYPE:", response.headers.get("Content-Type"))

response.raise_for_status()

batch_df = pd.read_csv(StringIO(response.text))

print("SHAPE:", batch_df.shape)
print("\nKOLOM:")
print(batch_df.columns.tolist())

display(batch_df.head())


STATUS: 200
CONTENT TYPE: text/csv
SHAPE: (1005, 44)

KOLOM:
['PUBCHEM_RESULT_TAG', 'PUBCHEM_SID', 'PUBCHEM_CID', 'PUBCHEM_EXT_DATASOURCE_SMILES', 'PUBCHEM_ACTIVITY_OUTCOME', 'PUBCHEM_ACTIVITY_SCORE', 'PUBCHEM_ACTIVITY_URL', 'PUBCHEM_ASSAYDATA_COMMENT', 'Phenotype', 'Potency', 'Efficacy', 'Analysis Comment', 'Activity_Score', 'Curve_Description', 'Fit_LogAC50', 'Fit_HillSlope', 'Fit_R2', 'Fit_InfiniteActivity', 'Fit_ZeroActivity', 'Fit_CurveClass', 'Excluded_Points', 'Max_Response', 'Activity at 0.00295 uM', 'Activity at 0.015 uM', 'Activity at 0.074 uM', 'Activity at 0.092 uM', 'Activity at 0.184 uM', 'Activity at 0.258 uM', 'Activity at 0.369 uM', 'Activity at 0.508 uM', 'Activity at 0.749 uM', 'Activity at 1.290 uM', 'Activity at 1.840 uM', 'Activity at 2.537 uM', 'Activity at 3.751 uM', 'Activity at 6.450 uM', 'Activity at 9.220 uM', 'Activity at 12.94 uM', 'Activity at 18.70 uM', 'Activity at 36.87 uM', 'Activity at 46.10 uM', 'Activity at 64.71 uM', 'Activity at 92.20 uM', 'Compo

,PUBCHEM_RESULT_TAG,PUBCHEM_SID,PUBCHEM_CID,PUBCHEM_EXT_DATASOURCE_SMILES,PUBCHEM_ACTIVITY_OUTCOME,PUBCHEM_ACTIVITY_SCORE,PUBCHEM_ACTIVITY_URL,PUBCHEM_ASSAYDATA_COMMENT,Phenotype,Potency,...,Activity at 3.751 uM,Activity at 6.450 uM,Activity at 9.220 uM,Activity at 12.94 uM,Activity at 18.70 uM,Activity at 36.87 uM,Activity at 46.10 uM,Activity at 64.71 uM,Activity at 92.20 uM,Compound QC
0,RESULT_TYPE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,STRING,FLOAT,...,FLOAT,FLOAT,FLOAT,FLOAT,FLOAT,FLOAT,FLOAT,FLOAT,FLOAT,STRING
1,RESULT_DESCR,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"Indicates type of activity observed: inhibitor, activator, fluorescent, cytotoxic, inactive, or inconclusive.","Concentration at which compound exhibits half-maximal efficacy, AC50. Extrapolated AC50s also include the highest efficacy observed and the concentration of compound at which it was observed.",...,% Activity at given concentration.,% Activity at given concentration.,% Activity at given concentration.,% Activity at given concentration.,% Activity at given concentration.,% Activity at given concentration.,% Activity at given concentration.,% Activity at given concentration.,% Activity at given concentration.,"NCGC designation for data stage: 'qHTS', 'qHTS Verification', 'Secondary Profiling'"
2,RESULT_UNIT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NONE,MICROMOLAR,...,PERCENT,PERCENT,PERCENT,PERCENT,PERCENT,PERCENT,PERCENT,PERCENT,PERCENT,NONE
3,RESULT_IS_ACTIVE_CONCENTRATION,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,TRUE,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,RESULT_ATTR_CONC_MICROMOL,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,3.75059,6.45,9.21976,12.9388,18.7029,36.8711,46.0969,64.715,92.2,NaN


In [ ]:

# Salin dataframe agar hasil asli tetap aman
batch_clean = batch_df.copy()

# Buang lima baris metadata
metadata_tags = [
    "RESULT_TYPE",
    "RESULT_DESCR",
    "RESULT_UNIT",
    "RESULT_IS_ACTIVE_CONCENTRATION",
    "RESULT_ATTR_CONC_MICROMOL"
]


batch_clean = batch_clean[
    ~batch_clean["PUBCHEM_RESULT_TAG"].isin(metadata_tags)
].copy()

# Reset index
batch_clean = batch_clean.reset_index(drop=True)

# Pastikan SID tersedia
batch_clean["PUBCHEM_SID"] = pd.to_numeric(
    batch_clean["PUBCHEM_SID"],
    errors="coerce"
)

batch_clean = batch_clean.dropna(
    subset=["PUBCHEM_SID"]
)

batch_clean["PUBCHEM_SID"] = (
    batch_clean["PUBCHEM_SID"].astype("int64")
)

print("HASIL CLEANING BATCH")
print("Jumlah record:", len(batch_clean))
print("SID unik:", batch_clean["PUBCHEM_SID"].nunique())
print("Duplikasi SID:", batch_clean["PUBCHEM_SID"].duplicated().sum())

display(
    batch_clean[
        [
            "PUBCHEM_SID",
            "PUBCHEM_CID",
            "PUBCHEM_ACTIVITY_OUTCOME",
            "Phenotype",
            "Potency",
            "Efficacy",
            "Curve_Description",
            "Fit_LogAC50",
            "Fit_R2"
        ]
    ].head(10)
)


HASIL CLEANING BATCH
Jumlah record: 1000
SID unik: 1000
Duplikasi SID: 0


,PUBCHEM_SID,PUBCHEM_CID,PUBCHEM_ACTIVITY_OUTCOME,Phenotype,Potency,Efficacy,Curve_Description,Fit_LogAC50,Fit_R2
0,125306346,53383630.0,Inactive,Inactive,NaN,NaN,NaN,NaN,NaN
1,125306347,53383631.0,Inactive,Inactive,NaN,NaN,NaN,-5.35,1
2,125306348,53383632.0,Inactive,Inactive,NaN,NaN,NaN,NaN,NaN
3,125306349,53383633.0,Inactive,Inactive,NaN,NaN,NaN,NaN,NaN
4,125306350,53383634.0,Inactive,Inactive,NaN,NaN,NaN,-5.2,0.4019
5,125306351,53383635.0,Inactive,Inactive,NaN,NaN,NaN,-5.2,0.7906
6,125306352,53383636.0,Inactive,Inactive,NaN,NaN,NaN,NaN,NaN
7,125306353,53383637.0,Inactive,Inactive,NaN,NaN,NaN,NaN,NaN
8,125306354,53383638.0,Inactive,Inactive,NaN,NaN,NaN,-4.45,0.5637
9,125306355,674370.0,Inactive,Inactive,NaN,NaN,NaN,-5.2,0.9991


In [ ]:
columns_to_check = [
    "PUBCHEM_ACTIVITY_OUTCOME",
    "Phenotype",
    "Curve_Description",
    "Compound QC"
]

for col in columns_to_check:
    print("\n" + "=" * 60)
    print(f"DISTRIBUSI: {col}")
    print("=" * 60)

    print(
        batch_clean[col]
        .value_counts(dropna=False)
        .to_string()
    )


DISTRIBUSI: PUBCHEM_ACTIVITY_OUTCOME
PUBCHEM_ACTIVITY_OUTCOME
Inactive        922
Inconclusive     67
Active           11

DISTRIBUSI: Phenotype
Phenotype
Inactive     902
Activator     78
Inhibitor     20

DISTRIBUSI: Curve_Description
Curve_Description
NaN                                          903
Single point of activity                      71
Partial curve; partial efficacy               18
Partial curve; high efficacy                   7
Partial curve; partial efficacy; poor fit      1

DISTRIBUSI: Compound QC
Compound QC
QC'd by Chris Hulme - Univ. of Arizona - MLI PSL                                 431
QC'd by Sergey Kozmin - Univ. of Chicago - MLI CMLD                              258
QC'd by Ohyun Kwon - Univ. of California; Los Angeles - MLI PSL                  147
QC'd by Vanderbilt Screening Center for GPCRs; Ion Channels; and Transporters     37
QC'd by Alexander Doemling - Univ. of Pittsburgh - MLI GCS                        35
QC'd by Jeffrey Johnson - Univ. of No

In [ ]:
# Hubungan antara PubChem Outcome dan Phenotype
print("CROSS-TAB OUTCOME vs PHENOTYPE")

display(
    pd.crosstab(
        batch_clean["PUBCHEM_ACTIVITY_OUTCOME"],
        batch_clean["Phenotype"],
        margins=True,
        dropna=False
    )
)

CROSS-TAB OUTCOME vs PHENOTYPE


Phenotype,Activator,Inactive,Inhibitor,All
PUBCHEM_ACTIVITY_OUTCOME,,,,
Active,11,0,0,11
Inactive,0,902,20,922
Inconclusive,67,0,0,67
All,78,902,20,1000


In [ ]:
inhibitor_batch = batch_clean[
    batch_clean["Phenotype"].astype(str).str.lower() == "inhibitor"
].copy()

print("JUMLAH PHENOTYPE INHIBITOR:", len(inhibitor_batch))

display(
    inhibitor_batch[
        [
            "PUBCHEM_SID",
            "PUBCHEM_CID",
            "PUBCHEM_ACTIVITY_OUTCOME",
            "Phenotype",
            "Potency",
            "Efficacy",
            "Curve_Description",
            "Fit_LogAC50",
            "Fit_HillSlope",
            "Fit_R2",
            "Compound QC"
        ]
    ]
)

JUMLAH PHENOTYPE INHIBITOR: 20


,PUBCHEM_SID,PUBCHEM_CID,PUBCHEM_ACTIVITY_OUTCOME,Phenotype,Potency,Efficacy,Curve_Description,Fit_LogAC50,Fit_HillSlope,Fit_R2,Compound QC
280,125306471,53383747.0,Inactive,Inhibitor,10,30.5485,Partial curve; partial efficacy,-5,3.5117,0.9893,QC'd by Chris Hulme - Univ. of Arizona - MLI PSL
369,125306726,53383999.0,Inactive,Inhibitor,22.3872,33.8225,Single point of activity,-4.65,3.6772,0.9661,QC'd by Chris Hulme - Univ. of Arizona - MLI PSL
487,125306316,53383607.0,Inactive,Inhibitor,3.1623,38.2901,Single point of activity,-5.5,3.6272,1,QC'd by Jeffrey Johnson - Univ. of North Carolina - MLI GCS
623,125306963,53384226.0,Inactive,Inhibitor,35.4813,77.4141,Single point of activity,-4.45,4.4495,0.9976,QC'd by Ohyun Kwon - Univ. of California; Los Angeles - MLI PSL
681,144203819,5154.0,Inactive,Inhibitor,35.4813,81.1237,Single point of activity,-4.45,4.9549,0.9889,QC'd by Prestwick Chemical; Inc.
788,125306570,53383843.0,Inactive,Inhibitor,3.1623,31.7836,Single point of activity,-5.5,4.5045,0.9999,QC'd by Sergey Kozmin - Univ. of Chicago - MLI CMLD
816,125306598,53383871.0,Inactive,Inhibitor,6.3096,29.324,Single point of activity,-5.2,4.9549,0.9643,QC'd by Sergey Kozmin - Univ. of Chicago - MLI CMLD
831,125306613,53383886.0,Inactive,Inhibitor,22.3872,50.1078,Single point of activity,-4.65,3.6772,0.9987,QC'd by Sergey Kozmin - Univ. of Chicago - MLI CMLD
841,125306623,53383896.0,Inactive,Inhibitor,35.4813,83.0988,Single point of activity,-4.45,4.9549,0.9792,QC'd by Sergey Kozmin - Univ. of Chicago - MLI CMLD
851,125306633,53383906.0,Inactive,Inhibitor,19.9526,32.2582,Partial curve; partial efficacy,-4.7,2.9523,0.9935,QC'd by Sergey Kozmin - Univ. of Chicago - MLI CMLD


In [ ]:
activator_batch = batch_clean[
    batch_clean["Phenotype"].astype(str).str.lower() == "activator"
].copy()

print("JUMLAH PHENOTYPE ACTIVATOR:", len(activator_batch))

print("\nDISTRIBUSI PUBCHEM OUTCOME:")
display(
    activator_batch["PUBCHEM_ACTIVITY_OUTCOME"]
    .value_counts(dropna=False)
)

print("\nDISTRIBUSI CURVE DESCRIPTION:")
display(
    activator_batch["Curve_Description"]
    .value_counts(dropna=False)
)

print("\nRINGKASAN POTENCY, EFFICACY, DAN FIT:")
display(
    activator_batch[
        ["Potency", "Efficacy", "Fit_LogAC50",
         "Fit_HillSlope", "Fit_R2"]
    ].describe()
)

JUMLAH PHENOTYPE ACTIVATOR: 78

DISTRIBUSI PUBCHEM OUTCOME:


PUBCHEM_ACTIVITY_OUTCOME
Inconclusive    67
Active          11
Name: count, dtype: int64


DISTRIBUSI CURVE DESCRIPTION:


Curve_Description
Single point of activity                     53
Partial curve; partial efficacy              16
Partial curve; high efficacy                  7
Partial curve; partial efficacy; poor fit     1
NaN                                           1
Name: count, dtype: int64


RINGKASAN POTENCY, EFFICACY, DAN FIT:


,Potency,Efficacy,Fit_LogAC50,Fit_HillSlope,Fit_R2
count,77,77,78,78,78
unique,20,77,20,18,40
top,5.6234,315.374,-5.25,4.9549,1
freq,13,1,13,26,12


In [ ]:
display(
    activator_batch[
        [
            "PUBCHEM_SID",
            "PUBCHEM_CID",
            "PUBCHEM_ACTIVITY_OUTCOME",
            "Phenotype",
            "Potency",
            "Efficacy",
            "Curve_Description",
            "Fit_LogAC50",
            "Fit_HillSlope",
            "Fit_R2",
            "Compound QC"
        ]
    ].sort_values(
        by=["PUBCHEM_ACTIVITY_OUTCOME", "Fit_R2"],
        ascending=[True, False]
    )
)

,PUBCHEM_SID,PUBCHEM_CID,PUBCHEM_ACTIVITY_OUTCOME,Phenotype,Potency,Efficacy,Curve_Description,Fit_LogAC50,Fit_HillSlope,Fit_R2,Compound QC
980,125306226,53383538.0,Active,Activator,11.2202,113.452,Partial curve; high efficacy,-4.95,1.1341,1,QC'd by Vanderbilt Screening Center for GPCRs; Ion Channels; and Transporters
975,125306221,53383533.0,Active,Activator,17.7828,376.817,Partial curve; high efficacy,-4.75,4.5045,0.9999,QC'd by Vanderbilt Screening Center for GPCRs; Ion Channels; and Transporters
199,125306185,53383512.0,Active,Activator,14.1254,128.584,Partial curve; high efficacy,-4.85,4.095,0.9997,QC'd by Chris Hulme - Univ. of Arizona - MLI PSL
107,125306089,53383416.0,Active,Activator,12.5893,67.1201,Partial curve; partial efficacy,-4.9,4.095,0.9996,QC'd by Chris Hulme - Univ. of Arizona - MLI PSL
466,125306923,53384189.0,Active,Activator,12.5893,112.881,Partial curve; high efficacy,-4.9,4.9549,0.9996,QC'd by Chris Hulme - Univ. of Arizona - MLI PSL
105,125306087,53383414.0,Active,Activator,7.0795,111.842,Partial curve; high efficacy,-5.15,1.8265,0.999,QC'd by Chris Hulme - Univ. of Arizona - MLI PSL
96,125306078,53383405.0,Active,Activator,8.9125,113.939,Partial curve; high efficacy,-5.05,2.3332,0.9986,QC'd by Chris Hulme - Univ. of Arizona - MLI PSL
106,125306088,53383415.0,Active,Activator,7.0795,84.0557,Partial curve; partial efficacy,-5.15,1.6266,0.9972,QC'd by Chris Hulme - Univ. of Arizona - MLI PSL
321,125306512,53383785.0,Active,Activator,15.8489,198.118,Partial curve; high efficacy,-4.8,3.2975,0.9965,QC'd by Chris Hulme - Univ. of Arizona - MLI PSL
73,125306055,53383382.0,Active,Activator,10,77.0859,Partial curve; partial efficacy,-5,2.9023,0.9962,QC'd by Chris Hulme - Univ. of Arizona - MLI PSL


In [ ]:
# Ringkasan berdasarkan PubChem Outcome

summary_activator = (
    activator_batch
    .groupby("PUBCHEM_ACTIVITY_OUTCOME")
    .agg(
        n=("PUBCHEM_SID", "count"),
        potency_mean=("Potency", "mean"),
        potency_median=("Potency", "median"),
        efficacy_median=("Efficacy", "median"),
        fit_r2_median=("Fit_R2", "median"),
        fit_r2_min=("Fit_R2", "min"),
        fit_r2_max=("Fit_R2", "max")
    )
    .reset_index()
)

display(summary_activator)

TypeError: dtype 'str' does not support operation 'mean'

In [ ]:
cols = [
    "PUBCHEM_SID",
    "Potency",
    "Efficacy",
    "Fit_R2",
    "Fit_HillSlope",
    "Fit_LogAC50"
]

display(activator_batch[cols].dtypes)
display(activator_batch[cols].head())
numeric_cols = [
    "PUBCHEM_SID",
    "Potency",
    "Efficacy",
    "Fit_R2",
    "Fit_HillSlope",
    "Fit_LogAC50"
]

activator_batch = activator_batch.copy()

for col in numeric_cols:
    activator_batch[col] = pd.to_numeric(
        activator_batch[col],
        errors="coerce"
    )
summary_activator = (
    activator_batch
    .groupby("PUBCHEM_ACTIVITY_OUTCOME")
    .agg(
        n=("PUBCHEM_SID", "count"),
        potency_mean=("Potency", "mean"),
        potency_median=("Potency", "median"),
        efficacy_median=("Efficacy", "median"),
        fit_r2_median=("Fit_R2", "median"),
        fit_r2_min=("Fit_R2", "min"),
        fit_r2_max=("Fit_R2", "max")
    )
    .reset_index()
)

display(summary_activator)

missing_activator = activator_batch[
    [
        "Potency",
        "Efficacy",
        "Curve_Description",
        "Fit_LogAC50",
        "Fit_HillSlope",
        "Fit_R2"
    ]
].isna().sum()

display(missing_activator)

PUBCHEM_SID        int64
Potency          float64
Efficacy         float64
Fit_R2           float64
Fit_HillSlope    float64
Fit_LogAC50      float64
dtype: object

,PUBCHEM_SID,Potency,Efficacy,Fit_R2,Fit_HillSlope,Fit_LogAC50
14,125306360,35.4813,315.3740,0.9992,4.4495,-4.45
22,125306368,31.6228,53.2648,0.9495,4.4495,-4.50
29,125306375,8.9125,31.0779,0.9983,2.4064,-5.05
39,125306195,31.6228,75.6055,0.9996,4.9549,-4.50
43,125306199,0.5012,29.0932,0.3953,4.9549,-6.30


,PUBCHEM_ACTIVITY_OUTCOME,n,potency_mean,potency_median,efficacy_median,fit_r2_median,fit_r2_min,fit_r2_max
0,Active,11,11.300627,11.22020,112.8810,0.9990,0.9947,1.0
1,Inconclusive,67,16.905189,14.98715,64.5029,0.9985,0.3953,1.0


Potency              1
Efficacy             1
Curve_Description    1
Fit_LogAC50          0
Fit_HillSlope        0
Fit_R2               0
dtype: int64

In [ ]:
import numpy as np

activator_batch = activator_batch.copy()

def classify_activator(row):
    outcome = row["PUBCHEM_ACTIVITY_OUTCOME"]
    r2 = row["Fit_R2"]
    curve = row["Curve_Description"]

    if outcome == "Active":
        return "Cellular_Rescue_Active_Provisional"

    elif outcome == "Inconclusive":
        if pd.notna(r2) and r2 < 0.5:
            return "Inconclusive_Poor_Fit"

        elif pd.isna(curve) or "Single point" in str(curve):
            return "Inconclusive_Single_Point"

        else:
            return "Inconclusive_Curve_Response"

    return "Other"

activator_batch["provisional_category"] = activator_batch.apply(
    classify_activator,
    axis=1
)

display(
    activator_batch["provisional_category"]
    .value_counts()
    .to_frame("record_count")
)

,record_count
provisional_category,
Inconclusive_Single_Point,54
Inconclusive_Curve_Response,12
Cellular_Rescue_Active_Provisional,11
Inconclusive_Poor_Fit,1


In [ ]:
review_cols = [
    "PUBCHEM_SID",
    "PUBCHEM_CID",
    "PUBCHEM_ACTIVITY_OUTCOME",
    "Phenotype",
    "Potency",
    "Efficacy",
    "Curve_Description",
    "Fit_LogAC50",
    "Fit_HillSlope",
    "Fit_R2",
    "provisional_category"
]

display(
    activator_batch[
        activator_batch["provisional_category"].isin([
            "Cellular_Rescue_Active_Provisional",
            "Inconclusive_Poor_Fit"
        ])
    ][review_cols]
)

,PUBCHEM_SID,PUBCHEM_CID,PUBCHEM_ACTIVITY_OUTCOME,Phenotype,Potency,Efficacy,Curve_Description,Fit_LogAC50,Fit_HillSlope,Fit_R2,provisional_category
43,125306199,53383525.0,Inconclusive,Activator,0.5012,29.0932,Partial curve; partial efficacy; poor fit,-6.30,4.9549,0.3953,Inconclusive_Poor_Fit
66,125306048,53383375.0,Active,Activator,7.0795,79.1328,Partial curve; partial efficacy,-5.15,1.8265,0.9947,Cellular_Rescue_Active_Provisional
73,125306055,53383382.0,Active,Activator,10.0000,77.0859,Partial curve; partial efficacy,-5.00,2.9023,0.9962,Cellular_Rescue_Active_Provisional
96,125306078,53383405.0,Active,Activator,8.9125,113.9390,Partial curve; high efficacy,-5.05,2.3332,0.9986,Cellular_Rescue_Active_Provisional
105,125306087,53383414.0,Active,Activator,7.0795,111.8420,Partial curve; high efficacy,-5.15,1.8265,0.9990,Cellular_Rescue_Active_Provisional
106,125306088,53383415.0,Active,Activator,7.0795,84.0557,Partial curve; partial efficacy,-5.15,1.6266,0.9972,Cellular_Rescue_Active_Provisional
107,125306089,53383416.0,Active,Activator,12.5893,67.1201,Partial curve; partial efficacy,-4.90,4.0950,0.9996,Cellular_Rescue_Active_Provisional
199,125306185,53383512.0,Active,Activator,14.1254,128.5840,Partial curve; high efficacy,-4.85,4.0950,0.9997,Cellular_Rescue_Active_Provisional
321,125306512,53383785.0,Active,Activator,15.8489,198.1180,Partial curve; high efficacy,-4.80,3.2975,0.9965,Cellular_Rescue_Active_Provisional
466,125306923,53384189.0,Active,Activator,12.5893,112.8810,Partial curve; high efficacy,-4.90,4.9549,0.9996,Cellular_Rescue_Active_Provisional


In [ ]:
active_provisional = activator_batch[
    activator_batch["provisional_category"]
    == "Cellular_Rescue_Active_Provisional"
].copy()

print("Jumlah record:", len(active_provisional))
print("Unique SID:", active_provisional["PUBCHEM_SID"].nunique())
print("Unique CID:", active_provisional["PUBCHEM_CID"].nunique())

print("\nCID duplikat:")
display(
    active_provisional[
        active_provisional["PUBCHEM_CID"].duplicated(keep=False)
    ].sort_values("PUBCHEM_CID")
)
print(
    "CID missing:",
    active_provisional["PUBCHEM_CID"].isna().sum()
)

Jumlah record: 11
Unique SID: 11
Unique CID: 11

CID duplikat:


,PUBCHEM_RESULT_TAG,PUBCHEM_SID,PUBCHEM_CID,PUBCHEM_EXT_DATASOURCE_SMILES,PUBCHEM_ACTIVITY_OUTCOME,PUBCHEM_ACTIVITY_SCORE,PUBCHEM_ACTIVITY_URL,PUBCHEM_ASSAYDATA_COMMENT,Phenotype,Potency,...,Activity at 6.450 uM,Activity at 9.220 uM,Activity at 12.94 uM,Activity at 18.70 uM,Activity at 36.87 uM,Activity at 46.10 uM,Activity at 64.71 uM,Activity at 92.20 uM,Compound QC,provisional_category


CID missing: 0


In [ ]:
from pathlib import Path

BATCH_OUTPUT_DIR = (
    ALPHA_CLEANED_DIR
    / "Unmatched_Audit"
    / "AID652106_Batch_001"
)

BATCH_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 1. Seluruh 1.000 record hasil batch setelah metadata rows dibuang
batch_clean.to_csv(
    BATCH_OUTPUT_DIR / "aid652106_batch001_all.csv",
    index=False
)

# 2. Seluruh 78 record dengan phenotype Activator
activator_batch.to_csv(
    BATCH_OUTPUT_DIR / "aid652106_batch001_activator_review.csv",
    index=False
)

# 3. Sebelas kandidat Active provisional
active_provisional.to_csv(
    BATCH_OUTPUT_DIR / "aid652106_batch001_active_provisional.csv",
    index=False
)

print("Files saved:")
for file in BATCH_OUTPUT_DIR.glob("*.csv"):
    print(file.name, "|", file.stat().st_size, "bytes")

Files saved:
aid652106_batch001_all.csv | 346314 bytes
aid652106_batch001_activator_review.csv | 34420 bytes
aid652106_batch001_active_provisional.csv | 5712 bytes


In [ ]:
# Read-back validation
saved_all = pd.read_csv(
    BATCH_OUTPUT_DIR / "aid652106_batch001_all.csv"
)

saved_activator = pd.read_csv(
    BATCH_OUTPUT_DIR / "aid652106_batch001_activator_review.csv"
)

saved_active = pd.read_csv(
    BATCH_OUTPUT_DIR / "aid652106_batch001_active_provisional.csv"
)

print("ALL:", saved_all.shape)
print("ACTIVATOR:", saved_activator.shape)
print("ACTIVE:", saved_active.shape)

print(
    "Unique SID all:",
    saved_all["PUBCHEM_SID"].nunique()
)

print(
    "Unique SID active:",
    saved_active["PUBCHEM_SID"].nunique()
)

ALL: (1000, 44)
ACTIVATOR: (78, 45)
ACTIVE: (11, 45)
Unique SID all: 1000
Unique SID active: 11


In [ ]:

import requests

AID = 652106

url = (
    f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/"
    f"assay/aid/{AID}/sids/JSON"
)

response = requests.get(
    url,
    params={"list_return": "listkey"},
    timeout=180
)

print("STATUS:", response.status_code)

response.raise_for_status()

result = response.json()

identifier_list = result["IdentifierList"]

NEW_LISTKEY = identifier_list["ListKey"]
NEW_TOTAL_SIDS = identifier_list["Size"]

print("NEW LISTKEY:", NEW_LISTKEY)
print("TOTAL SIDS:", NEW_TOTAL_SIDS)




STATUS: 200
NEW LISTKEY: 838465785031670392
TOTAL SIDS: 370177


In [ ]:

params = {
    "sid": "listkey",
    "listkey": NEW_LISTKEY,
    "listkey_start": 0,
    "listkey_count": 1000
}

test_response = requests.get(
    BASE_URL,
    params=params,
    timeout=180
)

print("STATUS:", test_response.status_code)
print(test_response.text[:1000])


STATUS: 200
PUBCHEM_RESULT_TAG,PUBCHEM_SID,PUBCHEM_CID,PUBCHEM_EXT_DATASOURCE_SMILES,PUBCHEM_ACTIVITY_OUTCOME,PUBCHEM_ACTIVITY_SCORE,PUBCHEM_ACTIVITY_URL,PUBCHEM_ASSAYDATA_COMMENT,Phenotype,Potency,Efficacy,Analysis Comment,Activity_Score,Curve_Description,Fit_LogAC50,Fit_HillSlope,Fit_R2,Fit_InfiniteActivity,Fit_ZeroActivity,Fit_CurveClass,Excluded_Points,Max_Response,Activity at 0.00295 uM,Activity at 0.015 uM,Activity at 0.074 uM,Activity at 0.092 uM,Activity at 0.184 uM,Activity at 0.258 uM,Activity at 0.369 uM,Activity at 0.508 uM,Activity at 0.749 uM,Activity at 1.290 uM,Activity at 1.840 uM,Activity at 2.537 uM,Activity at 3.751 uM,Activity at 6.450 uM,Activity at 9.220 uM,Activity at 12.94 uM,Activity at 18.70 uM,Activity at 36.87 uM,Activity at 46.10 uM,Activity at 64.71 uM,Activity at 92.20 uM,Compound QC
RESULT_TYPE,,,,,,,,STRING,FLOAT,FLOAT,STRING,INTEGER,STRING,FLOAT,FLOAT,FLOAT,FLOAT,FLOAT,FLOAT,STRING,FLOAT,FLOAT,FLOAT,FLOAT,FLOAT,FLOAT,FLOAT,FLOAT,FLOAT,FLOAT,FLOAT,FLOA

In [ ]:

from io import StringIO
import pandas as pd
import requests

# Ambil ulang 1.000 SID pertama dari ListKey baru
params = {
    "sid": "listkey",
    "listkey": NEW_LISTKEY,
    "listkey_start": 0,
    "listkey_count": 1000
}

response = requests.get(
    BASE_URL,
    params=params,
    timeout=180
)
response.raise_for_status()

new_df = pd.read_csv(
    StringIO(response.text),
    low_memory=False
)

# Hapus baris metadata PubChem
new_df = new_df[
    ~new_df["PUBCHEM_RESULT_TAG"].isin(METADATA_TAGS)
].copy()

new_df["PUBCHEM_SID"] = pd.to_numeric(
    new_df["PUBCHEM_SID"],
    errors="coerce"
)

new_df = new_df.dropna(subset=["PUBCHEM_SID"])
new_df["PUBCHEM_SID"] = new_df["PUBCHEM_SID"].astype("int64")

# Baca Batch 1 lama
old_df = pd.read_csv(batch001_file)

old_sids = set(old_df["PUBCHEM_SID"].astype(int))
new_sids = set(new_df["PUBCHEM_SID"].astype(int))

print("=" * 55)
print("BATCH 001 RECONCILIATION")
print("=" * 55)

print("Old Batch 1:", len(old_sids))
print("New ListKey first 1000:", len(new_sids))
print("Intersection:", len(old_sids & new_sids))
print("Old only:", len(old_sids - new_sids))
print("New only:", len(new_sids - old_sids))

if old_sids == new_sids:
    print("\nRESULT: MATCH - SAFE TO CONTINUE")
else:
    print("\nRESULT: MISMATCH - STOP AND REVIEW")


BATCH 001 RECONCILIATION
Old Batch 1: 1000
New ListKey first 1000: 1000
Intersection: 1000
Old only: 0
New only: 0

RESULT: MATCH - SAFE TO CONTINUE


In [ ]:

import requests
import pandas as pd
import time
from io import StringIO
from pathlib import Path

# =====================================================
# BATCH 002 - AID 652106
# =====================================================

AID = 652106
LISTKEY = "838465785031670392"

START = 1000
COUNT = 1000

BASE_URL = (
    f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/"
    f"assay/aid/{AID}/CSV"
)

METADATA_TAGS = [
    "RESULT_TYPE",
    "RESULT_DESCR",
    "RESULT_UNIT",
    "RESULT_IS_ACTIVE_CONCENTRATION",
    "RESULT_ATTR_CONC_MICROMOL"
]

# =====================================================
# 1. OUTPUT DIRECTORY
# =====================================================

BATCH_OUTPUT_DIR = (
    ALPHA_CLEANED_DIR
    / "Unmatched_Audit"
    / "AID652106_Batch_002"
)

BATCH_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

all_file = BATCH_OUTPUT_DIR / "aid652106_batch002_all.csv"
activator_file = BATCH_OUTPUT_DIR / "aid652106_batch002_activator_review.csv"
active_file = BATCH_OUTPUT_DIR / "aid652106_batch002_active_provisional.csv"

# Jangan menimpa hasil yang sudah ada
if any(p.exists() for p in [all_file, activator_file, active_file]):
    raise FileExistsError(
        "File Batch 002 sudah ada. Periksa dahulu sebelum menjalankan ulang."
    )

# =====================================================
# 2. REQUEST PUBCHEM
# =====================================================

params = {
    "sid": "listkey",
    "listkey": LISTKEY,
    "listkey_start": START,
    "listkey_count": COUNT
}

response = requests.get(
    BASE_URL,
    params=params,
    timeout=180
)

print("STATUS:", response.status_code)
response.raise_for_status()

# =====================================================
# 3. READ AND CLEAN
# =====================================================

batch_df = pd.read_csv(
    StringIO(response.text),
    low_memory=False
)

batch_clean = batch_df[
    ~batch_df["PUBCHEM_RESULT_TAG"].isin(METADATA_TAGS)
].copy()

batch_clean = batch_clean.reset_index(drop=True)

batch_clean["PUBCHEM_SID"] = pd.to_numeric(
    batch_clean["PUBCHEM_SID"],
    errors="coerce"
)

batch_clean = batch_clean.dropna(
    subset=["PUBCHEM_SID"]
)

batch_clean["PUBCHEM_SID"] = (
    batch_clean["PUBCHEM_SID"].astype("int64")
)

# =====================================================
# 4. VALIDATION
# =====================================================

if len(batch_clean) != COUNT:
    raise ValueError(
        f"Expected {COUNT}, got {len(batch_clean)}"
    )

if batch_clean["PUBCHEM_SID"].nunique() != len(batch_clean):
    raise ValueError("SID duplikat ditemukan dalam Batch 002!")

# Periksa agar tidak overlap dengan Batch 001
batch001_file = (
    ALPHA_CLEANED_DIR
    / "Unmatched_Audit"
    / "AID652106_Batch_001"
    / "aid652106_batch001_all.csv"
)

batch001 = pd.read_csv(batch001_file)

old_sids = set(
    batch001["PUBCHEM_SID"].astype(int)
)

new_sids = set(
    batch_clean["PUBCHEM_SID"].astype(int)
)

overlap = old_sids & new_sids

if overlap:
    raise ValueError(
        f"Ditemukan {len(overlap)} SID overlap dengan Batch 001!"
    )

# =====================================================
# 5. ACTIVATOR REVIEW
# =====================================================

activator_batch = batch_clean[
    batch_clean["Phenotype"] == "Activator"
].copy()

def classify_activator(row):

    outcome = row["PUBCHEM_ACTIVITY_OUTCOME"]

    r2 = pd.to_numeric(
        row.get("Fit_R2"),
        errors="coerce"
    )

    curve = str(row.get("Curve_Description", ""))

    if outcome == "Active":
        return "Cellular_Rescue_Active_Provisional"

    elif outcome == "Inconclusive":

        if pd.notna(r2) and r2 < 0.5:
            return "Inconclusive_Poor_Fit"

        elif pd.isna(row.get("Curve_Description")) or "Single point" in curve:
            return "Inconclusive_Single_Point"

        else:
            return "Inconclusive_Curve_Response"

    return "Other"


activator_batch["provisional_category"] = (
    activator_batch.apply(
        classify_activator,
        axis=1
    )
)

active_provisional = activator_batch[
    activator_batch["provisional_category"]
    == "Cellular_Rescue_Active_Provisional"
].copy()

# =====================================================
# 6. SAVE
# =====================================================

batch_clean.to_csv(all_file, index=False)
activator_batch.to_csv(activator_file, index=False)
active_provisional.to_csv(active_file, index=False)

# =====================================================
# 7. READ-BACK VALIDATION
# =====================================================

saved_all = pd.read_csv(all_file)
saved_activator = pd.read_csv(activator_file)
saved_active = pd.read_csv(active_file)

assert len(saved_all) == 1000
assert saved_all["PUBCHEM_SID"].nunique() == 1000
assert saved_active["PUBCHEM_SID"].nunique() == len(saved_active)

print("\n" + "=" * 55)
print("BATCH 002 COMPLETED")
print("=" * 55)

print("ALL:", saved_all.shape)
print("ACTIVATOR:", saved_activator.shape)
print("ACTIVE:", saved_active.shape)

print("Unique SID all:", saved_all["PUBCHEM_SID"].nunique())
print("Unique SID active:", saved_active["PUBCHEM_SID"].nunique())

print("Overlap with Batch 001:", len(overlap))

print("\nSAVED FILES:")
print(all_file)
print(activator_file)
print(active_file)

print("\nVALIDATION: PASSED")


STATUS: 200

BATCH 002 COMPLETED
ALL: (1000, 44)
ACTIVATOR: (61, 45)
ACTIVE: (5, 45)
Unique SID all: 1000
Unique SID active: 5
Overlap with Batch 001: 0

SAVED FILES:
H:\My Drive\PROJECT SAINS DATA\OUTPUT\01_Cleaned_Data\Alpha_Synuclein\Unmatched_Audit\AID652106_Batch_002\aid652106_batch002_all.csv
H:\My Drive\PROJECT SAINS DATA\OUTPUT\01_Cleaned_Data\Alpha_Synuclein\Unmatched_Audit\AID652106_Batch_002\aid652106_batch002_activator_review.csv
H:\My Drive\PROJECT SAINS DATA\OUTPUT\01_Cleaned_Data\Alpha_Synuclein\Unmatched_Audit\AID652106_Batch_002\aid652106_batch002_active_provisional.csv

VALIDATION: PASSED


In [ ]:

# ============================================================
# AUTOMATED PUBCHEM AID652106 PIPELINE
# Auricularia polytricha × Parkinson
# Batch 003 sampai batch terakhir
# ============================================================

import requests
import pandas as pd
import numpy as np
import json
import time
from io import StringIO
from pathlib import Path
from datetime import datetime

# ------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------

AID = 652106
LISTKEY = "838465785031670392"

TOTAL_SIDS = 370177
BATCH_SIZE = 1000

# Batch 001 dan 002 sudah selesai.
START_INDEX = 2000
START_BATCH = 3

BASE_URL = (
    f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/"
    f"assay/aid/{AID}/CSV"
)

# Gunakan folder output yang sama dengan Batch 001 dan 002.
ALPHA_CLEANED_DIR = Path(
    r"H:\My Drive\PROJECT SAINS DATA\OUTPUT"
    r"\01_Cleaned_Data\Alpha_Synuclein"
)

BATCH_ROOT = (
    ALPHA_CLEANED_DIR
    / "Unmatched_Audit"
)

BATCH_ROOT.mkdir(parents=True, exist_ok=True)

MANIFEST_PATH = BATCH_ROOT / "aid652106_pipeline_progress.json"
LOG_PATH = BATCH_ROOT / "aid652106_pipeline_log.csv"

print("=" * 65)
print("AUTOMATED AID652106 PIPELINE")
print("=" * 65)
print("Total SID       :", TOTAL_SIDS)
print("Batch size      :", BATCH_SIZE)
print("Starting index  :", START_INDEX)
print("Starting batch  :", START_BATCH)
print("Output          :", BATCH_ROOT)


# ------------------------------------------------------------
# 2. HELPER FUNCTIONS
# ------------------------------------------------------------

def get_sid_column(df):
    candidates = ["PUBCHEM_SID", "SID"]

    for col in candidates:
        if col in df.columns:
            return col

    raise ValueError(
        "Kolom SID tidak ditemukan. "
        f"Kolom tersedia: {df.columns.tolist()}"
    )


def read_pubchem_csv(response_text):

    df = pd.read_csv(
        StringIO(response_text),
        low_memory=False
    )

    sid_col = get_sid_column(df)

    # Menghapus baris metadata PubChem dan baris tanpa SID.
    df[sid_col] = pd.to_numeric(
        df[sid_col],
        errors="coerce"
    )

    df = df.loc[df[sid_col].notna()].copy()

    df[sid_col] = df[sid_col].astype("int64")

    return df, sid_col


def classify_activator(row):

    outcome = str(
        row.get("PUBCHEM_ACTIVITY_OUTCOME", "")
    ).strip().lower()

    if outcome == "active":
        return "Cellular_Rescue_Active_Provisional"

    if outcome == "inconclusive":

        fit_r2 = pd.to_numeric(
            row.get("Fit_R2"),
            errors="coerce"
        )

        if pd.notna(fit_r2) and fit_r2 < 0.5:
            return "Inconclusive_Poor_Fit"

        curve = row.get("Curve_Description")

        if pd.isna(curve):
            return "Inconclusive_Single_Point"

        curve = str(curve).strip().lower()

        if (
            curve == ""
            or "single" in curve
            or "point" in curve
        ):
            return "Inconclusive_Single_Point"

        return "Inconclusive_Curve_Response"

    return "Other_Outcome"


def download_batch(start, count):

    params = {
        "sid": "listkey",
        "listkey": LISTKEY,
        "listkey_start": start,
        "listkey_count": count
    }

    max_attempts = 5

    for attempt in range(1, max_attempts + 1):

        try:
            response = requests.get(
                BASE_URL,
                params=params,
                timeout=180
            )

            if response.status_code == 200:
                return response.text

            if response.status_code == 400:
                raise RuntimeError(
                    "HTTP 400: ListKey mungkin kedaluwarsa "
                    "atau parameter ditolak PubChem. "
                    "Pipeline dihentikan."
                )

            if response.status_code not in [
                429, 500, 502, 503, 504
            ]:
                response.raise_for_status()

            print(
                f"HTTP {response.status_code}. "
                f"Percobaan {attempt}/{max_attempts}"
            )

        except requests.exceptions.RequestException as exc:

            if attempt == max_attempts:
                raise RuntimeError(
                    f"Download gagal setelah {max_attempts} "
                    f"percobaan: {exc}"
                )

            print(
                f"Request error: {exc}. "
                f"Percobaan {attempt}/{max_attempts}"
            )

        if attempt < max_attempts:
            time.sleep(5 * attempt)

    raise RuntimeError("Download batch gagal.")


def load_existing_batches():

    existing = {}
    all_sids = set()

    files = sorted(
        BATCH_ROOT.glob(
            "AID652106_Batch_*/aid652106_batch*_all.csv"
        )
    )

    for file in files:

        df = pd.read_csv(file, low_memory=False)
        sid_col = get_sid_column(df)

        sids = set(
            pd.to_numeric(
                df[sid_col],
                errors="raise"
            ).astype("int64")
        )

        if len(sids) != len(df):
            raise RuntimeError(
                f"SID duplikat ditemukan di {file}"
            )

        overlap = all_sids.intersection(sids)

        if overlap:
            raise RuntimeError(
                f"SID overlap antarbatch ditemukan: "
                f"{len(overlap)}"
            )

        all_sids.update(sids)

        existing[file.parent.name] = {
            "file": file,
            "sids": sids,
            "records": len(df)
        }

    return existing, all_sids


def save_manifest(data):

    with open(
        MANIFEST_PATH,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            data,
            f,
            indent=4,
            ensure_ascii=False
        )


# ------------------------------------------------------------
# 3. VALIDATE PREVIOUS BATCHES
# ------------------------------------------------------------

existing_batches, global_sids = load_existing_batches()

print("\nEXISTING BATCH VALIDATION")
print("-" * 65)

for batch_name, info in existing_batches.items():
    print(
        f"{batch_name}: "
        f"{info['records']} records | "
        f"{len(info['sids'])} unique SID"
    )

expected_previous = [
    "AID652106_Batch_001",
    "AID652106_Batch_002"
]

for batch_name in expected_previous:

    if batch_name not in existing_batches:
        raise RuntimeError(
            f"File ALL untuk {batch_name} tidak ditemukan. "
            "Jangan lanjut sebelum file batch sebelumnya tersedia."
        )

    if existing_batches[batch_name]["records"] != 1000:
        raise RuntimeError(
            f"Jumlah record {batch_name} bukan 1000."
        )

if len(global_sids) != len(global_sids):
    raise RuntimeError("Validasi SID gagal.")

print("\nExisting batches:", len(existing_batches))
print("Global unique SID:", len(global_sids))
print("Previous batches: PASSED")


# ------------------------------------------------------------
# 4. INITIALIZE PIPELINE LOG
# ------------------------------------------------------------

if LOG_PATH.exists():
    log_df = pd.read_csv(LOG_PATH)
else:
    log_df = pd.DataFrame(
        columns=[
            "batch",
            "start_index",
            "expected_records",
            "actual_records",
            "activator_records",
            "active_records",
            "status",
            "timestamp"
        ]
    )


# ------------------------------------------------------------
# 5. AUTOMATED BATCH LOOP
# ------------------------------------------------------------

batch_number = START_BATCH
start_index = START_INDEX

session = requests.Session()

try:

    while start_index < TOTAL_SIDS:

        count = min(
            BATCH_SIZE,
            TOTAL_SIDS - start_index
        )

        batch_name = (
            f"AID652106_Batch_{batch_number:03d}"
        )

        batch_dir = BATCH_ROOT / batch_name

        all_file = (
            batch_dir
            / f"aid652106_batch{batch_number:03d}_all.csv"
        )

        activator_file = (
            batch_dir
            / f"aid652106_batch{batch_number:03d}_activator_review.csv"
        )

        active_file = (
            batch_dir
            / f"aid652106_batch{batch_number:03d}_active_provisional.csv"
        )

        print("\n" + "=" * 65)
        print(f"PROCESSING {batch_name}")
        print("=" * 65)

        # ----------------------------------------------------
        # A. RESUME CHECK
        # ----------------------------------------------------

        output_files = [
            all_file,
            activator_file,
            active_file
        ]

        existing_outputs = [
            f.exists() for f in output_files
        ]

        if all(existing_outputs):

            print("Batch sudah lengkap. Validasi ulang...")

            df_check = pd.read_csv(
                all_file,
                low_memory=False
            )

            sid_col = get_sid_column(df_check)

            sids_check = set(
                pd.to_numeric(
                    df_check[sid_col],
                    errors="raise"
                ).astype("int64")
            )

            if len(df_check) != count:
                raise RuntimeError(
                    "Jumlah record batch tersimpan tidak sesuai."
                )

            if len(sids_check) != count:
                raise RuntimeError(
                    "SID duplikat dalam batch tersimpan."
                )

            print("Existing batch valid. SKIPPED.")

            start_index += count
            batch_number += 1
            continue

        if any(existing_outputs):
            raise RuntimeError(
                f"{batch_name} memiliki file parsial. "
                "Tidak akan menimpa file yang ada."
            )

        batch_dir.mkdir(
            parents=True,
            exist_ok=True
        )

        # ----------------------------------------------------
        # B. DOWNLOAD
        # ----------------------------------------------------

        print(
            f"SID index: {start_index} - "
            f"{start_index + count - 1}"
        )

        csv_text = download_batch(
            start_index,
            count
        )

        df, sid_col = read_pubchem_csv(csv_text)

        # ----------------------------------------------------
        # C. VALIDATION
        # ----------------------------------------------------

        if len(df) != count:
            raise RuntimeError(
                f"Record mismatch: expected {count}, "
                f"received {len(df)}"
            )

        if df[sid_col].duplicated().any():
            raise RuntimeError(
                "SID duplikat ditemukan dalam batch."
            )

        current_sids = set(df[sid_col])

        overlap = global_sids.intersection(
            current_sids
        )

        if overlap:
            raise RuntimeError(
                f"Overlap dengan batch sebelumnya: "
                f"{len(overlap)} SID"
            )

        required_columns = [
            "PUBCHEM_ACTIVITY_OUTCOME",
            "Phenotype",
            "Fit_R2",
            "Curve_Description"
        ]

        missing_columns = [
            col for col in required_columns
            if col not in df.columns
        ]

        if missing_columns:
            raise RuntimeError(
                f"Kolom penting hilang: {missing_columns}"
            )

        print("Record count :", len(df))
        print("Unique SID   :", df[sid_col].nunique())
        print("Overlap      :", len(overlap))
        print("Validation   : PASSED")

        # ----------------------------------------------------
        # D. CLASSIFICATION
        # ----------------------------------------------------

        phenotype = (
            df["Phenotype"]
            .astype(str)
            .str.strip()
            .str.lower()
        )

        activator = df.loc[
            phenotype == "activator"
        ].copy()

        activator["review_category"] = (
            activator.apply(
                classify_activator,
                axis=1
            )
        )

        outcome = (
            activator["PUBCHEM_ACTIVITY_OUTCOME"]
            .astype(str)
            .str.strip()
            .str.lower()
        )

        active = activator.loc[
            outcome == "active"
        ].copy()

        print("Activator:", len(activator))
        print("Active provisional:", len(active))

        # ----------------------------------------------------
        # E. SAVE
        # ----------------------------------------------------

        df.to_csv(
            all_file,
            index=False
        )

        activator.to_csv(
            activator_file,
            index=False
        )

        active.to_csv(
            active_file,
            index=False
        )

        # ----------------------------------------------------
        # F. READ-BACK VALIDATION
        # ----------------------------------------------------

        check_all = pd.read_csv(
            all_file,
            low_memory=False
        )

        check_activator = pd.read_csv(
            activator_file,
            low_memory=False
        )

        check_active = pd.read_csv(
            active_file,
            low_memory=False
        )

        if len(check_all) != count:
            raise RuntimeError(
                "Read-back ALL gagal."
            )

        if check_all[sid_col].nunique() != count:
            raise RuntimeError(
                "Read-back ALL memiliki SID duplikat."
            )

        if len(check_activator) != len(activator):
            raise RuntimeError(
                "Read-back ACTIVATOR gagal."
            )

        if len(check_active) != len(active):
            raise RuntimeError(
                "Read-back ACTIVE gagal."
            )

        if check_active[sid_col].duplicated().any():
            raise RuntimeError(
                "Read-back ACTIVE memiliki SID duplikat."
            )

        # Tambahkan SID setelah seluruh validasi berhasil.
        global_sids.update(current_sids)

        # ----------------------------------------------------
        # G. UPDATE LOG
        # ----------------------------------------------------

        log_row = {
            "batch": batch_name,
            "start_index": start_index,
            "expected_records": count,
            "actual_records": len(df),
            "activator_records": len(activator),
            "active_records": len(active),
            "status": "PASSED",
            "timestamp": datetime.now().isoformat()
        }

        log_df = pd.concat(
            [
                log_df,
                pd.DataFrame([log_row])
            ],
            ignore_index=True
        )

        log_df.to_csv(
            LOG_PATH,
            index=False
        )

        manifest = {
            "aid": AID,
            "listkey": LISTKEY,
            "total_sids_expected": TOTAL_SIDS,
            "batch_size": BATCH_SIZE,
            "last_completed_batch": batch_name,
            "next_start_index": start_index + count,
            "completed_unique_sids": len(global_sids),
            "last_update": datetime.now().isoformat(),
            "status": "RUNNING"
        }

        save_manifest(manifest)

        # ----------------------------------------------------
        # H. BATCH SUMMARY
        # ----------------------------------------------------

        print("\nBATCH COMPLETED")
        print("-" * 65)
        print("ALL       :", check_all.shape)
        print("ACTIVATOR :", check_activator.shape)
        print("ACTIVE    :", check_active.shape)
        print("SID total :", len(global_sids))
        print("VALIDATION: PASSED")

        # Beri jeda singkat untuk menjaga request.
        time.sleep(0.5)

        start_index += count
        batch_number += 1

    # --------------------------------------------------------
    # 6. FINAL SUMMARY
    # --------------------------------------------------------

    final_manifest = {
        "aid": AID,
        "listkey": LISTKEY,
        "total_sids_expected": TOTAL_SIDS,
        "completed_unique_sids": len(global_sids),
        "last_completed_batch": f"AID652106_Batch_{batch_number-1:03d}",
        "completion_time": datetime.now().isoformat(),
        "status": (
            "COMPLETED"
            if len(global_sids) == TOTAL_SIDS
            else "COUNT_REVIEW_REQUIRED"
        )
    }

    save_manifest(final_manifest)

    print("\n" + "=" * 65)
    print("PIPELINE FINISHED")
    print("=" * 65)
    print("Expected SID :", TOTAL_SIDS)
    print("Collected SID:", len(global_sids))
    print("Status       :", final_manifest["status"])
    print("Log          :", LOG_PATH)
    print("Manifest     :", MANIFEST_PATH)

    if len(global_sids) != TOTAL_SIDS:
        print(
            "PERHATIAN: Jumlah SID belum sesuai target. "
            "Periksa manifest dan log."
        )

except Exception as error:

    print("\n" + "!" * 65)
    print("PIPELINE STOPPED")
    print("!" * 65)
    print("Error:", repr(error))
    print("Batch terakhir yang sedang diproses:",
          f"AID652106_Batch_{batch_number:03d}")
    print("Jangan hapus file batch yang sudah lolos validasi.")
    print("Perbaiki penyebab error sebelum menjalankan ulang.")

    raise

finally:
    session.close()


AUTOMATED AID652106 PIPELINE
Total SID       : 370177
Batch size      : 1000
Starting index  : 2000
Starting batch  : 3
Output          : H:\My Drive\PROJECT SAINS DATA\OUTPUT\01_Cleaned_Data\Alpha_Synuclein\Unmatched_Audit

EXISTING BATCH VALIDATION
-----------------------------------------------------------------
AID652106_Batch_001: 1000 records | 1000 unique SID
AID652106_Batch_002: 1000 records | 1000 unique SID

Existing batches: 2
Global unique SID: 2000
Previous batches: PASSED

PROCESSING AID652106_Batch_003
SID index: 2000 - 2999
Record count : 1000
Unique SID   : 1000
Overlap      : 0
Validation   : PASSED
Activator: 54
Active provisional: 4

BATCH COMPLETED
-----------------------------------------------------------------
ALL       : (1000, 44)
ACTIVATOR : (54, 45)
ACTIVE    : (4, 45)
SID total : 3000
VALIDATION: PASSED

PROCESSING AID652106_Batch_004
SID index: 3000 - 3999
Record count : 1000
Unique SID   : 1000
Overlap      : 0
Validation   : PASSED
Activator: 65
Active p

In [ ]:

from pathlib import Path
import pandas as pd
import numpy as np

BATCH_ROOT = (
    Path(r"H:\My Drive\PROJECT SAINS DATA\OUTPUT")
    / "01_Cleaned_Data"
    / "Alpha_Synuclein"
    / "Unmatched_Audit"
)

TOTAL_EXPECTED = 370177
BATCH_SIZE = 1000

all_sids = set()
audit_results = []

total_all = 0
total_activator = 0
total_active = 0

phenotype_summary = {}
outcome_summary = {}
category_summary = {}

errors = []

# --------------------------------------------------
# 1. AUDIT ALL BATCHES
# --------------------------------------------------

for batch_no in range(1, 372):

    batch_name = f"AID652106_Batch_{batch_no:03d}"
    batch_dir = BATCH_ROOT / batch_name

    all_file = batch_dir / (
        f"aid652106_batch{batch_no:03d}_all.csv"
    )

    activator_file = batch_dir / (
        f"aid652106_batch{batch_no:03d}_activator_review.csv"
    )

    active_file = batch_dir / (
        f"aid652106_batch{batch_no:03d}_active_provisional.csv"
    )

    expected = min(
        BATCH_SIZE,
        TOTAL_EXPECTED - (batch_no - 1) * BATCH_SIZE
    )

    if not all(f.exists() for f in [
        all_file, activator_file, active_file
    ]):
        errors.append(f"{batch_name}: Missing file")
        continue

    df = pd.read_csv(all_file, low_memory=False)
    act = pd.read_csv(activator_file, low_memory=False)
    active = pd.read_csv(active_file, low_memory=False)

    sid_col = (
        "PUBCHEM_SID"
        if "PUBCHEM_SID" in df.columns
        else "SID"
    )

    sids = set(
        pd.to_numeric(
            df[sid_col], errors="coerce"
        ).dropna().astype("int64")
    )

    # Record count
    if len(df) != expected:
        errors.append(
            f"{batch_name}: Expected {expected}, got {len(df)}"
        )

    # Within-batch duplicates
    if len(sids) != len(df):
        errors.append(
            f"{batch_name}: Duplicate or invalid SID"
        )

    # Across-batch duplicates
    overlap = all_sids.intersection(sids)

    if overlap:
        errors.append(
            f"{batch_name}: {len(overlap)} overlapping SID"
        )

    # Check activator subset
    if not set(act[sid_col]).issubset(sids):
        errors.append(
            f"{batch_name}: Activator SID outside ALL"
        )

    if not set(active[sid_col]).issubset(
        set(act[sid_col])
    ):
        errors.append(
            f"{batch_name}: Active SID outside Activator"
        )

    all_sids.update(sids)

    total_all += len(df)
    total_activator += len(act)
    total_active += len(active)

    if "Phenotype" in df.columns:
        counts = df["Phenotype"].value_counts(dropna=False)

        for key, value in counts.items():
            key = str(key)
            phenotype_summary[key] = (
                phenotype_summary.get(key, 0) + value
            )

    if "PUBCHEM_ACTIVITY_OUTCOME" in df.columns:
        counts = df[
            "PUBCHEM_ACTIVITY_OUTCOME"
        ].value_counts(dropna=False)

        for key, value in counts.items():
            key = str(key)
            outcome_summary[key] = (
                outcome_summary.get(key, 0) + value
            )

    if "review_category" in act.columns:
        counts = act[
            "review_category"
        ].value_counts(dropna=False)

        for key, value in counts.items():
            key = str(key)
            category_summary[key] = (
                category_summary.get(key, 0) + value
            )

    audit_results.append({
        "batch": batch_name,
        "expected": expected,
        "all": len(df),
        "activator": len(act),
        "active": len(active),
        "unique_sid": len(sids),
        "overlap": len(overlap)
    })

# --------------------------------------------------
# 2. FINAL RECONCILIATION
# --------------------------------------------------

audit_df = pd.DataFrame(audit_results)

print("=" * 65)
print("FINAL AID652106 POST-PIPELINE AUDIT")
print("=" * 65)

print("Batch audited:", len(audit_df))
print("Total ALL:", total_all)
print("Unique SID:", len(all_sids))
print("Total ACTIVATOR:", total_activator)
print("Total ACTIVE:", total_active)

print("\nPHENOTYPE SUMMARY")
print(pd.Series(phenotype_summary).sort_values(ascending=False))

print("\nOUTCOME SUMMARY")
print(pd.Series(outcome_summary).sort_values(ascending=False))

print("\nACTIVATOR CATEGORY SUMMARY")
print(pd.Series(category_summary).sort_values(ascending=False))

# --------------------------------------------------
# 3. FINAL STATUS
# --------------------------------------------------

if (
    len(audit_df) == 371
    and total_all == TOTAL_EXPECTED
    and len(all_sids) == TOTAL_EXPECTED
    and not errors
):
    print("\nFINAL AUDIT: PASSED")
else:
    print("\nFINAL AUDIT: REVIEW REQUIRED")

    for error in errors[:30]:
        print("-", error)

# --------------------------------------------------
# 4. SAVE AUDIT REPORT
# --------------------------------------------------

audit_file = BATCH_ROOT / "aid652106_final_batch_audit.csv"

audit_df.to_csv(audit_file, index=False)

print("\nAudit report saved:")
print(audit_file)


FINAL AID652106 POST-PIPELINE AUDIT
Batch audited: 371
Total ALL: 370177
Unique SID: 370177
Total ACTIVATOR: 6326
Total ACTIVE: 503

PHENOTYPE SUMMARY
Inactive     360022
Activator      6326
Inhibitor      3829
dtype: int64

OUTCOME SUMMARY
Inactive        363851
Inconclusive      5823
Active             503
dtype: int64

ACTIVATOR CATEGORY SUMMARY
Inconclusive_Single_Point             5008
Inconclusive_Curve_Response            655
Cellular_Rescue_Active_Provisional     487
Inconclusive_Poor_Fit                   37
dtype: int64

FINAL AUDIT: PASSED

Audit report saved:
H:\My Drive\PROJECT SAINS DATA\OUTPUT\01_Cleaned_Data\Alpha_Synuclein\Unmatched_Audit\aid652106_final_batch_audit.csv


In [ ]:

from pathlib import Path
import pandas as pd

ROOT = (
    Path(r"H:\My Drive\PROJECT SAINS DATA\OUTPUT")
    / "01_Cleaned_Data"
    / "Alpha_Synuclein"
    / "Unmatched_Audit"
)

for batch_no in [1, 2]:

    folder = ROOT / f"AID652106_Batch_{batch_no:03d}"

    file = folder / (
        f"aid652106_batch{batch_no:03d}_activator_review.csv"
    )

    df = pd.read_csv(file, low_memory=False)

    print(f"\nBATCH {batch_no:03d}")
    print("Rows:", len(df))
    print("Columns:", df.columns.tolist())
    print("Has review_category:", "review_category" in df.columns)

    if "review_category" in df.columns:
        print(df["review_category"].value_counts(dropna=False))



BATCH 001
Rows: 78
Columns: ['PUBCHEM_RESULT_TAG', 'PUBCHEM_SID', 'PUBCHEM_CID', 'PUBCHEM_EXT_DATASOURCE_SMILES', 'PUBCHEM_ACTIVITY_OUTCOME', 'PUBCHEM_ACTIVITY_SCORE', 'PUBCHEM_ACTIVITY_URL', 'PUBCHEM_ASSAYDATA_COMMENT', 'Phenotype', 'Potency', 'Efficacy', 'Analysis Comment', 'Activity_Score', 'Curve_Description', 'Fit_LogAC50', 'Fit_HillSlope', 'Fit_R2', 'Fit_InfiniteActivity', 'Fit_ZeroActivity', 'Fit_CurveClass', 'Excluded_Points', 'Max_Response', 'Activity at 0.00295 uM', 'Activity at 0.015 uM', 'Activity at 0.074 uM', 'Activity at 0.092 uM', 'Activity at 0.184 uM', 'Activity at 0.258 uM', 'Activity at 0.369 uM', 'Activity at 0.508 uM', 'Activity at 0.749 uM', 'Activity at 1.290 uM', 'Activity at 1.840 uM', 'Activity at 2.537 uM', 'Activity at 3.751 uM', 'Activity at 6.450 uM', 'Activity at 9.220 uM', 'Activity at 12.94 uM', 'Activity at 18.70 uM', 'Activity at 36.87 uM', 'Activity at 46.10 uM', 'Activity at 64.71 uM', 'Activity at 92.20 uM', 'Compound QC', 'provisional_category']

In [ ]:

from pathlib import Path
import pandas as pd

ROOT = (
    Path(r"H:\My Drive\PROJECT SAINS DATA\OUTPUT")
    / "01_Cleaned_Data"
    / "Alpha_Synuclein"
    / "Unmatched_Audit"
)

for batch_no in [1, 2]:

    file = (
        ROOT
        / f"AID652106_Batch_{batch_no:03d}"
        / f"aid652106_batch{batch_no:03d}_activator_review.csv"
    )

    df = pd.read_csv(file, low_memory=False)

    print("\n" + "=" * 60)
    print(f"BATCH {batch_no:03d} - PROVISIONAL CATEGORY")
    print("=" * 60)

    print("\nDistribusi provisional_category:")
    print(
        df["provisional_category"]
        .value_counts(dropna=False)
    )

    print("\nDistribusi PUBCHEM_ACTIVITY_OUTCOME:")
    print(
        df["PUBCHEM_ACTIVITY_OUTCOME"]
        .value_counts(dropna=False)
    )

    print("\nContoh kombinasi kategori dan outcome:")
    print(
        pd.crosstab(
            df["provisional_category"],
            df["PUBCHEM_ACTIVITY_OUTCOME"],
            dropna=False
        )
    )



BATCH 001 - PROVISIONAL CATEGORY

Distribusi provisional_category:
provisional_category
Inconclusive_Single_Point             54
Inconclusive_Curve_Response           12
Cellular_Rescue_Active_Provisional    11
Inconclusive_Poor_Fit                  1
Name: count, dtype: int64

Distribusi PUBCHEM_ACTIVITY_OUTCOME:
PUBCHEM_ACTIVITY_OUTCOME
Inconclusive    67
Active          11
Name: count, dtype: int64

Contoh kombinasi kategori dan outcome:
PUBCHEM_ACTIVITY_OUTCOME            Active  Inconclusive
provisional_category                                    
Cellular_Rescue_Active_Provisional      11             0
Inconclusive_Curve_Response              0            12
Inconclusive_Poor_Fit                    0             1
Inconclusive_Single_Point                0            54

BATCH 002 - PROVISIONAL CATEGORY

Distribusi provisional_category:
provisional_category
Inconclusive_Single_Point             50
Inconclusive_Curve_Response            6
Cellular_Rescue_Active_Provisional     5


In [ ]:

from pathlib import Path
import pandas as pd

ROOT = (
    Path(r"H:\My Drive\PROJECT SAINS DATA\OUTPUT")
    / "01_Cleaned_Data"
    / "Alpha_Synuclein"
    / "Unmatched_Audit"
)

MASTER_FILE = ROOT / "aid652106_all_activator_review.csv"
SUMMARY_FILE = ROOT / "aid652106_activator_category_summary.csv"

if MASTER_FILE.exists() or SUMMARY_FILE.exists():
    raise FileExistsError(
        "File output sudah ada. Jangan overwrite. "
        "Periksa atau ubah nama output terlebih dahulu."
    )

frames = []

# --------------------------------------------------
# 1. LOAD AND STANDARDIZE ALL ACTIVATOR FILES
# --------------------------------------------------

for batch_no in range(1, 372):

    file = (
        ROOT
        / f"AID652106_Batch_{batch_no:03d}"
        / f"aid652106_batch{batch_no:03d}_activator_review.csv"
    )

    if not file.exists():
        raise FileNotFoundError(file)

    df = pd.read_csv(file, low_memory=False)

    # Ambil kategori sesuai skema masing-masing batch.
    if "provisional_category" in df.columns:
        source_col = "provisional_category"

    elif "review_category" in df.columns:
        source_col = "review_category"

    else:
        raise ValueError(
            f"Kolom kategori tidak ditemukan: {file}"
        )

    # Jangan mengubah kategori asli.
    df["review_category"] = df[source_col]

    df["source_batch"] = f"Batch_{batch_no:03d}"

    frames.append(df)

# --------------------------------------------------
# 2. COMBINE
# --------------------------------------------------

master = pd.concat(
    frames,
    ignore_index=True,
    sort=False
)

print("MASTER SHAPE:", master.shape)

# --------------------------------------------------
# 3. VALIDATION
# --------------------------------------------------

required_categories = {
    "Inconclusive_Single_Point",
    "Inconclusive_Curve_Response",
    "Cellular_Rescue_Active_Provisional",
    "Inconclusive_Poor_Fit"
}

sid = pd.to_numeric(
    master["PUBCHEM_SID"],
    errors="coerce"
)

if sid.isna().any():
    raise ValueError("Ada SID kosong atau tidak valid.")

if sid.duplicated().any():
    raise ValueError("Ada SID duplikat pada master.")

if len(master) != 6326:
    raise ValueError(
        f"Jumlah master tidak sesuai: {len(master)}"
    )

if master["review_category"].isna().any():
    raise ValueError("Ada kategori yang kosong.")

observed_categories = set(
    master["review_category"].unique()
)

if observed_categories != required_categories:
    raise ValueError(
        f"Kategori tidak sesuai: {observed_categories}"
    )

# Active outcome harus sesuai dengan kategori kandidat.
active_mask = (
    master["review_category"]
    == "Cellular_Rescue_Active_Provisional"
)

outcome = (
    master["PUBCHEM_ACTIVITY_OUTCOME"]
    .astype(str)
    .str.strip()
    .str.lower()
)

if not (outcome[active_mask] == "active").all():
    raise ValueError(
        "Ada kandidat Cellular Rescue dengan outcome "
        "bukan Active."
    )

if active_mask.sum() != 503:
    raise ValueError(
        "Jumlah kandidat Active tidak sesuai."
    )

# --------------------------------------------------
# 4. SUMMARY
# --------------------------------------------------

summary = (
    master.groupby(
        ["source_batch", "review_category"],
        dropna=False
    )
    .size()
    .reset_index(name="record_count")
)

category_summary = (
    master["review_category"]
    .value_counts()
    .rename_axis("review_category")
    .reset_index(name="record_count")
)

print("\nCATEGORY SUMMARY")
print(category_summary.to_string(index=False))

print("\nTOTAL ACTIVATOR:", len(master))
print("UNIQUE SID:", sid.nunique())
print("ACTIVE PROVISIONAL:", active_mask.sum())

# --------------------------------------------------
# 5. SAVE
# --------------------------------------------------

master.to_csv(
    MASTER_FILE,
    index=False
)

category_summary.to_csv(
    SUMMARY_FILE,
    index=False
)

print("\nSAVED:")
print(MASTER_FILE)
print(SUMMARY_FILE)

# --------------------------------------------------
# 6. READ-BACK CHECK
# --------------------------------------------------

check = pd.read_csv(
    MASTER_FILE,
    low_memory=False
)

assert len(check) == 6326
assert check["PUBCHEM_SID"].nunique() == 6326
assert check["review_category"].notna().all()

print("\nMASTER VALIDATION: PASSED")


MASTER SHAPE: (6326, 47)

CATEGORY SUMMARY
                   review_category  record_count
         Inconclusive_Single_Point          5112
       Inconclusive_Curve_Response           673
Cellular_Rescue_Active_Provisional           503
             Inconclusive_Poor_Fit            38

TOTAL ACTIVATOR: 6326
UNIQUE SID: 6326
ACTIVE PROVISIONAL: 503

SAVED:
H:\My Drive\PROJECT SAINS DATA\OUTPUT\01_Cleaned_Data\Alpha_Synuclein\Unmatched_Audit\aid652106_all_activator_review.csv
H:\My Drive\PROJECT SAINS DATA\OUTPUT\01_Cleaned_Data\Alpha_Synuclein\Unmatched_Audit\aid652106_activator_category_summary.csv

MASTER VALIDATION: PASSED


In [ ]:

from pathlib import Path
import pandas as pd

ROOT = (
    Path(r"H:\My Drive\PROJECT SAINS DATA\OUTPUT")
    / "01_Cleaned_Data"
    / "Alpha_Synuclein"
    / "Unmatched_Audit"
)

MASTER_FILE = ROOT / "aid652106_all_activator_review.csv"

df = pd.read_csv(MASTER_FILE, low_memory=False)

print("=" * 65)
print("AID652106 COMPOUND IDENTITY AUDIT")
print("=" * 65)

# --------------------------------------------------
# 1. REQUIRED COLUMNS
# --------------------------------------------------

required = [
    "PUBCHEM_SID",
    "PUBCHEM_CID",
    "PUBCHEM_EXT_DATASOURCE_SMILES"
]

missing_columns = [
    col for col in required
    if col not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Kolom tidak ditemukan: {missing_columns}"
    )

# --------------------------------------------------
# 2. MISSING VALUES
# --------------------------------------------------

print("\nMISSING VALUE SUMMARY")

for col in required:
    missing = df[col].isna().sum()

    print(
        f"{col}: {missing} "
        f"({missing / len(df) * 100:.2f}%)"
    )

# --------------------------------------------------
# 3. VALID SID AND CID
# --------------------------------------------------

df["SID_numeric"] = pd.to_numeric(
    df["PUBCHEM_SID"],
    errors="coerce"
)

df["CID_numeric"] = pd.to_numeric(
    df["PUBCHEM_CID"],
    errors="coerce"
)

invalid_sid = (
    df["SID_numeric"].isna()
    | (df["SID_numeric"] <= 0)
)

invalid_cid = (
    df["CID_numeric"].isna()
    | (df["CID_numeric"] <= 0)
)

print("\nIDENTIFIER QUALITY")

print("Invalid SID:", invalid_sid.sum())
print("Invalid CID:", invalid_cid.sum())

# --------------------------------------------------
# 4. UNIQUE IDENTIFIERS
# --------------------------------------------------

print("\nUNIQUE IDENTIFIERS")

print(
    "Total records:",
    len(df)
)

print(
    "Unique SID:",
    df["SID_numeric"].nunique()
)

print(
    "Unique CID:",
    df["CID_numeric"].nunique()
)

# --------------------------------------------------
# 5. SMILES QUALITY
# --------------------------------------------------

smiles = (
    df["PUBCHEM_EXT_DATASOURCE_SMILES"]
    .astype("string")
    .str.strip()
)

missing_smiles = (
    smiles.isna()
    | smiles.eq("")
)

print("\nSMILES QUALITY")

print("Missing/empty SMILES:", missing_smiles.sum())
print(
    "Unique nonmissing SMILES:",
    smiles[~missing_smiles].nunique()
)

# --------------------------------------------------
# 6. SID-CID RELATIONSHIP
# --------------------------------------------------

sid_cid = (
    df.loc[
        df["SID_numeric"].notna()
        & df["CID_numeric"].notna(),
        ["SID_numeric", "CID_numeric"]
    ]
    .drop_duplicates()
)

cid_sid_count = (
    sid_cid.groupby("CID_numeric")["SID_numeric"]
    .nunique()
)

print("\nSID-CID RELATIONSHIP")

print(
    "CID with multiple SID:",
    (cid_sid_count > 1).sum()
)

print(
    "Maximum SID per CID:",
    cid_sid_count.max()
)

# --------------------------------------------------
# 7. SUMMARY
# --------------------------------------------------

print("\nFINAL SUMMARY")

print({
    "records": len(df),
    "unique_sid": df["SID_numeric"].nunique(),
    "unique_cid": df["CID_numeric"].nunique(),
    "missing_cid": int(invalid_cid.sum()),
    "missing_smiles": int(missing_smiles.sum()),
    "cid_with_multiple_sid": int(
        (cid_sid_count > 1).sum()
    )
})


AID652106 COMPOUND IDENTITY AUDIT

MISSING VALUE SUMMARY
PUBCHEM_SID: 0 (0.00%)
PUBCHEM_CID: 0 (0.00%)
PUBCHEM_EXT_DATASOURCE_SMILES: 0 (0.00%)

IDENTIFIER QUALITY
Invalid SID: 0
Invalid CID: 0

UNIQUE IDENTIFIERS
Total records: 6326
Unique SID: 6326
Unique CID: 6310

SMILES QUALITY
Missing/empty SMILES: 0
Unique nonmissing SMILES: 6310

SID-CID RELATIONSHIP
CID with multiple SID: 14
Maximum SID per CID: 3

FINAL SUMMARY
{'records': 6326, 'unique_sid': 6326, 'unique_cid': 6310, 'missing_cid': 0, 'missing_smiles': 0, 'cid_with_multiple_sid': 14}


In [ ]:
from pathlib import Path
import pandas as pd

# Cari file master secara otomatis
target_filename = "aid652106_all_activator_review.csv"

matches = list(ALPHA_CLEANED_DIR.rglob(target_filename))

print("=" * 65)
print("MASTER FILE SEARCH")
print("=" * 65)

print("Jumlah file ditemukan:", len(matches))

for i, path in enumerate(matches, start=1):
    print(f"{i}. {path}")

if len(matches) == 1:
    master_path = matches[0]
    df = pd.read_csv(master_path, low_memory=False)

    print("\nFILE BERHASIL DIBACA")
    print("Path:", master_path)
    print("Shape:", df.shape)

elif len(matches) == 0:
    raise FileNotFoundError(
        f"{target_filename} tidak ditemukan di {ALPHA_CLEANED_DIR}"
    )

else:
    raise ValueError(
        "Ditemukan lebih dari satu file master. "
        "Tentukan file yang benar terlebih dahulu."
    )

MASTER FILE SEARCH
Jumlah file ditemukan: 1
1. H:\My Drive\PROJECT SAINS DATA\OUTPUT\01_Cleaned_Data\Alpha_Synuclein\Unmatched_Audit\aid652106_all_activator_review.csv

FILE BERHASIL DIBACA
Path: H:\My Drive\PROJECT SAINS DATA\OUTPUT\01_Cleaned_Data\Alpha_Synuclein\Unmatched_Audit\aid652106_all_activator_review.csv
Shape: (6326, 47)


In [ ]:
# ============================================================
# AID652106 COMPOUND IDENTITY CONSISTENCY AUDIT
# ============================================================

required_cols = [
    "PUBCHEM_SID",
    "PUBCHEM_CID",
    "PUBCHEM_EXT_DATASOURCE_SMILES"
]

missing_cols = [
    col for col in required_cols
    if col not in df.columns
]

if missing_cols:
    raise ValueError(f"Kolom tidak ditemukan: {missing_cols}")

# ============================================================
# 1. CID WITH MULTIPLE SID
# ============================================================

cid_sid_audit = (
    df.groupby("PUBCHEM_CID")
    .agg(
        sid_count=("PUBCHEM_SID", "nunique"),
        record_count=("PUBCHEM_SID", "size"),
        smiles_count=("PUBCHEM_EXT_DATASOURCE_SMILES", "nunique")
    )
    .reset_index()
)

multiple_sid = cid_sid_audit[
    cid_sid_audit["sid_count"] > 1
].copy()

print("=" * 65)
print("CID WITH MULTIPLE SID")
print("=" * 65)

print("Jumlah CID:", len(multiple_sid))

print(
    multiple_sid.sort_values(
        "sid_count",
        ascending=False
    ).to_string(index=False)
)

# ============================================================
# 2. CID WITH MULTIPLE SMILES
# ============================================================

multiple_smiles = cid_sid_audit[
    cid_sid_audit["smiles_count"] > 1
].copy()

print("\nCID WITH MULTIPLE SMILES:", len(multiple_smiles))

print(multiple_smiles.to_string(index=False))

# ============================================================
# 3. SMILES WITH MULTIPLE CID
# ============================================================

smiles_cid_audit = (
    df.groupby("PUBCHEM_EXT_DATASOURCE_SMILES")
    .agg(
        cid_count=("PUBCHEM_CID", "nunique"),
        sid_count=("PUBCHEM_SID", "nunique")
    )
    .reset_index()
)

multiple_cid_per_smiles = smiles_cid_audit[
    smiles_cid_audit["cid_count"] > 1
].copy()

print("\nSMILES WITH MULTIPLE CID:", len(multiple_cid_per_smiles))

print(multiple_cid_per_smiles.head(20).to_string(index=False))

# ============================================================
# 4. DETAIL MULTIPLE SID
# ============================================================

multiple_sid_detail = df[
    df["PUBCHEM_CID"].isin(multiple_sid["PUBCHEM_CID"])
][required_cols].drop_duplicates()

print("\nMULTIPLE SID DETAIL")
print("Records:", len(multiple_sid_detail))

print(multiple_sid_detail.head(30).to_string(index=False))

# ============================================================
# 5. SAVE AUDIT RESULTS
# ============================================================

audit_dir = ALPHA_CLEANED_DIR / "Unmatched_Audit"
audit_dir.mkdir(parents=True, exist_ok=True)

multiple_sid.to_csv(
    audit_dir / "aid652106_cid_multiple_sid_audit.csv",
    index=False
)

multiple_sid_detail.to_csv(
    audit_dir / "aid652106_cid_multiple_sid_detail.csv",
    index=False
)

multiple_smiles.to_csv(
    audit_dir / "aid652106_cid_multiple_smiles.csv",
    index=False
)

multiple_cid_per_smiles.to_csv(
    audit_dir / "aid652106_smiles_multiple_cid.csv",
    index=False
)

print("\n" + "=" * 65)
print("AUDIT SAVED")
print("=" * 65)
print("Output:", audit_dir)


CID WITH MULTIPLE SID
Jumlah CID: 14
 PUBCHEM_CID  sid_count  record_count  smiles_count
      5712.0          3             3             1
    456201.0          3             3             1
      3117.0          2             2             1
      4878.0          2             2             1
      5059.0          2             2             1
      2788.0          2             2             1
     47576.0          2             2             1
    712700.0          2             2             1
    739449.0          2             2             1
   2999845.0          2             2             1
   5281787.0          2             2             1
   5282106.0          2             2             1
   6247192.0          2             2             1
  11509354.0          2             2             1

CID WITH MULTIPLE SMILES: 0
Empty DataFrame
Columns: [PUBCHEM_CID, sid_count, record_count, smiles_count]
Index: []

SMILES WITH MULTIPLE CID: 0
Empty DataFrame
Columns: [PUBCHEM_EXT

In [ ]:

# ============================================================
# AID652106 COMPOUND IDENTITY REGISTRY
# ============================================================

# Gunakan df yang sudah berhasil dibaca dan diaudit sebelumnya

registry_source = df.copy()

# Pastikan CID valid dan dapat dikonversi menjadi integer
registry_source["PUBCHEM_CID"] = pd.to_numeric(
    registry_source["PUBCHEM_CID"],
    errors="coerce"
)

assert registry_source["PUBCHEM_CID"].notna().all()
assert (
    registry_source["PUBCHEM_CID"]
    == registry_source["PUBCHEM_CID"].round()
).all()

registry_source["PUBCHEM_CID"] = (
    registry_source["PUBCHEM_CID"].astype("int64")
)

# Pastikan setiap CID hanya memiliki satu SMILES
smiles_consistency = (
    registry_source.groupby("PUBCHEM_CID")[
        "PUBCHEM_EXT_DATASOURCE_SMILES"
    ].nunique()
)

assert (smiles_consistency == 1).all(), (
    "Ditemukan CID dengan lebih dari satu SMILES!"
)

# Buat registry: satu baris per CID
compound_registry = (
    registry_source.groupby("PUBCHEM_CID")
    .agg(
        PUBCHEM_EXT_DATASOURCE_SMILES=(
            "PUBCHEM_EXT_DATASOURCE_SMILES",
            "first"
        ),
        SID_COUNT=("PUBCHEM_SID", "nunique"),
        SID_LIST=(
            "PUBCHEM_SID",
            lambda x: ";".join(
                sorted(set(x.astype(str)))
            )
        ),
        ACTIVITY_RECORD_COUNT=("PUBCHEM_SID", "size")
    )
    .reset_index()
)

# ============================================================
# VALIDATION
# ============================================================

print("=" * 65)
print("COMPOUND IDENTITY REGISTRY VALIDATION")
print("=" * 65)

print("Source records:", len(registry_source))
print("Source unique SID:", registry_source["PUBCHEM_SID"].nunique())
print("Registry records:", len(compound_registry))
print("Registry unique CID:", compound_registry["PUBCHEM_CID"].nunique())
print(
    "Missing SMILES:",
    compound_registry[
        "PUBCHEM_EXT_DATASOURCE_SMILES"
    ].isna().sum()
)

assert len(compound_registry) == 6310
assert compound_registry["PUBCHEM_CID"].is_unique
assert compound_registry[
    "PUBCHEM_EXT_DATASOURCE_SMILES"
].notna().all()

print("\nVALIDATION: PASSED")

# ============================================================
# SAVE
# ============================================================

audit_dir = ALPHA_CLEANED_DIR / "Unmatched_Audit"
audit_dir.mkdir(parents=True, exist_ok=True)

registry_path = (
    audit_dir / "aid652106_compound_identity_registry.csv"
)

compound_registry.to_csv(registry_path, index=False)

# Read-back validation
registry_check = pd.read_csv(registry_path)

assert len(registry_check) == 6310
assert registry_check["PUBCHEM_CID"].is_unique

print("\nSAVED:", registry_path)
print("READ-BACK SHAPE:", registry_check.shape)


COMPOUND IDENTITY REGISTRY VALIDATION
Source records: 6326
Source unique SID: 6326
Registry records: 6310
Registry unique CID: 6310
Missing SMILES: 0

VALIDATION: PASSED

SAVED: H:\My Drive\PROJECT SAINS DATA\OUTPUT\01_Cleaned_Data\Alpha_Synuclein\Unmatched_Audit\aid652106_compound_identity_registry.csv
READ-BACK SHAPE: (6310, 5)


In [ ]:
# ============================================================
# AID652106 METADATA AVAILABILITY AUDIT
# ============================================================

print("=" * 65)
print("MASTER METADATA COLUMNS")
print("=" * 65)

for col in df.columns:
    print(col)

print("\n" + "=" * 65)
print("REGISTRY COLUMNS")
print("=" * 65)

for col in compound_registry.columns:
    print(col)

print("\n" + "=" * 65)
print("REGISTRY SAMPLE")
print("=" * 65)

display(compound_registry.head(5))

MASTER METADATA COLUMNS
PUBCHEM_RESULT_TAG
PUBCHEM_SID
PUBCHEM_CID
PUBCHEM_EXT_DATASOURCE_SMILES
PUBCHEM_ACTIVITY_OUTCOME
PUBCHEM_ACTIVITY_SCORE
PUBCHEM_ACTIVITY_URL
PUBCHEM_ASSAYDATA_COMMENT
Phenotype
Potency
Efficacy
Analysis Comment
Activity_Score
Curve_Description
Fit_LogAC50
Fit_HillSlope
Fit_R2
Fit_InfiniteActivity
Fit_ZeroActivity
Fit_CurveClass
Excluded_Points
Max_Response
Activity at 0.00295 uM
Activity at 0.015 uM
Activity at 0.074 uM
Activity at 0.092 uM
Activity at 0.184 uM
Activity at 0.258 uM
Activity at 0.369 uM
Activity at 0.508 uM
Activity at 0.749 uM
Activity at 1.290 uM
Activity at 1.840 uM
Activity at 2.537 uM
Activity at 3.751 uM
Activity at 6.450 uM
Activity at 9.220 uM
Activity at 12.94 uM
Activity at 18.70 uM
Activity at 36.87 uM
Activity at 46.10 uM
Activity at 64.71 uM
Activity at 92.20 uM
Compound QC
provisional_category
review_category
source_batch

REGISTRY COLUMNS
PUBCHEM_CID
PUBCHEM_EXT_DATASOURCE_SMILES
SID_COUNT
SID_LIST
ACTIVITY_RECORD_COUNT

REGISTRY 

,PUBCHEM_CID,PUBCHEM_EXT_DATASOURCE_SMILES,SID_COUNT,SID_LIST,ACTIVITY_RECORD_COUNT
0,204,C1(C(=O)NC(=O)N1)NC(=O)N,1,26665884,1
1,1189,C1=NC2=C(N1C3C(C(C(O3)CO)O)O)NC(=O)NC2=O,1,26755589,1
2,1337,C1=CC(=CC=C1CCSC(=N)N)CCSC(=N)N,1,90341115,1
3,1570,C1=CC(=S)N(C=C1)O,1,57260394,1
4,1643,CCN(C1=CC=CC=C1)S(=O)(=O)C2=CC(=C(C=C2)OC3=CC=CC=C3)N,1,89854411,1


In [ ]:

# ============================================================
# AID652106 ACTIVITY METADATA QUALITY AUDIT
# ============================================================

import pandas as pd
import numpy as np

audit_cols = [
    "PUBCHEM_ACTIVITY_OUTCOME",
    "Phenotype",
    "Potency",
    "Efficacy",
    "Fit_LogAC50",
    "Fit_HillSlope",
    "Fit_R2",
    "Fit_CurveClass",
    "Max_Response",
    "review_category",
    "Compound QC"
]

print("=" * 70)
print("AID652106 ACTIVITY METADATA AUDIT")
print("=" * 70)

# ------------------------------------------------------------
# 1. COLUMN AVAILABILITY
# ------------------------------------------------------------

available_cols = [c for c in audit_cols if c in df.columns]
missing_cols = [c for c in audit_cols if c not in df.columns]

print("\nAVAILABLE COLUMNS:")
print(available_cols)

print("\nMISSING COLUMNS:")
print(missing_cols)

# ------------------------------------------------------------
# 2. MISSING VALUE SUMMARY
# ------------------------------------------------------------

missing_summary = pd.DataFrame({
    "missing_count": df[available_cols].isna().sum(),
    "missing_percent": (
        df[available_cols].isna().mean() * 100
    ).round(2),
    "data_type": df[available_cols].dtypes.astype(str)
})

print("\nMISSING VALUE SUMMARY")
display(missing_summary)

# ------------------------------------------------------------
# 3. NUMERIC VARIABLE SUMMARY
# ------------------------------------------------------------

numeric_cols = [
    "Potency",
    "Efficacy",
    "Fit_LogAC50",
    "Fit_HillSlope",
    "Fit_R2",
    "Max_Response"
]

numeric_cols = [c for c in numeric_cols if c in df.columns]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

print("\nNUMERIC DESCRIPTIVE STATISTICS")
display(df[numeric_cols].describe().T)

# ------------------------------------------------------------
# 4. CATEGORICAL DISTRIBUTION
# ------------------------------------------------------------

categorical_cols = [
    "PUBCHEM_ACTIVITY_OUTCOME",
    "Phenotype",
    "Fit_CurveClass",
    "review_category",
    "Compound QC"
]

for col in categorical_cols:
    if col in df.columns:
        print(f"\nDISTRIBUTION: {col}")
        display(
            df[col].value_counts(dropna=False).to_frame(
                "record_count"
            )
        )

# ------------------------------------------------------------
# 5. POTENCY AND FIT LOG AC50 AVAILABILITY
# ------------------------------------------------------------

print("\nPOTENCY vs FIT_LOGAC50")

print(
    "Potency available:",
    df["Potency"].notna().sum()
)

print(
    "Fit_LogAC50 available:",
    df["Fit_LogAC50"].notna().sum()
)

print(
    "Both available:",
    (df["Potency"].notna() & df["Fit_LogAC50"].notna()).sum()
)

# ------------------------------------------------------------
# 6. FIT_R2 QUALITY CHECK
# ------------------------------------------------------------

if "Fit_R2" in df.columns:
    print("\nFIT_R2 OUTSIDE 0-1:")
    invalid_r2 = df[
        df["Fit_R2"].notna()
        & ~df["Fit_R2"].between(0, 1)
    ]

    print("Count:", len(invalid_r2))

# ------------------------------------------------------------
# 7. SAVE AUDIT
# ------------------------------------------------------------

audit_dir = ALPHA_CLEANED_DIR / "Unmatched_Audit"
audit_dir.mkdir(parents=True, exist_ok=True)

missing_summary.to_csv(
    audit_dir / "aid652106_activity_missing_audit.csv"
)

df[numeric_cols].describe().T.to_csv(
    audit_dir / "aid652106_activity_numeric_summary.csv"
)

print("\nAUDIT SUMMARY SAVED:", audit_dir)


AID652106 ACTIVITY METADATA AUDIT

AVAILABLE COLUMNS:
['PUBCHEM_ACTIVITY_OUTCOME', 'Phenotype', 'Potency', 'Efficacy', 'Fit_LogAC50', 'Fit_HillSlope', 'Fit_R2', 'Fit_CurveClass', 'Max_Response', 'review_category', 'Compound QC']

MISSING COLUMNS:
[]

MISSING VALUE SUMMARY


,missing_count,missing_percent,data_type
PUBCHEM_ACTIVITY_OUTCOME,0,0.00,str
Phenotype,0,0.00,str
Potency,136,2.15,float64
Efficacy,136,2.15,float64
Fit_LogAC50,23,0.36,float64
Fit_HillSlope,23,0.36,float64
Fit_R2,23,0.36,float64
Fit_CurveClass,0,0.00,float64
Max_Response,0,0.00,float64
review_category,0,0.00,str



NUMERIC DESCRIPTIVE STATISTICS


,count,mean,std,min,25%,50%,75%,max
Potency,6190.0,18.966337,11.872282,0.3548,5.623400,17.7828,31.6228,56.2341
Efficacy,6190.0,161.268191,230.515293,8.5000,45.327125,70.5060,165.1205,2118.5600
Fit_LogAC50,6303.0,-4.847056,0.377568,-6.4500,-5.250000,-4.7500,-4.5000,-4.2500
Fit_HillSlope,6303.0,4.283550,0.822626,0.5000,4.095000,4.4495,4.9549,4.9549
Fit_R2,6303.0,0.981587,0.066354,0.3490,0.992800,0.9987,0.9999,1.0000
Max_Response,6326.0,94.658449,174.189103,-96.8540,14.360275,43.2991,90.5069,1676.8700



DISTRIBUTION: PUBCHEM_ACTIVITY_OUTCOME


,record_count
PUBCHEM_ACTIVITY_OUTCOME,
Inconclusive,5823
Active,503


... (5 outputs left out)

,record_count
review_category,
Inconclusive_Single_Point,5112
Inconclusive_Curve_Response,673
Cellular_Rescue_Active_Provisional,503
Inconclusive_Poor_Fit,38



DISTRIBUTION: Compound QC


,record_count
Compound QC,
QC'd by DPISMR,762
QC'd by Enamine,719
QC'd by Chem Div,626
QC'd by Asinex Ltd.,440
QC'd by Evotec San Francisco,403
QC'd by ChemBridge,384
NaN,315
QC'd by InterBioScreen,292
QC'd by NCI Chemotherapeutic Agents Repository,276



POTENCY vs FIT_LOGAC50
Potency available: 6190
Fit_LogAC50 available: 6303
Both available: 6190

FIT_R2 OUTSIDE 0-1:
Count: 0

AUDIT SUMMARY SAVED: H:\My Drive\PROJECT SAINS DATA\OUTPUT\01_Cleaned_Data\Alpha_Synuclein\Unmatched_Audit


In [ ]:

# ============================================================
# AID652106 MISSINGNESS BY REVIEW CATEGORY
# ============================================================

category_col = "review_category"

required_cols = [
    category_col,
    "Potency",
    "Fit_LogAC50",
    "Fit_R2",
    "Efficacy"
]

assert all(col in df.columns for col in required_cols)

# Ringkasan per kategori
category_missingness = (
    df.groupby(category_col, dropna=False)
    .agg(
        total_records=("PUBCHEM_SID", "size"),

        potency_available=("Potency", "count"),
        potency_missing=("Potency", lambda x: x.isna().sum()),

        fit_logac50_available=("Fit_LogAC50", "count"),
        fit_logac50_missing=("Fit_LogAC50", lambda x: x.isna().sum()),

        fit_r2_available=("Fit_R2", "count"),
        efficacy_available=("Efficacy", "count")
    )
    .reset_index()
)

category_missingness["potency_missing_pct"] = (
    category_missingness["potency_missing"]
    / category_missingness["total_records"] * 100
).round(2)

category_missingness["fit_logac50_missing_pct"] = (
    category_missingness["fit_logac50_missing"]
    / category_missingness["total_records"] * 100
).round(2)

print("=" * 70)
print("MISSINGNESS BY REVIEW CATEGORY")
print("=" * 70)

display(category_missingness)

# ============================================================
# SAVE
# ============================================================

audit_dir = ALPHA_CLEANED_DIR / "Unmatched_Audit"

category_missingness.to_csv(
    audit_dir / "aid652106_missingness_by_category.csv",
    index=False
)

print("SAVED:", audit_dir / "aid652106_missingness_by_category.csv")


MISSINGNESS BY REVIEW CATEGORY


,review_category,total_records,potency_available,potency_missing,fit_logac50_available,fit_logac50_missing,fit_r2_available,efficacy_available,potency_missing_pct,fit_logac50_missing_pct
0,Cellular_Rescue_Active_Provisional,503,503,0,503,0,503,503,0.00,0.00
1,Inconclusive_Curve_Response,673,673,0,673,0,673,673,0.00,0.00
2,Inconclusive_Poor_Fit,38,20,18,38,0,38,20,47.37,0.00
3,Inconclusive_Single_Point,5112,4994,118,5089,23,5089,4994,2.31,0.45


SAVED: H:\My Drive\PROJECT SAINS DATA\OUTPUT\01_Cleaned_Data\Alpha_Synuclein\Unmatched_Audit\aid652106_missingness_by_category.csv


In [ ]:

# ============================================================
# AID652106 MISSING RECORD INVESTIGATION
# ============================================================

# 1. Record missing Potency
missing_potency = df[
    df["Potency"].isna()
].copy()

# 2. Record missing Fit_LogAC50
missing_fitlog = df[
    df["Fit_LogAC50"].isna()
].copy()

# 3. Record missing both
missing_both = df[
    df["Potency"].isna()
    & df["Fit_LogAC50"].isna()
].copy()

# 4. Record missing Potency but has Fit_LogAC50
missing_potency_only = df[
    df["Potency"].isna()
    & df["Fit_LogAC50"].notna()
].copy()

# ============================================================
# SUMMARY
# ============================================================

print("=" * 70)
print("MISSING RECORD INVESTIGATION")
print("=" * 70)

print("Missing Potency:", len(missing_potency))
print("Missing Fit_LogAC50:", len(missing_fitlog))
print("Missing Both:", len(missing_both))
print("Missing Potency Only:", len(missing_potency_only))

# ============================================================
# REVIEW IMPORTANT COLUMNS
# ============================================================

review_cols = [
    "PUBCHEM_SID",
    "PUBCHEM_CID",
    "Phenotype",
    "PUBCHEM_ACTIVITY_OUTCOME",
    "review_category",
    "Potency",
    "Efficacy",
    "Fit_LogAC50",
    "Fit_R2",
    "Fit_CurveClass",
    "Curve_Description",
    "Analysis Comment"
]

review_cols = [
    col for col in review_cols
    if col in df.columns
]

print("\nMISSING POTENCY ONLY")
display(
    missing_potency_only[review_cols].head(20)
)

print("\nMISSING BOTH")
display(
    missing_both[review_cols].head(20)
)

print("\nMISSING FIT_LOGAC50")
display(
    missing_fitlog[review_cols].head(20)
)

# ============================================================
# SAVE
# ============================================================

audit_dir = ALPHA_CLEANED_DIR / "Unmatched_Audit"

missing_potency[review_cols].to_csv(
    audit_dir / "aid652106_missing_potency_records.csv",
    index=False
)

missing_fitlog[review_cols].to_csv(
    audit_dir / "aid652106_missing_fitlog_records.csv",
    index=False
)

missing_both[review_cols].to_csv(
    audit_dir / "aid652106_missing_both_records.csv",
    index=False
)

print("\nMISSING RECORD AUDIT SAVED")


MISSING RECORD INVESTIGATION
Missing Potency: 136
Missing Fit_LogAC50: 23
Missing Both: 23
Missing Potency Only: 113

MISSING POTENCY ONLY


,PUBCHEM_SID,PUBCHEM_CID,Phenotype,PUBCHEM_ACTIVITY_OUTCOME,review_category,Potency,Efficacy,Fit_LogAC50,Fit_R2,Fit_CurveClass,Curve_Description,Analysis Comment
73,125306222,53383534.0,Activator,Inconclusive,Inconclusive_Single_Point,NaN,NaN,-4.45,0.5348,5.0,NaN,NaN
140,124894702,6249420.0,Activator,Inconclusive,Inconclusive_Poor_Fit,NaN,NaN,-5.20,0.3819,5.0,NaN,NaN
245,124881567,3117.0,Activator,Inconclusive,Inconclusive_Single_Point,NaN,NaN,-5.25,0.7826,5.0,NaN,NaN
255,124893543,4041506.0,Activator,Inconclusive,Inconclusive_Single_Point,NaN,NaN,-5.00,0.8506,5.0,NaN,NaN
280,124755424,53301253.0,Activator,Inconclusive,Inconclusive_Single_Point,NaN,NaN,-4.45,0.8171,5.0,NaN,NaN
396,121286188,51361436.0,Activator,Inconclusive,Inconclusive_Single_Point,NaN,NaN,-5.15,0.7483,5.0,NaN,NaN
407,121286573,138252.0,Activator,Inconclusive,Inconclusive_Single_Point,NaN,NaN,-4.45,0.6365,5.0,NaN,NaN
422,121285984,51361301.0,Activator,Inconclusive,Inconclusive_Single_Point,NaN,NaN,-4.50,0.9693,5.0,NaN,NaN
507,121283926,51359461.0,Activator,Inconclusive,Inconclusive_Poor_Fit,NaN,NaN,-5.15,0.3866,5.0,NaN,NaN
558,103163193,49795551.0,Activator,Inconclusive,Inconclusive_Single_Point,NaN,NaN,-4.45,0.7505,5.0,NaN,NaN



MISSING BOTH


,PUBCHEM_SID,PUBCHEM_CID,Phenotype,PUBCHEM_ACTIVITY_OUTCOME,review_category,Potency,Efficacy,Fit_LogAC50,Fit_R2,Fit_CurveClass,Curve_Description,Analysis Comment
481,121285697,51361033.0,Activator,Inconclusive,Inconclusive_Single_Point,NaN,NaN,NaN,NaN,5.0,NaN,NaN
574,121282678,51358928.0,Activator,Inconclusive,Inconclusive_Single_Point,NaN,NaN,NaN,NaN,5.0,NaN,NaN
598,103162462,3978434.0,Activator,Inconclusive,Inconclusive_Single_Point,NaN,NaN,NaN,NaN,5.0,NaN,NaN
1008,99494343,46947906.0,Activator,Inconclusive,Inconclusive_Single_Point,NaN,NaN,NaN,NaN,5.0,NaN,NaN
1667,85270011,7679149.0,Activator,Inconclusive,Inconclusive_Single_Point,NaN,NaN,NaN,NaN,3.0,Single point of activity,NaN
1701,85200187,2367156.0,Activator,Inconclusive,Inconclusive_Single_Point,NaN,NaN,NaN,NaN,3.0,Single point of activity,NaN
1722,85199669,3713455.0,Activator,Inconclusive,Inconclusive_Single_Point,NaN,NaN,NaN,NaN,5.0,NaN,NaN
1739,85199006,7906517.0,Activator,Inconclusive,Inconclusive_Single_Point,NaN,NaN,NaN,NaN,3.0,Single point of activity,NaN
1799,57268534,25163622.0,Activator,Inconclusive,Inconclusive_Single_Point,NaN,NaN,NaN,NaN,3.0,Single point of activity,NaN
1825,57267952,9418814.0,Activator,Inconclusive,Inconclusive_Single_Point,NaN,NaN,NaN,NaN,3.0,Single point of activity,NaN



MISSING FIT_LOGAC50


,PUBCHEM_SID,PUBCHEM_CID,Phenotype,PUBCHEM_ACTIVITY_OUTCOME,review_category,Potency,Efficacy,Fit_LogAC50,Fit_R2,Fit_CurveClass,Curve_Description,Analysis Comment
481,121285697,51361033.0,Activator,Inconclusive,Inconclusive_Single_Point,NaN,NaN,NaN,NaN,5.0,NaN,NaN
574,121282678,51358928.0,Activator,Inconclusive,Inconclusive_Single_Point,NaN,NaN,NaN,NaN,5.0,NaN,NaN
598,103162462,3978434.0,Activator,Inconclusive,Inconclusive_Single_Point,NaN,NaN,NaN,NaN,5.0,NaN,NaN
1008,99494343,46947906.0,Activator,Inconclusive,Inconclusive_Single_Point,NaN,NaN,NaN,NaN,5.0,NaN,NaN
1667,85270011,7679149.0,Activator,Inconclusive,Inconclusive_Single_Point,NaN,NaN,NaN,NaN,3.0,Single point of activity,NaN
1701,85200187,2367156.0,Activator,Inconclusive,Inconclusive_Single_Point,NaN,NaN,NaN,NaN,3.0,Single point of activity,NaN
1722,85199669,3713455.0,Activator,Inconclusive,Inconclusive_Single_Point,NaN,NaN,NaN,NaN,5.0,NaN,NaN
1739,85199006,7906517.0,Activator,Inconclusive,Inconclusive_Single_Point,NaN,NaN,NaN,NaN,3.0,Single point of activity,NaN
1799,57268534,25163622.0,Activator,Inconclusive,Inconclusive_Single_Point,NaN,NaN,NaN,NaN,3.0,Single point of activity,NaN
1825,57267952,9418814.0,Activator,Inconclusive,Inconclusive_Single_Point,NaN,NaN,NaN,NaN,3.0,Single point of activity,NaN



MISSING RECORD AUDIT SAVED


In [ ]:

# ============================================================
# AID652106 POTENCY vs FIT_LOGAC50 CONSISTENCY AUDIT
# ============================================================

import numpy as np
import pandas as pd

# Ambil record yang memiliki kedua parameter
comparison = df[
    df["Potency"].notna()
    & df["Fit_LogAC50"].notna()
].copy()

comparison["Potency"] = pd.to_numeric(
    comparison["Potency"], errors="coerce"
)

comparison["Fit_LogAC50"] = pd.to_numeric(
    comparison["Fit_LogAC50"], errors="coerce"
)

comparison = comparison.dropna(
    subset=["Potency", "Fit_LogAC50"]
)

# Konversi log AC50 (M) menjadi AC50 (nM)
comparison["Potency_calculated_nM"] = (
    10 ** (comparison["Fit_LogAC50"] + 9)
)

# Hitung selisih
comparison["absolute_difference"] = (
    comparison["Potency"]
    - comparison["Potency_calculated_nM"]
).abs()

comparison["relative_difference_pct"] = (
    comparison["absolute_difference"]
    / comparison["Potency_calculated_nM"]
    * 100
)

# Rasio aktual terhadap hasil konversi
comparison["potency_ratio"] = (
    comparison["Potency"]
    / comparison["Potency_calculated_nM"]
)

# ============================================================
# SUMMARY
# ============================================================

print("=" * 70)
print("POTENCY vs FIT_LOGAC50 CONSISTENCY")
print("=" * 70)

print("Records compared:", len(comparison))

display(
    comparison[
        [
            "Potency",
            "Fit_LogAC50",
            "Potency_calculated_nM",
            "absolute_difference",
            "relative_difference_pct"
        ]
    ].describe().T
)

# Pemeriksaan toleransi relatif 1%
comparison["within_1_percent"] = (
    comparison["relative_difference_pct"] <= 1
)

print("\nWITHIN 1%:", comparison["within_1_percent"].sum())
print("OUTSIDE 1%:", (~comparison["within_1_percent"]).sum())

print(
    "Median relative difference (%):",
    comparison["relative_difference_pct"].median()
)

# Tampilkan contoh ketidaksesuaian terbesar
print("\nLARGEST DIFFERENCES")

display(
    comparison.sort_values(
        "relative_difference_pct",
        ascending=False
    )[
        [
            "PUBCHEM_SID",
            "PUBCHEM_CID",
            "Potency",
            "Fit_LogAC50",
            "Potency_calculated_nM",
            "relative_difference_pct"
        ]
    ].head(15)
)

# ============================================================
# SAVE
# ============================================================

audit_dir = ALPHA_CLEANED_DIR / "Unmatched_Audit"

comparison.to_csv(
    audit_dir / "aid652106_potency_fitlog_consistency.csv",
    index=False
)

print("\nSAVED:")
print(audit_dir / "aid652106_potency_fitlog_consistency.csv")


POTENCY vs FIT_LOGAC50 CONSISTENCY
Records compared: 6190


,count,mean,std,min,25%,50%,75%,max
Potency,6190.0,18.966337,1.187228e+01,0.354800,5.623400,17.7828,31.622800,56.234100
Fit_LogAC50,6190.0,-4.849212,3.779355e-01,-6.450000,-5.250000,-4.7500,-4.500000,-4.250000
Potency_calculated_nM,6190.0,18966.343998,1.187229e+04,354.813389,5623.413252,17782.7941,31622.776602,56234.132519
absolute_difference,6190.0,18947.377661,1.186042e+04,354.458589,5617.789852,17765.0113,31591.153802,56177.898419
relative_difference_pct,6190.0,99.900000,6.934434e-07,99.899993,99.900000,99.9000,99.900000,99.900007



WITHIN 1%: 0
OUTSIDE 1%: 6190
Median relative difference (%): 99.90000010400519

LARGEST DIFFERENCES


,PUBCHEM_SID,PUBCHEM_CID,Potency,Fit_LogAC50,Potency_calculated_nM,relative_difference_pct
4982,17416266,135400342.0,0.5623,-6.25,562.341325,99.900007
6094,3717785,3000139.0,0.5623,-6.25,562.341325,99.900007
4263,24708106,5459622.0,0.5623,-6.25,562.341325,99.900007
4377,24386635,16010133.0,0.5623,-6.25,562.341325,99.900007
5146,17401889,6619501.0,0.5623,-6.25,562.341325,99.900007
1504,89854188,258584.0,0.5623,-6.25,562.341325,99.900007
5478,14728678,6096756.0,0.5623,-6.25,562.341325,99.900007
3810,24817100,5773281.0,0.5623,-6.25,562.341325,99.900007
6247,855519,68589.0,0.5623,-6.25,562.341325,99.900007
3384,26666864,313619.0,0.5623,-6.25,562.341325,99.900007



SAVED:
H:\My Drive\PROJECT SAINS DATA\OUTPUT\01_Cleaned_Data\Alpha_Synuclein\Unmatched_Audit\aid652106_potency_fitlog_consistency.csv


In [ ]:
import numpy as np
import pandas as pd

# Gunakan master activator yang sudah berhasil dibaca
df_check = df.copy()

# Pastikan kolom numerik
df_check["Potency"] = pd.to_numeric(
    df_check["Potency"], errors="coerce"
)

df_check["Fit_LogAC50"] = pd.to_numeric(
    df_check["Fit_LogAC50"], errors="coerce"
)

# Konversi logAC50 (log10 M) menjadi µM
df_check["AC50_calculated_uM"] = (
    10 ** df_check["Fit_LogAC50"] * 1e6
)

# Hanya record dengan kedua nilai tersedia
valid = df_check.dropna(
    subset=["Potency", "Fit_LogAC50"]
).copy()

# Hitung selisih
valid["absolute_difference"] = (
    valid["Potency"] - valid["AC50_calculated_uM"]
).abs()

valid["relative_difference_pct"] = (
    valid["absolute_difference"]
    / valid["AC50_calculated_uM"]
) * 100

# Ringkasan
within_1pct = (
    valid["relative_difference_pct"] <= 1
).sum()

print("=" * 65)
print("POTENCY vs FIT_LOGAC50 — µM UNIT VALIDATION")
print("=" * 65)

print("Records compared:", len(valid))
print("Within 1%:", within_1pct)
print("Outside 1%:", len(valid) - within_1pct)

print(
    "Median relative difference (%):",
    valid["relative_difference_pct"].median()
)

print("\nSAMPLE COMPARISON:")
display(
    valid[
        [
            "PUBCHEM_SID",
            "PUBCHEM_CID",
            "Potency",
            "Fit_LogAC50",
            "AC50_calculated_uM",
            "relative_difference_pct"
        ]
    ].head(15)
)

# Simpan hasil audit
audit_path = UNMATCHED_AUDIT_DIR / (
    "aid652106_potency_fitlog_consistency_uM.csv"
)

valid.to_csv(audit_path, index=False)

print("\nSAVED:", audit_path)

POTENCY vs FIT_LOGAC50 — µM UNIT VALIDATION
Records compared: 6190
Within 1%: 6190
Outside 1%: 0
Median relative difference (%): 0.00011602328482168746

SAMPLE COMPARISON:


,PUBCHEM_SID,PUBCHEM_CID,Potency,Fit_LogAC50,AC50_calculated_uM,relative_difference_pct
0,125306360,53383643.0,35.4813,-4.45,35.481339,0.000110
1,125306368,53383649.0,31.6228,-4.50,31.622777,0.000074
2,125306375,53383656.0,8.9125,-5.05,8.912509,0.000105
3,125306195,53383521.0,31.6228,-4.50,31.622777,0.000074
4,125306199,53383525.0,0.5012,-6.30,0.501187,0.002547
5,125306038,53383365.0,12.5893,-4.90,12.589254,0.000364
6,125306039,53383366.0,7.0795,-5.15,7.079458,0.000595
7,125306041,53383368.0,15.8489,-4.80,15.848932,0.000201
8,125306046,53383373.0,15.8489,-4.80,15.848932,0.000201
9,125306048,53383375.0,7.0795,-5.15,7.079458,0.000595


NameError: name 'UNMATCHED_AUDIT_DIR' is not defined

In [ ]:
from pathlib import Path

# Definisikan folder output audit
UNMATCHED_AUDIT_DIR = Path(
    r"H:\My Drive\PROJECT SAINS DATA\OUTPUT"
    r"\01_Cleaned_Data\Alpha_Synuclein\Unmatched_Audit"
)

# Pastikan folder tersedia
UNMATCHED_AUDIT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# Simpan hasil validasi
audit_path = (
    UNMATCHED_AUDIT_DIR
    / "aid652106_potency_fitlog_consistency_uM.csv"
)

valid.to_csv(audit_path, index=False)

# Read-back validation
check_saved = pd.read_csv(audit_path)

print("=" * 65)
print("SAVE VALIDATION")
print("=" * 65)

print("Saved:", audit_path)
print("Original shape:", valid.shape)
print("Read-back shape:", check_saved.shape)
print(
    "SID unique:",
    check_saved["PUBCHEM_SID"].nunique()
)

assert len(check_saved) == len(valid)
assert check_saved["PUBCHEM_SID"].nunique() == len(valid)

print("\nVALIDATION: PASSED")

SAVE VALIDATION
Saved: H:\My Drive\PROJECT SAINS DATA\OUTPUT\01_Cleaned_Data\Alpha_Synuclein\Unmatched_Audit\aid652106_potency_fitlog_consistency_uM.csv
Original shape: (6190, 50)
Read-back shape: (6190, 50)
SID unique: 6190

VALIDATION: PASSED


In [ ]:
import pandas as pd
from pathlib import Path

# Lokasi master activator
MASTER_PATH = (
    UNMATCHED_AUDIT_DIR
    / "aid652106_all_activator_review.csv"
)

df_master = pd.read_csv(MASTER_PATH)

print("=" * 65)
print("ENDPOINT & ELIGIBILITY AUDIT")
print("=" * 65)

print("Shape:", df_master.shape)
print("Unique SID:", df_master["PUBCHEM_SID"].nunique())

# 1. Distribusi kategori
print("\n1. REVIEW CATEGORY")
print(df_master["review_category"].value_counts(dropna=False))

# 2. Distribusi phenotype
print("\n2. PHENOTYPE")
print(df_master["Phenotype"].value_counts(dropna=False))

# 3. Distribusi activity outcome
print("\n3. ACTIVITY OUTCOME")
print(
    df_master["PUBCHEM_ACTIVITY_OUTCOME"]
    .value_counts(dropna=False)
)

# 4. Cross-tab kategori vs phenotype
print("\n4. CATEGORY vs PHENOTYPE")
category_phenotype = pd.crosstab(
    df_master["review_category"],
    df_master["Phenotype"],
    margins=True
)
display(category_phenotype)

# 5. Cross-tab phenotype vs outcome
print("\n5. PHENOTYPE vs ACTIVITY OUTCOME")
phenotype_outcome = pd.crosstab(
    df_master["Phenotype"],
    df_master["PUBCHEM_ACTIVITY_OUTCOME"],
    margins=True
)
display(phenotype_outcome)

# 6. Simpan hasil audit
category_phenotype.to_csv(
    UNMATCHED_AUDIT_DIR / "aid652106_category_phenotype_audit.csv"
)

phenotype_outcome.to_csv(
    UNMATCHED_AUDIT_DIR / "aid652106_phenotype_outcome_audit.csv"
)

print("\nAUDIT TABLES SAVED")

ENDPOINT & ELIGIBILITY AUDIT
Shape: (6326, 47)
Unique SID: 6326

1. REVIEW CATEGORY
review_category
Inconclusive_Single_Point             5112
Inconclusive_Curve_Response            673
Cellular_Rescue_Active_Provisional     503
Inconclusive_Poor_Fit                   38
Name: count, dtype: int64

2. PHENOTYPE
Phenotype
Activator    6326
Name: count, dtype: int64

3. ACTIVITY OUTCOME
PUBCHEM_ACTIVITY_OUTCOME
Inconclusive    5823
Active           503
Name: count, dtype: int64

4. CATEGORY vs PHENOTYPE


Phenotype,Activator,All
review_category,,
Cellular_Rescue_Active_Provisional,503,503
Inconclusive_Curve_Response,673,673
Inconclusive_Poor_Fit,38,38
Inconclusive_Single_Point,5112,5112
All,6326,6326



5. PHENOTYPE vs ACTIVITY OUTCOME


PUBCHEM_ACTIVITY_OUTCOME,Active,Inconclusive,All
Phenotype,,,
Activator,503,5823,6326
All,503,5823,6326



AUDIT TABLES SAVED


In [ ]:
# ============================================================
# AID652106 — ACTIVE PROVISIONAL QUALITY AUDIT
# ============================================================

active = df_master[
    df_master["review_category"]
    == "Cellular_Rescue_Active_Provisional"
].copy()

print("=" * 65)
print("ACTIVE PROVISIONAL QUALITY AUDIT")
print("=" * 65)

print("Records:", len(active))
print("Unique SID:", active["PUBCHEM_SID"].nunique())

# 1. Distribusi curve class
print("\n1. FIT CURVE CLASS")
display(
    active["Fit_CurveClass"]
    .value_counts(dropna=False)
    .rename_axis("Fit_CurveClass")
    .reset_index(name="count")
)

# 2. Distribusi curve description
print("\n2. CURVE DESCRIPTION")
display(
    active["Curve_Description"]
    .value_counts(dropna=False)
    .rename_axis("Curve_Description")
    .reset_index(name="count")
)

# 3. Ketersediaan parameter
parameter_cols = [
    "Potency",
    "Fit_LogAC50",
    "Fit_R2",
    "Fit_HillSlope",
    "Efficacy",
    "Max_Response",
]

print("\n3. PARAMETER AVAILABILITY")

availability = pd.DataFrame({
    "available": active[parameter_cols].notna().sum(),
    "missing": active[parameter_cols].isna().sum(),
    "missing_pct": (
        active[parameter_cols].isna().mean() * 100
    ).round(2)
})

display(availability)

# 4. Activity outcome
print("\n4. ACTIVITY OUTCOME")
print(
    active["PUBCHEM_ACTIVITY_OUTCOME"]
    .value_counts(dropna=False)
)

# 5. Numeric summary
numeric_cols = [
    "Potency",
    "Fit_LogAC50",
    "Fit_R2",
    "Fit_HillSlope",
    "Efficacy",
    "Max_Response",
]

print("\n5. NUMERIC SUMMARY")
display(active[numeric_cols].describe().T)

# 6. Simpan hasil audit
quality_path = (
    UNMATCHED_AUDIT_DIR
    / "aid652106_active_provisional_quality_audit.csv"
)

active.to_csv(quality_path, index=False)

print("\nSaved:", quality_path)
print("Audit records:", len(active))
print("Unique SID:", active["PUBCHEM_SID"].nunique())

ACTIVE PROVISIONAL QUALITY AUDIT
Records: 503
Unique SID: 503

1. FIT CURVE CLASS


,Fit_CurveClass,count
0,2.1,380
1,2.2,71
2,1.1,46
3,1.2,6



2. CURVE DESCRIPTION


,Curve_Description,count
0,Partial curve; high efficacy,380
1,Partial curve; partial efficacy,71
2,Complete curve; high efficacy,46
3,Complete curve; partial efficacy,6



3. PARAMETER AVAILABILITY


,available,missing,missing_pct
Potency,503,0,0.0
Fit_LogAC50,503,0,0.0
Fit_R2,503,0,0.0
Fit_HillSlope,503,0,0.0
Efficacy,503,0,0.0
Max_Response,503,0,0.0



4. ACTIVITY OUTCOME
PUBCHEM_ACTIVITY_OUTCOME
Active    503
Name: count, dtype: int64

5. NUMERIC SUMMARY


,count,mean,std,min,25%,50%,75%,max
Potency,503.0,13.474535,7.395081,0.5623,7.07950,12.5893,17.7828,35.4813
Fit_LogAC50,503.0,-4.970186,0.344966,-6.2500,-5.15000,-4.9000,-4.7500,-4.4500
Fit_R2,503.0,0.995545,0.012017,0.9012,0.99670,0.9992,0.9999,1.0000
Fit_HillSlope,503.0,3.747829,0.968087,0.6000,3.29750,4.0950,4.5045,4.9549
Efficacy,503.0,263.281294,275.653167,8.5000,97.93855,152.9760,320.2425,1857.1100
Max_Response,503.0,206.603376,250.358859,-72.8557,74.26150,119.4140,245.8645,1486.2000



Saved: H:\My Drive\PROJECT SAINS DATA\OUTPUT\01_Cleaned_Data\Alpha_Synuclein\Unmatched_Audit\aid652106_active_provisional_quality_audit.csv
Audit records: 503
Unique SID: 503


In [ ]:
# ============================================================
# AID652106 — RESPONSE & CURVE PARAMETER AUDIT
# ============================================================

import numpy as np
import pandas as pd

active = df_master[
    df_master["review_category"]
    == "Cellular_Rescue_Active_Provisional"
].copy()

# ------------------------------------------------------------
# 1. Numeric conversion untuk parameter audit
# ------------------------------------------------------------

audit_cols = [
    "Fit_R2",
    "Fit_HillSlope",
    "Efficacy",
    "Max_Response",
    "Potency",
    "Fit_LogAC50",
    "Fit_InfiniteActivity",
    "Fit_ZeroActivity",
    "Excluded_Points",
    "Activity_Score",
]

for col in audit_cols:
    if col in active.columns:
        active[col] = pd.to_numeric(
            active[col],
            errors="coerce"
        )

# ------------------------------------------------------------
# 2. Descriptive statistics
# ------------------------------------------------------------

available_cols = [
    col for col in audit_cols
    if col in active.columns
]

print("=" * 65)
print("RESPONSE PARAMETER SUMMARY")
print("=" * 65)

display(
    active[available_cols]
    .describe(percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99])
    .T
)

# ------------------------------------------------------------
# 3. Inspect extreme efficacy
# ------------------------------------------------------------

print("\nTOP 15 HIGHEST EFFICACY")

display(
    active.nlargest(15, "Efficacy")[
        [
            "PUBCHEM_SID",
            "PUBCHEM_CID",
            "Efficacy",
            "Max_Response",
            "Fit_R2",
            "Fit_HillSlope",
            "Fit_CurveClass",
            "Curve_Description",
        ]
    ]
)

# ------------------------------------------------------------
# 4. Inspect lowest Max_Response
# ------------------------------------------------------------

print("\nLOWEST 15 MAX RESPONSE")

display(
    active.nsmallest(15, "Max_Response")[
        [
            "PUBCHEM_SID",
            "PUBCHEM_CID",
            "Efficacy",
            "Max_Response",
            "Fit_R2",
            "Fit_HillSlope",
            "Fit_CurveClass",
            "Curve_Description",
        ]
    ]
)

# ------------------------------------------------------------
# 5. Inspect HillSlope extremes
# ------------------------------------------------------------

print("\nHIGHEST 15 HILL SLOPE")

display(
    active.nlargest(15, "Fit_HillSlope")[
        [
            "PUBCHEM_SID",
            "PUBCHEM_CID",
            "Fit_HillSlope",
            "Fit_R2",
            "Efficacy",
            "Max_Response",
            "Fit_CurveClass",
        ]
    ]
)

# ------------------------------------------------------------
# 6. Inspect concentration response columns
# ------------------------------------------------------------

response_cols = [
    col for col in active.columns
    if "uM" in col
]

print("\nCONCENTRATION RESPONSE COLUMNS")
print("Number of columns:", len(response_cols))
print(response_cols)

# ------------------------------------------------------------
# 7. Check additional QC fields
# ------------------------------------------------------------

qc_cols = [
    col for col in [
        "Compound QC",
        "Excluded_Points",
        "Fit_InfiniteActivity",
        "Fit_ZeroActivity",
    ]
    if col in active.columns
]

print("\nADDITIONAL QC DISTRIBUTION")

for col in qc_cols:
    print(f"\n{col}")
    print(active[col].value_counts(dropna=False).head(20))

RESPONSE PARAMETER SUMMARY


,count,mean,std,min,1%,5%,25%,50%,75%,95%,99%,max
Fit_R2,503.0,0.995545,0.012017,0.9012,0.925962,0.97863,0.99670,0.9992,0.9999,1.00000,1.000000,1.0000
Fit_HillSlope,503.0,3.747829,0.968087,0.6000,1.211552,1.69845,3.29750,4.0950,4.5045,4.95490,4.954900,4.9549
Efficacy,503.0,263.281294,275.653167,8.5000,63.002720,69.88430,97.93855,152.9760,320.2425,886.09600,1391.228800,1857.1100
Max_Response,503.0,206.603376,250.358859,-72.8557,-51.261088,-0.82285,74.26150,119.4140,245.8645,774.76850,1262.306000,1486.2000
Potency,503.0,13.474535,7.395081,0.5623,0.562300,3.16230,7.07950,12.5893,17.7828,25.11890,31.622800,35.4813
Fit_LogAC50,503.0,-4.970186,0.344966,-6.2500,-6.250000,-5.50000,-5.15000,-4.9000,-4.7500,-4.60000,-4.500000,-4.4500
Fit_InfiniteActivity,503.0,264.020444,273.824734,61.8657,66.429298,72.24481,100.10650,154.7500,322.4135,885.15880,1379.488400,1854.7800
Fit_ZeroActivity,503.0,1.158587,12.484901,-75.3452,-26.283520,-8.11735,-1.82065,1.1853,4.3546,11.30929,18.557236,167.3700
Excluded_Points,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Activity_Score,503.0,49.210736,14.545550,40.0000,40.000000,41.00000,42.00000,43.0000,48.0000,88.00000,100.000000,100.0000



TOP 15 HIGHEST EFFICACY


,PUBCHEM_SID,PUBCHEM_CID,Efficacy,Max_Response,Fit_R2,Fit_HillSlope,Fit_CurveClass,Curve_Description
3937,24810018,3138340.0,1857.11,1486.2000,1.0000,3.6772,2.1,Partial curve; high efficacy
6244,856259,135449532.0,1609.15,1419.0100,1.0000,4.0950,2.1,Partial curve; high efficacy
3511,26658341,933665.0,1484.08,1447.1700,1.0000,4.9549,2.1,Partial curve; high efficacy
2807,49669245,4429074.0,1451.42,1279.4400,1.0000,4.0950,2.1,Partial curve; high efficacy
1467,90340832,24867847.0,1424.78,1100.3600,0.9650,0.6000,2.1,Partial curve; high efficacy
5581,14721281,1933256.0,1391.82,1370.9400,0.9999,4.5045,2.1,Partial curve; high efficacy
176,124894477,16445890.0,1362.26,1091.1600,1.0000,3.6772,2.1,Partial curve; high efficacy
2111,51090338,4100358.0,1329.68,1264.9200,1.0000,4.0950,2.1,Partial curve; high efficacy
5596,14721179,3138330.0,1219.10,1120.8800,1.0000,4.0450,2.1,Partial curve; high efficacy
931,99495270,135438745.0,1216.80,1134.2200,0.9995,3.6772,2.1,Partial curve; high efficacy



LOWEST 15 MAX RESPONSE


,PUBCHEM_SID,PUBCHEM_CID,Efficacy,Max_Response,Fit_R2,Fit_HillSlope,Fit_CurveClass,Curve_Description
4671,22401490,135504329.0,75.3953,-72.8557,1.0000,4.0450,1.2,Complete curve; partial efficacy
1391,92764047,3246517.0,77.5066,-65.2780,1.0000,1.3723,2.2,Partial curve; partial efficacy
1656,85271222,2468342.0,516.0190,-58.6658,1.0000,4.0950,2.1,Partial curve; high efficacy
6219,858743,5389212.0,301.0830,-55.3404,1.0000,4.0950,2.1,Partial curve; high efficacy
1387,92763868,122815.0,1154.2800,-55.1652,0.9981,4.9549,1.1,Complete curve; high efficacy
3428,26665251,75253.0,339.6610,-51.3435,1.0000,1.9673,2.1,Partial curve; high efficacy
2666,49677916,20914270.0,113.7580,-47.2229,0.9996,4.9549,1.1,Complete curve; high efficacy
5736,7970078,2557756.0,131.1440,-45.8681,1.0000,4.0450,2.1,Partial curve; high efficacy
4920,17504202,691106.0,109.7620,-45.4878,1.0000,2.1876,2.1,Partial curve; high efficacy
3402,26665685,16409834.0,163.7460,-43.2808,1.0000,4.4495,2.1,Partial curve; high efficacy



HIGHEST 15 HILL SLOPE


,PUBCHEM_SID,PUBCHEM_CID,Fit_HillSlope,Fit_R2,Efficacy,Max_Response,Fit_CurveClass
49,125306923,53384189.0,4.9549,0.9996,112.8810,122.3310,2.1
118,125305192,53382548.0,4.9549,1.0000,285.4710,-2.4312,2.1
359,124753741,53299763.0,4.9549,1.0000,178.5040,5.0199,2.1
477,121285611,51360947.0,4.9549,0.9992,573.2490,559.8340,1.1
613,103162579,49795170.0,4.9549,0.9997,101.4360,112.5540,2.1
710,103075883,49790789.0,4.9549,0.9973,93.0073,105.9310,2.1
1061,99456139,46943636.0,4.9549,0.9990,91.1886,92.6643,2.1
1101,99361138,46904988.0,4.9549,0.9994,77.9440,73.8386,2.2
1218,99358329,4042376.0,4.9549,0.9999,134.5810,137.8420,1.1
1387,92763868,122815.0,4.9549,0.9981,1154.2800,-55.1652,1.1



CONCENTRATION RESPONSE COLUMNS
Number of columns: 21
['Activity at 0.00295 uM', 'Activity at 0.015 uM', 'Activity at 0.074 uM', 'Activity at 0.092 uM', 'Activity at 0.184 uM', 'Activity at 0.258 uM', 'Activity at 0.369 uM', 'Activity at 0.508 uM', 'Activity at 0.749 uM', 'Activity at 1.290 uM', 'Activity at 1.840 uM', 'Activity at 2.537 uM', 'Activity at 3.751 uM', 'Activity at 6.450 uM', 'Activity at 9.220 uM', 'Activity at 12.94 uM', 'Activity at 18.70 uM', 'Activity at 36.87 uM', 'Activity at 46.10 uM', 'Activity at 64.71 uM', 'Activity at 92.20 uM']

ADDITIONAL QC DISTRIBUTION

Compound QC
Compound QC
QC'd by DPISMR                                        75
QC'd by Enamine                                       46
QC'd by Chem Div                                      44
QC'd by Asinex Ltd.                                   41
QC'd by NCI Chemotherapeutic Agents Repository        37
QC'd by Evotec San Francisco                          32
QC'd by Life Chemicals                      

In [ ]:
# ============================================================
# AID652106 — CONCENTRATION RESPONSE AUDIT
# ============================================================

import numpy as np
import pandas as pd

active = df_master[
    df_master["review_category"]
    == "Cellular_Rescue_Active_Provisional"
].copy()

# Identifikasi kolom respons per konsentrasi
response_cols = [
    col for col in active.columns
    if col.startswith("Activity at ") and "uM" in col
]

# Konversi menjadi numerik
active[response_cols] = active[response_cols].apply(
    pd.to_numeric,
    errors="coerce"
)

# ------------------------------------------------------------
# 1. Missingness respons per konsentrasi
# ------------------------------------------------------------

print("=" * 65)
print("1. RESPONSE DATA COMPLETENESS")
print("=" * 65)

response_missing = pd.DataFrame({
    "available": active[response_cols].notna().sum(),
    "missing": active[response_cols].isna().sum(),
    "missing_pct": (
        active[response_cols].isna().mean() * 100
    ).round(2)
})

display(response_missing)

# ------------------------------------------------------------
# 2. Jumlah titik respons per senyawa
# ------------------------------------------------------------

active["response_points_available"] = (
    active[response_cols].notna().sum(axis=1)
)

print("\n2. RESPONSE POINTS PER RECORD")

print(
    active["response_points_available"]
    .value_counts()
    .sort_index()
)

# ------------------------------------------------------------
# 3. Statistik respons per konsentrasi
# ------------------------------------------------------------

print("\n3. RESPONSE DISTRIBUTION BY CONCENTRATION")

display(
    active[response_cols].describe().T
)

# ------------------------------------------------------------
# 4. Ringkasan respons setiap record
# ------------------------------------------------------------

active["response_min"] = active[response_cols].min(axis=1)
active["response_max"] = active[response_cols].max(axis=1)
active["response_mean"] = active[response_cols].mean(axis=1)
active["negative_response_count"] = (
    active[response_cols].lt(0).sum(axis=1)
)

print("\n4. NEGATIVE RESPONSE SUMMARY")

print(
    active["negative_response_count"]
    .value_counts()
    .sort_index()
)

# ------------------------------------------------------------
# 5. Record dengan efficacy tertinggi
# ------------------------------------------------------------

print("\n5. HIGHEST EFFICACY — RAW RESPONSE")

display(
    active.nlargest(10, "Efficacy")[
        [
            "PUBCHEM_SID",
            "PUBCHEM_CID",
            "Efficacy",
            "Max_Response",
            "Fit_R2",
            "Fit_HillSlope",
            "response_points_available",
            *response_cols
        ]
    ]
)

# ------------------------------------------------------------
# 6. Record dengan Max_Response negatif terendah
# ------------------------------------------------------------

print("\n6. NEGATIVE MAX_RESPONSE — RAW RESPONSE")

display(
    active.nsmallest(10, "Max_Response")[
        [
            "PUBCHEM_SID",
            "PUBCHEM_CID",
            "Efficacy",
            "Max_Response",
            "Fit_R2",
            "Fit_HillSlope",
            "response_points_available",
            *response_cols
        ]
    ]
)

# ------------------------------------------------------------
# 7. Investigasi HillSlope yang mendekati nilai maksimum
# ------------------------------------------------------------

print("\n7. HILLSLOPE FREQUENCY")

display(
    active["Fit_HillSlope"]
    .round(4)
    .value_counts()
    .head(15)
    .rename_axis("HillSlope")
    .reset_index(name="count")
)

# ------------------------------------------------------------
# 8. Simpan hasil audit tambahan
# ------------------------------------------------------------

response_audit = active[
    [
        "PUBCHEM_SID",
        "PUBCHEM_CID",
        "Efficacy",
        "Max_Response",
        "Fit_R2",
        "Fit_HillSlope",
        "response_points_available",
        "response_min",
        "response_max",
        "response_mean",
        "negative_response_count",
        *response_cols
    ]
].copy()

response_audit_path = (
    UNMATCHED_AUDIT_DIR
    / "aid652106_concentration_response_audit.csv"
)

response_audit.to_csv(
    response_audit_path,
    index=False
)

print("\nSAVED:", response_audit_path)
print("Shape:", response_audit.shape)
print("Unique SID:", response_audit["PUBCHEM_SID"].nunique())

1. RESPONSE DATA COMPLETENESS


,available,missing,missing_pct
Activity at 0.00295 uM,7,496,98.61
Activity at 0.015 uM,7,496,98.61
Activity at 0.074 uM,7,496,98.61
Activity at 0.092 uM,0,503,100.00
Activity at 0.184 uM,0,503,100.00
Activity at 0.258 uM,0,503,100.00
Activity at 0.369 uM,503,0,0.00
Activity at 0.508 uM,0,503,100.00
Activity at 0.749 uM,0,503,100.00
Activity at 1.290 uM,0,503,100.00



2. RESPONSE POINTS PER RECORD
response_points_available
3      2
4    494
7      7
Name: count, dtype: int64

3. RESPONSE DISTRIBUTION BY CONCENTRATION


,count,mean,std,min,25%,50%,75%,max
Activity at 0.00295 uM,7.0,0.170871,3.876338,-4.3825,-1.86725,-0.4587,1.06255,7.6467
Activity at 0.015 uM,7.0,-0.771586,0.995431,-2.3750,-1.32695,-0.3992,-0.09375,0.2145
Activity at 0.074 uM,7.0,1.143143,4.551161,-5.0911,0.33710,0.5902,0.77415,10.2804
Activity at 0.092 uM,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Activity at 0.184 uM,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Activity at 0.258 uM,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Activity at 0.369 uM,503.0,3.029305,28.603208,-15.3864,-1.30040,0.6517,3.81415,622.4360
Activity at 0.508 uM,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Activity at 0.749 uM,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Activity at 1.290 uM,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN



4. NEGATIVE RESPONSE SUMMARY
negative_response_count
0    224
1    187
2     88
3      4
Name: count, dtype: int64

5. HIGHEST EFFICACY — RAW RESPONSE


,PUBCHEM_SID,PUBCHEM_CID,Efficacy,Max_Response,Fit_R2,Fit_HillSlope,response_points_available,Activity at 0.00295 uM,Activity at 0.015 uM,Activity at 0.074 uM,...,Activity at 2.537 uM,Activity at 3.751 uM,Activity at 6.450 uM,Activity at 9.220 uM,Activity at 12.94 uM,Activity at 18.70 uM,Activity at 36.87 uM,Activity at 46.10 uM,Activity at 64.71 uM,Activity at 92.20 uM
3937,24810018,3138340.0,1857.11,1486.20,1.0000,3.6772,4,NaN,NaN,NaN,...,NaN,NaN,NaN,15.3841,NaN,NaN,NaN,1486.20,NaN,NaN
6244,856259,135449532.0,1609.15,1419.01,1.0000,4.0950,4,NaN,NaN,NaN,...,NaN,NaN,NaN,15.9286,NaN,NaN,NaN,1419.01,NaN,NaN
3511,26658341,933665.0,1484.08,1447.17,1.0000,4.9549,4,NaN,NaN,NaN,...,NaN,NaN,NaN,21.0872,NaN,NaN,NaN,1447.17,NaN,NaN
2807,49669245,4429074.0,1451.42,1279.44,1.0000,4.0950,4,NaN,NaN,NaN,...,NaN,NaN,NaN,13.2696,NaN,NaN,NaN,1279.44,NaN,NaN
1467,90340832,24867847.0,1424.78,1100.36,0.9650,0.6000,7,1.4148,-0.741,0.8314,...,NaN,NaN,NaN,616.0020,NaN,NaN,NaN,1100.36,NaN,NaN
5581,14721281,1933256.0,1391.82,1370.94,0.9999,4.5045,4,NaN,NaN,NaN,...,NaN,NaN,NaN,50.9047,NaN,NaN,NaN,1370.94,NaN,NaN
176,124894477,16445890.0,1362.26,1091.16,1.0000,3.6772,4,NaN,NaN,NaN,...,NaN,NaN,NaN,15.5455,NaN,NaN,NaN,1091.16,NaN,NaN
2111,51090338,4100358.0,1329.68,1264.92,1.0000,4.0950,4,NaN,NaN,NaN,...,NaN,NaN,NaN,28.4769,NaN,NaN,NaN,1264.92,NaN,NaN
5596,14721179,3138330.0,1219.10,1120.88,1.0000,4.0450,4,NaN,NaN,NaN,...,NaN,NaN,NaN,18.0638,NaN,NaN,NaN,1120.88,NaN,NaN
931,99495270,135438745.0,1216.80,1134.22,0.9995,3.6772,4,NaN,NaN,NaN,...,NaN,NaN,NaN,65.8160,NaN,NaN,NaN,1134.22,NaN,NaN



6. NEGATIVE MAX_RESPONSE — RAW RESPONSE


,PUBCHEM_SID,PUBCHEM_CID,Efficacy,Max_Response,Fit_R2,Fit_HillSlope,response_points_available,Activity at 0.00295 uM,Activity at 0.015 uM,Activity at 0.074 uM,...,Activity at 2.537 uM,Activity at 3.751 uM,Activity at 6.450 uM,Activity at 9.220 uM,Activity at 12.94 uM,Activity at 18.70 uM,Activity at 36.87 uM,Activity at 46.10 uM,Activity at 64.71 uM,Activity at 92.20 uM
4671,22401490,135504329.0,75.3953,-72.8557,1.0000,4.0450,4,NaN,NaN,NaN,...,NaN,NaN,NaN,73.9537,NaN,NaN,NaN,-72.8557,NaN,NaN
1391,92764047,3246517.0,77.5066,-65.2780,1.0000,1.3723,4,NaN,NaN,NaN,...,NaN,NaN,NaN,66.5556,NaN,NaN,NaN,-65.2780,NaN,NaN
1656,85271222,2468342.0,516.0190,-58.6658,1.0000,4.0950,4,NaN,NaN,NaN,...,NaN,NaN,NaN,488.2010,NaN,NaN,NaN,-58.6658,NaN,NaN
6219,858743,5389212.0,301.0830,-55.3404,1.0000,4.0950,4,NaN,NaN,NaN,...,NaN,NaN,NaN,294.5720,NaN,NaN,NaN,-55.3404,NaN,NaN
1387,92763868,122815.0,1154.2800,-55.1652,0.9981,4.9549,4,NaN,NaN,NaN,...,NaN,NaN,NaN,1059.0700,NaN,NaN,NaN,-55.1652,NaN,NaN
3428,26665251,75253.0,339.6610,-51.3435,1.0000,1.9673,4,NaN,NaN,NaN,...,NaN,NaN,NaN,309.5040,NaN,NaN,NaN,-51.3435,NaN,NaN
2666,49677916,20914270.0,113.7580,-47.2229,0.9996,4.9549,4,NaN,NaN,NaN,...,NaN,NaN,NaN,89.3118,NaN,NaN,NaN,-47.2229,NaN,NaN
5736,7970078,2557756.0,131.1440,-45.8681,1.0000,4.0450,4,NaN,NaN,NaN,...,NaN,NaN,NaN,129.9720,NaN,NaN,NaN,-45.8681,NaN,NaN
4920,17504202,691106.0,109.7620,-45.4878,1.0000,2.1876,4,NaN,NaN,NaN,...,NaN,NaN,NaN,92.0490,NaN,NaN,NaN,-45.4878,NaN,NaN
3402,26665685,16409834.0,163.7460,-43.2808,1.0000,4.4495,4,NaN,NaN,NaN,...,NaN,NaN,NaN,163.2520,NaN,NaN,NaN,-43.2808,NaN,NaN



7. HILLSLOPE FREQUENCY


,HillSlope,count
0,4.0950,120
1,4.9549,81
2,3.6772,74
3,3.2975,64
4,4.5045,51
5,4.0450,9
6,1.9673,8
7,4.4495,7
8,1.8851,7
9,2.0437,7



SAVED: H:\My Drive\PROJECT SAINS DATA\OUTPUT\01_Cleaned_Data\Alpha_Synuclein\Unmatched_Audit\aid652106_concentration_response_audit.csv
Shape: (503, 32)
Unique SID: 503


In [ ]:
from pathlib import Path
import pandas as pd

# Sesuaikan jika nama/path file berbeda
metadata_path = (
    ALPHA_CLEANED_DIR
    / "Unmatched_Audit"
    / "AID652106_Batch_001"
    / "aid652106_batch001_all.csv"
)

meta = pd.read_csv(metadata_path, low_memory=False)

print("SHAPE:", meta.shape)

for col in [
    "PUBCHEM_ASSAYDATA_COMMENT",
    "PUBCHEM_RESULT_TAG",
    "PUBCHEM_RESULT_DESCR",
    "PUBCHEM_RESULT_UNIT",
    "PUBCHEM_RESULT_IS_ACTIVE_CONCENTRATION",
    "PUBCHEM_ACTIVITY_URL",
]:
    if col in meta.columns:
        print(f"\n===== {col} =====")
        print(meta[col].dropna().astype(str).drop_duplicates().head(30).to_string(index=False))

SHAPE: (1000, 44)

===== PUBCHEM_ASSAYDATA_COMMENT =====
Series([], )

===== PUBCHEM_RESULT_TAG =====
259
260
261
262
263
264
265
266
267
268
269
270
271
272
273
274
275
276
277
278
279
280
281
282
283
284
285
286
287
288

===== PUBCHEM_ACTIVITY_URL =====
http://assay.nih.gov/htsws/rest/display/alpha-syn-p2/MLS003675879-01/142480784
http://assay.nih.gov/htsws/rest/display/alpha-syn-p2/MLS003675880-01/142676511
http://assay.nih.gov/htsws/rest/display/alpha-syn-p2/MLS003675881-01/142480456
http://assay.nih.gov/htsws/rest/display/alpha-syn-p2/MLS003675882-01/142480187
http://assay.nih.gov/htsws/rest/display/alpha-syn-p2/MLS003675883-01/142481219
http://assay.nih.gov/htsws/rest/display/alpha-syn-p2/MLS003675884-01/142480387
http://assay.nih.gov/htsws/rest/display/alpha-syn-p2/MLS003675885-01/142480522
http://assay.nih.gov/htsws/rest/display/alpha-syn-p2/MLS003675886-01/142480355
http://assay.nih.gov/htsws/rest/display/alpha-syn-p2/MLS003675887-01/142481113
http://assay.nih.gov/htsws/rest/d

In [ ]:
from pathlib import Path
import pandas as pd

# Folder sumber data mentah
ALPHA_RAW_DIR = DATA_DIR / "α-Synuclein"

metadata_path = (
    ALPHA_RAW_DIR
    / "alpha_synuclein_assay_metadata_raw.csv"
)

print("PATH:", metadata_path)
print("FILE EXISTS:", metadata_path.exists())

assay_meta = pd.read_csv(metadata_path, low_memory=False)

print("SHAPE:", assay_meta.shape)

print("\nCOLUMNS:")
for col in assay_meta.columns:
    print("-", col)

display(assay_meta.head())

PATH: H:\My Drive\PROJECT SAINS DATA\DATA\α-Synuclein\α-Synuclein\alpha_synuclein_assay_metadata_raw.csv
FILE EXISTS: False


FileNotFoundError: [Errno 2] No such file or directory: 'H:\\My Drive\\PROJECT SAINS DATA\\DATA\\α-Synuclein\\α-Synuclein\\alpha_synuclein_assay_metadata_raw.csv'

In [ ]:
from pathlib import Path

target_name = "alpha_synuclein_assay_metadata_raw.csv"

matches = list(DATA_DIR.rglob(target_name))

print("JUMLAH FILE DITEMUKAN:", len(matches))

for i, path in enumerate(matches, start=1):
    print(f"{i}. {path}")

JUMLAH FILE DITEMUKAN: 1
1. H:\My Drive\PROJECT SAINS DATA\DATA\α-Synuclein\alpha_synuclein_assay_metadata_raw.csv


In [ ]:
if len(matches) == 1:
    metadata_path = matches[0]

    print("PATH:", metadata_path)
    print("EXISTS:", metadata_path.exists())

    assay_meta = pd.read_csv(
        metadata_path,
        low_memory=False
    )

    print("SHAPE:", assay_meta.shape)
    print("COLUMNS:", assay_meta.columns.tolist())

    display(assay_meta.head())

elif len(matches) == 0:
    print("File tidak ditemukan di dalam DATA_DIR.")

else:
    print("Ditemukan lebih dari satu file. Periksa lokasi masing-masing.")

PATH: H:\My Drive\PROJECT SAINS DATA\DATA\α-Synuclein\alpha_synuclein_assay_metadata_raw.csv
EXISTS: True
SHAPE: (429, 29)
COLUMNS: ['aidx', 'assay_category', 'assay_cell_type', 'assay_chembl_id', 'assay_classifications', 'assay_group', 'assay_organism', 'assay_parameters', 'assay_strain', 'assay_subcellular_fraction', 'assay_tax_id', 'assay_test_type', 'assay_tissue', 'assay_type', 'assay_type_description', 'bao_format', 'bao_label', 'cell_chembl_id', 'confidence_description', 'confidence_score', 'description', 'document_chembl_id', 'relationship_description', 'relationship_type', 'src_assay_id', 'src_id', 'target_chembl_id', 'tissue_chembl_id', 'variant_sequence']


,aidx,assay_category,assay_cell_type,assay_chembl_id,assay_classifications,assay_group,assay_organism,assay_parameters,assay_strain,assay_subcellular_fraction,...,confidence_score,description,document_chembl_id,relationship_description,relationship_type,src_assay_id,src_id,target_chembl_id,tissue_chembl_id,variant_sequence
0,CLD0,NaN,NaN,CHEMBL945801,[],NaN,NaN,[],NaN,NaN,...,8,Binding affinity to alpha-synuclein fibrils by fluorescent titration,CHEMBL1143231,Homologous protein target assigned,H,NaN,1,CHEMBL6152,NaN,NaN
1,CLD0,NaN,NaN,CHEMBL1218779,[],NaN,Homo sapiens,[],NaN,NaN,...,9,Inhibition of alpha-synuclein-mediated toxicity in yeast transfected with human alpha-synuclein by spotting assay,CHEMBL1212724,Direct protein target assigned,D,NaN,1,CHEMBL6152,NaN,NaN
2,CLD0,NaN,NaN,CHEMBL1218780,[],NaN,Homo sapiens,[],NaN,NaN,...,9,Inhibition of human alpha-synuclein-mediated accumulation of stalled vesicle pools in yeast transfected with human alpha-synuclein after 8 to 12 hrs by electron microscopy,CHEMBL1212724,Direct protein target assigned,D,NaN,1,CHEMBL6152,NaN,NaN
3,CLD0,NaN,NaN,CHEMBL1218781,[],NaN,Homo sapiens,[],NaN,NaN,...,9,Inhibition of alpha-synuclein-mediated dopaminergic neurodegeneration in Caenorhabditis elegans transfected with human alpha-synuclein,CHEMBL1212724,Direct protein target assigned,D,NaN,1,CHEMBL6152,NaN,NaN
4,CLD0,NaN,NaN,CHEMBL1250260,[],NaN,NaN,[],NaN,NaN,...,8,Induction of alpha-synuclein aggregation in PBS buffer assessed as fluorescence after 48 hrs by Thioflavin T fluorescence assay,CHEMBL1240308,Homologous protein target assigned,H,NaN,1,CHEMBL6152,NaN,NaN


In [ ]:
assay_id = "CHEMBL2354282"

assay_info = assay_meta.loc[
    assay_meta["assay_chembl_id"].astype(str).eq(assay_id)
].copy()

print("=" * 70)
print("ASSAY METADATA AUDIT")
print("=" * 70)

print("Assay ID:", assay_id)
print("Matched rows:", len(assay_info))

if len(assay_info) > 0:
    cols_to_inspect = [
        "aidx",
        "assay_chembl_id",
        "assay_type",
        "assay_type_description",
        "assay_organism",
        "assay_test_type",
        "assay_category",
        "assay_cell_type",
        "assay_strain",
        "assay_parameters",
        "description",
        "document_chembl_id",
        "relationship_description",
        "relationship_type",
        "confidence_score"
    ]

    cols_to_inspect = [
        col for col in cols_to_inspect
        if col in assay_info.columns
    ]

    display(assay_info[cols_to_inspect].T)
else:
    print("Assay ID tidak ditemukan dalam metadata.")

ASSAY METADATA AUDIT
Assay ID: CHEMBL2354282
Matched rows: 1


,22
aidx,CLD0
assay_chembl_id,CHEMBL2354282
assay_type,F
assay_type_description,Functional
assay_organism,NaN
assay_test_type,NaN
assay_category,confirmatory
assay_cell_type,NaN
assay_strain,NaN
assay_parameters,[]


In [ ]:
if len(assay_info) > 0:

    for col in [
        "description",
        "assay_parameters",
        "assay_type_description"
    ]:
        if col in assay_info.columns:
            print(f"\n{'=' * 20} {col} {'=' * 20}")

            for value in assay_info[col].dropna():
                print(value)


==================== description ====================
PubChem BioAssay. qHTS of alpha-syn Inhibitors. (Class of assay: confirmatory) 

==================== assay_parameters ====================
[]

==================== assay_type_description ====================
Functional


In [ ]:
activity_path = (
    ALPHA_RAW_DIR
    / "alpha_synuclein_activity_raw.csv"
)

alpha_raw = pd.read_csv(
    activity_path,
    low_memory=False
)

assay_records = alpha_raw.loc[
    alpha_raw["assay_chembl_id"].astype(str).eq(assay_id)
].copy()

print("ASSAY:", assay_id)
print("RECORDS:", len(assay_records))
print(
    "UNIQUE MOLECULES:",
    assay_records["molecule_chembl_id"].nunique()
)

print("\nSTANDARD TYPE:")
display(
    assay_records["standard_type"]
    .value_counts(dropna=False)
)

print("\nSTANDARD UNITS:")
display(
    assay_records["standard_units"]
    .value_counts(dropna=False)
)

print("\nACTIVITY COMMENT:")
display(
    assay_records["activity_comment"]
    .value_counts(dropna=False)
)

FileNotFoundError: [Errno 2] No such file or directory: 'H:\\My Drive\\PROJECT SAINS DATA\\DATA\\α-Synuclein\\α-Synuclein\\alpha_synuclein_activity_raw.csv'

In [ ]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path(
    r"H:\My Drive\PROJECT SAINS DATA\DATA"
)

ALPHA_RAW_DIR = DATA_DIR / "α-Synuclein"

activity_path = (
    ALPHA_RAW_DIR / "alpha_synuclein_activity_raw.csv"
)

print("ACTIVITY PATH:", activity_path)
print("FILE EXISTS:", activity_path.exists())

ACTIVITY PATH: H:\My Drive\PROJECT SAINS DATA\DATA\α-Synuclein\alpha_synuclein_activity_raw.csv
FILE EXISTS: True


In [ ]:
if activity_path.exists():
    alpha_raw = pd.read_csv(
        activity_path,
        low_memory=False
    )

    print("FILE SUCCESSFULLY LOADED")
    print("SHAPE:", alpha_raw.shape)
    print("UNIQUE ACTIVITY ID:",
          alpha_raw["activity_id"].nunique())
else:
    print("FILE NOT FOUND. CHECK THE GOOGLE DRIVE PATH.")

FILE SUCCESSFULLY LOADED
SHAPE: (12221, 47)
UNIQUE ACTIVITY ID: 12221


In [ ]:
assay_id = "CHEMBL2354282"

assay_detail = assay_meta.loc[
    assay_meta["assay_chembl_id"] == assay_id
].copy()

metadata_columns = [
    "assay_chembl_id",
    "assay_type",
    "assay_type_description",
    "assay_organism",
    "assay_test_type",
    "assay_category",
    "assay_cell_type",
    "assay_strain",
    "assay_parameters",
    "description",
    "document_chembl_id",
]

print("METADATA RECORD:", len(assay_detail))

display(
    assay_detail[
        [c for c in metadata_columns if c in assay_detail.columns]
    ].T
)

METADATA RECORD: 1


,22
assay_chembl_id,CHEMBL2354282
assay_type,F
assay_type_description,Functional
assay_organism,NaN
assay_test_type,NaN
assay_category,confirmatory
assay_cell_type,NaN
assay_strain,NaN
assay_parameters,[]
description,PubChem BioAssay. qHTS of alpha-syn Inhibitors. (Class of assay: confirmatory)


In [ ]:
assay_activity = alpha_raw.loc[
    alpha_raw["assay_chembl_id"] == assay_id
].copy()

print("ASSAY:", assay_id)
print("TOTAL RECORDS:", len(assay_activity))
print(
    "UNIQUE MOLECULES:",
    assay_activity["molecule_chembl_id"].nunique()
)

print("\nSTANDARD TYPE:")
display(assay_activity["standard_type"].value_counts(dropna=False))

print("\nSTANDARD UNITS:")
display(assay_activity["standard_units"].value_counts(dropna=False))

print("\nACTIVITY COMMENT:")
display(assay_activity["activity_comment"].value_counts(dropna=False))

ASSAY: CHEMBL2354282
TOTAL RECORDS: 10019
UNIQUE MOLECULES: 9955

STANDARD TYPE:


standard_type
Potency    10019
Name: count, dtype: int64


STANDARD UNITS:


standard_units
nM    10019
Name: count, dtype: int64


ACTIVITY COMMENT:


activity_comment
inconclusive    5687
inactive        3829
active           503
Name: count, dtype: int64

In [ ]:
activity_columns = [
    "type",
    "value",
    "units",
    "standard_type",
    "standard_value",
    "standard_units",
    "pchembl_value",
    "activity_comment",
]

display(
    assay_activity[
        [c for c in activity_columns if c in assay_activity.columns]
    ].head(10)
)

,type,value,units,standard_type,standard_value,standard_units,pchembl_value,activity_comment
116,Potency,1.5849,uM,Potency,1584.9,nM,NaN,active
117,Potency,35.4813,uM,Potency,35481.3,nM,NaN,inactive
118,Potency,14.1254,uM,Potency,14125.4,nM,NaN,inconclusive
119,Potency,35.4813,uM,Potency,35481.3,nM,NaN,inactive
120,Potency,15.8489,uM,Potency,15848.9,nM,NaN,inactive
121,Potency,35.4813,uM,Potency,35481.3,nM,NaN,inactive
122,Potency,25.1189,uM,Potency,25118.9,nM,NaN,inconclusive
123,Potency,12.5893,uM,Potency,12589.3,nM,NaN,inconclusive
124,Potency,17.7828,uM,Potency,17782.8,nM,NaN,active
125,Potency,10.0000,uM,Potency,10000.0,nM,NaN,inactive


In [ ]:
import numpy as np
import pandas as pd

assay_check = assay_activity.copy()

# Pastikan kolom numerik
assay_check["value"] = pd.to_numeric(
    assay_check["value"], errors="coerce"
)

assay_check["standard_value"] = pd.to_numeric(
    assay_check["standard_value"], errors="coerce"
)

# Hitung nilai standard berdasarkan konversi uM ke nM
assay_check["expected_standard_value"] = (
    assay_check["value"] * 1000
)

# Hitung selisih relatif
valid = assay_check[
    ["value", "standard_value"]
].notna().all(axis=1)

assay_check["relative_difference"] = np.nan

assay_check.loc[valid, "relative_difference"] = (
    (
        assay_check.loc[valid, "standard_value"]
        - assay_check.loc[valid, "expected_standard_value"]
    ).abs()
    / assay_check.loc[valid, "expected_standard_value"].abs()
) * 100

print("TOTAL RECORD:", len(assay_check))
print("RECORD VALID:", valid.sum())

print(
    "MATCH WITHIN 1%:",
    (assay_check["relative_difference"] <= 1).sum()
)

print(
    "MISMATCH ABOVE 1%:",
    (assay_check["relative_difference"] > 1).sum()
)

print(
    "MISSING VALUE:",
    assay_check["relative_difference"].isna().sum()
)

print("\nRELATION DISTRIBUTION:")
display(
    assay_check["standard_relation"].value_counts(dropna=False)
)

print("\nUNIT DISTRIBUTION:")
display(
    assay_check["units"].value_counts(dropna=False)
)

TOTAL RECORD: 10019
RECORD VALID: 10019
MATCH WITHIN 1%: 10019
MISMATCH ABOVE 1%: 0
MISSING VALUE: 0

RELATION DISTRIBUTION:


standard_relation
NaN    10019
Name: count, dtype: int64


UNIT DISTRIBUTION:


units
uM    10019
Name: count, dtype: int64

In [ ]:
print("=" * 65)
print("RELATION AUDIT - CHEMBL2354282")
print("=" * 65)

print("\nRAW RELATION:")
display(
    assay_check["relation"].value_counts(dropna=False)
)

print("\nSTANDARD RELATION:")
display(
    assay_check["standard_relation"].value_counts(dropna=False)
)

print("\nDATA VALIDITY COMMENT:")
display(
    assay_check["data_validity_comment"].value_counts(
        dropna=False
    )
)

print("\nSTANDARD FLAG:")
display(
    assay_check["standard_flag"].value_counts(
        dropna=False
    )
)

RELATION AUDIT - CHEMBL2354282

RAW RELATION:


relation
NaN    10019
Name: count, dtype: int64


STANDARD RELATION:


standard_relation
NaN    10019
Name: count, dtype: int64


DATA VALIDITY COMMENT:


data_validity_comment
NaN    10019
Name: count, dtype: int64


STANDARD FLAG:


standard_flag
1    10019
Name: count, dtype: int64

In [ ]:
assay_audit_summary = pd.DataFrame([{
    "assay_chembl_id": "CHEMBL2354282",
    "assay_category": "Cellular_Rescue",
    "total_records": len(assay_check),
    "unique_molecules": assay_check["molecule_chembl_id"].nunique(),
    "endpoint": "Potency",
    "original_units": "uM",
    "standard_units": "nM",
    "unit_conversion_validated": True,
    "conversion_match_count": 10019,
    "conversion_mismatch_count": 0,
    "missing_relation_count": assay_check["relation"].isna().sum(),
    "missing_standard_relation_count": assay_check["standard_relation"].isna().sum(),
    "missing_validity_comment_count": assay_check["data_validity_comment"].isna().sum(),
    "standard_flag_1_count": (assay_check["standard_flag"] == 1).sum(),
    "active_count": (assay_check["activity_comment"] == "active").sum(),
    "inactive_count": (assay_check["activity_comment"] == "inactive").sum(),
    "inconclusive_count": (assay_check["activity_comment"] == "inconclusive").sum(),
    "preprocessing_status": "Audited_Keep_Separate"
}])

display(assay_audit_summary)

,assay_chembl_id,assay_category,total_records,unique_molecules,endpoint,original_units,standard_units,unit_conversion_validated,conversion_match_count,conversion_mismatch_count,missing_relation_count,missing_standard_relation_count,missing_validity_comment_count,standard_flag_1_count,active_count,inactive_count,inconclusive_count,preprocessing_status
0,CHEMBL2354282,Cellular_Rescue,10019,9955,Potency,uM,nM,True,10019,0,10019,10019,10019,10019,503,3829,5687,Audited_Keep_Separate


In [ ]:
audit_path = (
    ALPHA_CLEANED_DIR
    / "alpha_chembl2354282_assay_audit_summary.csv"
)

audit_path.parent.mkdir(parents=True, exist_ok=True)

assay_audit_summary.to_csv(
    audit_path,
    index=False
)

print("SAVED:", audit_path)

SAVED: H:\My Drive\PROJECT SAINS DATA\OUTPUT\01_Cleaned_Data\Alpha_Synuclein\alpha_chembl2354282_assay_audit_summary.csv


In [ ]:
audit_path = (
    ALPHA_CLEANED_DIR
    / "alpha_chembl2354282_assay_audit_summary.csv"
)

audit_path.parent.mkdir(parents=True, exist_ok=True)

assay_audit_summary.to_csv(
    audit_path,
    index=False
)

print("SAVED:", audit_path)
print("EXISTS:", audit_path.exists())

SAVED: H:\My Drive\PROJECT SAINS DATA\OUTPUT\01_Cleaned_Data\Alpha_Synuclein\alpha_chembl2354282_assay_audit_summary.csv
EXISTS: True


In [ ]:
audit_readback = pd.read_csv(audit_path)

print("READ-BACK SHAPE:", audit_readback.shape)
display(audit_readback)

READ-BACK SHAPE: (1, 18)


,assay_chembl_id,assay_category,total_records,unique_molecules,endpoint,original_units,standard_units,unit_conversion_validated,conversion_match_count,conversion_mismatch_count,missing_relation_count,missing_standard_relation_count,missing_validity_comment_count,standard_flag_1_count,active_count,inactive_count,inconclusive_count,preprocessing_status
0,CHEMBL2354282,Cellular_Rescue,10019,9955,Potency,uM,nM,True,10019,0,10019,10019,10019,10019,503,3829,5687,Audited_Keep_Separate


In [ ]:
# ============================================================
# ALPHA-SYNUCLEIN ASSAY INVENTORY
# ============================================================

# Ringkasan jumlah record dari raw activity
assay_record_counts = (
    alpha_raw
    .groupby("assay_chembl_id")
    .agg(
        record_count=("activity_id", "count"),
        unique_molecules=("molecule_chembl_id", "nunique"),
        endpoint_count=("standard_type", "nunique")
    )
    .reset_index()
)

# Ringkasan endpoint per assay
assay_endpoint_summary = (
    alpha_raw
    .groupby("assay_chembl_id")
    .agg(
        standard_types=(
            "standard_type",
            lambda x: " | ".join(
                sorted(x.dropna().astype(str).unique())
            )
        ),
        standard_units=(
            "standard_units",
            lambda x: " | ".join(
                sorted(x.dropna().astype(str).unique())
            )
        )
    )
    .reset_index()
)

# Gabungkan ringkasan aktivitas dan metadata
assay_inventory = (
    assay_meta
    .merge(
        assay_record_counts,
        on="assay_chembl_id",
        how="left",
        validate="one_to_one"
    )
    .merge(
        assay_endpoint_summary,
        on="assay_chembl_id",
        how="left",
        validate="one_to_one"
    )
)

assay_inventory["record_count"] = (
    assay_inventory["record_count"].fillna(0).astype(int)
)

assay_inventory = assay_inventory.sort_values(
    "record_count",
    ascending=False
).reset_index(drop=True)

# Ringkasan validasi
print("=" * 70)
print("ALPHA-SYNUCLEIN ASSAY INVENTORY")
print("=" * 70)

print("METADATA ASSAYS:", len(assay_meta))
print("INVENTORY ASSAYS:", len(assay_inventory))
print("RAW ACTIVITY RECORDS:", len(alpha_raw))
print(
    "TOTAL INVENTORY RECORDS:",
    assay_inventory["record_count"].sum()
)
print(
    "ASSAYS WITHOUT ACTIVITY:",
    (assay_inventory["record_count"] == 0).sum()
)

print("\nTOP 20 ASSAYS:")
display(
    assay_inventory[
        [
            "assay_chembl_id",
            "record_count",
            "unique_molecules",
            "standard_types",
            "standard_units",
            "assay_type_description",
            "assay_category",
            "description",
            "document_chembl_id"
        ]
    ].head(20)
)

ALPHA-SYNUCLEIN ASSAY INVENTORY
METADATA ASSAYS: 429
INVENTORY ASSAYS: 429
RAW ACTIVITY RECORDS: 12221
TOTAL INVENTORY RECORDS: 12221
ASSAYS WITHOUT ACTIVITY: 0

TOP 20 ASSAYS:


,assay_chembl_id,record_count,unique_molecules,standard_types,standard_units,assay_type_description,assay_category,description,document_chembl_id
0,CHEMBL2354282,10019,9955,Potency,nM,Functional,confirmatory,PubChem BioAssay. qHTS of alpha-syn Inhibitors. (Class of assay: confirmatory),CHEMBL1201862
1,CHEMBL3888008,90,89,EC50,nM,Binding,NaN,"In Vitro Inhibition Assay: α-Synucleinopathies are characterised by aggregation of α-synuclein in neurons. Aggregation of purified α-synuclein is performed essentially as described by Gerard et al. FASEB. 20(3):524-6 (2006). 20-100 μg purified α-synuclein (Sigma; S7820) at a concentration of about 2.5 μg/mL is incubated in the presence of spermin (250 μM) or paraquat (32 mM) or 6-hydroxydopamine (400 μM) or vehicle in a 384 well plate. Spermin, paraquat and 6-hydroxydopamine promote the α-synuclein aggregation process. Aggregation kinetics is determined by measuring turbidity at 340 nm, every 1-15 minutes for at least one hour.",CHEMBL3886537
2,CHEMBL5736665,75,25,IC50 | k_off | kon,nM | s-1,Binding,NaN,"In-Vitro Fluorescence Polarization Assay: The fluorescence polarization assay tests the ability of compounds to inhibit the self-aggregation of α-synuclein peptide fragments. Peptides were incubated for 120 min at room temperature in the presence or absence of test compounds (compound concentrations were 33.3 to 0.015 μM). Samples were read on a Beckman Coulter DTX 880 plate reader in fluorescence polarization mode using excitation at 485 nm and emission at 520 nm. Data was analyzed using a four-parameter logistic fit (XLFit, IDBS Software). Peptide 4F (CTGFVKKDQLGK (SEQ ID NO: 1)) was prepared by American Peptide. Fresh peptide samples were reconstituted in purified water at 5 mM and diluted into 50 mM HEPES pH 7.4 with 50 mM NaCl to 100 nM final concentration. Solid compounds were dissolved in DMSO (10 mM), and then diluted serially in DMSO (300×) followed by dilution in buffer (1×) to provide solutions with a consistent final DMSO concentration of 0.33%.",CHEMBL5728395
3,CHEMBL5524380,68,68,Activity,%,Binding,NaN,Inhibition of recombinant alpha-Synuclein (unknown origin) aggregation assessed as reduction in fluorescence intensity by measuring residual activity at 100 uM measured every 20 mins for 40 to 70 hrs by Thioflavin T based fluorescence analysis relative to control,CHEMBL5523695
4,CHEMBL5736378,57,11,Kd | k_off | kon,nM | s-1,Binding,NaN,"In Vitro Direct Determination By Fluorescence: Table 6 & 7: Fresh solutions of each test compound (2.5 mM in DMSO) are diluted in PBS pH=7.4 at concentrations between 0.5 nM and 5 μM in the presence or absence of α-syn (200 nM), Aβ1-42 (500 nM) or Tau (200 nM) fibers in a final 2 mL volume. After 1 h of incubation at RT, the binding of each compound to α-syn, A131-42 or Tau fibers is measured by fluorescence using the spectral changes of the compounds during their binding to the fibers, using the appropriate excitation/emission wavelengths (those giving the highest fluorescence variation during binding to the fibres). For each concentration of compound added to the fibres, the bound and free fractions are measured by fluorescence according to the following equation:F m =F B B+F F×(T−B).",CHEMBL5728251
5,CHEMBL4628227,47,47,Activity,RU,Binding,NaN,Binding affinity to human wlid-type alpha-synuclein at 50 uM by SPR analysis,CHEMBL4627295
6,CHEMBL4625544,47,47,Inhibition,%,Binding,NaN,Inhibition of alpha-synuclein aggregation (unknown origin) at 30 uM incubated for 3 days by thioflavin T based fluorescence assay relative to control,CHEMBL4622956
7,CHEMBL4628228,47,47,Kd,nM,Binding,NaN,Binding affinity to human wlid-type alpha-synuclein by SPR analysis,CHEMBL4627295
8,CHEMBL3594881,45,45,Ki,nM,Binding,NaN,Displacement of Thio-T from recombinant alpha-synuclein (unknown origin) expressed in Escherichia coli after 1.5 hrs by fluorescence assay,CHEMBL3593202
9,CHEMBL5736397,42,14,IC50 | k_off | kon,nM | s-1,Binding,NaN,"In-Vitro Fluorescence Polarizati

In [ ]:
inventory_path = (
    ALPHA_CLEANED_DIR
    / "alpha_synuclein_assay_inventory.csv"
)

assay_inventory.to_csv(
    inventory_path,
    index=False
)

print("SAVED:", inventory_path)
print("EXISTS:", inventory_path.exists())

# Read-back validation
inventory_readback = pd.read_csv(inventory_path)

print("READ-BACK SHAPE:", inventory_readback.shape)
print(
    "READ-BACK RECORD TOTAL:",
    inventory_readback["record_count"].sum()
)

SAVED: H:\My Drive\PROJECT SAINS DATA\OUTPUT\01_Cleaned_Data\Alpha_Synuclein\alpha_synuclein_assay_inventory.csv
EXISTS: True
READ-BACK SHAPE: (429, 34)
READ-BACK RECORD TOTAL: 12221


In [ ]:
import re
import numpy as np
import pandas as pd

def classify_alpha_assay(description):
    if pd.isna(description):
        return "Unresolved"

    text = str(description).lower()

    # Cellular rescue / neuroprotection
    if re.search(
        r"yeast|cellular toxicity|cell toxicity|"
        r"neuroprotection|neurodegeneration|"
        r"cell viability|cell growth|cell survival|"
        r"cytotoxicity|cell rescue",
        text
    ):
        return "Cellular_Rescue_Neuroprotection"

    # Fibril depolymerization
    if re.search(
        r"depolymerization|depolymerisation|"
        r"disassembly of.*fibril|fibril disassembly",
        text
    ):
        return "Fibril_Depolymerization"

    # Direct aggregation inhibition
    if re.search(
        r"inhibit.*aggregation|aggregation inhibition|"
        r"inhibition of.*aggregation|"
        r"inhibit.*self-aggregation|"
        r"inhibitor.*aggregation|"
        r"aggregation inhibitory",
        text
    ):
        return "Aggregation_Inhibition"

    # Aggregation induction / formation
    if re.search(
        r"induction of.*aggregation|"
        r"induce.*aggregation|"
        r"aggregation kinetics|"
        r"aggregation formation|"
        r"promote.*aggregation",
        text
    ):
        return "Aggregation_Formation"

    # Binding to alpha-synuclein or fibrils
    if re.search(
        r"binding affinity|"
        r"bind.*fibril|"
        r"binding to.*fibril|"
        r"fibril binding|"
        r"competitive binding|"
        r"displacement of",
        text
    ):
        return "Binding_Affinity"

    # Expression / regulation
    if re.search(
        r"expression of.*alpha|"
        r"alpha.*expression|"
        r"transcription|"
        r"protein expression|"
        r"protein level",
        text
    ):
        return "Expression_Regulation"

    return "Other_Unresolved"


assay_inventory["initial_category"] = (
    assay_inventory["description"]
    .apply(classify_alpha_assay)
)

print("=" * 65)
print("INITIAL ASSAY CLASSIFICATION")
print("=" * 65)

category_summary = (
    assay_inventory
    .groupby("initial_category")
    .agg(
        assay_count=("assay_chembl_id", "count"),
        activity_records=("record_count", "sum")
    )
    .sort_values("activity_records", ascending=False)
    .reset_index()
)

display(category_summary)

INITIAL ASSAY CLASSIFICATION


,initial_category,assay_count,activity_records
0,Other_Unresolved,194,10574
1,Aggregation_Inhibition,140,1132
2,Binding_Affinity,54,378
3,Aggregation_Formation,34,79
4,Fibril_Depolymerization,3,46
5,Cellular_Rescue_Neuroprotection,4,12


In [ ]:
unresolved_assays = assay_inventory.loc[
    assay_inventory["initial_category"].isin([
        "Other_Unresolved",
        "Unresolved"
    ])
].copy()

print("UNRESOLVED ASSAYS:", len(unresolved_assays))
print(
    "UNRESOLVED RECORDS:",
    unresolved_assays["record_count"].sum()
)

display(
    unresolved_assays[
        [
            "assay_chembl_id",
            "record_count",
            "standard_types",
            "standard_units",
            "description",
            "document_chembl_id"
        ]
    ].sort_values(
        "record_count",
        ascending=False
    )
)

UNRESOLVED ASSAYS: 194
UNRESOLVED RECORDS: 10574


,assay_chembl_id,record_count,standard_types,standard_units,description,document_chembl_id
0,CHEMBL2354282,10019,Potency,nM,PubChem BioAssay. qHTS of alpha-syn Inhibitors. (Class of assay: confirmatory),CHEMBL1201862
4,CHEMBL5736378,57,Kd | k_off | kon,nM | s-1,"In Vitro Direct Determination By Fluorescence: Table 6 & 7: Fresh solutions of each test compound (2.5 mM in DMSO) are diluted in PBS pH=7.4 at concentrations between 0.5 nM and 5 μM in the presence or absence of α-syn (200 nM), Aβ1-42 (500 nM) or Tau (200 nM) fibers in a final 2 mL volume. After 1 h of incubation at RT, the binding of each compound to α-syn, A131-42 or Tau fibers is measured by fluorescence using the spectral changes of the compounds during their binding to the fibers, using the appropriate excitation/emission wavelengths (those giving the highest fluorescence variation during binding to the fibres). For each concentration of compound added to the fibres, the bound and free fractions are measured by fluorescence according to the following equation:F m =F B B+F F×(T−B).",CHEMBL5728251
15,CHEMBL1613965,37,Potency,nM,PUBCHEM_BIOASSAY: Alpha-synuclein ThS binding assay. (Class of assay: confirmatory) [Related pubchem assays: 596 ],CHEMBL1201862
22,CHEMBL5736375,27,Ki | k_off | kon,nM | s-1,"In Vitro Inhibition Assay: Table 5: Fresh solutions of the test compounds (2.5 mM in DMSO) are diluted in PBS pH7.4, at concentrations between 0.5 nM and 5 μM in the presence of α-syn (200 nM), Aβ1-42 (500 nM) or Tau (200 nM) fibres and ThT 500 nM in a final 2 mL volume. The samples are incubated for 1 h at room temperature (RT) to reach equilibrium. The binding of ThT to α-syn, Aβ1-42 or Tau fibres is measured by fluorescence (excitation: 440 nm, emission: 480 nm). Inhibition of ThT binding by increasing concentrations of each test compound is measured by fluorescence reduction. Due to the partial superposition between the emission spectra of some compounds and ThT, the compounds are also incubated with α-syn, Aβ1-42 or Tau fibers in the absence of ThT. For each concentration, the fluorescence values in the absence of ThT are subtracted from those in the presence of ThT. The results are expressed as a percentage of the maximum fluorescence value of the ThT measured in the absence of a competitor (100% bind).",CHEMBL5728251
24,CHEMBL6104001,27,Flu intensity,%,Inhibition of recombinant alpha-synuclein (unknown origin) fibril formation assessed as thioflavin T fluorescence intensity at 100 uM measured every 20 mins by Thioflavin T fluorescence assay (Rvb = 100 %),CHEMBL6103543
25,CHEMBL6104006,27,Inhibition,%,Inhibition of alpha-synuclein (unknown origin) oligomer formation assessed as reduction in cross linking at 100 uM preincubated with compound followed by Ru(BPY3) addition and measured after 1 sec light irradiation by coomassie blue staining based PICUP assay,CHEMBL6103543
30,CHEMBL4700551,19,IC50,nM,Inhibition of human alpha-synuclein filament formation expressed in Escherichia coli BL21(DE3) cells incubated for 72 hrs by thioflavin S based fluorescence assay,CHEMBL4699438
36,CHEMBL4700552,12,IC50,nM,Inhibition of alpha-synuclein fibril formation (unknown origin) incubated for 24 hrs to 7 days by thioflavin S based fluorescence assay,CHEMBL4699438
42,CHEMBL5518528,9,Flu intensity,%,Inhibition of recombinant alpha-synuclein (unknown origin) fibrillation assessed as fluorescence intensity at 100 uM measured every 24 hrs by ThT fluorescence based microplate reader assay relative to control,CHEMBL5518163
48,CHEMBL5325295,8,Activity | DC50,nM,Induction of alpha-synuclein (unknown origin) degradation overexpressed in HEK293T cells infected with adenovirus expressing mCherry-EGFP-LC3B incubated for 24 hrs by immunoblot analysis,CHEMBL5321295


In [ ]:
print("=" * 70)
print("ASSAY AUDIT SUMMARY")
print("=" * 70)

display(assay_audit_summary.T)

ASSAY AUDIT SUMMARY


,0
assay_chembl_id,CHEMBL2354282
assay_category,Cellular_Rescue
total_records,10019
unique_molecules,9955
endpoint,Potency
original_units,uM
standard_units,nM
unit_conversion_validated,True
conversion_match_count,10019
conversion_mismatch_count,0


In [ ]:
print("=" * 70)
print("ACTIVITY RECORD DISTRIBUTION")
print("=" * 70)

print("Total records:", len(assay_data))
print("Unique molecules:", assay_data["molecule_chembl_id"].nunique())

print("\nSTANDARD TYPE:")
display(assay_data["standard_type"].value_counts(dropna=False))

print("\nSTANDARD UNITS:")
display(assay_data["standard_units"].value_counts(dropna=False))

print("\nSTANDARD RELATION:")
display(assay_data["standard_relation"].value_counts(dropna=False))

print("\nDATA VALIDITY:")
display(assay_data["data_validity_comment"].value_counts(dropna=False))

ACTIVITY RECORD DISTRIBUTION
Total records: 10019
Unique molecules: 9955

STANDARD TYPE:


standard_type
Potency    10019
Name: count, dtype: int64


STANDARD UNITS:


standard_units
nM    10019
Name: count, dtype: int64


STANDARD RELATION:


standard_relation
NaN    10019
Name: count, dtype: int64


DATA VALIDITY:


data_validity_comment
NaN    10019
Name: count, dtype: int64

In [ ]:

import requests
import json
from pprint import pprint

# ============================================================
# SOURCE AUDIT: CHEMBL2354282
# ============================================================

BASE_CHEMBL = "https://www.ebi.ac.uk/chembl/api/data"
BASE_PUBCHEM = "https://pubchem.ncbi.nlm.nih.gov/rest/pug"

ASSAY_ID = "CHEMBL2354282"
DOCUMENT_ID = "CHEMBL1201862"
TARGET_ID = "CHEMBL6152"
PUBCHEM_AID = "652106"

def get_json(url):
    response = requests.get(url, timeout=60)
    print(f"STATUS: {response.status_code}")
    response.raise_for_status()
    return response.json()


# 1. ChEMBL assay details
assay_data = get_json(
    f"{BASE_CHEMBL}/assay/{ASSAY_ID}.json"
)

# 2. ChEMBL document details
document_data = get_json(
    f"{BASE_CHEMBL}/document/{DOCUMENT_ID}.json"
)

# 3. ChEMBL target details
target_data = get_json(
    f"{BASE_CHEMBL}/target/{TARGET_ID}.json"
)

# 4. PubChem BioAssay description
pubchem_data = get_json(
    f"{BASE_PUBCHEM}/assay/aid/{PUBCHEM_AID}/description/JSON"
)

print("\n" + "=" * 70)
print("1. ASSAY DETAILS")
print("=" * 70)

assay_fields = [
    "assay_chembl_id",
    "description",
    "assay_type",
    "assay_type_description",
    "assay_category",
    "assay_organism",
    "assay_tax_id",
    "target_chembl_id",
    "relationship_type",
    "relationship_description",
    "confidence_score",
    "document_chembl_id",
    "src_assay_id",
    "src_id",
]

for field in assay_fields:
    print(f"{field}: {assay_data.get(field)}")


print("\n" + "=" * 70)
print("2. DOCUMENT DETAILS")
print("=" * 70)

document_fields = [
    "document_chembl_id",
    "title",
    "authors",
    "journal",
    "year",
    "doi",
    "pubmed_id",
    "abstract",
]

for field in document_fields:
    print(f"\n{field}:")
    pprint(document_data.get(field))


print("\n" + "=" * 70)
print("3. TARGET DETAILS")
print("=" * 70)

target_fields = [
    "target_chembl_id",
    "pref_name",
    "target_type",
    "organism",
    "tax_id",
    "target_components",
]

for field in target_fields:
    print(f"\n{field}:")
    pprint(target_data.get(field))


print("\n" + "=" * 70)
print("4. PUBCHEM BIOASSAY DESCRIPTION")
print("=" * 70)

# Simpan respons mentah agar seluruh informasi tetap bisa diperiksa.
with open(
    "results_alpha_syn_CHEMBL2354282_pubchem_description.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(pubchem_data, file, indent=2, ensure_ascii=False)

print(json.dumps(pubchem_data, indent=2, ensure_ascii=False)[:15000])


STATUS: 200
STATUS: 200
STATUS: 200
STATUS: 200

1. ASSAY DETAILS
assay_chembl_id: CHEMBL2354282
description: PubChem BioAssay. qHTS of alpha-syn Inhibitors. (Class of assay: confirmatory) 
assay_type: F
assay_type_description: Functional
assay_category: confirmatory
assay_organism: None
assay_tax_id: None
target_chembl_id: CHEMBL6152
relationship_type: H
relationship_description: Homologous protein target assigned
confidence_score: 8
document_chembl_id: CHEMBL1201862
src_assay_id: 652106
src_id: 7

2. DOCUMENT DETAILS

document_chembl_id:
'CHEMBL1201862'

title:
'PubChem BioAssay data set'

authors:
None

journal:
None

year:
None

doi:
None

pubmed_id:
None

abstract:
''

3. TARGET DETAILS

target_chembl_id:
'CHEMBL6152'

pref_name:
'Alpha-synuclein'

target_type:
'SINGLE PROTEIN'

organism:
'Homo sapiens'

tax_id:
9606

target_components:
[{'accession': 'P37840',
  'component_description': 'Alpha-synuclein',
  'component_id': 4717,
  'component_type': 'PROTEIN',
  'relationship': 'S

In [ ]:
# Menampilkan semua variabel DataFrame yang tersedia
import pandas as pd

dataframe_vars = {
    name: obj.shape
    for name, obj in globals().items()
    if isinstance(obj, pd.DataFrame)
}

print("DATAFRAME YANG TERSEDIA:")
for name, shape in dataframe_vars.items():
    print(f"{name}: {shape}")

DATAFRAME YANG TERSEDIA:
df: (6326, 47)
classification: (313, 9)
manual_review: (114, 9)
review_display: (114, 8)
group_summary: (14, 4)
review_df: (64, 61)
suggestions: (114, 3)
summary: (923, 3)
subset: (1, 11)
summary_df: (4, 5)
temp: (64, 2)
alpha_activity: (12221, 47)
alpha_metadata_1: (429, 29)
alpha_metadata_2: (429, 29)
activity: (12221, 47)
review: (114, 19)
activity_reviewed: (12221, 50)
unmatched: (11858, 47)
actual_counts: (114, 3)
comparison: (6190, 52)
mismatches: (1, 23)
assay_records: (8, 47)
assay_review: (1, 19)
actual_inventory: (429, 3)
quantitative: (11444, 47)
quantitative_inventory: (219, 2)
qualitative: (510, 47)
qualitative_inventory: (162, 2)
reconciliation_updated: (114, 23)
audit: (363, 54)
provisional_ids: (299, 2)
duplicate_ids: (0, 2)
audit_only_df: (64, 54)
provisional_only_df: (0, 2)
metadata: (429, 29)
metadata_check: (8, 29)
data: (64, 63)
document_df: (2, 12)
candidate_df: (4, 58)
reviewed: (64, 61)
raw: (12221, 47)
problem_records: (3, 61)
raw_assay

In [ ]:
# ============================================================
# VALIDASI PEMETAAN ENDPOINT PUBCHEM - CHEMBL2354282
# ============================================================

df = assay.copy()

candidate_columns = [
    "molecule_chembl_id",
    "assay_chembl_id",
    "standard_type",
    "standard_value",
    "standard_units",
    "standard_relation",
    "Potency",
    "Phenotype",
    "Efficacy",
    "Curve_Description",
    "Fit_LogAC50",
    "Fit_R2",
    "Fit_CurveClass"
]

available_columns = [
    col for col in candidate_columns
    if col in df.columns
]

print("=" * 70)
print("ENDPOINT VALIDATION - CHEMBL2354282")
print("=" * 70)

print("\nDATASET")
print("Shape:", df.shape)
print("Unique molecules:", df["molecule_chembl_id"].nunique())
print("Unique assays:", df["assay_chembl_id"].nunique())

print("\nAVAILABLE COLUMNS")
print(available_columns)

print("\nMISSING VALUES")
print(df[available_columns].isna().sum())

print("\nSTANDARD TYPE DISTRIBUTION")
if "standard_type" in df.columns:
    print(df["standard_type"].value_counts(dropna=False))

print("\nSTANDARD UNITS DISTRIBUTION")
if "standard_units" in df.columns:
    print(df["standard_units"].value_counts(dropna=False))

print("\nSAMPLE RECORDS")
display(df[available_columns].head(10))

ENDPOINT VALIDATION - CHEMBL2354282

DATASET
Shape: (10019, 47)
Unique molecules: 9955
Unique assays: 1

AVAILABLE COLUMNS
['molecule_chembl_id', 'assay_chembl_id', 'standard_type', 'standard_value', 'standard_units', 'standard_relation']

MISSING VALUES
molecule_chembl_id        0
assay_chembl_id           0
standard_type             0
standard_value            0
standard_units            0
standard_relation     10019
dtype: int64

STANDARD TYPE DISTRIBUTION
standard_type
Potency    10019
Name: count, dtype: int64

STANDARD UNITS DISTRIBUTION
standard_units
nM    10019
Name: count, dtype: int64

SAMPLE RECORDS


,molecule_chembl_id,assay_chembl_id,standard_type,standard_value,standard_units,standard_relation
78,CHEMBL1416096,CHEMBL2354282,Potency,1584.9,nM,NaN
79,CHEMBL1699082,CHEMBL2354282,Potency,35481.3,nM,NaN
80,CHEMBL1543347,CHEMBL2354282,Potency,14125.4,nM,NaN
81,CHEMBL1425843,CHEMBL2354282,Potency,35481.3,nM,NaN
82,CHEMBL1699039,CHEMBL2354282,Potency,15848.9,nM,NaN
83,CHEMBL1555805,CHEMBL2354282,Potency,35481.3,nM,NaN
84,CHEMBL1885038,CHEMBL2354282,Potency,25118.9,nM,NaN
85,CHEMBL1521536,CHEMBL2354282,Potency,12589.3,nM,NaN
86,CHEMBL3199539,CHEMBL2354282,Potency,17782.8,nM,NaN
87,CHEMBL5483017,CHEMBL2354282,Potency,10000.0,nM,NaN


In [ ]:
# ============================================================
# AUDIT NILAI AC50 - CHEMBL2354282
# ============================================================

import numpy as np
import pandas as pd

df = assay.copy()

# Pastikan standard_value numerik
df["standard_value_numeric"] = pd.to_numeric(
    df["standard_value"],
    errors="coerce"
)

values = df["standard_value_numeric"]

print("=" * 70)
print("AC50 NUMERICAL QUALITY AUDIT")
print("=" * 70)

print("\n1. VALIDITY CHECK")
print("Total records:", len(df))
print("Missing/non-numeric:", values.isna().sum())
print("Non-positive values:", (values <= 0).sum())
print("Infinite values:", np.isinf(values).sum())

print("\n2. DESCRIPTIVE STATISTICS (nM)")
display(values.describe(percentiles=[
    0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99
]).to_frame("AC50_nM"))

print("\n3. MOLECULE DUPLICATION")
molecule_counts = df["molecule_chembl_id"].value_counts()

print("Unique molecules:", molecule_counts.size)
print("Molecules with multiple records:", (molecule_counts > 1).sum())
print("Maximum records per molecule:", molecule_counts.max())

print("\n4. DUPLICATE MOLECULE EXAMPLES")
display(
    molecule_counts[molecule_counts > 1]
    .head(20)
    .rename("record_count")
    .to_frame()
)

print("\n5. EXTREME VALUE REVIEW")
display(
    df.loc[
        values.notna(),
        ["molecule_chembl_id", "standard_value", "standard_units"]
    ]
    .sort_values("standard_value", ascending=False)
    .head(10)
)

AC50 NUMERICAL QUALITY AUDIT

1. VALIDITY CHECK
Total records: 10019
Missing/non-numeric: 0
Non-positive values: 0
Infinite values: 0

2. DESCRIPTIVE STATISTICS (nM)


,AC50_nM
count,10019.000000
mean,19293.595119
std,11894.624262
min,20.000000
1%,891.300000
5%,3162.300000
25%,7943.300000
50%,17782.800000
75%,31622.800000
95%,35481.300000



3. MOLECULE DUPLICATION
Unique molecules: 9955
Molecules with multiple records: 58
Maximum records per molecule: 5

4. DUPLICATE MOLECULE EXAMPLES


,record_count
molecule_chembl_id,
CHEMBL157101,5
CHEMBL51085,3
CHEMBL333985,3
CHEMBL140,3
CHEMBL5483017,2
CHEMBL417799,2
CHEMBL78150,2
CHEMBL2002411,2
CHEMBL1326058,2



5. EXTREME VALUE REVIEW


,molecule_chembl_id,standard_value,standard_units
3614,CHEMBL1590946,56234.1,nM
633,CHEMBL1502467,39810.7,nM
7624,CHEMBL490129,39810.7,nM
83,CHEMBL1555805,35481.3,nM
89,CHEMBL2131053,35481.3,nM
10073,CHEMBL1384408,35481.3,nM
8560,CHEMBL1879251,35481.3,nM
6050,CHEMBL1433075,35481.3,nM
79,CHEMBL1699082,35481.3,nM
6018,CHEMBL1406387,35481.3,nM


In [ ]:
# ============================================================
# INVESTIGASI RECORD BERULANG - CHEMBL2354282
# ============================================================

import pandas as pd

df = assay.copy()

# Kolom identitas dan pengukuran yang tersedia
inspect_cols = [
    "molecule_chembl_id",
    "assay_chembl_id",
    "standard_type",
    "standard_value",
    "standard_units",
    "standard_relation",
    "document_chembl_id",
    "activity_id"
]

inspect_cols = [c for c in inspect_cols if c in df.columns]

# A. Pemeriksaan duplikasi seluruh kolom
print("=" * 70)
print("A. EXACT DUPLICATE RECORDS")
print("=" * 70)

print("Exact duplicate rows:", df.duplicated().sum())

# B. Ambil molekul yang memiliki lebih dari satu record
counts = df["molecule_chembl_id"].value_counts()
repeated_ids = counts[counts > 1].index

repeated_records = (
    df[df["molecule_chembl_id"].isin(repeated_ids)]
    .sort_values(["molecule_chembl_id", "standard_value"])
)

print("\nB. REPEATED MOLECULES")
print("Repeated molecules:", len(repeated_ids))
print("Records belonging to repeated molecules:",
      len(repeated_records))

display(repeated_records[inspect_cols].head(30))

# C. Ringkasan nilai AC50 per molekul
print("\nC. AC50 VARIATION PER MOLECULE")

duplicate_summary = (
    repeated_records
    .groupby("molecule_chembl_id")
    .agg(
        record_count=("standard_value", "size"),
        unique_ac50=("standard_value", "nunique"),
        min_ac50=("standard_value", "min"),
        max_ac50=("standard_value", "max")
    )
    .reset_index()
)

duplicate_summary["ac50_range"] = (
    duplicate_summary["max_ac50"] -
    duplicate_summary["min_ac50"]
)

display(duplicate_summary.sort_values(
    ["unique_ac50", "record_count"],
    ascending=False
))

print("\nD. DUPLICATION CATEGORY")
print(
    duplicate_summary["unique_ac50"]
    .value_counts()
    .sort_index()
    .rename_axis("unique_ac50_per_molecule")
    .to_frame("number_of_molecules")
)

A. EXACT DUPLICATE RECORDS
Exact duplicate rows: 0

B. REPEATED MOLECULES
Repeated molecules: 58
Records belonging to repeated molecules: 122


,molecule_chembl_id,assay_chembl_id,standard_type,standard_value,standard_units,standard_relation,document_chembl_id,activity_id
793,CHEMBL1034,CHEMBL2354282,Potency,16360.1,nM,NaN,CHEMBL1201862,13139747
9575,CHEMBL1034,CHEMBL2354282,Potency,35481.3,nM,NaN,CHEMBL1201862,13191485
3024,CHEMBL1076347,CHEMBL2354282,Potency,631.0,nM,NaN,CHEMBL1201862,13150993
7784,CHEMBL1076347,CHEMBL2354282,Potency,1000.0,nM,NaN,CHEMBL1201862,13180571
938,CHEMBL120734,CHEMBL2354282,Potency,2238.7,nM,NaN,CHEMBL1201862,13139892
2956,CHEMBL120734,CHEMBL2354282,Potency,3162.3,nM,NaN,CHEMBL1201862,13150925
4318,CHEMBL1208858,CHEMBL2354282,Potency,11220.2,nM,NaN,CHEMBL1201862,13158314
2575,CHEMBL1208858,CHEMBL2354282,Potency,22387.2,nM,NaN,CHEMBL1201862,13147533
6168,CHEMBL1256028,CHEMBL2354282,Potency,29092.9,nM,NaN,CHEMBL1201862,13166197
5744,CHEMBL1256028,CHEMBL2354282,Potency,35481.3,nM,NaN,CHEMBL1201862,13165773



C. AC50 VARIATION PER MOLECULE


,molecule_chembl_id,record_count,unique_ac50,min_ac50,max_ac50,ac50_range
25,CHEMBL157101,5,4,1122.0,5173.5,4051.5
12,CHEMBL140,3,3,12589.3,25118.9,12529.6
44,CHEMBL333985,3,3,3981.1,35481.3,31500.2
49,CHEMBL51085,3,3,3162.3,5804.8,2642.5
0,CHEMBL1034,2,2,16360.1,35481.3,19121.2
1,CHEMBL1076347,2,2,631.0,1000.0,369.0
2,CHEMBL120734,2,2,2238.7,3162.3,923.6
3,CHEMBL1208858,2,2,11220.2,22387.2,11167.0
4,CHEMBL1256028,2,2,29092.9,35481.3,6388.4
5,CHEMBL1256719,2,2,16360.1,22387.2,6027.1



D. DUPLICATION CATEGORY
                          number_of_molecules
unique_ac50_per_molecule                     
1                                           4
2                                          50
3                                           3
4                                           1


In [ ]:
# ============================================================
# FREQUENCY DISTRIBUTION OF AC50
# ============================================================

ac50_frequency = (
    df["standard_value"]
    .value_counts()
    .rename_axis("AC50_nM")
    .reset_index(name="record_count")
    .sort_values("record_count", ascending=False)
)

print("TOP 20 MOST FREQUENT AC50 VALUES")
display(ac50_frequency.head(20))

print("\nNUMBER OF UNIQUE AC50 VALUES:",
      df["standard_value"].nunique())

TOP 20 MOST FREQUENT AC50 VALUES


,AC50_nM,record_count
0,35481.3,2212
1,5623.4,856
2,31622.8,782
3,22387.2,688
4,12589.3,517
5,11220.2,512
6,25118.9,461
7,17782.8,417
8,19952.6,400
9,15848.9,394



NUMBER OF UNIQUE AC50 VALUES: 70


In [ ]:
# ============================================================
# INSPEKSI KETERSEDIAAN METADATA KURVA
# ============================================================

datasets_to_check = {
    "assay": assay,
    "assay_check": assay_check,
    "activity_reviewed": activity_reviewed,
    "alpha_activity": alpha_activity,
}

keywords = [
    "phenotype",
    "potency",
    "efficacy",
    "curve",
    "fit_logac50",
    "fit_r2",
    "hill",
    "activity_id",
    "pubchem",
    "sid",
    "aid"
]

for name, frame in datasets_to_check.items():
    print("\n" + "=" * 70)
    print(f"DATASET: {name} | SHAPE: {frame.shape}")
    print("=" * 70)

    relevant_cols = [
        col for col in frame.columns
        if any(keyword in col.lower() for keyword in keywords)
    ]

    print("Relevant columns:")
    print(relevant_cols)


DATASET: assay | SHAPE: (10019, 47)
Relevant columns:
['activity_id']

DATASET: assay_check | SHAPE: (10019, 49)
Relevant columns:
['activity_id']

DATASET: activity_reviewed | SHAPE: (12221, 50)
Relevant columns:
['activity_id']

DATASET: alpha_activity | SHAPE: (12221, 47)
Relevant columns:
['activity_id']


In [ ]:
datasets_to_check = {
    "assay": assay,
    "assay_check": assay_check,
    "activity_reviewed": activity_reviewed,
    "alpha_activity": alpha_activity,
}

for name, df in datasets_to_check.items():
    print("\n" + "=" * 70)
    print(f"DATASET: {name} | SHAPE: {df.shape}")
    print("=" * 70)

    print("ALL COLUMNS:")
    for col in df.columns:
        print(f"- {col}")


DATASET: assay | SHAPE: (10019, 47)
ALL COLUMNS:
- action_type
- activity_comment
- activity_id
- activity_properties
- assay_chembl_id
- assay_description
- assay_type
- assay_variant_accession
- assay_variant_mutation
- bao_endpoint
- bao_format
- bao_label
- canonical_smiles
- data_validity_comment
- data_validity_description
- document_chembl_id
- document_journal
- document_year
- ligand_efficiency
- modality
- molecule_chembl_id
- molecule_pref_name
- parent_molecule_chembl_id
- pchembl_value
- potential_duplicate
- qudt_units
- record_id
- relation
- src_id
- standard_flag
- standard_relation
- standard_text_value
- standard_type
- standard_units
- standard_upper_value
- standard_value
- target_chembl_id
- target_organism
- target_pref_name
- target_tax_id
- text_value
- toid
- type
- units
- uo_units
- upper_value
- value

DATASET: assay_check | SHAPE: (10019, 49)
ALL COLUMNS:
- action_type
- activity_comment
- activity_id
- activity_properties
- assay_chembl_id
- assay_descri

In [ ]:
import pandas as pd

df = assay.copy()

columns_to_inspect = [
    "activity_properties",
    "activity_comment",
    "action_type",
    "src_id",
    "record_id",
    "toid",
    "text_value",
    "type",
    "units",
    "value",
]

for col in columns_to_inspect:
    print("\n" + "=" * 75)
    print(f"COLUMN: {col}")
    print("=" * 75)

    if col not in df.columns:
        print("Column tidak ditemukan.")
        continue

    print("Missing:", df[col].isna().sum())
    print("Unique values:", df[col].nunique(dropna=True))

    print("\nContoh nilai:")
    display(df[col].dropna().head(5))


COLUMN: activity_properties
Missing: 0
Unique values: 1

Contoh nilai:


78    []
79    []
80    []
81    []
82    []
Name: activity_properties, dtype: str


COLUMN: activity_comment
Missing: 0
Unique values: 3

Contoh nilai:


78          active
79        inactive
80    inconclusive
81        inactive
82        inactive
Name: activity_comment, dtype: str


COLUMN: action_type
Missing: 10019
Unique values: 0

Contoh nilai:


Series([], Name: action_type, dtype: str)

... (10 outputs left out)


COLUMN: units
Missing: 0
Unique values: 1

Contoh nilai:


78    uM
79    uM
80    uM
81    uM
82    uM
Name: units, dtype: str


COLUMN: value
Missing: 0
Unique values: 70

Contoh nilai:


78     1.5849
79    35.4813
80    14.1254
81    35.4813
82    15.8489
Name: value, dtype: float64

In [ ]:
print("Tipe data activity_properties:")

display(
    assay["activity_properties"]
    .dropna()
    .map(type)
    .value_counts()
)

print("\nContoh activity_properties:")
for value in assay["activity_properties"].dropna().head(10):
    print(repr(value))
    print("-" * 60)

Tipe data activity_properties:


activity_properties
<class 'str'>    10019
Name: count, dtype: int64


Contoh activity_properties:
'[]'
------------------------------------------------------------
'[]'
------------------------------------------------------------
'[]'
------------------------------------------------------------
'[]'
------------------------------------------------------------
'[]'
------------------------------------------------------------
'[]'
------------------------------------------------------------
'[]'
------------------------------------------------------------
'[]'
------------------------------------------------------------
'[]'
------------------------------------------------------------
'[]'
------------------------------------------------------------


In [ ]:
print("DISTRIBUSI ACTIVITY COMMENT")
display(
    assay["activity_comment"]
    .value_counts(dropna=False)
    .rename_axis("activity_comment")
    .reset_index(name="record_count")
)

repeated_ids = (
    assay.groupby("molecule_chembl_id")
    .filter(lambda x: len(x) > 1)
)

print("Jumlah record dari molekul berulang:", len(repeated_ids))
print("Jumlah molekul berulang:", repeated_ids["molecule_chembl_id"].nunique())

display(
    pd.crosstab(
        repeated_ids["molecule_chembl_id"],
        repeated_ids["activity_comment"]
    ).head(15)
)

DISTRIBUSI ACTIVITY COMMENT


,activity_comment,record_count
0,inconclusive,5687
1,inactive,3829
2,active,503


Jumlah record dari molekul berulang: 122
Jumlah molekul berulang: 58


activity_comment,active,inactive,inconclusive
molecule_chembl_id,,,
CHEMBL1034,0,2,0
CHEMBL1076347,0,2,0
CHEMBL120734,0,2,0
CHEMBL1208858,0,2,0
CHEMBL1256028,0,2,0
CHEMBL1256719,0,2,0
CHEMBL1314461,0,2,0
CHEMBL1316480,0,2,0
CHEMBL1326058,0,2,0


In [ ]:
# Ringkasan jumlah record dan variasi label per molekul
repeated_audit = (
    repeated_ids
    .groupby("molecule_chembl_id")
    .agg(
        record_count=("activity_id", "count"),
        unique_labels=("activity_comment", "nunique"),
        unique_ac50=("standard_value", "nunique"),
        labels=("activity_comment", lambda x: ", ".join(sorted(x.unique())))
    )
    .reset_index()
)

# Kategori konsistensi label
repeated_audit["label_consistency"] = repeated_audit[
    "unique_labels"
].map({
    1: "Consistent",
}).fillna("Mixed")

print("DISTRIBUSI KONSISTENSI LABEL")
display(
    repeated_audit["label_consistency"]
    .value_counts()
    .rename_axis("consistency")
    .reset_index(name="molecule_count")
)

print("\nMOLEKUL DENGAN LABEL BERBEDA")
display(
    repeated_audit[
        repeated_audit["label_consistency"] == "Mixed"
    ]
)

print("\nRINGKASAN SELURUH MOLEKUL BERULANG")
display(repeated_audit)

DISTRIBUSI KONSISTENSI LABEL


,consistency,molecule_count
0,Consistent,53
1,Mixed,5



MOLEKUL DENGAN LABEL BERBEDA


,molecule_chembl_id,record_count,unique_labels,unique_ac50,labels,label_consistency
19,CHEMBL1468522,2,2,2,"active, inconclusive",Mixed
25,CHEMBL157101,5,2,4,"active, inconclusive",Mixed
26,CHEMBL1604119,2,2,2,"inactive, inconclusive",Mixed
44,CHEMBL333985,3,2,3,"active, inconclusive",Mixed
45,CHEMBL406845,2,2,2,"active, inconclusive",Mixed



RINGKASAN SELURUH MOLEKUL BERULANG


,molecule_chembl_id,record_count,unique_labels,unique_ac50,labels,label_consistency
0,CHEMBL1034,2,1,2,inactive,Consistent
1,CHEMBL1076347,2,1,2,inactive,Consistent
2,CHEMBL120734,2,1,2,inactive,Consistent
3,CHEMBL1208858,2,1,2,inactive,Consistent
4,CHEMBL1256028,2,1,2,inactive,Consistent
5,CHEMBL1256719,2,1,2,inactive,Consistent
6,CHEMBL1314461,2,1,2,inactive,Consistent
7,CHEMBL1316480,2,1,1,inactive,Consistent
8,CHEMBL1326058,2,1,2,inactive,Consistent
9,CHEMBL1335945,2,1,2,active,Consistent


In [ ]:
# Ambil ID kelima molekul dengan label campuran
mixed_molecules = repeated_audit.loc[
    repeated_audit["label_consistency"] == "Mixed",
    "molecule_chembl_id"
].tolist()

# Tampilkan detail setiap record
mixed_detail = (
    assay[
        assay["molecule_chembl_id"].isin(mixed_molecules)
    ][
        [
            "molecule_chembl_id",
            "activity_id",
            "record_id",
            "value",
            "units",
            "standard_value",
            "standard_units",
            "relation",
            "standard_relation",
            "activity_comment",
        ]
    ]
    .sort_values(["molecule_chembl_id", "standard_value"])
)

display(mixed_detail)

,molecule_chembl_id,activity_id,record_id,value,units,standard_value,standard_units,relation,standard_relation,activity_comment
5835,CHEMBL1468522,13165864,1009935,5.0119,uM,5011.9,nM,NaN,NaN,active
1693,CHEMBL1468522,13143650,1492157,17.7828,uM,17782.8,nM,NaN,NaN,inconclusive
9885,CHEMBL157101,13191795,1490996,1.1220,uM,1122.0,nM,NaN,NaN,inconclusive
6019,CHEMBL157101,13166048,1381857,1.9953,uM,1995.3,nM,NaN,NaN,inconclusive
6095,CHEMBL157101,13166124,1856597,3.1623,uM,3162.3,nM,NaN,NaN,active
6348,CHEMBL157101,13169384,1263470,3.1623,uM,3162.3,nM,NaN,NaN,inconclusive
7143,CHEMBL157101,13173405,1309631,5.1735,uM,5173.5,nM,NaN,NaN,active
5641,CHEMBL1604119,13165670,994149,12.5893,uM,12589.3,nM,NaN,NaN,inconclusive
3051,CHEMBL1604119,13151020,1345082,35.4813,uM,35481.3,nM,NaN,NaN,inactive
7685,CHEMBL333985,13180472,1490967,3.9811,uM,3981.1,nM,NaN,NaN,active


In [ ]:
# Ringkasan kategori assay pada dataset master
print("MASTER DATASET:", activity_reviewed.shape)

display(
    activity_reviewed
    .groupby(
        ["reviewed_assay_category", "reviewed_assay_decision"],
        dropna=False
    )
    .agg(
        record_count=("activity_id", "count"),
        unique_molecules=("molecule_chembl_id", "nunique"),
        unique_assays=("assay_chembl_id", "nunique")
    )
    .reset_index()
    .sort_values("record_count", ascending=False)
)

# Pastikan assay yang sedang diaudit tetap teridentifikasi
assay_master = activity_reviewed[
    activity_reviewed["assay_chembl_id"] == "CHEMBL2354282"
]

print("\nCHEMBL2354282 DI MASTER DATASET")
print("Records:", len(assay_master))
print("Unique activity IDs:", assay_master["activity_id"].nunique())
print("Unique molecules:", assay_master["molecule_chembl_id"].nunique())

MASTER DATASET: (12221, 50)


,reviewed_assay_category,reviewed_assay_decision,record_count,unique_molecules,unique_assays
12,NaN,NaN,11983,11043,315
1,Aggregation_Inhibition_Candidate,Candidate_Separate_Endpoint,111,33,42
8,Fibril_Formation_Inhibition,Candidate_Separate_Endpoint,53,32,25
10,Protein_Degradation_Clearance,Separate_Analysis,19,7,11
2,Cellular_Inclusion_Modulation,Separate_Analysis,15,13,7
5,Conformational_Modulation,Separate_Analysis,14,5,14
0,Aggregation_Induction,Exclude_from_Inhibition_Analysis,9,4,6
3,Cellular_Toxicity_or_Protection,Separate_Analysis,6,6,2
9,Oligomer_Modulation,Separate_Analysis,3,3,3
11,Unresolved_Keep_Quarantine,Pending_Review,3,3,1



CHEMBL2354282 DI MASTER DATASET
Records: 10019
Unique activity IDs: 10019
Unique molecules: 9955


In [ ]:
# ============================================================
# AUDIT RECORDS YANG BELUM MEMILIKI KATEGORI REVIEW
# ============================================================

unreviewed = activity_reviewed[
    activity_reviewed["reviewed_assay_category"].isna()
].copy()

print("=" * 70)
print("UNREVIEWED RECORD AUDIT")
print("=" * 70)

print("Records:", len(unreviewed))
print("Unique activity IDs:", unreviewed["activity_id"].nunique())
print("Unique assays:", unreviewed["assay_chembl_id"].nunique())
print("Unique molecules:", unreviewed["molecule_chembl_id"].nunique())

# Distribusi assay dengan jumlah record terbanyak
unreviewed_assay_summary = (
    unreviewed
    .groupby(
        ["assay_chembl_id", "document_chembl_id"],
        dropna=False
    )
    .agg(
        record_count=("activity_id", "count"),
        unique_molecules=("molecule_chembl_id", "nunique"),
        standard_types=("standard_type", "nunique"),
        assay_descriptions=("assay_description", "nunique")
    )
    .reset_index()
    .sort_values("record_count", ascending=False)
)

display(unreviewed_assay_summary.head(30))

UNREVIEWED RECORD AUDIT
Records: 11983
Unique activity IDs: 11983
Unique assays: 315
Unique molecules: 11043


,assay_chembl_id,document_chembl_id,record_count,unique_molecules,standard_types,assay_descriptions
13,CHEMBL2354282,CHEMBL1201862,10019,9955,1,1
19,CHEMBL3888008,CHEMBL3886537,90,89,1,1
296,CHEMBL5736665,CHEMBL5728395,75,25,3,1
236,CHEMBL5524380,CHEMBL5523695,68,68,1,1
294,CHEMBL5736378,CHEMBL5728251,57,11,3,1
35,CHEMBL4625544,CHEMBL4622956,47,47,1,1
38,CHEMBL4628227,CHEMBL4627295,47,47,1,1
39,CHEMBL4628228,CHEMBL4627295,47,47,1,1
15,CHEMBL3594881,CHEMBL3593202,45,45,1,1
295,CHEMBL5736397,CHEMBL5728260,42,14,3,1


In [ ]:
print(
    "CHEMBL2354282 masuk kategori NaN:",
    (
        unreviewed["assay_chembl_id"]
        == "CHEMBL2354282"
    ).sum()
)

CHEMBL2354282 masuk kategori NaN: 10019


In [ ]:
# ============================================================
# INTEGRASI HASIL REVIEW CHEMBL2354282 KE MASTER DATASET
# ============================================================

assay_id = "CHEMBL2354282"

# Salin agar objek hasil sebelumnya tetap aman
activity_reviewed_final = activity_reviewed.copy()

# Pastikan assay memang ada
mask = (
    activity_reviewed_final["assay_chembl_id"]
    == assay_id
)

print("Records yang akan diperbarui:", mask.sum())

# Pastikan jumlahnya sesuai hasil audit
assert mask.sum() == 10019, (
    "Jumlah record tidak sesuai audit sebelumnya."
)

# Integrasikan kategori dan keputusan
activity_reviewed_final.loc[
    mask, "reviewed_assay_category"
] = "Cellular_Toxicity_or_Protection"

activity_reviewed_final.loc[
    mask, "reviewed_assay_decision"
] = "Separate_Analysis"

# Verifikasi hasil
print("\nHASIL INTEGRASI")
print(
    activity_reviewed_final.loc[
        mask,
        [
            "assay_chembl_id",
            "reviewed_assay_category",
            "reviewed_assay_decision"
        ]
    ].value_counts(dropna=False)
)

print("\nSISA RECORD TANPA KATEGORI:")
print(
    activity_reviewed_final[
        "reviewed_assay_category"
    ].isna().sum()
)

Records yang akan diperbarui: 10019

HASIL INTEGRASI
assay_chembl_id  reviewed_assay_category          reviewed_assay_decision
CHEMBL2354282    Cellular_Toxicity_or_Protection  Separate_Analysis          10019
Name: count, dtype: int64

SISA RECORD TANPA KATEGORI:
1964


In [ ]:
# ============================================================
# AUDIT SISA ASSAY YANG BELUM DIREVIEW
# ============================================================

remaining_unreviewed = activity_reviewed_final[
    activity_reviewed_final[
        "reviewed_assay_category"
    ].isna()
].copy()

print("=" * 70)
print("REMAINING UNREVIEWED DATA")
print("=" * 70)

print("Records:", len(remaining_unreviewed))
print(
    "Unique assays:",
    remaining_unreviewed["assay_chembl_id"].nunique()
)
print(
    "Unique molecules:",
    remaining_unreviewed["molecule_chembl_id"].nunique()
)

remaining_summary = (
    remaining_unreviewed
    .groupby(
        ["assay_chembl_id", "document_chembl_id"],
        dropna=False
    )
    .agg(
        record_count=("activity_id", "count"),
        unique_molecules=("molecule_chembl_id", "nunique"),
        standard_types=("standard_type", "nunique"),
        standard_units=("standard_units", "nunique"),
        assay_descriptions=("assay_description", "nunique")
    )
    .reset_index()
    .sort_values("record_count", ascending=False)
)

display(remaining_summary.head(40))

REMAINING UNREVIEWED DATA
Records: 1964
Unique assays: 314
Unique molecules: 1093


,assay_chembl_id,document_chembl_id,record_count,unique_molecules,standard_types,standard_units,assay_descriptions
18,CHEMBL3888008,CHEMBL3886537,90,89,1,1,1
295,CHEMBL5736665,CHEMBL5728395,75,25,3,2,1
235,CHEMBL5524380,CHEMBL5523695,68,68,1,1,1
293,CHEMBL5736378,CHEMBL5728251,57,11,3,2,1
38,CHEMBL4628228,CHEMBL4627295,47,47,1,1,1
34,CHEMBL4625544,CHEMBL4622956,47,47,1,1,1
37,CHEMBL4628227,CHEMBL4627295,47,47,1,1,1
14,CHEMBL3594881,CHEMBL3593202,45,45,1,1,1
294,CHEMBL5736397,CHEMBL5728260,42,14,3,2,1
240,CHEMBL5551997,CHEMBL5550158,41,41,1,1,1


In [ ]:
# ============================================================
# AUDIT DETAIL METADATA 15 ASSAY TERATAS
# ============================================================

top_assay_ids = (
    remaining_summary
    .head(15)["assay_chembl_id"]
    .tolist()
)

metadata_columns = [
    "assay_chembl_id",
    "assay_description",
    "assay_type",
    "bao_endpoint",
    "bao_format",
    "bao_label",
    "target_chembl_id",
    "target_pref_name",
    "target_organism",
    "standard_type",
    "standard_units",
    "type",
    "units",
    "document_chembl_id"
]

metadata_columns = [
    col for col in metadata_columns
    if col in remaining_unreviewed.columns
]

assay_metadata_audit = (
    remaining_unreviewed[
        remaining_unreviewed["assay_chembl_id"].isin(top_assay_ids)
    ][metadata_columns]
    .drop_duplicates()
    .sort_values("assay_chembl_id")
)

print("Jumlah kombinasi metadata unik:", len(assay_metadata_audit))

display(assay_metadata_audit)

Jumlah kombinasi metadata unik: 27


,assay_chembl_id,assay_description,assay_type,bao_endpoint,bao_format,bao_label,target_chembl_id,target_pref_name,target_organism,standard_type,standard_units,type,units,document_chembl_id
35,CHEMBL1613965,PUBCHEM_BIOASSAY: Alpha-synuclein ThS binding assay. (Class of assay: confirmatory) [Related pubchem assays: 596 ],B,BAO_0000186,BAO_0000357,single protein format,CHEMBL6152,Alpha-synuclein,Homo sapiens,Potency,nM,Potency,um,CHEMBL1201862
72,CHEMBL2065090,Binding affinity to human alpha-synuclein expressed in Escherichia coli BL21 (DE3) cells after 1 hr by thioflavin T fluorescence assay,B,BAO_0000192,BAO_0000219,cell-based format,CHEMBL6152,Alpha-synuclein,Homo sapiens,Ki,nM,Ki,nM,CHEMBL2062425
91,CHEMBL2065090,Binding affinity to human alpha-synuclein expressed in Escherichia coli BL21 (DE3) cells after 1 hr by thioflavin T fluorescence assay,B,BAO_0000190,BAO_0000219,cell-based format,CHEMBL6152,Alpha-synuclein,Homo sapiens,IC50,nM,IC50,nM,CHEMBL2062425
10143,CHEMBL3594881,Displacement of Thio-T from recombinant alpha-synuclein (unknown origin) expressed in Escherichia coli after 1.5 hrs by fluorescence assay,B,BAO_0000192,BAO_0000019,assay format,CHEMBL6152,Alpha-synuclein,Homo sapiens,Ki,nM,Ki,nM,CHEMBL3593202
10221,CHEMBL3888008,"In Vitro Inhibition Assay: α-Synucleinopathies are characterised by aggregation of α-synuclein in neurons. Aggregation of purified α-synuclein is performed essentially as described by Gerard et al. FASEB. 20(3):524-6 (2006). 20-100 μg purified α-synuclein (Sigma; S7820) at a concentration of about 2.5 μg/mL is incubated in the presence of spermin (250 μM) or paraquat (32 mM) or 6-hydroxydopamine (400 μM) or vehicle in a 384 well plate. Spermin, paraquat and 6-hydroxydopamine promote the α-synuclein aggregation process. Aggregation kinetics is determined by measuring turbidity at 340 nm, every 1-15 minutes for at least one hour.",B,BAO_0000188,BAO_0000357,single protein format,CHEMBL6152,Alpha-synuclein,Homo sapiens,EC50,nM,EC50,nM,CHEMBL3886537
10404,CHEMBL4625544,Inhibition of alpha-synuclein aggregation (unknown origin) at 30 uM incubated for 3 days by thioflavin T based fluorescence assay relative to control,B,BAO_0000201,BAO_0000357,single protein format,CHEMBL6152,Alpha-synuclein,Homo sapiens,Inhibition,%,INH,%,CHEMBL4622956
10428,CHEMBL4625544,Inhibition of alpha-synuclein aggregation (unknown origin) at 30 uM incubated for 3 days by thioflavin T based fluorescence assay relative to control,B,BAO_0000201,BAO_0000357,single protein format,CHEMBL6152,Alpha-synuclein,Homo sapiens,Inhibition,%,INH,NaN,CHEMBL4622956
10467,CHEMBL4628227,Binding affinity to human wlid-type alpha-synuclein at 50 uM by SPR analysis,B,BAO_0000181,BAO_0000357,single protein format,CHEMBL6152,Alpha-synuclein,Homo sapiens,Activity,RU,Activity,RU,CHEMBL4627295
10514,CHEMBL4628228,Binding affinity to human wlid-type alpha-synuclein by SPR analysis,B,BAO_0000034,BAO_0000357,single protein format,CHEMBL6152,Alpha-synuclein,Homo sapiens,Kd,nM,Kd,nM,CHEMBL4627295
10515,CHEMBL4628228,Binding affinity to human wlid-type alpha-synuclein by SPR analysis,B,BAO_0000034,BAO_0000357,single protein format,CHEMBL6152,Alpha-synuclein,Homo sapiens,Kd,NaN,Kd,NaN,CHEMBL4627295


In [ ]:
# ============================================================
# DISTRIBUSI ENDPOINT DAN SATUAN PER ASSAY
# ============================================================

endpoint_audit = (
    remaining_unreviewed
    .groupby(
        [
            "assay_chembl_id",
            "standard_type",
            "standard_units"
        ],
        dropna=False
    )
    .agg(
        record_count=("activity_id", "count"),
        unique_molecules=("molecule_chembl_id", "nunique")
    )
    .reset_index()
    .sort_values(
        ["assay_chembl_id", "record_count"],
        ascending=[True, False]
    )
)

display(
    endpoint_audit[
        endpoint_audit["assay_chembl_id"].isin(top_assay_ids)
    ]
)

,assay_chembl_id,standard_type,standard_units,record_count,unique_molecules
10,CHEMBL1613965,Potency,nM,37,37
11,CHEMBL2065090,IC50,nM,19,19
12,CHEMBL2065090,Ki,nM,19,19
16,CHEMBL3594881,Ki,nM,45,45
20,CHEMBL3888008,EC50,nM,90,89
37,CHEMBL4625544,Inhibition,%,47,47
40,CHEMBL4628227,Activity,RU,47,47
42,CHEMBL4628228,Kd,NaN,33,33
41,CHEMBL4628228,Kd,nM,14,14
150,CHEMBL5347686,Inhibition,%,40,40


In [ ]:
# ============================================================
# AUDIT DESKRIPSI EKSPERIMEN 15 ASSAY TERATAS
# ============================================================

selected_assays = [
    "CHEMBL1613965",
    "CHEMBL2065090",
    "CHEMBL3594881",
    "CHEMBL3888008",
    "CHEMBL4625544",
    "CHEMBL4628227",
    "CHEMBL4628228",
    "CHEMBL5347686",
    "CHEMBL5524380",
    "CHEMBL5551997",
    "CHEMBL5580095",
    "CHEMBL5736378",
    "CHEMBL5736397",
    "CHEMBL5736665",
    "CHEMBL5737082"
]

description_columns = [
    "assay_chembl_id",
    "document_chembl_id",
    "assay_description",
    "assay_type",
    "bao_endpoint",
    "bao_format",
    "bao_label",
    "target_chembl_id",
    "target_pref_name",
    "target_organism"
]

description_columns = [
    col for col in description_columns
    if col in remaining_unreviewed.columns
]

description_audit = (
    remaining_unreviewed[
        remaining_unreviewed["assay_chembl_id"].isin(
            selected_assays
        )
    ][description_columns]
    .drop_duplicates()
    .sort_values("assay_chembl_id")
)

display(description_audit.to_string(index=False))

'assay_chembl_id document_chembl_id                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                           assay_desc

In [ ]:
# ============================================================
# AUDIT RECORD DETAIL ASSAY MULTI-ENDPOINT
# ============================================================

kinetic_assays = [
    "CHEMBL5736378",
    "CHEMBL5736397",
    "CHEMBL5736665",
    "CHEMBL5737082"
]

detail_columns = [
    "assay_chembl_id",
    "molecule_chembl_id",
    "standard_type",
    "standard_relation",
    "standard_value",
    "standard_units",
    "type",
    "relation",
    "value",
    "units"
]

detail_columns = [
    col for col in detail_columns
    if col in remaining_unreviewed.columns
]

kinetic_detail = (
    remaining_unreviewed[
        remaining_unreviewed["assay_chembl_id"].isin(
            kinetic_assays
        )
    ][detail_columns]
    .sort_values(
        ["assay_chembl_id", "molecule_chembl_id", "standard_type"]
    )
)

display(kinetic_detail.head(60))

,assay_chembl_id,molecule_chembl_id,standard_type,standard_relation,standard_value,standard_units,type,relation,value,units
11876,CHEMBL5736378,CHEMBL1908919,Kd,=,24.30,nM,Kd,=,24.30,nM
11909,CHEMBL5736378,CHEMBL1908919,Kd,=,24.30,nM,Kd,=,24.30,nM
11878,CHEMBL5736378,CHEMBL1908919,k_off,=,NaN,s-1,koff,=,NaN,s-1
11911,CHEMBL5736378,CHEMBL1908919,k_off,=,NaN,s-1,koff,=,NaN,s-1
11877,CHEMBL5736378,CHEMBL1908919,kon,=,NaN,NaN,kon,=,NaN,M-1-s-1
11910,CHEMBL5736378,CHEMBL1908919,kon,=,NaN,NaN,kon,=,NaN,M-1-s-1
11879,CHEMBL5736378,CHEMBL207649,Kd,=,16.10,nM,Kd,=,16.10,nM
11912,CHEMBL5736378,CHEMBL207649,Kd,=,16.10,nM,Kd,=,16.10,nM
11881,CHEMBL5736378,CHEMBL207649,k_off,=,NaN,s-1,koff,=,NaN,s-1
11914,CHEMBL5736378,CHEMBL207649,k_off,=,NaN,s-1,koff,=,NaN,s-1


In [ ]:
# ============================================================
# AUDIT KELENGKAPAN DAN DUPLIKASI ASSAY KINETIKA
# ============================================================

kinetic_assays = [
    "CHEMBL5736378",
    "CHEMBL5736397",
    "CHEMBL5736665",
    "CHEMBL5737082"
]

kinetic_data = remaining_unreviewed[
    remaining_unreviewed["assay_chembl_id"].isin(
        kinetic_assays
    )
].copy()

# 1. Kelengkapan nilai per assay dan endpoint
completeness = (
    kinetic_data
    .groupby(
        ["assay_chembl_id", "standard_type"],
        dropna=False
    )
    .agg(
        records=("activity_id", "count"),
        unique_molecules=("molecule_chembl_id", "nunique"),
        numeric_values=("standard_value", "count"),
        missing_values=("standard_value", lambda x: x.isna().sum()),
        units=("standard_units", lambda x: x.dropna().unique().tolist())
    )
    .reset_index()
)

display(completeness)

# 2. Periksa kolom identitas record
identity_columns = [
    "activity_id",
    "record_id",
    "assay_chembl_id",
    "molecule_chembl_id",
    "standard_type",
    "standard_value",
    "standard_units",
    "document_chembl_id"
]

identity_columns = [
    col for col in identity_columns
    if col in kinetic_data.columns
]

print("\nIDENTITY AUDIT")
for col in ["activity_id", "record_id"]:
    if col in kinetic_data.columns:
        print(
            f"{col}:",
            "missing =", kinetic_data[col].isna().sum(),
            "| unique =", kinetic_data[col].nunique()
        )

# 3. Cari baris dengan kombinasi molekul dan endpoint berulang
repeated_measurements = (
    kinetic_data
    .groupby(
        [
            "assay_chembl_id",
            "molecule_chembl_id",
            "standard_type"
        ],
        dropna=False
    )
    .filter(lambda x: len(x) > 1)
    [identity_columns]
    .sort_values(
        [
            "assay_chembl_id",
            "molecule_chembl_id",
            "standard_type"
        ]
    )
)

print("\nREPEATED MEASUREMENTS")
print("Records:", len(repeated_measurements))
display(repeated_measurements.head(50))

# 4. Duplikasi berdasarkan identitas aktivitas
print("\nDUPLICATE ACTIVITY ID:")
print(kinetic_data["activity_id"].duplicated().sum())

if "record_id" in kinetic_data.columns:
    print(
        "DUPLICATE RECORD ID:",
        kinetic_data["record_id"].duplicated().sum()
    )

,assay_chembl_id,standard_type,records,unique_molecules,numeric_values,missing_values,units
0,CHEMBL5736378,Kd,19,11,19,0,[nM]
1,CHEMBL5736378,k_off,19,11,0,19,[s-1]
2,CHEMBL5736378,kon,19,11,0,19,[]
3,CHEMBL5736397,IC50,14,14,14,0,[nM]
4,CHEMBL5736397,k_off,14,14,0,14,[s-1]
5,CHEMBL5736397,kon,14,14,0,14,[]
6,CHEMBL5736665,IC50,25,25,25,0,[nM]
7,CHEMBL5736665,k_off,25,25,0,25,[s-1]
8,CHEMBL5736665,kon,25,25,0,25,[]
9,CHEMBL5737082,IC50,13,13,13,0,[nM]



IDENTITY AUDIT
activity_id: missing = 0 | unique = 213
record_id: missing = 0 | unique = 63

REPEATED MEASUREMENTS
Records: 48


,activity_id,record_id,assay_chembl_id,molecule_chembl_id,standard_type,standard_value,standard_units,document_chembl_id
11876,28237931,4473787,CHEMBL5736378,CHEMBL1908919,Kd,24.30,nM,CHEMBL5728251
11909,28237997,4473787,CHEMBL5736378,CHEMBL1908919,Kd,24.30,nM,CHEMBL5728251
11878,28237933,4473787,CHEMBL5736378,CHEMBL1908919,k_off,NaN,s-1,CHEMBL5728251
11911,28237999,4473787,CHEMBL5736378,CHEMBL1908919,k_off,NaN,s-1,CHEMBL5728251
11877,28237932,4473787,CHEMBL5736378,CHEMBL1908919,kon,NaN,NaN,CHEMBL5728251
11910,28237998,4473787,CHEMBL5736378,CHEMBL1908919,kon,NaN,NaN,CHEMBL5728251
11879,28237937,4473788,CHEMBL5736378,CHEMBL207649,Kd,16.10,nM,CHEMBL5728251
11912,28238003,4473788,CHEMBL5736378,CHEMBL207649,Kd,16.10,nM,CHEMBL5728251
11881,28237939,4473788,CHEMBL5736378,CHEMBL207649,k_off,NaN,s-1,CHEMBL5728251
11914,28238005,4473788,CHEMBL5736378,CHEMBL207649,k_off,NaN,s-1,CHEMBL5728251



DUPLICATE ACTIVITY ID:
0
DUPLICATE RECORD ID: 150


In [ ]:
# ============================================================
# DUPLICATE RECORD_ID AUDIT
# ============================================================

duplicate_record_ids = (
    remaining_unreviewed[
        remaining_unreviewed["record_id"].duplicated(keep=False)
    ]
    .sort_values(["record_id", "standard_type", "activity_id"])
    .copy()
)

print("TOTAL ROWS WITH REPEATED RECORD_ID:",
      len(duplicate_record_ids))

print("UNIQUE REPEATED RECORD_ID:",
      duplicate_record_ids["record_id"].nunique())


# Ringkasan jumlah baris per record_id
record_id_summary = (
    duplicate_record_ids
    .groupby("record_id", dropna=False)
    .agg(
        row_count=("activity_id", "count"),
        unique_activity=("activity_id", "nunique"),
        unique_assay=("assay_chembl_id", "nunique"),
        unique_molecule=("molecule_chembl_id", "nunique"),
        unique_endpoint=("standard_type", "nunique"),
        unique_values=("standard_value", "nunique")
    )
    .reset_index()
)

display(record_id_summary.head(30))


# Periksa kombinasi identitas yang sama
comparison_columns = [
    "record_id",
    "assay_chembl_id",
    "molecule_chembl_id",
    "standard_type",
    "standard_value",
    "standard_units",
    "standard_relation",
    "activity_id"
]

comparison_columns = [
    col for col in comparison_columns
    if col in duplicate_record_ids.columns
]

display(
    duplicate_record_ids[comparison_columns]
    .head(60)
)

TOTAL ROWS WITH REPEATED RECORD_ID: 1166
UNIQUE REPEATED RECORD_ID: 315


,record_id,row_count,unique_activity,unique_assay,unique_molecule,unique_endpoint,unique_values
0,953293,8,8,8,1,1,7
1,953424,8,8,8,1,1,6
2,1659761,2,2,1,1,2,2
3,1659762,2,2,1,1,2,2
4,1659763,2,2,1,1,2,2
5,1659764,2,2,1,1,2,2
6,1659765,2,2,1,1,2,2
7,1659766,3,3,2,1,3,2
8,1659767,2,2,1,1,2,2
9,1659768,2,2,1,1,2,2


,record_id,assay_chembl_id,molecule_chembl_id,standard_type,standard_value,standard_units,standard_relation,activity_id
23,953293,CHEMBL1250260,CHEMBL1242187,Activity,220.0,a.u.,=,3446908
24,953293,CHEMBL1250261,CHEMBL1242187,Activity,330.0,a.u.,=,3446909
25,953293,CHEMBL1250262,CHEMBL1242187,Activity,410.0,a.u.,=,3446910
26,953293,CHEMBL1250263,CHEMBL1242187,Activity,500.0,a.u.,=,3446911
27,953293,CHEMBL1250264,CHEMBL1242187,Activity,40.0,%,=,3446912
28,953293,CHEMBL1250266,CHEMBL1242187,Activity,60.0,%,=,3446913
29,953293,CHEMBL1250267,CHEMBL1242187,Activity,100.0,%,=,3446914
34,953293,CHEMBL1250273,CHEMBL1242187,Activity,NaN,NaN,NaN,3448364
12,953424,CHEMBL1250260,CHEMBL112570,Activity,0.0,a.u.,=,3446897
13,953424,CHEMBL1250261,CHEMBL112570,Activity,25.0,a.u.,=,3446898


In [ ]:
# ============================================================
# SAME RECORD + SAME MEASUREMENT AUDIT
# ============================================================

measurement_columns = [
    "record_id",
    "assay_chembl_id",
    "molecule_chembl_id",
    "standard_type",
    "standard_value",
    "standard_units",
    "standard_relation"
]

measurement_columns = [
    col for col in measurement_columns
    if col in remaining_unreviewed.columns
]

same_measurement_groups = (
    remaining_unreviewed
    .groupby(measurement_columns, dropna=False)
    .filter(lambda group: len(group) > 1)
    .sort_values(["record_id", "standard_type"])
)

print("ROWS IN REPEATED MEASUREMENT GROUPS:",
      len(same_measurement_groups))

print("UNIQUE RECORD IDS:",
      same_measurement_groups["record_id"].nunique())

display(
    same_measurement_groups[
        measurement_columns + ["activity_id"]
    ].head(60)
)

ROWS IN REPEATED MEASUREMENT GROUPS: 48
UNIQUE RECORD IDS: 8


,record_id,assay_chembl_id,molecule_chembl_id,standard_type,standard_value,standard_units,standard_relation,activity_id
11876,4473787,CHEMBL5736378,CHEMBL1908919,Kd,24.30,nM,=,28237931
11909,4473787,CHEMBL5736378,CHEMBL1908919,Kd,24.30,nM,=,28237997
11878,4473787,CHEMBL5736378,CHEMBL1908919,k_off,NaN,s-1,=,28237933
11911,4473787,CHEMBL5736378,CHEMBL1908919,k_off,NaN,s-1,=,28237999
11877,4473787,CHEMBL5736378,CHEMBL1908919,kon,NaN,NaN,=,28237932
11910,4473787,CHEMBL5736378,CHEMBL1908919,kon,NaN,NaN,=,28237998
11879,4473788,CHEMBL5736378,CHEMBL207649,Kd,16.10,nM,=,28237937
11912,4473788,CHEMBL5736378,CHEMBL207649,Kd,16.10,nM,=,28238003
11881,4473788,CHEMBL5736378,CHEMBL207649,k_off,NaN,s-1,=,28237939
11914,4473788,CHEMBL5736378,CHEMBL207649,k_off,NaN,s-1,=,28238005


In [ ]:
kinetic_record_ids = [
    4473787, 4473788, 4473789, 4473790,
    4473791, 4473796, 4473797, 4473800
]

audit_columns = [
    "activity_id",
    "record_id",
    "assay_chembl_id",
    "molecule_chembl_id",
    "standard_type",
    "standard_value",
    "standard_units",
    "standard_relation",
    "document_chembl_id",
]

audit_columns = [
    col for col in audit_columns
    if col in remaining_unreviewed.columns
]

kinetic_record_audit = (
    remaining_unreviewed[
        remaining_unreviewed["record_id"].isin(
            kinetic_record_ids
        )
    ][audit_columns]
    .sort_values(
        ["record_id", "standard_type", "activity_id"]
    )
)

display(kinetic_record_audit)

,activity_id,record_id,assay_chembl_id,molecule_chembl_id,standard_type,standard_value,standard_units,standard_relation,document_chembl_id
11876,28237931,4473787,CHEMBL5736378,CHEMBL1908919,Kd,24.30,nM,=,CHEMBL5728251
11909,28237997,4473787,CHEMBL5736378,CHEMBL1908919,Kd,24.30,nM,=,CHEMBL5728251
11849,28237859,4473787,CHEMBL5736375,CHEMBL1908919,Ki,76.60,nM,=,CHEMBL5728251
11851,28237861,4473787,CHEMBL5736375,CHEMBL1908919,k_off,NaN,s-1,=,CHEMBL5728251
11878,28237933,4473787,CHEMBL5736378,CHEMBL1908919,k_off,NaN,s-1,=,CHEMBL5728251
11911,28237999,4473787,CHEMBL5736378,CHEMBL1908919,k_off,NaN,s-1,=,CHEMBL5728251
11850,28237860,4473787,CHEMBL5736375,CHEMBL1908919,kon,NaN,NaN,=,CHEMBL5728251
11877,28237932,4473787,CHEMBL5736378,CHEMBL1908919,kon,NaN,NaN,=,CHEMBL5728251
11910,28237998,4473787,CHEMBL5736378,CHEMBL1908919,kon,NaN,NaN,=,CHEMBL5728251
11879,28237937,4473788,CHEMBL5736378,CHEMBL207649,Kd,16.10,nM,=,CHEMBL5728251


In [ ]:
kinetic_record_audit["activity_id_group"] = (
    kinetic_record_audit["activity_id"]
    .astype(str)
    .str[:5]
)

display(
    kinetic_record_audit.groupby(
        ["record_id", "standard_type", "activity_id_group"],
        dropna=False
    ).agg(
        rows=("activity_id", "count"),
        values=("standard_value", "unique"),
        activities=("activity_id", "unique")
    ).reset_index()
)

,record_id,standard_type,activity_id_group,rows,values,activities
0,4473787,Kd,28237,2,[24.3],"[28237931, 28237997]"
1,4473787,Ki,28237,1,[76.6],[28237859]
2,4473787,k_off,28237,3,[nan],"[28237861, 28237933, 28237999]"
3,4473787,kon,28237,3,[nan],"[28237860, 28237932, 28237998]"
4,4473788,Kd,28237,1,[16.1],[28237937]
5,4473788,Kd,28238,1,[16.1],[28238003]
6,4473788,Ki,28237,1,[25.1],[28237868]
7,4473788,k_off,28237,2,[nan],"[28237870, 28237939]"
8,4473788,k_off,28238,1,[nan],[28238005]
9,4473788,kon,28237,2,[nan],"[28237869, 28237938]"


In [ ]:
# ============================================================
# POTENTIAL DUPLICATE FLAG
# ============================================================

duplicate_key = [
    "record_id",
    "assay_chembl_id",
    "molecule_chembl_id",
    "standard_type",
    "standard_value",
    "standard_units",
    "standard_relation",
    "document_chembl_id",
]

# Identifikasi kelompok dengan kombinasi atribut yang sama
remaining_unreviewed["potential_duplicate"] = (
    remaining_unreviewed
    .duplicated(subset=duplicate_key, keep=False)
)

# Ringkasan
potential_duplicates = remaining_unreviewed[
    remaining_unreviewed["potential_duplicate"]
].copy()

print("=" * 65)
print("POTENTIAL DUPLICATE SUMMARY")
print("=" * 65)

print("Total rows flagged:", len(potential_duplicates))

print(
    "Unique record IDs:",
    potential_duplicates["record_id"].nunique()
)

print(
    "Unique activity IDs:",
    potential_duplicates["activity_id"].nunique()
)

print(
    "Unique assay IDs:",
    potential_duplicates["assay_chembl_id"].nunique()
)

display(
    potential_duplicates[
        duplicate_key + ["activity_id"]
    ].sort_values(
        ["assay_chembl_id", "record_id", "standard_type"]
    ).head(30)
)

POTENTIAL DUPLICATE SUMMARY
Total rows flagged: 48
Unique record IDs: 8
Unique activity IDs: 48
Unique assay IDs: 1


,record_id,assay_chembl_id,molecule_chembl_id,standard_type,standard_value,standard_units,standard_relation,document_chembl_id,activity_id
11876,4473787,CHEMBL5736378,CHEMBL1908919,Kd,24.30,nM,=,CHEMBL5728251,28237931
11909,4473787,CHEMBL5736378,CHEMBL1908919,Kd,24.30,nM,=,CHEMBL5728251,28237997
11878,4473787,CHEMBL5736378,CHEMBL1908919,k_off,NaN,s-1,=,CHEMBL5728251,28237933
11911,4473787,CHEMBL5736378,CHEMBL1908919,k_off,NaN,s-1,=,CHEMBL5728251,28237999
11877,4473787,CHEMBL5736378,CHEMBL1908919,kon,NaN,NaN,=,CHEMBL5728251,28237932
11910,4473787,CHEMBL5736378,CHEMBL1908919,kon,NaN,NaN,=,CHEMBL5728251,28237998
11879,4473788,CHEMBL5736378,CHEMBL207649,Kd,16.10,nM,=,CHEMBL5728251,28237937
11912,4473788,CHEMBL5736378,CHEMBL207649,Kd,16.10,nM,=,CHEMBL5728251,28238003
11881,4473788,CHEMBL5736378,CHEMBL207649,k_off,NaN,s-1,=,CHEMBL5728251,28237939
11914,4473788,CHEMBL5736378,CHEMBL207649,k_off,NaN,s-1,=,CHEMBL5728251,28238005


In [ ]:
# ============================================================
# ASSAY METADATA AUDIT
# ============================================================

assay_ids_to_check = [
    "CHEMBL5736375",
    "CHEMBL5736378"
]

assay_metadata_columns = [
    "assay_chembl_id",
    "assay_type",
    "description",
    "assay_organism",
    "assay_tax_id",
    "bao_endpoint",
    "bao_format",
    "document_chembl_id",
]

available_columns = [
    col for col in assay_metadata_columns
    if col in assay_metadata.columns
]

assay_description_audit = (
    assay_metadata[
        assay_metadata["assay_chembl_id"].isin(
            assay_ids_to_check
        )
    ][available_columns]
    .drop_duplicates()
)

display(assay_description_audit)

NameError: name 'assay_metadata' is not defined

In [ ]:
print("ASSAY_META SHAPE:", assay_meta.shape)
print("\nCOLUMNS:")
print(assay_meta.columns.tolist())

display(assay_meta.head(3))

ASSAY_META SHAPE: (429, 29)

COLUMNS:
['aidx', 'assay_category', 'assay_cell_type', 'assay_chembl_id', 'assay_classifications', 'assay_group', 'assay_organism', 'assay_parameters', 'assay_strain', 'assay_subcellular_fraction', 'assay_tax_id', 'assay_test_type', 'assay_tissue', 'assay_type', 'assay_type_description', 'bao_format', 'bao_label', 'cell_chembl_id', 'confidence_description', 'confidence_score', 'description', 'document_chembl_id', 'relationship_description', 'relationship_type', 'src_assay_id', 'src_id', 'target_chembl_id', 'tissue_chembl_id', 'variant_sequence']


,aidx,assay_category,assay_cell_type,assay_chembl_id,assay_classifications,assay_group,assay_organism,assay_parameters,assay_strain,assay_subcellular_fraction,...,confidence_score,description,document_chembl_id,relationship_description,relationship_type,src_assay_id,src_id,target_chembl_id,tissue_chembl_id,variant_sequence
0,CLD0,NaN,NaN,CHEMBL945801,[],NaN,NaN,[],NaN,NaN,...,8,Binding affinity to alpha-synuclein fibrils by fluorescent titration,CHEMBL1143231,Homologous protein target assigned,H,NaN,1,CHEMBL6152,NaN,NaN
1,CLD0,NaN,NaN,CHEMBL1218779,[],NaN,Homo sapiens,[],NaN,NaN,...,9,Inhibition of alpha-synuclein-mediated toxicity in yeast transfected with human alpha-synuclein by spotting assay,CHEMBL1212724,Direct protein target assigned,D,NaN,1,CHEMBL6152,NaN,NaN
2,CLD0,NaN,NaN,CHEMBL1218780,[],NaN,Homo sapiens,[],NaN,NaN,...,9,Inhibition of human alpha-synuclein-mediated accumulation of stalled vesicle pools in yeast transfected with human alpha-synuclein after 8 to 12 hrs by electron microscopy,CHEMBL1212724,Direct protein target assigned,D,NaN,1,CHEMBL6152,NaN,NaN


In [ ]:
assay_ids_to_check = [
    "CHEMBL5736375",
    "CHEMBL5736378"
]

assay_metadata_columns = [
    "assay_chembl_id",
    "assay_type",
    "description",
    "assay_organism",
    "assay_tax_id",
    "bao_endpoint",
    "bao_format",
    "document_chembl_id",
]

available_columns = [
    col for col in assay_metadata_columns
    if col in assay_meta.columns
]

assay_description_audit = (
    assay_meta[
        assay_meta["assay_chembl_id"].isin(assay_ids_to_check)
    ][available_columns]
    .drop_duplicates()
)

display(assay_description_audit)

,assay_chembl_id,assay_type,description,assay_organism,assay_tax_id,bao_format,document_chembl_id
404,CHEMBL5736375,B,"In Vitro Inhibition Assay: Table 5: Fresh solutions of the test compounds (2.5 mM in DMSO) are diluted in PBS pH7.4, at concentrations between 0.5 nM and 5 μM in the presence of α-syn (200 nM), Aβ1-42 (500 nM) or Tau (200 nM) fibres and ThT 500 nM in a final 2 mL volume. The samples are incubated for 1 h at room temperature (RT) to reach equilibrium. The binding of ThT to α-syn, Aβ1-42 or Tau fibres is measured by fluorescence (excitation: 440 nm, emission: 480 nm). Inhibition of ThT binding by increasing concentrations of each test compound is measured by fluorescence reduction. Due to the partial superposition between the emission spectra of some compounds and ThT, the compounds are also incubated with α-syn, Aβ1-42 or Tau fibers in the absence of ThT. For each concentration, the fluorescence values in the absence of ThT are subtracted from those in the presence of ThT. The results are expressed as a percentage of the maximum fluorescence value of the ThT measured in the absence of a competitor (100% bind).",NaN,NaN,BAO_0000357,CHEMBL5728251
405,CHEMBL5736378,B,"In Vitro Direct Determination By Fluorescence: Table 6 & 7: Fresh solutions of each test compound (2.5 mM in DMSO) are diluted in PBS pH=7.4 at concentrations between 0.5 nM and 5 μM in the presence or absence of α-syn (200 nM), Aβ1-42 (500 nM) or Tau (200 nM) fibers in a final 2 mL volume. After 1 h of incubation at RT, the binding of each compound to α-syn, A131-42 or Tau fibers is measured by fluorescence using the spectral changes of the compounds during their binding to the fibers, using the appropriate excitation/emission wavelengths (those giving the highest fluorescence variation during binding to the fibres). For each concentration of compound added to the fibres, the bound and free fractions are measured by fluorescence according to the following equation:F m =F B B+F F×(T−B).",NaN,NaN,BAO_0000357,CHEMBL5728251


In [ ]:
# Kolom penting untuk membandingkan pengukuran
audit_columns = [
    "record_id",
    "activity_id",
    "assay_chembl_id",
    "molecule_chembl_id",
    "standard_type",
    "standard_relation",
    "standard_value",
    "standard_units",
    "type",
    "relation",
    "value",
    "units",
    "document_chembl_id",
]

available_audit_columns = [
    col for col in audit_columns
    if col in potential_duplicates.columns
]

potential_duplicate_detail = (
    potential_duplicates[available_audit_columns]
    .sort_values([
        "record_id",
        "molecule_chembl_id",
        "standard_type",
        "activity_id"
    ])
)

display(potential_duplicate_detail)

,record_id,activity_id,assay_chembl_id,molecule_chembl_id,standard_type,standard_relation,standard_value,standard_units,type,relation,value,units,document_chembl_id
11876,4473787,28237931,CHEMBL5736378,CHEMBL1908919,Kd,=,24.30,nM,Kd,=,24.30,nM,CHEMBL5728251
11909,4473787,28237997,CHEMBL5736378,CHEMBL1908919,Kd,=,24.30,nM,Kd,=,24.30,nM,CHEMBL5728251
11878,4473787,28237933,CHEMBL5736378,CHEMBL1908919,k_off,=,NaN,s-1,koff,=,NaN,s-1,CHEMBL5728251
11911,4473787,28237999,CHEMBL5736378,CHEMBL1908919,k_off,=,NaN,s-1,koff,=,NaN,s-1,CHEMBL5728251
11877,4473787,28237932,CHEMBL5736378,CHEMBL1908919,kon,=,NaN,NaN,kon,=,NaN,M-1-s-1,CHEMBL5728251
11910,4473787,28237998,CHEMBL5736378,CHEMBL1908919,kon,=,NaN,NaN,kon,=,NaN,M-1-s-1,CHEMBL5728251
11879,4473788,28237937,CHEMBL5736378,CHEMBL207649,Kd,=,16.10,nM,Kd,=,16.10,nM,CHEMBL5728251
11912,4473788,28238003,CHEMBL5736378,CHEMBL207649,Kd,=,16.10,nM,Kd,=,16.10,nM,CHEMBL5728251
11881,4473788,28237939,CHEMBL5736378,CHEMBL207649,k_off,=,NaN,s-1,koff,=,NaN,s-1,CHEMBL5728251
11914,4473788,28238005,CHEMBL5736378,CHEMBL207649,k_off,=,NaN,s-1,koff,=,NaN,s-1,CHEMBL5728251


In [ ]:
duplicate_group_summary = (
    potential_duplicates
    .groupby(
        [
            "record_id",
            "assay_chembl_id",
            "molecule_chembl_id",
            "standard_type",
        ],
        dropna=False
    )
    .agg(
        row_count=("activity_id", "size"),
        unique_activity_ids=("activity_id", "nunique"),
        unique_standard_values=("standard_value", "nunique"),
        unique_relations=("standard_relation", "nunique"),
        unique_units=("standard_units", "nunique"),
    )
    .reset_index()
)

display(duplicate_group_summary)

,record_id,assay_chembl_id,molecule_chembl_id,standard_type,row_count,unique_activity_ids,unique_standard_values,unique_relations,unique_units
0,4473787,CHEMBL5736378,CHEMBL1908919,Kd,2,2,1,1,1
1,4473787,CHEMBL5736378,CHEMBL1908919,k_off,2,2,0,1,1
2,4473787,CHEMBL5736378,CHEMBL1908919,kon,2,2,0,1,0
3,4473788,CHEMBL5736378,CHEMBL207649,Kd,2,2,1,1,1
4,4473788,CHEMBL5736378,CHEMBL207649,k_off,2,2,0,1,1
5,4473788,CHEMBL5736378,CHEMBL207649,kon,2,2,0,1,0
6,4473789,CHEMBL5736378,CHEMBL5771524,Kd,2,2,1,1,1
7,4473789,CHEMBL5736378,CHEMBL5771524,k_off,2,2,0,1,1
8,4473789,CHEMBL5736378,CHEMBL5771524,kon,2,2,0,1,0
9,4473790,CHEMBL5736378,CHEMBL6035240,Kd,2,2,1,1,1


In [ ]:
# Periksa kolom sumber yang tersedia
source_columns = [
    "record_id",
    "activity_id",
    "assay_chembl_id",
    "molecule_chembl_id",
    "standard_type",
    "value",
    "units",
    "relation",
    "standard_value",
    "standard_units",
    "standard_relation",
    "src_id",
    "src_assay_id",
    "document_chembl_id",
]

available_columns = [
    col for col in source_columns
    if col in potential_duplicates.columns
]

display(
    potential_duplicates[available_columns]
    .sort_values(["record_id", "standard_type", "activity_id"])
    .to_string(index=False)
)

' record_id  activity_id assay_chembl_id molecule_chembl_id standard_type  value   units relation  standard_value standard_units standard_relation  src_id document_chembl_id\n   4473787     28237931   CHEMBL5736378      CHEMBL1908919            Kd  24.30      nM        =           24.30             nM                 =      37      CHEMBL5728251\n   4473787     28237997   CHEMBL5736378      CHEMBL1908919            Kd  24.30      nM        =           24.30             nM                 =      37      CHEMBL5728251\n   4473787     28237933   CHEMBL5736378      CHEMBL1908919         k_off    NaN     s-1        =             NaN            s-1                 =      37      CHEMBL5728251\n   4473787     28237999   CHEMBL5736378      CHEMBL1908919         k_off    NaN     s-1        =             NaN            s-1                 =      37      CHEMBL5728251\n   4473787     28237932   CHEMBL5736378      CHEMBL1908919           kon    NaN M-1-s-1        =             NaN            NaN  

In [ ]:
# Perbandingan isi setiap pasangan
comparison_columns = [
    col for col in available_columns
    if col not in ["activity_id"]
]

pair_comparison = (
    potential_duplicates
    .groupby(
        ["record_id", "standard_type"],
        dropna=False
    )[comparison_columns]
    .nunique(dropna=False)
)

display(pair_comparison)

record_id  assay_chembl_id  molecule_chembl_id  standard_type  value  units  relation  standard_value  standard_units  standard_relation  src_id  document_chembl_id
record_id standard_type                                                                                                                                                                      
4473787   Kd                     1                1                   1              1      1      1         1               1               1                  1       1                   1
          k_off                  1                1                   1              1      1      1         1               1               1                  1       1                   1
          kon                    1                1                   1              1      1      1         1               1               1                  1       1                   1
4473788   Kd                     1                1                   1              1      1      1         1               1               1                  1       1                   1
          k_off                  1                1                   1              1      1      1         1               1               1                  1       1                   1
          kon                    1                1                   1              1      1      1         1               1               1                  1       1                   1
4473789   Kd                     1                1                   1              1      1      1         1               1               1                  1       1                   1
          k_off                  1                1                   1              1      1      1         1               1               1                  1       1                   1
          kon                    1                1                   1              1      1      1         1               1               1                  1       1                   1
4473790   Kd                     1                1                   1              1      1      1         1               1               1                  1       1                   1
          k_off                  1                1                   1              1      1      1         1               1               1                  1       1                   1
          kon                    1                1                   1              1      1      1         1               1               1                  1       1                   1
4473791   Kd                     1                1                   1              1      1      1         1               1               1                  1       1                   1
          k_off                  1                1                   1              1      1      1         1               1               1                  1       1                   1
          kon                    1                1                   1              1      1      1         1               1               1                  1       1                   1
4473796   Kd                     1                1                   1              1      1      1         1               1               1                  1       1                   1
          k_off                  1                1                   1              1      1      1         1               1               1                  1       1                   1
          kon                    1                1                   1              1      1      1         1               1               1                  1       1                   1
4473797   Kd                     1                1                   1              1      1      1         1               1               1                  1       1                   1
          k_off                  1 

In [ ]:
# Kolom untuk mengidentifikasi isi pengukuran yang identik
duplicate_key = [
    "record_id",
    "assay_chembl_id",
    "molecule_chembl_id",
    "standard_type",
    "value",
    "units",
    "relation",
    "standard_value",
    "standard_units",
    "standard_relation",
    "src_id",
    "document_chembl_id",
]

# Pastikan seluruh kolom tersedia
missing_columns = [
    col for col in duplicate_key
    if col not in remaining_unreviewed.columns
]

if missing_columns:
    raise KeyError(f"Kolom tidak ditemukan: {missing_columns}")

# Identifikasi baris duplikat hanya pada kelompok yang sudah diaudit
duplicate_rows_to_remove = potential_duplicates.duplicated(
    subset=duplicate_key,
    keep="first"
)

duplicate_indices = potential_duplicates.index[
    duplicate_rows_to_remove
]

# Buat salinan untuk analisis
remaining_unreviewed_analysis = remaining_unreviewed.drop(
    index=duplicate_indices
).copy()

# Ringkasan
print("BEFORE:", remaining_unreviewed.shape)
print("REMOVED FROM ANALYSIS:", len(duplicate_indices))
print("AFTER:", remaining_unreviewed_analysis.shape)
print("RAW DATA UNCHANGED:", remaining_unreviewed.shape)

BEFORE: (1964, 50)
REMOVED FROM ANALYSIS: 24
AFTER: (1940, 50)
RAW DATA UNCHANGED: (1964, 50)


In [ ]:
# Pastikan tidak ada lagi pasangan identik dalam kelompok yang diaudit
remaining_audited = remaining_unreviewed_analysis.loc[
    remaining_unreviewed_analysis.index.intersection(
        potential_duplicates.index
    )
]

print(
    "REMAINING IDENTICAL ROWS:",
    remaining_audited.duplicated(
        subset=duplicate_key,
        keep=False
    ).sum()
)

print(
    "UNIQUE RECORD IDs IN AUDITED DATA:",
    remaining_audited["record_id"].nunique()
)

print(
    "AUDITED ROWS RETAINED:",
    len(remaining_audited)
)

REMAINING IDENTICAL ROWS: 0
UNIQUE RECORD IDs IN AUDITED DATA: 8
AUDITED ROWS RETAINED: 24


#### Audit Potential Duplicate Entries

Ditemukan 48 baris yang terindikasi sebagai potential duplicate entries, terdiri dari 8 `record_id` pada assay CHEMBL5736378. Setiap pasangan memiliki atribut pengukuran yang identik, tetapi memiliki `activity_id` berbeda.

Setelah pemeriksaan, satu baris dari setiap pasangan dikeluarkan dari salinan dataset analisis sehingga 24 baris dihilangkan. Dataset analisis sementara berubah dari 1.964 menjadi 1.940 baris. Validasi menunjukkan tidak ada lagi baris identik berdasarkan kombinasi kolom audit.

Data hasil integrasi sebelum audit tetap dipertahankan tanpa perubahan. Pengurangan dilakukan hanya pada salinan untuk analisis. Karena `activity_id` berbeda, hasil ini dicatat sebagai potential duplicate entries, bukan bukti pasti duplikasi pada sumber asli.

In [ ]:

# ============================================================
# ASSAY INVENTORY - ALPHA-SYNUCLEIN
# ============================================================

import pandas as pd

# 1. Pastikan kolom penting tersedia
required_activity_cols = [
    "assay_chembl_id",
    "activity_id",
    "molecule_chembl_id",
    "standard_type",
    "standard_units",
]

missing_cols = [
    col for col in required_activity_cols
    if col not in remaining_unreviewed_analysis.columns
]

if missing_cols:
    raise KeyError(f"Kolom aktivitas tidak ditemukan: {missing_cols}")

# 2. Ringkasan aktivitas per assay
assay_inventory = (
    remaining_unreviewed_analysis
    .groupby("assay_chembl_id", dropna=False)
    .agg(
        total_records=("activity_id", "count"),
        unique_molecules=("molecule_chembl_id", "nunique"),
        endpoint_types=(
            "standard_type",
            lambda x: sorted(x.dropna().astype(str).unique().tolist())
        ),
        units=(
            "standard_units",
            lambda x: sorted(x.dropna().astype(str).unique().tolist())
        ),
        missing_endpoint=("standard_type", lambda x: x.isna().sum()),
    )
    .reset_index()
)

# 3. Ambil metadata assay
metadata_cols = [
    "assay_chembl_id",
    "assay_type",
    "assay_type_description",
    "description",
    "assay_organism",
    "assay_tax_id",
    "bao_format",
    "bao_label",
    "document_chembl_id",
]

metadata_cols = [
    col for col in metadata_cols
    if col in assay_meta.columns
]

assay_metadata = assay_meta[metadata_cols].copy()

# 4. Pastikan satu assay hanya memiliki satu baris metadata
metadata_duplicates = assay_metadata[
    assay_metadata["assay_chembl_id"].duplicated(keep=False)
]

print("DUPLICATE ASSAY METADATA ROWS:", len(metadata_duplicates))

if len(metadata_duplicates) > 0:
    display(
        metadata_duplicates.sort_values("assay_chembl_id").head(20)
    )
    raise ValueError(
        "Metadata memiliki assay_chembl_id berulang. "
        "Periksa sebelum melakukan merge."
    )

# 5. Gabungkan ringkasan aktivitas dan metadata
assay_inventory = assay_inventory.merge(
    assay_metadata,
    on="assay_chembl_id",
    how="left",
    validate="one_to_one",
    indicator=True
)

print("=" * 65)
print("ASSAY INVENTORY SUMMARY")
print("=" * 65)

print("Total assay:", len(assay_inventory))
print(
    "Assay dengan metadata:",
    (assay_inventory["_merge"] == "both").sum()
)
print(
    "Assay tanpa metadata:",
    (assay_inventory["_merge"] == "left_only").sum()
)

# 6. Urutkan berdasarkan jumlah record
assay_inventory = (
    assay_inventory
    .drop(columns="_merge")
    .sort_values("total_records", ascending=False)
    .reset_index(drop=True)
)

display(assay_inventory.head(30))


DUPLICATE ASSAY METADATA ROWS: 0
ASSAY INVENTORY SUMMARY
Total assay: 314
Assay dengan metadata: 314
Assay tanpa metadata: 0


,assay_chembl_id,total_records,unique_molecules,endpoint_types,units,missing_endpoint,assay_type,assay_type_description,description,assay_organism,assay_tax_id,bao_format,bao_label,document_chembl_id
0,CHEMBL3888008,90,89,[EC50],[nM],0,B,Binding,"In Vitro Inhibition Assay: α-Synucleinopathies are characterised by aggregation of α-synuclein in neurons. Aggregation of purified α-synuclein is performed essentially as described by Gerard et al. FASEB. 20(3):524-6 (2006). 20-100 μg purified α-synuclein (Sigma; S7820) at a concentration of about 2.5 μg/mL is incubated in the presence of spermin (250 μM) or paraquat (32 mM) or 6-hydroxydopamine (400 μM) or vehicle in a 384 well plate. Spermin, paraquat and 6-hydroxydopamine promote the α-synuclein aggregation process. Aggregation kinetics is determined by measuring turbidity at 340 nm, every 1-15 minutes for at least one hour.",NaN,NaN,BAO_0000357,single protein format,CHEMBL3886537
1,CHEMBL5736665,75,25,"[IC50, k_off, kon]","[nM, s-1]",0,B,Binding,"In-Vitro Fluorescence Polarization Assay: The fluorescence polarization assay tests the ability of compounds to inhibit the self-aggregation of α-synuclein peptide fragments. Peptides were incubated for 120 min at room temperature in the presence or absence of test compounds (compound concentrations were 33.3 to 0.015 μM). Samples were read on a Beckman Coulter DTX 880 plate reader in fluorescence polarization mode using excitation at 485 nm and emission at 520 nm. Data was analyzed using a four-parameter logistic fit (XLFit, IDBS Software). Peptide 4F (CTGFVKKDQLGK (SEQ ID NO: 1)) was prepared by American Peptide. Fresh peptide samples were reconstituted in purified water at 5 mM and diluted into 50 mM HEPES pH 7.4 with 50 mM NaCl to 100 nM final concentration. Solid compounds were dissolved in DMSO (10 mM), and then diluted serially in DMSO (300×) followed by dilution in buffer (1×) to provide solutions with a consistent final DMSO concentration of 0.33%.",NaN,NaN,BAO_0000357,single protein format,CHEMBL5728395
2,CHEMBL5524380,68,68,[Activity],[%],0,B,Binding,Inhibition of recombinant alpha-Synuclein (unknown origin) aggregation assessed as reduction in fluorescence intensity by measuring residual activity at 100 uM measured every 20 mins for 40 to 70 hrs by Thioflavin T based fluorescence analysis relative to control,Homo sapiens,9606.0,BAO_0000357,single protein format,CHEMBL5523695
3,CHEMBL4625544,47,47,[Inhibition],[%],0,B,Binding,Inhibition of alpha-synuclein aggregation (unknown origin) at 30 uM incubated for 3 days by thioflavin T based fluorescence assay relative to control,Homo sapiens,9606.0,BAO_0000357,single protein format,CHEMBL4622956
4,CHEMBL4628228,47,47,[Kd],[nM],0,B,Binding,Binding affinity to human wlid-type alpha-synuclein by SPR analysis,Homo sapiens,9606.0,BAO_0000357,single protein format,CHEMBL4627295
5,CHEMBL4628227,47,47,[Activity],[RU],0,B,Binding,Binding affinity to human wlid-type alpha-synuclein at 50 uM by SPR analysis,Homo sapiens,9606.0,BAO_0000357,single protein format,CHEMBL4627295
6,CHEMBL3594881,45,45,[Ki],[nM],0,B,Binding,Displacement of Thio-T from recombinant alpha-synuclein (unknown origin) expressed in Escherichia coli after 1.5 hrs by fluorescence assay,Homo sapiens,9606.0,BAO_0000019,assay format,CHEMBL3593202
7,CHEMBL5736397,42,14,"[IC50, k_off, kon]","[nM, s-1]",0,B,Binding,"In-Vitro Fluorescence Polarization Assay: The fluorescence polarization assay tests the ability of compounds to inhibit the self-aggregation of α-synuclein peptide fragments. Peptides were incubated for 120 min at room temperature in the presence or absence of test compounds (compound concentrations were 33.3 to 0.015 DM). Samples were read on a Beckman Coulter DTX 880 plate reader in fluorescence polarization mode using excitation at 485 nm and emission at 520 nm. Data was analyzed using a four-parameter logistic fit (XLFit, IDBS Software). Peptide 4F (CTGFVKKDQLGK (SEQ ID NO: 1)) was prepared by American Peptide. Fresh

In [ ]:

# ============================================================
# ENDPOINT DISTRIBUTION - REMAINING ASSAYS
# ============================================================

endpoint_distribution = (
    remaining_unreviewed_analysis
    .groupby(
        ["standard_type", "standard_units"],
        dropna=False
    )
    .agg(
        total_records=("activity_id", "count"),
        unique_assays=("assay_chembl_id", "nunique"),
        unique_molecules=("molecule_chembl_id", "nunique")
    )
    .reset_index()
    .sort_values("total_records", ascending=False)
)

display(endpoint_distribution)


,standard_type,standard_units,total_records,unique_assays,unique_molecules
9,Inhibition,%,595,77,451
0,Activity,%,198,46,149
4,Activity,NaN,196,73,89
8,IC50,nM,192,29,176
16,Ki,nM,142,8,141
14,Kd,nM,99,23,92
5,EC50,nM,92,2,91
25,kon,NaN,76,6,70
24,k_off,s-1,76,6,70
7,Flu intensity,%,59,24,39


In [ ]:

# ============================================================
# INVESTIGASI ASSAY CHEMBL3888008
# ============================================================

target_assay = "CHEMBL3888008"

# 1. Ambil metadata assay
assay_detail = assay_inventory[
    assay_inventory["assay_chembl_id"] == target_assay
]

if assay_detail.empty:
    raise ValueError(f"Metadata {target_assay} tidak ditemukan.")

print("=" * 70)
print("ASSAY METADATA")
print("=" * 70)

for col in [
    "assay_chembl_id",
    "assay_type",
    "assay_type_description",
    "description",
    "assay_organism",
    "assay_tax_id",
    "bao_format",
    "bao_label",
    "document_chembl_id",
]:
    if col in assay_detail.columns:
        print(f"\n{col}:")
        print(assay_detail.iloc[0][col])

# 2. Ambil seluruh record aktivitas
assay_records = remaining_unreviewed_analysis[
    remaining_unreviewed_analysis["assay_chembl_id"] == target_assay
].copy()

print("\n" + "=" * 70)
print("ACTIVITY RECORD SUMMARY")
print("=" * 70)

print("Total records:", len(assay_records))
print("Unique molecules:", assay_records["molecule_chembl_id"].nunique())

# 3. Periksa endpoint dan qualifier
for col in [
    "standard_type",
    "standard_relation",
    "standard_units",
    "relation",
    "units",
]:
    if col in assay_records.columns:
        print(f"\nDISTRIBUTION: {col}")
        print(assay_records[col].value_counts(dropna=False))

# 4. Periksa statistik nilai aktivitas
for col in ["standard_value", "value"]:
    if col in assay_records.columns:
        print(f"\nSTATISTICS: {col}")
        print(assay_records[col].describe())

# 5. Tampilkan contoh record secara lengkap
detail_cols = [
    "activity_id",
    "record_id",
    "molecule_chembl_id",
    "standard_type",
    "standard_relation",
    "standard_value",
    "standard_units",
    "type",
    "relation",
    "value",
    "units",
    "pchembl_value",
    "document_chembl_id",
]

detail_cols = [
    col for col in detail_cols
    if col in assay_records.columns
]

print("\n" + "=" * 70)
print("SAMPLE ACTIVITY RECORDS")
print("=" * 70)

display(
    assay_records[detail_cols]
    .sort_values("molecule_chembl_id")
    .head(20)
)


ASSAY METADATA

assay_chembl_id:
CHEMBL3888008

assay_type:
B

assay_type_description:
Binding

description:
In Vitro Inhibition Assay: α-Synucleinopathies are characterised by aggregation of α-synuclein in neurons. Aggregation of purified α-synuclein is performed essentially as described by Gerard et al. FASEB. 20(3):524-6 (2006). 20-100 μg purified α-synuclein (Sigma; S7820) at a concentration of about 2.5 μg/mL is incubated in the presence of spermin (250 μM) or paraquat (32 mM) or 6-hydroxydopamine (400 μM) or vehicle in a 384 well plate. Spermin, paraquat and 6-hydroxydopamine promote the α-synuclein aggregation process. Aggregation kinetics is determined by measuring turbidity at 340 nm, every 1-15 minutes for at least one hour.

assay_organism:
nan

assay_tax_id:
nan

bao_format:
BAO_0000357

bao_label:
single protein format

document_chembl_id:
CHEMBL3886537

ACTIVITY RECORD SUMMARY
Total records: 90
Unique molecules: 89

DISTRIBUTION: standard_type
standard_type
EC50    90
Nam

,activity_id,record_id,molecule_chembl_id,standard_type,standard_relation,standard_value,standard_units,type,relation,value,units,pchembl_value,document_chembl_id
10223,17742269,2863021,CHEMBL2153080,EC50,=,78.0,nM,EC50,=,78.0,nM,7.11,CHEMBL3886537
10292,17742338,2863089,CHEMBL2153082,EC50,=,161.0,nM,EC50,=,161.0,nM,6.79,CHEMBL3886537
10236,17742282,2863034,CHEMBL2153083,EC50,=,16.0,nM,EC50,=,16.0,nM,7.80,CHEMBL3886537
10237,17742283,2863035,CHEMBL2153084,EC50,=,14.0,nM,EC50,=,14.0,nM,7.85,CHEMBL3886537
10250,17742296,2863048,CHEMBL2153085,EC50,=,7.0,nM,EC50,=,7.0,nM,8.15,CHEMBL3886537
10283,17742329,2863050,CHEMBL2153086,EC50,=,26.0,nM,EC50,=,26.0,nM,7.58,CHEMBL3886537
10252,17742298,2863050,CHEMBL2153086,EC50,=,13.0,nM,EC50,=,13.0,nM,7.89,CHEMBL3886537
10259,17742305,2863057,CHEMBL2153087,EC50,=,70.0,nM,EC50,=,70.0,nM,7.16,CHEMBL3886537
10271,17742317,2863069,CHEMBL2153088,EC50,=,90.0,nM,EC50,=,90.0,nM,7.05,CHEMBL3886537
10258,17742304,2863056,CHEMBL3889741,EC50,=,36.0,nM,EC50,=,36.0,nM,7.44,CHEMBL3886537


In [ ]:

# ============================================================
# AUDIT REPEATED MOLECULES - CHEMBL3888008
# ============================================================

# 1. Cari molekul dengan lebih dari satu record
repeated_molecules = (
    assay_records
    .groupby("molecule_chembl_id", dropna=False)
    .agg(
        total_records=("activity_id", "count"),
        unique_activity_ids=("activity_id", "nunique"),
        unique_record_ids=("record_id", "nunique"),
        values=("standard_value", lambda x: x.tolist()),
        endpoints=("standard_type", lambda x: x.unique().tolist())
    )
    .query("total_records > 1")
    .reset_index()
)

print("REPEATED MOLECULES:", len(repeated_molecules))
display(repeated_molecules)

# 2. Periksa detail semua record molekul berulang
if not repeated_molecules.empty:
    repeated_ids = repeated_molecules["molecule_chembl_id"].tolist()

    repeated_details = assay_records[
        assay_records["molecule_chembl_id"].isin(repeated_ids)
    ][[
        "activity_id",
        "record_id",
        "assay_chembl_id",
        "molecule_chembl_id",
        "standard_type",
        "standard_relation",
        "standard_value",
        "standard_units",
        "pchembl_value",
        "document_chembl_id"
    ]].sort_values(["molecule_chembl_id", "standard_value"])

    display(repeated_details)

    # 3. Periksa apakah kombinasi pengukuran identik
    duplicate_check_cols = [
        "record_id",
        "assay_chembl_id",
        "molecule_chembl_id",
        "standard_type",
        "standard_relation",
        "standard_value",
        "standard_units",
        "document_chembl_id"
    ]

    print(
        "IDENTICAL MEASUREMENT ROWS:",
        repeated_details.duplicated(
            subset=duplicate_check_cols,
            keep=False
        ).sum()
    )


REPEATED MOLECULES: 1


,molecule_chembl_id,total_records,unique_activity_ids,unique_record_ids,values,endpoints
0,CHEMBL2153086,2,2,1,"[13.0, 26.0]",[EC50]


,activity_id,record_id,assay_chembl_id,molecule_chembl_id,standard_type,standard_relation,standard_value,standard_units,pchembl_value,document_chembl_id
10252,17742298,2863050,CHEMBL3888008,CHEMBL2153086,EC50,=,13.0,nM,7.89,CHEMBL3886537
10283,17742329,2863050,CHEMBL3888008,CHEMBL2153086,EC50,=,26.0,nM,7.58,CHEMBL3886537


IDENTICAL MEASUREMENT ROWS: 0


In [ ]:
assay_id = "CHEMBL5736665"

assay_records = remaining_unreviewed_analysis[
    remaining_unreviewed_analysis["assay_chembl_id"] == assay_id
].copy()

print("=" * 70)
print("ASSAY RECORD OVERVIEW")
print("=" * 70)

print("Shape:", assay_records.shape)
print("Unique molecules:", assay_records["molecule_chembl_id"].nunique())
print("Unique activity IDs:", assay_records["activity_id"].nunique())
print("Unique record IDs:", assay_records["record_id"].nunique())

print("\nSTANDARD TYPE DISTRIBUTION")
display(
    assay_records.groupby(
        ["standard_type", "standard_units"], dropna=False
    ).agg(
        records=("activity_id", "count"),
        molecules=("molecule_chembl_id", "nunique"),
        missing_value=("standard_value", lambda x: x.isna().sum()),
        missing_pchembl=("pchembl_value", lambda x: x.isna().sum())
    ).reset_index()
)

print("\nRELATION DISTRIBUTION")
display(
    assay_records.groupby(
        ["standard_type", "standard_relation"], dropna=False
    ).size().reset_index(name="records")
)

print("\nSAMPLE RECORDS")
display(
    assay_records[
        [
            "activity_id",
            "record_id",
            "molecule_chembl_id",
            "standard_type",
            "standard_relation",
            "standard_value",
            "standard_units",
            "pchembl_value"
        ]
    ].head(15)
)

ASSAY RECORD OVERVIEW
Shape: (75, 50)
Unique molecules: 25
Unique activity IDs: 75
Unique record IDs: 25

STANDARD TYPE DISTRIBUTION


,standard_type,standard_units,records,molecules,missing_value,missing_pchembl
0,IC50,nM,25,25,0,0
1,k_off,s-1,25,25,25,25
2,kon,NaN,25,25,25,25



RELATION DISTRIBUTION


,standard_type,standard_relation,records
0,IC50,=,25
1,k_off,=,25
2,kon,=,25



SAMPLE RECORDS


,activity_id,record_id,molecule_chembl_id,standard_type,standard_relation,standard_value,standard_units,pchembl_value
11975,28310777,4489238,CHEMBL5813355,IC50,=,1400.0,nM,5.85
11976,28310778,4489238,CHEMBL5813355,kon,=,NaN,NaN,NaN
11977,28310779,4489238,CHEMBL5813355,k_off,=,NaN,s-1,NaN
11978,28310780,4489239,CHEMBL5938529,IC50,=,1200.0,nM,5.92
11979,28310781,4489239,CHEMBL5938529,kon,=,NaN,NaN,NaN
11980,28310782,4489239,CHEMBL5938529,k_off,=,NaN,s-1,NaN
11981,28310783,4489240,CHEMBL5859369,IC50,=,1100.0,nM,5.96
11982,28310784,4489240,CHEMBL5859369,kon,=,NaN,NaN,NaN
11983,28310785,4489240,CHEMBL5859369,k_off,=,NaN,s-1,NaN
11984,28310786,4489241,CHEMBL5857516,IC50,=,1500.0,nM,5.82


In [ ]:
# Pisahkan IC50 untuk pemeriksaan kuantitatif
ic50_records = assay_records[
    assay_records["standard_type"].eq("IC50")
].copy()

print("=" * 70)
print("IC50 SUMMARY - CHEMBL5736665")
print("=" * 70)

display(
    ic50_records["standard_value"].describe()
)

print("\nP-CHEMBL SUMMARY")
display(
    ic50_records["pchembl_value"].describe()
)

print("\nIC50 VALUES")
display(
    ic50_records[
        [
            "molecule_chembl_id",
            "standard_relation",
            "standard_value",
            "standard_units",
            "pchembl_value",
            "activity_id",
            "record_id"
        ]
    ].sort_values("standard_value")
)

print("\nASSAY METADATA")
display(
    assay_inventory[
        assay_inventory["assay_chembl_id"].eq("CHEMBL5736665")
    ].T
)

IC50 SUMMARY - CHEMBL5736665


count      25.000000
mean     1708.000000
std      1020.996246
min       900.000000
25%      1200.000000
50%      1400.000000
75%      1700.000000
max      5000.000000
Name: standard_value, dtype: float64


P-CHEMBL SUMMARY


count    25.00000
mean      5.81520
std       0.18855
min       5.30000
25%       5.77000
50%       5.85000
75%       5.92000
max       6.05000
Name: pchembl_value, dtype: float64


IC50 VALUES


,molecule_chembl_id,standard_relation,standard_value,standard_units,pchembl_value,activity_id,record_id
11987,CHEMBL5959127,=,900.0,nM,6.05,28310789,4489242
11999,CHEMBL5786903,=,1000.0,nM,6.00,28310801,4489246
12002,CHEMBL5861526,=,1000.0,nM,6.00,28310804,4489247
11981,CHEMBL5859369,=,1100.0,nM,5.96,28310783,4489240
11990,CHEMBL5807178,=,1100.0,nM,5.96,28310792,4489243
12038,CHEMBL6036868,=,1100.0,nM,5.96,28310840,4489259
11993,CHEMBL6023473,=,1200.0,nM,5.92,28310795,4489244
11996,CHEMBL5829029,=,1200.0,nM,5.92,28310798,4489245
12020,CHEMBL5740848,=,1200.0,nM,5.92,28310822,4489253
12011,CHEMBL6049446,=,1200.0,nM,5.92,28310813,4489250



ASSAY METADATA


,1
assay_chembl_id,CHEMBL5736665
total_records,75
unique_molecules,25
endpoint_types,"[IC50, k_off, kon]"
units,"[nM, s-1]"
missing_endpoint,0
assay_type,B
assay_type_description,Binding
description,"In-Vitro Fluorescence Polarization Assay: The fluorescence polarization assay tests the ability of compounds to inhibit the self-aggregation of α-synuclein peptide fragments. Peptides were incubated for 120 min at room temperature in the presence or absence of test compounds (compound concentrations were 33.3 to 0.015 μM). Samples were read on a Beckman Coulter DTX 880 plate reader in fluorescence polarization mode using excitation at 485 nm and emission at 520 nm. Data was analyzed using a four-parameter logistic fit (XLFit, IDBS Software). Peptide 4F (CTGFVKKDQLGK (SEQ ID NO: 1)) was prepared by American Peptide. Fresh peptide samples were reconstituted in purified water at 5 mM and diluted into 50 mM HEPES pH 7.4 with 50 mM NaCl to 100 nM final concentration. Solid compounds were dissolved in DMSO (10 mM), and then diluted serially in DMSO (300×) followed by dilution in buffer (1×) to provide solutions with a consistent final DMSO concentration of 0.33%."
assay_organism,NaN
